# Notebook 02 · Slice operations and measure original prose

Fresh-run release. Follow the input and review gates in RUN_ORDER.md.

In [1]:
# NOTEBOOK 02 — CELL 1 (CORRECTED)
# Locate and qualify the Stage 00 and Stage 01 inputs.

from pathlib import Path
import json
import os
import platform
import sys


KAGGLE_INPUT_ROOT = Path("/kaggle/input")

PROJECT_ROOT = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

STAGE_02_OUTPUT_ROOT = Path(
    "/kaggle/working/stage_02_fresh_dataset"
)


def locate_dataset_root(
    dataset_slug,
    external_manifest_name,
):
    candidates = [
        (
            KAGGLE_INPUT_ROOT
            / "datasets"
            / "anonuser01"
            / dataset_slug
        ),
        (
            KAGGLE_INPUT_ROOT
            / "anonuser01"
            / dataset_slug
        ),
        KAGGLE_INPUT_ROOT / dataset_slug,
    ]

    for candidate in candidates:
        if (
            candidate.is_dir()
            and (
                candidate
                / external_manifest_name
            ).is_file()
        ):
            return candidate

    matches = [
        path.parent
        for path in KAGGLE_INPUT_ROOT.rglob(
            external_manifest_name
        )
        if dataset_slug in path.as_posix() or external_manifest_name == "stage_00_manifest.json"
    ]

    matches = sorted(
        set(matches),
        key=lambda path: path.as_posix(),
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Could not uniquely locate "
            f"{dataset_slug!r}. Found: "
            f"{[str(path) for path in matches]}"
        )

    return matches[0]


def locate_project_root(
    dataset_root,
    internal_manifest_name,
):
    matches = [
        path
        for path in dataset_root.rglob(
            internal_manifest_name
        )
        if path.parent.name == "registries"
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "Could not uniquely locate "
            f"{internal_manifest_name!r}. Found: "
            f"{[str(path) for path in matches]}"
        )

    return matches[0].parent.parent


def manifest_status(manifest):
    status = manifest.get("status")

    if status is None:
        status = manifest.get("stage_status")

    return status


STAGE_00_DATASET_ROOT = locate_dataset_root(
    "stage-00",
    "stage_00_manifest.json",
)

STAGE_01_DATASET_ROOT = locate_dataset_root(
    "stage-01",
    "stage_01_manifest.json",
)

STAGE_00_PROJECT_ROOT = locate_project_root(
    STAGE_00_DATASET_ROOT,
    "00_manifest.json",
)

STAGE_01_PROJECT_ROOT = locate_project_root(
    STAGE_01_DATASET_ROOT,
    "01_manifest.json",
)

STAGE_00_EXTERNAL_MANIFEST_PATH = (
    STAGE_00_DATASET_ROOT
    / "stage_00_manifest.json"
)

STAGE_01_EXTERNAL_MANIFEST_PATH = (
    STAGE_01_DATASET_ROOT
    / "stage_01_manifest.json"
)

STAGE_00_INTERNAL_MANIFEST_PATH = (
    STAGE_00_PROJECT_ROOT
    / "registries"
    / "00_manifest.json"
)

STAGE_01_INTERNAL_MANIFEST_PATH = (
    STAGE_01_PROJECT_ROOT
    / "registries"
    / "01_manifest.json"
)

FINAL_ELIGIBLE_REGISTRY_PATH = (
    STAGE_01_PROJECT_ROOT
    / "data"
    / "inventory"
    / "final_eligible_specifications.jsonl"
)

STAGE_00_EXTERNAL_MANIFEST = json.loads(
    STAGE_00_EXTERNAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

STAGE_01_EXTERNAL_MANIFEST = json.loads(
    STAGE_01_EXTERNAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

STAGE_00_INTERNAL_MANIFEST = json.loads(
    STAGE_00_INTERNAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

STAGE_01_INTERNAL_MANIFEST = json.loads(
    STAGE_01_INTERNAL_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

stage_00_external_status = manifest_status(
    STAGE_00_EXTERNAL_MANIFEST
)

stage_00_internal_status = manifest_status(
    STAGE_00_INTERNAL_MANIFEST
)

stage_01_external_status = manifest_status(
    STAGE_01_EXTERNAL_MANIFEST
)

stage_01_internal_status = manifest_status(
    STAGE_01_INTERNAL_MANIFEST
)

eligible_corpus = (
    STAGE_01_EXTERNAL_MANIFEST.get(
        "eligible_corpus",
        {},
    )
)

problems = []

status_checks = {
    "Stage 00 external manifest":
        stage_00_external_status,
    "Stage 00 internal manifest":
        stage_00_internal_status,
    "Stage 01 external manifest":
        stage_01_external_status,
    "Stage 01 internal manifest":
        stage_01_internal_status,
}

for label, observed_status in (
    status_checks.items()
):
    if observed_status != "PASS":
        problems.append(
            f"{label} is not PASS: "
            f"{observed_status!r}."
        )

if not FINAL_ELIGIBLE_REGISTRY_PATH.is_file():
    problems.append(
        "Final Stage 01 eligible registry "
        "is missing."
    )

if (
    eligible_corpus.get("specification_count")
    != 1970
):
    problems.append(
        "Unexpected Stage 01 specification count."
    )

if (
    eligible_corpus.get("operation_count")
    != 58585
):
    problems.append(
        "Unexpected Stage 01 operation count."
    )

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STAGE_02_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

report = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "kaggle_kernel_run_type": (
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE",
            "Interactive",
        )
    ),
    "accelerator_required": False,
    "stage_00_dataset_root": str(
        STAGE_00_DATASET_ROOT
    ),
    "stage_00_project_root": str(
        STAGE_00_PROJECT_ROOT
    ),
    "stage_00_external_status": (
        stage_00_external_status
    ),
    "stage_00_internal_status": (
        stage_00_internal_status
    ),
    "stage_01_dataset_root": str(
        STAGE_01_DATASET_ROOT
    ),
    "stage_01_project_root": str(
        STAGE_01_PROJECT_ROOT
    ),
    "stage_01_external_status": (
        stage_01_external_status
    ),
    "stage_01_internal_status": (
        stage_01_internal_status
    ),
    "eligible_specification_count": (
        eligible_corpus.get(
            "specification_count"
        )
    ),
    "eligible_operation_count": (
        eligible_corpus.get(
            "operation_count"
        )
    ),
    "native_openapi_count": (
        eligible_corpus.get(
            "native_openapi_count"
        )
    ),
    "converted_swagger_count": (
        eligible_corpus.get(
            "converted_swagger_count"
        )
    ),
    "final_eligible_registry": str(
        FINAL_ELIGIBLE_REGISTRY_PATH
    ),
    "project_root": str(PROJECT_ROOT),
    "stage_02_output_root": str(
        STAGE_02_OUTPUT_ROOT
    ),
    "problems": problems,
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

print(
    json.dumps(
        report,
        indent=2,
    )
)

if problems:
    raise RuntimeError(
        "Notebook 02 input qualification failed:\n"
        + "\n".join(problems)
    )

print(
    "\nNOTEBOOK 02 — CELL 1: PASS"
)
if STAGE_01_INTERNAL_MANIFEST.get("design_id") != "openapi-fresh-pilot-v1":
    raise RuntimeError("Attach the fresh Stage 01 release with the verified family registry.")
import hashlib
if (STAGE_01_INTERNAL_MANIFEST["upstream_manifest_sha256"]["00"]
        != hashlib.sha256(STAGE_00_INTERNAL_MANIFEST_PATH.read_bytes()).hexdigest()):
    raise RuntimeError("Stage 01 belongs to a different Stage 00 release.")


{
  "python_version": "3.13.18",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "kaggle_kernel_run_type": "Interactive",
  "accelerator_required": false,
  "stage_00_dataset_root": "/kaggle/input/datasets/anonuser01/stage-00",
  "stage_00_project_root": "/kaggle/input/datasets/anonuser01/stage-00/stage_00_protocol_runtime_v1/prose_operation_experiment",
  "stage_00_external_status": "PASS",
  "stage_00_internal_status": "PASS",
  "stage_01_dataset_root": "/kaggle/input/datasets/anonuser01/stage-01",
  "stage_01_project_root": "/kaggle/input/datasets/anonuser01/stage-01/stage_01_corpus_preparation_v1/prose_operation_experiment",
  "stage_01_external_status": "PASS",
  "stage_01_internal_status": "PASS",
  "eligible_specification_count": 1970,
  "eligible_operation_count": 58585,
  "native_openapi_count": 1757,
  "converted_swagger_count": 213,
  "final_eligible_registry": "/kaggle/input/datasets/anonuser01/stage-01/stage_01_corpus_preparation_v1/prose_operation_experiment/data/

In [2]:
# NOTEBOOK 02 — CELL 2
# Independently verify the expanded Stage 00 and Stage 01 artifacts.

from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import time


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
        newline="\n",
    )

    os.replace(temporary_path, path)


def normalized_relative_path(value):
    if not isinstance(value, str) or not value:
        raise RuntimeError(
            f"Invalid artifact path: {value!r}"
        )

    value = value.replace("\\", "/")
    path = PurePosixPath(value)

    if (
        path.is_absolute()
        or ".." in path.parts
    ):
        raise RuntimeError(
            f"Unsafe artifact path: {value!r}"
        )

    return path.as_posix()


def expected_internal_manifest_hash(
    external_manifest,
):
    direct_value = external_manifest.get(
        "internal_manifest_sha256"
    )

    if direct_value:
        return direct_value

    internal_record = external_manifest.get(
        "internal_manifest",
        {},
    )

    if isinstance(internal_record, dict):
        return internal_record.get("sha256")

    return None


def verify_expanded_stage(
    stage_label,
    project_root,
    internal_manifest_path,
    internal_manifest,
    external_manifest,
):
    start = time.perf_counter()
    problems = []

    expected_manifest_sha256 = (
        expected_internal_manifest_hash(
            external_manifest
        )
    )

    observed_manifest_sha256 = sha256_file(
        internal_manifest_path
    )

    if expected_manifest_sha256 is None:
        problems.append(
            "External manifest does not record "
            "the internal-manifest SHA-256."
        )
    elif (
        observed_manifest_sha256
        != expected_manifest_sha256
    ):
        problems.append(
            "Internal-manifest SHA-256 mismatch: "
            f"expected {expected_manifest_sha256}, "
            f"observed {observed_manifest_sha256}."
        )

    artifacts = internal_manifest.get(
        "artifacts"
    )

    if not isinstance(artifacts, list):
        artifacts = internal_manifest.get("files")

    if not isinstance(artifacts, list):
        raise RuntimeError(
            f"{stage_label} internal manifest "
            "contains no artifact list."
        )

    declared_artifact_count = int(
        internal_manifest.get(
            "artifact_count",
            len(artifacts),
        )
    )

    if len(artifacts) != declared_artifact_count:
        problems.append(
            "Declared artifact count does not "
            "match the artifact list."
        )

    verified_count = 0
    verified_bytes = 0
    seen_paths = set()

    for index, artifact in enumerate(
        artifacts,
        start=1,
    ):
        relative_value = (
            artifact.get("relative_path")
            or artifact.get("path")
            or artifact.get("artifact_path")
        )

        try:
            relative_path = (
                normalized_relative_path(
                    relative_value
                )
            )
        except Exception as error:
            problems.append(str(error))
            continue

        if relative_path in seen_paths:
            problems.append(
                "Duplicate artifact path: "
                f"{relative_path}"
            )
            continue

        seen_paths.add(relative_path)

        artifact_path = (
            project_root / relative_path
        )

        expected_sha256 = artifact.get(
            "sha256"
        )

        expected_bytes = artifact.get(
            "bytes"
        )

        if expected_bytes is None:
            expected_bytes = artifact.get(
                "size_bytes"
            )

        if not artifact_path.is_file():
            problems.append(
                "Missing artifact: "
                f"{relative_path}"
            )
            continue

        observed_bytes = (
            artifact_path.stat().st_size
        )

        if (
            expected_bytes is not None
            and observed_bytes
            != int(expected_bytes)
        ):
            problems.append(
                "Byte-count mismatch for "
                f"{relative_path}: expected "
                f"{expected_bytes}, observed "
                f"{observed_bytes}."
            )
            continue

        observed_sha256 = sha256_file(
            artifact_path
        )

        if observed_sha256 != expected_sha256:
            problems.append(
                "SHA-256 mismatch for "
                f"{relative_path}: expected "
                f"{expected_sha256}, observed "
                f"{observed_sha256}."
            )
            continue

        verified_count += 1
        verified_bytes += observed_bytes

        if (
            index % 1000 == 0
            or index == len(artifacts)
        ):
            print(
                json.dumps(
                    {
                        "stage": stage_label,
                        "checked": index,
                        "total": len(artifacts),
                        "verified": (
                            verified_count
                        ),
                        "problem_count": len(
                            problems
                        ),
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - start,
                            1,
                        ),
                    }
                )
            )

    return {
        "stage": stage_label,
        "verification_mode": (
            "expanded_internal_manifest"
        ),
        "internal_manifest_path": str(
            internal_manifest_path
        ),
        "expected_internal_manifest_sha256":
            expected_manifest_sha256,
        "observed_internal_manifest_sha256":
            observed_manifest_sha256,
        "expected_artifact_count":
            declared_artifact_count,
        "verified_artifact_count":
            verified_count,
        "verified_artifact_bytes":
            verified_bytes,
        "verification_problems":
            problems,
        "duration_seconds": round(
            time.perf_counter() - start,
            3,
        ),
        "status": (
            "PASS"
            if (
                not problems
                and verified_count
                == declared_artifact_count
            )
            else "BLOCKED"
        ),
    }


stage_00_verification = verify_expanded_stage(
    stage_label="00",
    project_root=STAGE_00_PROJECT_ROOT,
    internal_manifest_path=(
        STAGE_00_INTERNAL_MANIFEST_PATH
    ),
    internal_manifest=(
        STAGE_00_INTERNAL_MANIFEST
    ),
    external_manifest=(
        STAGE_00_EXTERNAL_MANIFEST
    ),
)

stage_01_verification = verify_expanded_stage(
    stage_label="01",
    project_root=STAGE_01_PROJECT_ROOT,
    internal_manifest_path=(
        STAGE_01_INTERNAL_MANIFEST_PATH
    ),
    internal_manifest=(
        STAGE_01_INTERNAL_MANIFEST
    ),
    external_manifest=(
        STAGE_01_EXTERNAL_MANIFEST
    ),
)

UPSTREAM_VERIFICATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "upstream_stage_00_01_verification.json"
)

combined_report = {
    "stage_00": stage_00_verification,
    "stage_01": stage_01_verification,
    "total_expected_artifacts": (
        stage_00_verification[
            "expected_artifact_count"
        ]
        + stage_01_verification[
            "expected_artifact_count"
        ]
    ),
    "total_verified_artifacts": (
        stage_00_verification[
            "verified_artifact_count"
        ]
        + stage_01_verification[
            "verified_artifact_count"
        ]
    ),
    "status": (
        "PASS"
        if (
            stage_00_verification["status"]
            == "PASS"
            and stage_01_verification["status"]
            == "PASS"
        )
        else "BLOCKED"
    ),
}

write_json_atomic(
    UPSTREAM_VERIFICATION_PATH,
    combined_report,
)

combined_report[
    "verification_report_path"
] = str(UPSTREAM_VERIFICATION_PATH)

combined_report[
    "verification_report_sha256"
] = sha256_file(
    UPSTREAM_VERIFICATION_PATH
)

print(
    json.dumps(
        combined_report,
        indent=2,
    )
)

if combined_report["status"] != "PASS":
    raise RuntimeError(
        "Upstream artifact verification failed."
    )

print(
    "\nNOTEBOOK 02 — CELL 2: PASS"
)

{"stage": "00", "checked": 32, "total": 32, "verified": 32, "problem_count": 0, "elapsed_seconds": 1.0}
{"stage": "01", "checked": 1000, "total": 10335, "verified": 1000, "problem_count": 0, "elapsed_seconds": 34.9}
{"stage": "01", "checked": 2000, "total": 10335, "verified": 2000, "problem_count": 0, "elapsed_seconds": 67.7}
{"stage": "01", "checked": 3000, "total": 10335, "verified": 3000, "problem_count": 0, "elapsed_seconds": 100.5}
{"stage": "01", "checked": 4000, "total": 10335, "verified": 4000, "problem_count": 0, "elapsed_seconds": 134.7}
{"stage": "01", "checked": 5000, "total": 10335, "verified": 5000, "problem_count": 0, "elapsed_seconds": 168.3}
{"stage": "01", "checked": 6000, "total": 10335, "verified": 6000, "problem_count": 0, "elapsed_seconds": 200.7}
{"stage": "01", "checked": 7000, "total": 10335, "verified": 7000, "problem_count": 0, "elapsed_seconds": 233.8}
{"stage": "01", "checked": 8000, "total": 10335, "verified": 8000, "problem_count": 0, "elapsed_seconds": 2

In [3]:
# NOTEBOOK 02 — CELL 3
# Audit and qualify the Stage 02 software dependencies.

from importlib import import_module
from importlib import invalidate_caches
from importlib.metadata import (
    PackageNotFoundError,
    version,
)
from pathlib import Path
import json
import subprocess
import sys
import time


REQUIRED_PACKAGES = [
    {
        "import_name": "yaml",
        "distribution": "PyYAML",
        "install_spec": "PyYAML",
        "required_version": None,
    },
    {
        "import_name": "pandas",
        "distribution": "pandas",
        "install_spec": "pandas",
        "required_version": None,
    },
    {
        "import_name": "numpy",
        "distribution": "numpy",
        "install_spec": "numpy",
        "required_version": None,
    },
    {
        "import_name": "matplotlib",
        "distribution": "matplotlib",
        "install_spec": "matplotlib",
        "required_version": None,
    },
    {
        "import_name": "jsonschema",
        "distribution": "jsonschema",
        "install_spec": "jsonschema",
        "required_version": None,
    },
    {
        "import_name": "referencing",
        "distribution": "referencing",
        "install_spec": "referencing",
        "required_version": None,
    },
    {
        "import_name": "openapi_spec_validator",
        "distribution": "openapi-spec-validator",
        "install_spec": (
            "openapi-spec-validator==0.9.0"
        ),
        "required_version": "0.9.0",
    },
    {
        "import_name": "transformers",
        "distribution": "transformers",
        "install_spec": "transformers",
        "required_version": None,
    },
    {
        "import_name": "tokenizers",
        "distribution": "tokenizers",
        "install_spec": "tokenizers",
        "required_version": None,
    },
    {
        "import_name": "huggingface_hub",
        "distribution": "huggingface-hub",
        "install_spec": "huggingface-hub",
        "required_version": None,
    },
    {
        "import_name": "tqdm",
        "distribution": "tqdm",
        "install_spec": "tqdm",
        "required_version": None,
    },
]


def inspect_package(package):
    import_name = package["import_name"]
    distribution = package["distribution"]
    required_version = package[
        "required_version"
    ]

    diagnostic = None
    available = False
    installed_version = None
    import_path = None

    try:
        module = import_module(import_name)
        available = True
        import_path = getattr(
            module,
            "__file__",
            None,
        )
    except Exception as error:
        diagnostic = repr(error)

    try:
        installed_version = version(distribution)
    except PackageNotFoundError:
        installed_version = None

    version_matches = (
        required_version is None
        or installed_version == required_version
    )

    return {
        "import_name": import_name,
        "distribution": distribution,
        "available": available,
        "installed_version": installed_version,
        "required_version": required_version,
        "version_matches": version_matches,
        "import_path": import_path,
        "diagnostic": diagnostic,
    }


initial_results = [
    inspect_package(package)
    for package in REQUIRED_PACKAGES
]

install_specs = []

for package, result in zip(
    REQUIRED_PACKAGES,
    initial_results,
):
    if (
        not result["available"]
        or not result["version_matches"]
    ):
        install_specs.append(
            package["install_spec"]
        )

installation = {
    "required": bool(install_specs),
    "install_specs": install_specs,
    "returncode": None,
    "stdout": "",
    "stderr": "",
    "duration_seconds": 0.0,
}

if install_specs:
    install_start = time.perf_counter()

    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-input",
            *install_specs,
        ],
        text=True,
        capture_output=True,
        check=False,
    )

    installation.update(
        {
            "returncode": completed.returncode,
            "stdout": completed.stdout,
            "stderr": completed.stderr,
            "duration_seconds": round(
                time.perf_counter()
                - install_start,
                3,
            ),
        }
    )

    print(completed.stdout)

    if completed.stderr:
        print(
            completed.stderr,
            file=sys.stderr,
        )

    if completed.returncode != 0:
        raise RuntimeError(
            "Dependency installation failed."
        )

    invalidate_caches()

final_results = [
    inspect_package(package)
    for package in REQUIRED_PACKAGES
]

problems = []

for result in final_results:
    if not result["available"]:
        problems.append(
            f"{result['distribution']} cannot "
            "be imported."
        )

    if not result["version_matches"]:
        problems.append(
            f"{result['distribution']} version "
            f"{result['installed_version']!r} does "
            f"not match required version "
            f"{result['required_version']!r}."
        )

DEPENDENCY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "runtime"
    / "dependency_qualification.json"
)

dependency_report = {
    "python_version": sys.version,
    "initial_results": initial_results,
    "installation": installation,
    "final_results": final_results,
    "problems": problems,
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

write_json_atomic(
    DEPENDENCY_REPORT_PATH,
    dependency_report,
)

output = {
    "python_version": sys.version.split()[0],
    "installation_required": (
        installation["required"]
    ),
    "installed_specs": install_specs,
    "packages": final_results,
    "report_path": str(
        DEPENDENCY_REPORT_PATH
    ),
    "report_sha256": sha256_file(
        DEPENDENCY_REPORT_PATH
    ),
    "status": dependency_report["status"],
}

print(
    json.dumps(
        output,
        indent=2,
    )
)

if problems:
    raise RuntimeError(
        "Dependency qualification failed:\n"
        + "\n".join(problems)
    )

print(
    "\nNOTEBOOK 02 — CELL 3: PASS"
)



{
  "python_version": "3.13.18",
  "installation_required": true,
  "installed_specs": [
    "openapi-spec-validator==0.9.0"
  ],
  "packages": [
    {
      "import_name": "yaml",
      "distribution": "PyYAML",
      "available": true,
      "installed_version": "6.0.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/dist-packages/yaml/__init__.py",
      "diagnostic": null
    },
    {
      "import_name": "pandas",
      "distribution": "pandas",
      "available": true,
      "installed_version": "2.3.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/dist-packages/pandas/__init__.py",
      "diagnostic": null
    },
    {
      "import_name": "numpy",
      "distribution": "numpy",
      "available": true,
      "installed_version": "2.1.3",
      "required_version": null,
      "version_matches": true,
      "import_path": "/usr/local/lib/python3.13/d

In [4]:
# NOTEBOOK 02 — CELL 4
# Create the workspace and freeze the Stage 02 policy and environment.

from importlib.metadata import version
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import yaml


STAGE_02_DIRECTORIES = [
    PROJECT_ROOT / "config",
    PROJECT_ROOT / "config" / "upstream",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "inventory",
    PROJECT_ROOT / "data" / "slices",
    PROJECT_ROOT / "data" / "slices" / "L0",
    PROJECT_ROOT / "data" / "audits",
    PROJECT_ROOT / "data" / "checkpoints",
    PROJECT_ROOT / "reports",
    PROJECT_ROOT / "reports" / "audit",
    PROJECT_ROOT / "reports" / "figures",
    PROJECT_ROOT / "reports" / "runtime",
    PROJECT_ROOT / "registries",
]

for directory in STAGE_02_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def write_bytes_atomic(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    temporary_path.write_bytes(content)

    os.replace(
        temporary_path,
        path,
    )


def write_text_atomic(path, content):
    write_bytes_atomic(
        path,
        content.encode("utf-8"),
    )


def write_yaml_atomic(path, value):
    serialized = yaml.safe_dump(
        value,
        sort_keys=False,
        allow_unicode=True,
        default_flow_style=False,
    )

    write_text_atomic(path, serialized)


STAGE_02_POLICY = {
    "stage": "02",
    "policy_version": "1.0",
    "title": (
        "Operation input construction and "
        "prose measurement"
    ),
    "upstream_requirements": {
        "stage_00_status": "PASS",
        "stage_01_status": "PASS",
        "expected_specification_count": 1970,
        "expected_operation_count": 58585,
    },
    "operation_enumeration": {
        "container": "paths",
        "supported_http_methods": [
            "get",
            "put",
            "post",
            "delete",
            "options",
            "head",
            "patch",
            "trace",
        ],
        "path_item_metadata_is_not_operation": [
            "$ref",
            "summary",
            "description",
            "servers",
            "parameters",
        ],
        "callbacks_are_not_independent_targets": True,
        "webhooks_are_not_target_operations": True,
    },
    "slice_policy": {
        "slice_name": "L0",
        "serialization": "canonical_compact_json",
        "preserve_source_openapi_version": True,
        "fixed_info": {
            "title": "Operation slice",
            "version": "1.0.0",
        },
        "preserve_original_api_identity_in_provenance":
            True,
        "include_only_target_path": True,
        "include_only_target_operation": True,
        "path_item_fields_retained": [
            "summary",
            "description",
            "x-*",
        ],
        "operation_fields": (
            "preserve all source operation fields "
            "except servers; effective parameters "
            "and security are materialized"
        ),
        "parameter_policy": {
            "merge_path_and_operation_parameters":
                True,
            "override_key": [
                "name",
                "in",
            ],
            "operation_parameter_overrides_path":
                True,
            "materialize_effective_parameters_at":
                "operation",
            "preserve_effective_order": (
                "path parameters followed by "
                "operation additions or overrides"
            ),
        },
        "security_policy": {
            "operation_security_overrides_root":
                True,
            "materialize_inherited_root_security":
                True,
            "include_referenced_security_schemes":
                True,
        },
        "reference_policy": {
            "local_references_only": True,
            "retain_references": True,
            "include_each_reachable_component_once":
                True,
            "recursively_include_reachable_components":
                True,
            "preserve_original_component_paths":
                True,
            "record_slice_to_source_pointer_map":
                True,
        },
        "tag_policy": {
            "retain_operation_tag_names": True,
            "include_matching_top_level_tag_objects":
                True,
            "omit_unused_top_level_tag_objects":
                True,
        },
        "root_fields": {
            "retain": [
                "openapi",
                "jsonSchemaDialect",
                "paths",
                "components",
                "tags",
            ],
            "omit": [
                "servers",
                "externalDocs",
                "webhooks",
                "x-*",
            ],
        },
        "omit_all_source_servers": True,
        "retain_operation_external_docs": True,
        "retain_callbacks_inside_target_operation":
            True,
        "no_truncation": True,
        "validation_required": True,
    },
    "prose_measurement": {
        "grouping_feature": "prose_tokens_L0",
        "measured_field_names": [
            "summary",
            "description",
        ],
        "require_schema_defined_annotation":
            True,
        "require_string_value": True,
        "preserve_original_whitespace": True,
        "empty_string_token_count": 0,
        "invalid_nonstring_value": (
            "diagnostic_not_stringification"
        ),
        "count_identical_text_at_distinct_positions":
            True,
        "count_shared_component_once_per_slice":
            True,
        "exclude_fields": [
            "title",
            "enum",
            "default",
            "pattern",
            "example payload properties",
            "OAuth scope descriptions",
            "extension-defined strings",
        ],
        "tokenization": {
            "model_id": (
                STAGE_00_EXTERNAL_MANIFEST[
                    "model_id"
                ]
            ),
            "revision": (
                STAGE_00_EXTERNAL_MANIFEST[
                    "model_revision"
                ]
            ),
            "add_special_tokens": False,
            "formula": (
                "len(tokenizer.encode("
                "value, add_special_tokens=False))"
            ),
        },
        "also_record_full_rendered_prompt_tokens":
            True,
    },
    "checkpointing": {
        "enabled": True,
        "checkpoint_every_operations": 250,
        "atomic_writes": True,
        "resume_only_after_hash_verification": True,
    },
    "outputs": {
        "slices": "data/slices/L0/<op_id>.json",
        "pointer_maps": (
            "data/audits/slice_pointer_maps.jsonl"
        ),
        "prose_fields": (
            "data/audits/prose_fields.csv"
        ),
        "operations": (
            "data/audits/operations.csv"
        ),
    },
}

STAGE_02_POLICY_PATH = (
    PROJECT_ROOT
    / "config"
    / "stage_02.yaml"
)

write_yaml_atomic(
    STAGE_02_POLICY_PATH,
    STAGE_02_POLICY,
)

upstream_config_sources = {
    "stage_00_study.yaml": (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "study.yaml"
    ),
    "stage_00_runtime.yaml": (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "runtime.yaml"
    ),
    "stage_00_prose_counting.yaml": (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "prose_counting.yaml"
    ),
    "stage_00_environment.json": (
        STAGE_00_PROJECT_ROOT
        / "config"
        / "environment.json"
    ),
    "stage_01_policy.yaml": (
        STAGE_01_PROJECT_ROOT
        / "config"
        / "stage_01.yaml"
    ),
}

upstream_config_records = []
missing_upstream_configs = []

for destination_name, source_path in (
    upstream_config_sources.items()
):
    if not source_path.is_file():
        missing_upstream_configs.append(
            str(source_path)
        )
        continue

    content = source_path.read_bytes()

    destination_path = (
        PROJECT_ROOT
        / "config"
        / "upstream"
        / destination_name
    )

    write_bytes_atomic(
        destination_path,
        content,
    )

    source_sha256 = sha256_file(
        source_path
    )

    destination_sha256 = sha256_file(
        destination_path
    )

    if source_sha256 != destination_sha256:
        raise RuntimeError(
            "Upstream configuration copy "
            f"mismatch for {source_path}."
        )

    upstream_config_records.append(
        {
            "source_path": str(source_path),
            "destination_path": str(
                destination_path
            ),
            "bytes": len(content),
            "sha256": source_sha256,
        }
    )

if missing_upstream_configs:
    raise RuntimeError(
        "Missing upstream configuration files:\n"
        + "\n".join(missing_upstream_configs)
    )

package_distributions = [
    "PyYAML",
    "pandas",
    "numpy",
    "matplotlib",
    "jsonschema",
    "referencing",
    "openapi-spec-validator",
    "transformers",
    "tokenizers",
    "huggingface-hub",
    "tqdm",
]

qualified_package_versions = {
    distribution: version(distribution)
    for distribution in package_distributions
}

disk_usage = shutil.disk_usage(
    "/kaggle/working"
)

ENVIRONMENT_PATH = (
    PROJECT_ROOT
    / "config"
    / "environment.json"
)

environment_record = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "kaggle_kernel_run_type": (
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE",
            "Interactive",
        )
    ),
    "accelerator_required": False,
    "packages": qualified_package_versions,
    "model_id": (
        STAGE_00_EXTERNAL_MANIFEST[
            "model_id"
        ]
    ),
    "model_revision": (
        STAGE_00_EXTERNAL_MANIFEST[
            "model_revision"
        ]
    ),
    "stage_00_internal_manifest_sha256":
        sha256_file(
            STAGE_00_INTERNAL_MANIFEST_PATH
        ),
    "stage_01_internal_manifest_sha256":
        sha256_file(
            STAGE_01_INTERNAL_MANIFEST_PATH
        ),
    "working_storage": {
        "total_bytes": disk_usage.total,
        "used_bytes": disk_usage.used,
        "free_bytes": disk_usage.free,
    },
}

write_json_atomic(
    ENVIRONMENT_PATH,
    environment_record,
)

freeze_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "freeze",
        "--all",
    ],
    text=True,
    capture_output=True,
    check=False,
)

if freeze_result.returncode != 0:
    raise RuntimeError(
        "pip freeze failed:\n"
        + freeze_result.stderr
    )

REQUIREMENTS_LOCK_PATH = (
    PROJECT_ROOT
    / "config"
    / "requirements.lock.txt"
)

write_text_atomic(
    REQUIREMENTS_LOCK_PATH,
    freeze_result.stdout,
)

UPSTREAM_CONFIG_RECORD_PATH = (
    PROJECT_ROOT
    / "config"
    / "upstream_configuration_hashes.json"
)

upstream_config_report = {
    "records": upstream_config_records,
    "stage_00_internal_manifest_sha256":
        sha256_file(
            STAGE_00_INTERNAL_MANIFEST_PATH
        ),
    "stage_01_internal_manifest_sha256":
        sha256_file(
            STAGE_01_INTERNAL_MANIFEST_PATH
        ),
    "status": "PASS",
}

write_json_atomic(
    UPSTREAM_CONFIG_RECORD_PATH,
    upstream_config_report,
)

configuration_files = [
    STAGE_02_POLICY_PATH,
    ENVIRONMENT_PATH,
    REQUIREMENTS_LOCK_PATH,
    UPSTREAM_CONFIG_RECORD_PATH,
    *[
        Path(record["destination_path"])
        for record in upstream_config_records
    ],
]

configuration_records = [
    {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in configuration_files
]

configuration_hash = hashlib.sha256(
    json.dumps(
        configuration_records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()

result = {
    "status": "PASS",
    "created_directory_count": len(
        STAGE_02_DIRECTORIES
    ),
    "stage_02_policy": {
        "path": str(STAGE_02_POLICY_PATH),
        "sha256": sha256_file(
            STAGE_02_POLICY_PATH
        ),
    },
    "environment": {
        "path": str(ENVIRONMENT_PATH),
        "sha256": sha256_file(
            ENVIRONMENT_PATH
        ),
    },
    "requirements_lock": {
        "path": str(
            REQUIREMENTS_LOCK_PATH
        ),
        "bytes": (
            REQUIREMENTS_LOCK_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            REQUIREMENTS_LOCK_PATH
        ),
    },
    "upstream_configuration_count": len(
        upstream_config_records
    ),
    "upstream_configurations": (
        upstream_config_records
    ),
    "configuration_hash": configuration_hash,
    "working_storage_free_gib": round(
        disk_usage.free / (1024 ** 3),
        3,
    ),
}

print(
    json.dumps(
        result,
        indent=2,
    )
)

print(
    "\nNOTEBOOK 02 — CELL 4: PASS"
)

{
  "status": "PASS",
  "created_directory_count": 13,
  "stage_02_policy": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/config/stage_02.yaml",
    "sha256": "94ccff4d1c864bd7ec9e3c000b9d0689f22aa236127cb67ab7b47d5af94569f6"
  },
  "environment": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/config/environment.json",
    "sha256": "3925dc4d3ea70c25bedc372bd553e8474aeeef4a2a152496b61cab83190048fe"
  },
  "requirements_lock": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/config/requirements.lock.txt",
    "bytes": 16754,
    "sha256": "055e33a7ebdb491c3d4e264df7ee5c4bd31d4afc75d095f230dab4b38f911c8f"
  },
  "upstream_configuration_count": 5,
  "upstream_configurations": [
    {
      "source_path": "/kaggle/input/datasets/anonuser01/stage-00/stage_00_protocol_runtime_v1/prose_operation_experiment/config/study.yaml",
      "destination_path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/config/

In [5]:
# NOTEBOOK 02 — CELL 5 (CORRECTED)
# Qualify and freeze the Stage 01 eligible specification registry.

from collections import Counter
import json
import time


EXPECTED_SPECIFICATION_COUNT = 1970
EXPECTED_OPERATION_COUNT = 58585
EXPECTED_NATIVE_COUNT = 1757
EXPECTED_CONVERTED_SWAGGER_COUNT = 213

SPECIFICATION_INPUTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "eligible_specifications.jsonl"
)

SPECIFICATION_INPUT_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "stage_02_specification_input_qualification.json"
)

qualification_start = time.perf_counter()

stage_01_artifacts = (
    STAGE_01_INTERNAL_MANIFEST.get("artifacts")
)

if not isinstance(stage_01_artifacts, list):
    stage_01_artifacts = (
        STAGE_01_INTERNAL_MANIFEST.get("files")
    )

if not isinstance(stage_01_artifacts, list):
    raise RuntimeError(
        "Stage 01 manifest contains no "
        "artifact list."
    )

stage_01_artifact_map = {}

for artifact in stage_01_artifacts:
    relative_value = (
        artifact.get("relative_path")
        or artifact.get("path")
        or artifact.get("artifact_path")
    )

    relative_path = normalized_relative_path(
        relative_value
    )

    if relative_path in stage_01_artifact_map:
        raise RuntimeError(
            "Duplicate Stage 01 manifest path: "
            f"{relative_path}"
        )

    stage_01_artifact_map[
        relative_path
    ] = artifact

registry_relative_path = (
    FINAL_ELIGIBLE_REGISTRY_PATH
    .relative_to(STAGE_01_PROJECT_ROOT)
    .as_posix()
)

registry_manifest_record = (
    stage_01_artifact_map.get(
        registry_relative_path
    )
)

if registry_manifest_record is None:
    raise RuntimeError(
        "The eligible registry is absent from "
        "the Stage 01 manifest."
    )

source_registry_sha256 = sha256_file(
    FINAL_ELIGIBLE_REGISTRY_PATH
)

if (
    source_registry_sha256
    != registry_manifest_record["sha256"]
):
    raise RuntimeError(
        "The eligible-registry hash does not "
        "match the Stage 01 manifest."
    )

records = []

with FINAL_ELIGIBLE_REGISTRY_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    for line_number, line in enumerate(
        handle,
        start=1,
    ):
        if not line.strip():
            continue

        try:
            records.append(json.loads(line))
        except Exception as error:
            raise RuntimeError(
                "Invalid JSON at eligible-registry "
                f"line {line_number}: {error}"
            ) from error

required_fields = {
    "spec_id",
    "raw_id",
    "provider_directory",
    "relative_path",
    "source_class",
    "source_dialect",
    "canonical_openapi_version",
    "canonical_relative_path",
    "canonical_file_sha256",
    "canonical_semantic_sha256",
    "operation_count",
    "eligibility_basis",
}

problems = []

spec_ids = set()
raw_ids = set()
canonical_paths = set()
semantic_hashes = set()

zero_operation_spec_ids = []

source_class_counts = Counter()
dialect_counts = Counter()
version_counts = Counter()
eligibility_basis_counts = Counter()
provider_counts = Counter()

operation_total = 0
canonical_file_present_count = 0
canonical_manifest_match_count = 0

for index, record in enumerate(
    records,
    start=1,
):
    missing_fields = sorted(
        required_fields - set(record)
    )

    if missing_fields:
        problems.append(
            {
                "record_index": index,
                "problem": "MISSING_FIELDS",
                "fields": missing_fields,
            }
        )
        continue

    spec_id = record["spec_id"]
    raw_id = record["raw_id"]

    canonical_relative_path = (
        normalized_relative_path(
            record[
                "canonical_relative_path"
            ]
        )
    )

    duplicate_checks = [
        (
            spec_id in spec_ids,
            "DUPLICATE_SPEC_ID",
            spec_id,
        ),
        (
            raw_id in raw_ids,
            "DUPLICATE_RAW_ID",
            raw_id,
        ),
        (
            canonical_relative_path
            in canonical_paths,
            "DUPLICATE_CANONICAL_PATH",
            canonical_relative_path,
        ),
        (
            record[
                "canonical_semantic_sha256"
            ]
            in semantic_hashes,
            "DUPLICATE_SEMANTIC_HASH",
            record[
                "canonical_semantic_sha256"
            ],
        ),
    ]

    for detected, problem_name, value in (
        duplicate_checks
    ):
        if detected:
            problems.append(
                {
                    "spec_id": spec_id,
                    "problem": problem_name,
                    "value": value,
                }
            )

    spec_ids.add(spec_id)
    raw_ids.add(raw_id)
    canonical_paths.add(
        canonical_relative_path
    )
    semantic_hashes.add(
        record["canonical_semantic_sha256"]
    )

    operation_count = int(
        record["operation_count"]
    )

    if operation_count < 0:
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "NEGATIVE_OPERATION_COUNT"
                ),
                "operation_count":
                    operation_count,
            }
        )
    elif operation_count == 0:
        zero_operation_spec_ids.append(
            spec_id
        )

    operation_total += operation_count

    source_class_counts[
        record["source_class"]
    ] += 1

    dialect_counts[
        record["source_dialect"]
    ] += 1

    version_counts[
        record["canonical_openapi_version"]
    ] += 1

    eligibility_basis_counts[
        record["eligibility_basis"]
    ] += 1

    provider_counts[
        record["provider_directory"]
    ] += 1

    canonical_path = (
        STAGE_01_PROJECT_ROOT
        / canonical_relative_path
    )

    if not canonical_path.is_file():
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "CANONICAL_FILE_MISSING"
                ),
                "canonical_relative_path":
                    canonical_relative_path,
            }
        )
        continue

    canonical_file_present_count += 1

    manifest_record = (
        stage_01_artifact_map.get(
            canonical_relative_path
        )
    )

    if manifest_record is None:
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "CANONICAL_FILE_NOT_IN_MANIFEST"
                ),
                "canonical_relative_path":
                    canonical_relative_path,
            }
        )
        continue

    if (
        manifest_record.get("sha256")
        != record["canonical_file_sha256"]
    ):
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "CANONICAL_HASH_RECORD_MISMATCH"
                ),
                "registry_sha256": record[
                    "canonical_file_sha256"
                ],
                "manifest_sha256":
                    manifest_record.get(
                        "sha256"
                    ),
            }
        )
        continue

    canonical_manifest_match_count += 1

expected_checks = [
    (
        len(records),
        EXPECTED_SPECIFICATION_COUNT,
        "SPECIFICATION_COUNT_MISMATCH",
    ),
    (
        operation_total,
        EXPECTED_OPERATION_COUNT,
        "OPERATION_COUNT_MISMATCH",
    ),
    (
        source_class_counts.get(
            "native_openapi",
            0,
        ),
        EXPECTED_NATIVE_COUNT,
        "NATIVE_SOURCE_COUNT_MISMATCH",
    ),
    (
        source_class_counts.get(
            "converted_swagger",
            0,
        ),
        EXPECTED_CONVERTED_SWAGGER_COUNT,
        "CONVERTED_SWAGGER_COUNT_MISMATCH",
    ),
]

for observed, expected, problem_name in (
    expected_checks
):
    if observed != expected:
        problems.append(
            {
                "problem": problem_name,
                "expected": expected,
                "observed": observed,
            }
        )

write_bytes_atomic(
    SPECIFICATION_INPUTS_PATH,
    FINAL_ELIGIBLE_REGISTRY_PATH.read_bytes(),
)

frozen_registry_sha256 = sha256_file(
    SPECIFICATION_INPUTS_PATH
)

if (
    frozen_registry_sha256
    != source_registry_sha256
):
    raise RuntimeError(
        "Frozen registry does not match the "
        "Stage 01 registry."
    )

qualification_report = {
    "source_registry": {
        "path": str(
            FINAL_ELIGIBLE_REGISTRY_PATH
        ),
        "sha256": source_registry_sha256,
    },
    "frozen_registry": {
        "path": str(
            SPECIFICATION_INPUTS_PATH
        ),
        "sha256": (
            frozen_registry_sha256
        ),
        "bytes": (
            SPECIFICATION_INPUTS_PATH
            .stat()
            .st_size
        ),
    },
    "specification_count": len(records),
    "operation_bearing_specification_count": (
        len(records)
        - len(zero_operation_spec_ids)
    ),
    "zero_operation_specification_count": len(
        zero_operation_spec_ids
    ),
    "zero_operation_spec_ids": sorted(
        zero_operation_spec_ids
    ),
    "zero_operation_policy": (
        "Retain for specification-level "
        "accounting; generate no operation slices."
    ),
    "operation_count": operation_total,
    "canonical_file_present_count":
        canonical_file_present_count,
    "canonical_manifest_match_count":
        canonical_manifest_match_count,
    "unique_spec_id_count": len(spec_ids),
    "unique_raw_id_count": len(raw_ids),
    "unique_canonical_path_count": len(
        canonical_paths
    ),
    "unique_semantic_hash_count": len(
        semantic_hashes
    ),
    "source_class_counts": dict(
        sorted(source_class_counts.items())
    ),
    "dialect_counts": dict(
        sorted(dialect_counts.items())
    ),
    "canonical_version_counts": dict(
        sorted(version_counts.items())
    ),
    "eligibility_basis_counts": dict(
        sorted(
            eligibility_basis_counts.items()
        )
    ),
    "provider_directory_count": len(
        provider_counts
    ),
    "problems": problems,
    "duration_seconds": round(
        time.perf_counter()
        - qualification_start,
        3,
    ),
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

write_json_atomic(
    SPECIFICATION_INPUT_REPORT_PATH,
    qualification_report,
)

output = dict(qualification_report)

output["report_path"] = str(
    SPECIFICATION_INPUT_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    SPECIFICATION_INPUT_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if problems:
    raise RuntimeError(
        "Specification input qualification "
        "failed. Review the reported problems."
    )

print(
    "\nNOTEBOOK 02 — CELL 5: PASS"
)

{
  "source_registry": {
    "path": "/kaggle/input/datasets/anonuser01/stage-01/stage_01_corpus_preparation_v1/prose_operation_experiment/data/inventory/final_eligible_specifications.jsonl",
    "sha256": "6f2229de06ffa55ca342e1666bae4cfa0c3b8bab71ace204c93cafcd4b64f055"
  },
  "frozen_registry": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/inventory/eligible_specifications.jsonl",
    "sha256": "6f2229de06ffa55ca342e1666bae4cfa0c3b8bab71ace204c93cafcd4b64f055",
    "bytes": 1338051
  },
  "specification_count": 1970,
  "operation_bearing_specification_count": 1956,
  "zero_operation_specification_count": 14,
  "zero_operation_spec_ids": [
    "spec_18c337d71480eb4f9cfa7fec",
    "spec_214e3ce4fb9a5fe428d4a047",
    "spec_2171e07dba20f93a15237d46",
    "spec_2e9310119bd64497fed5042c",
    "spec_69795d74aa81f03691bde450",
    "spec_6f9e2dbbfe3e70ab5154fd00",
    "spec_700691eaeb2175df2c26cf97",
    "spec_80e7bf83edd7aa40354b381d",
    "spec_9c90d42ccd99

In [6]:
# NOTEBOOK 02 — CELL 6
# Enumerate genuine HTTP operations and assign stable operation IDs.

from collections import Counter
from pathlib import Path
import csv
import hashlib
import json
import os
import time


SUPPORTED_HTTP_METHODS = [
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
]

OPERATION_INVENTORY_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "operations_enumerated.jsonl"
)

SPECIFICATION_COUNTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "inventory"
    / "specification_operation_counts.csv"
)

ENUMERATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "operation_enumeration.json"
)


def json_pointer_escape(value):
    return (
        str(value)
        .replace("~", "~0")
        .replace("/", "~1")
    )


def canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def make_operation_id(
    spec_id,
    path_template,
    method,
):
    identity = canonical_json_bytes(
        [
            spec_id,
            path_template,
            method,
        ]
    )

    return (
        "op_"
        + hashlib.sha256(identity)
        .hexdigest()[:24]
    )


def write_jsonl_atomic(path, rows):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as handle:
        for row in rows:
            handle.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                    allow_nan=False,
                )
            )
            handle.write("\n")

    os.replace(
        temporary_path,
        path,
    )


def write_csv_atomic(
    path,
    rows,
    fieldnames,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
            extrasaction="raise",
            lineterminator="\n",
        )

        writer.writeheader()
        writer.writerows(rows)

    os.replace(
        temporary_path,
        path,
    )


enumeration_start = time.perf_counter()

with (STAGE_01_PROJECT_ROOT / "data/inventory/families.csv").open(newline="", encoding="utf-8") as family_stream:
    family_rows = list(csv.DictReader(family_stream))
family_ids_by_spec = {row["spec_id"]: row["family_id"] for row in family_rows}
if (len(family_ids_by_spec) != len(family_rows)
        or set(family_ids_by_spec) != {row["spec_id"] for row in records}
        or any(row["review_status"] not in {"AUTO_VERIFIED", "HUMAN_VERIFIED"}
               for row in family_rows)):
    raise RuntimeError("Operation enumeration requires a verified family for each source specification.")

operation_records = []
specification_count_rows = []
problems = []

seen_operation_ids = set()
method_counts = Counter()
source_class_operation_counts = Counter()
dialect_operation_counts = Counter()
version_operation_counts = Counter()

global_operation_index = 0

for specification_index, spec_record in enumerate(
    records,
    start=1,
):
    spec_id = spec_record["spec_id"]

    canonical_relative_path = (
        spec_record[
            "canonical_relative_path"
        ]
    )

    canonical_path = (
        STAGE_01_PROJECT_ROOT
        / canonical_relative_path
    )

    try:
        document = json.loads(
            canonical_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as error:
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "CANONICAL_JSON_LOAD_FAILURE"
                ),
                "diagnostic": repr(error),
            }
        )
        continue

    paths_object = document.get("paths")

    if paths_object is None:
        paths_object = {}

    if not isinstance(paths_object, dict):
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "PATHS_IS_NOT_OBJECT"
                ),
                "observed_type": (
                    type(paths_object).__name__
                ),
            }
        )
        continue

    observed_spec_operation_count = 0
    nonoperation_path_item_keys = Counter()

    for path_template in sorted(
        paths_object,
        key=str,
    ):
        path_item = paths_object[
            path_template
        ]

        if not isinstance(path_template, str):
            problems.append(
                {
                    "spec_id": spec_id,
                    "problem": (
                        "NONSTRING_PATH_TEMPLATE"
                    ),
                    "value": repr(path_template),
                }
            )
            continue

        if not isinstance(path_item, dict):
            problems.append(
                {
                    "spec_id": spec_id,
                    "path": path_template,
                    "problem": (
                        "PATH_ITEM_IS_NOT_OBJECT"
                    ),
                    "observed_type": (
                        type(path_item).__name__
                    ),
                }
            )
            continue

        for key in path_item:
            if key not in SUPPORTED_HTTP_METHODS:
                nonoperation_path_item_keys[
                    str(key)
                ] += 1

        for method in SUPPORTED_HTTP_METHODS:
            if method not in path_item:
                continue

            operation = path_item[method]

            if not isinstance(operation, dict):
                problems.append(
                    {
                        "spec_id": spec_id,
                        "path": path_template,
                        "method": method,
                        "problem": (
                            "OPERATION_IS_NOT_OBJECT"
                        ),
                        "observed_type": (
                            type(operation).__name__
                        ),
                    }
                )
                continue

            operation_id = make_operation_id(
                spec_id,
                path_template,
                method,
            )

            if operation_id in seen_operation_ids:
                problems.append(
                    {
                        "spec_id": spec_id,
                        "path": path_template,
                        "method": method,
                        "op_id": operation_id,
                        "problem": (
                            "OPERATION_ID_COLLISION"
                        ),
                    }
                )
                continue

            seen_operation_ids.add(
                operation_id
            )

            observed_spec_operation_count += 1
            global_operation_index += 1

            path_pointer = (
                "/paths/"
                + json_pointer_escape(
                    path_template
                )
            )

            operation_pointer = (
                path_pointer
                + "/"
                + method
            )

            path_parameters = path_item.get(
                "parameters",
                [],
            )

            operation_parameters = (
                operation.get(
                    "parameters",
                    [],
                )
            )

            raw_path_parameter_count = (
                len(path_parameters)
                if isinstance(
                    path_parameters,
                    list,
                )
                else None
            )

            raw_operation_parameter_count = (
                len(operation_parameters)
                if isinstance(
                    operation_parameters,
                    list,
                )
                else None
            )

            source_operation_id = (
                operation.get("operationId")
            )

            operation_record = {
                "op_id": operation_id,
                "global_operation_index":
                    global_operation_index,
                "spec_operation_index":
                    observed_spec_operation_count,
                "spec_id": spec_id,
                "family_id": family_ids_by_spec[spec_id],
                "raw_id": spec_record[
                    "raw_id"
                ],
                "provider_directory":
                    spec_record[
                        "provider_directory"
                    ],
                "source_relative_path":
                    spec_record[
                        "relative_path"
                    ],
                "source_class": spec_record[
                    "source_class"
                ],
                "source_dialect":
                    spec_record[
                        "source_dialect"
                    ],
                "canonical_openapi_version":
                    spec_record[
                        "canonical_openapi_version"
                    ],
                "canonical_relative_path":
                    canonical_relative_path,
                "canonical_file_sha256":
                    spec_record[
                        "canonical_file_sha256"
                    ],
                "path": path_template,
                "method": method,
                "source_operation_id":
                    source_operation_id,
                "source_path_pointer":
                    path_pointer,
                "source_operation_pointer":
                    operation_pointer,
                "raw_path_parameter_count":
                    raw_path_parameter_count,
                "raw_operation_parameter_count":
                    raw_operation_parameter_count,
                "has_request_body": (
                    "requestBody" in operation
                ),
                "response_entry_count": (
                    len(
                        operation.get(
                            "responses",
                            {},
                        )
                    )
                    if isinstance(
                        operation.get(
                            "responses",
                            {},
                        ),
                        dict,
                    )
                    else None
                ),
                "tag_count": (
                    len(
                        operation.get(
                            "tags",
                            [],
                        )
                    )
                    if isinstance(
                        operation.get(
                            "tags",
                            [],
                        ),
                        list,
                    )
                    else None
                ),
                "security_declared_at_operation":
                    (
                        "security" in operation
                    ),
                "operation_sha256":
                    hashlib.sha256(
                        canonical_json_bytes(
                            operation
                        )
                    ).hexdigest(),
            }

            operation_records.append(
                operation_record
            )

            method_counts[method] += 1

            source_class_operation_counts[
                spec_record["source_class"]
            ] += 1

            dialect_operation_counts[
                spec_record["source_dialect"]
            ] += 1

            version_operation_counts[
                spec_record[
                    "canonical_openapi_version"
                ]
            ] += 1

    declared_operation_count = int(
        spec_record["operation_count"]
    )

    count_matches = (
        observed_spec_operation_count
        == declared_operation_count
    )

    if not count_matches:
        problems.append(
            {
                "spec_id": spec_id,
                "problem": (
                    "SPECIFICATION_OPERATION_COUNT_"
                    "MISMATCH"
                ),
                "declared_operation_count":
                    declared_operation_count,
                "observed_operation_count":
                    observed_spec_operation_count,
            }
        )

    specification_count_rows.append(
        {
            "spec_id": spec_id,
            "provider_directory":
                spec_record[
                    "provider_directory"
                ],
            "source_class":
                spec_record["source_class"],
            "source_dialect":
                spec_record["source_dialect"],
            "canonical_openapi_version":
                spec_record[
                    "canonical_openapi_version"
                ],
            "declared_operation_count":
                declared_operation_count,
            "observed_operation_count":
                observed_spec_operation_count,
            "count_matches":
                count_matches,
            "nonoperation_path_item_key_count":
                sum(
                    nonoperation_path_item_keys
                    .values()
                ),
            "nonoperation_path_item_keys":
                json.dumps(
                    dict(
                        sorted(
                            nonoperation_path_item_keys
                            .items()
                        )
                    ),
                    ensure_ascii=False,
                    sort_keys=True,
                ),
        }
    )

    if (
        specification_index % 100 == 0
        or specification_index
        == len(records)
    ):
        print(
            json.dumps(
                {
                    "specifications_processed":
                        specification_index,
                    "specification_total":
                        len(records),
                    "operations_enumerated":
                        len(operation_records),
                    "problem_count":
                        len(problems),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - enumeration_start,
                        1,
                    ),
                }
            )
        )

if (
    len(operation_records)
    != EXPECTED_OPERATION_COUNT
):
    problems.append(
        {
            "problem": (
                "TOTAL_OPERATION_COUNT_MISMATCH"
            ),
            "expected": EXPECTED_OPERATION_COUNT,
            "observed": len(
                operation_records
            ),
        }
    )

if (
    len(seen_operation_ids)
    != len(operation_records)
):
    problems.append(
        {
            "problem": (
                "OPERATION_ID_UNIQUENESS_FAILURE"
            ),
            "record_count": len(
                operation_records
            ),
            "unique_id_count": len(
                seen_operation_ids
            ),
        }
    )

write_jsonl_atomic(
    OPERATION_INVENTORY_PATH,
    operation_records,
)

write_csv_atomic(
    SPECIFICATION_COUNTS_PATH,
    specification_count_rows,
    [
        "spec_id",
        "provider_directory",
        "source_class",
        "source_dialect",
        "canonical_openapi_version",
        "declared_operation_count",
        "observed_operation_count",
        "count_matches",
        "nonoperation_path_item_key_count",
        "nonoperation_path_item_keys",
    ],
)

enumeration_report = {
    "specification_count": len(records),
    "operation_bearing_specification_count":
        sum(
            1
            for row
            in specification_count_rows
            if row[
                "observed_operation_count"
            ]
            > 0
        ),
    "zero_operation_specification_count":
        sum(
            1
            for row
            in specification_count_rows
            if row[
                "observed_operation_count"
            ]
            == 0
        ),
    "operation_count": len(
        operation_records
    ),
    "unique_operation_id_count": len(
        seen_operation_ids
    ),
    "method_counts": dict(
        sorted(method_counts.items())
    ),
    "source_class_operation_counts": dict(
        sorted(
            source_class_operation_counts
            .items()
        )
    ),
    "dialect_operation_counts": dict(
        sorted(
            dialect_operation_counts.items()
        )
    ),
    "canonical_version_operation_counts":
        dict(
            sorted(
                version_operation_counts.items()
            )
        ),
    "operation_inventory": {
        "path": str(
            OPERATION_INVENTORY_PATH
        ),
        "bytes": (
            OPERATION_INVENTORY_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            OPERATION_INVENTORY_PATH
        ),
    },
    "specification_counts": {
        "path": str(
            SPECIFICATION_COUNTS_PATH
        ),
        "bytes": (
            SPECIFICATION_COUNTS_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            SPECIFICATION_COUNTS_PATH
        ),
    },
    "problems": problems,
    "duration_seconds": round(
        time.perf_counter()
        - enumeration_start,
        3,
    ),
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

write_json_atomic(
    ENUMERATION_REPORT_PATH,
    enumeration_report,
)

output = dict(enumeration_report)

output["report_path"] = str(
    ENUMERATION_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    ENUMERATION_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if problems:
    raise RuntimeError(
        "Operation enumeration failed. "
        "Review the reported problems."
    )

print(
    "\nNOTEBOOK 02 — CELL 6: PASS"
)

{"specifications_processed": 100, "specification_total": 1970, "operations_enumerated": 1399, "problem_count": 0, "elapsed_seconds": 0.4}
{"specifications_processed": 200, "specification_total": 1970, "operations_enumerated": 5204, "problem_count": 0, "elapsed_seconds": 1.0}
{"specifications_processed": 300, "specification_total": 1970, "operations_enumerated": 9917, "problem_count": 0, "elapsed_seconds": 1.8}
{"specifications_processed": 400, "specification_total": 1970, "operations_enumerated": 13455, "problem_count": 0, "elapsed_seconds": 2.4}
{"specifications_processed": 500, "specification_total": 1970, "operations_enumerated": 14255, "problem_count": 0, "elapsed_seconds": 2.6}
{"specifications_processed": 600, "specification_total": 1970, "operations_enumerated": 14488, "problem_count": 0, "elapsed_seconds": 2.7}
{"specifications_processed": 700, "specification_total": 1970, "operations_enumerated": 18273, "problem_count": 0, "elapsed_seconds": 3.2}
{"specifications_processed": 8

In [7]:
# NOTEBOOK 02 — CELL 7
# Implement and qualify the portable L0 operation slicer.

from copy import deepcopy
from pathlib import Path
from urllib.parse import unquote
import hashlib
import json
import time

from openapi_spec_validator import validate


SLICE_QUALIFICATION_ROOT = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "slice_qualification"
)

SLICE_QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_builder_qualification.json"
)

SLICE_QUALIFICATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def pointer_from_tokens(tokens):
    if not tokens:
        return ""

    return "/" + "/".join(
        json_pointer_escape(token)
        for token in tokens
    )


def tokens_from_local_reference(reference):
    if reference == "#":
        return []

    if not (
        isinstance(reference, str)
        and reference.startswith("#/")
    ):
        return None

    fragment = unquote(reference[1:])

    return [
        token
        .replace("~1", "/")
        .replace("~0", "~")
        for token in fragment[1:].split("/")
    ]


def resolve_pointer_tokens(document, tokens):
    current = document

    for token in tokens:
        if isinstance(current, dict):
            if token not in current:
                raise KeyError(token)

            current = current[token]

        elif isinstance(current, list):
            current = current[int(token)]

        else:
            raise TypeError(
                "Cannot traverse through "
                f"{type(current).__name__}."
            )

    return current


def resolve_local_reference(
    document,
    reference,
):
    tokens = tokens_from_local_reference(
        reference
    )

    if tokens is None:
        raise ValueError(
            f"Not a local reference: {reference!r}"
        )

    return resolve_pointer_tokens(
        document,
        tokens,
    )


def is_example_object(value):
    if not isinstance(value, dict):
        return False

    standard_keys = {
        "summary",
        "description",
        "value",
        "externalValue",
        "$ref",
    }

    return all(
        key in standard_keys
        or str(key).startswith("x-")
        for key in value
    )


def collect_dependency_references(value):
    references = set()

    def walk(node):
        if isinstance(node, dict):
            for reference_keyword in (
                "$ref",
                "$dynamicRef",
                "$recursiveRef",
            ):
                reference = node.get(
                    reference_keyword
                )

                if isinstance(reference, str):
                    references.add(reference)

            discriminator = node.get(
                "discriminator"
            )

            if isinstance(discriminator, dict):
                mapping = discriminator.get(
                    "mapping"
                )

                if isinstance(mapping, dict):
                    for reference in (
                        mapping.values()
                    ):
                        if (
                            isinstance(
                                reference,
                                str,
                            )
                            and reference.startswith(
                                "#/"
                            )
                        ):
                            references.add(
                                reference
                            )

            for key, child in node.items():
                if key == "example":
                    continue

                if (
                    key == "value"
                    and is_example_object(node)
                ):
                    continue

                if key == "examples":
                    if isinstance(child, list):
                        continue

                    if isinstance(child, dict):
                        for example_value in (
                            child.values()
                        ):
                            if isinstance(
                                example_value,
                                dict,
                            ):
                                reference = (
                                    example_value.get(
                                        "$ref"
                                    )
                                )

                                if isinstance(
                                    reference,
                                    str,
                                ):
                                    references.add(
                                        reference
                                    )

                                if not (
                                    "value"
                                    in example_value
                                ):
                                    walk(
                                        example_value
                                    )
                        continue

                walk(child)

        elif isinstance(node, list):
            for child in node:
                walk(child)

    walk(value)

    return references


def parameter_identity(
    parameter,
    source_document,
):
    resolved = parameter
    seen_references = set()

    while (
        isinstance(resolved, dict)
        and isinstance(
            resolved.get("$ref"),
            str,
        )
    ):
        reference = resolved["$ref"]

        if reference in seen_references:
            break

        seen_references.add(reference)

        try:
            resolved = resolve_local_reference(
                source_document,
                reference,
            )
        except Exception:
            break

    if isinstance(resolved, dict):
        name = resolved.get("name")
        location = resolved.get("in")

        if (
            isinstance(name, str)
            and isinstance(location, str)
        ):
            return (
                name,
                location,
            )

    if (
        isinstance(parameter, dict)
        and isinstance(
            parameter.get("$ref"),
            str,
        )
    ):
        return (
            "$ref",
            parameter["$ref"],
        )

    return (
        "$unresolved",
        hashlib.sha256(
            canonical_json_bytes(parameter)
        ).hexdigest(),
    )


def merge_effective_parameters(
    source_document,
    path_item,
    operation,
    source_path_pointer,
    source_operation_pointer,
):
    path_parameters = path_item.get(
        "parameters",
        [],
    )

    operation_parameters = operation.get(
        "parameters",
        [],
    )

    if not isinstance(path_parameters, list):
        raise TypeError(
            "Path-level parameters must be a list."
        )

    if not isinstance(
        operation_parameters,
        list,
    ):
        raise TypeError(
            "Operation parameters must be a list."
        )

    operation_identities = {
        parameter_identity(
            parameter,
            source_document,
        )
        for parameter in operation_parameters
    }

    effective_parameters = []
    parameter_sources = []

    for index, parameter in enumerate(
        path_parameters
    ):
        identity = parameter_identity(
            parameter,
            source_document,
        )

        if identity in operation_identities:
            continue

        effective_parameters.append(
            deepcopy(parameter)
        )

        parameter_sources.append(
            (
                f"{source_path_pointer}"
                f"/parameters/{index}"
            )
        )

    for index, parameter in enumerate(
        operation_parameters
    ):
        effective_parameters.append(
            deepcopy(parameter)
        )

        parameter_sources.append(
            (
                f"{source_operation_pointer}"
                f"/parameters/{index}"
            )
        )

    return (
        effective_parameters,
        parameter_sources,
    )


def strip_servers_from_operation(
    operation,
):
    if not isinstance(operation, dict):
        return

    operation.pop("servers", None)

    callbacks = operation.get("callbacks")

    if not isinstance(callbacks, dict):
        return

    for callback in callbacks.values():
        strip_servers_from_callback(
            callback
        )


def strip_servers_from_path_item(
    path_item,
):
    if not isinstance(path_item, dict):
        return

    path_item.pop("servers", None)

    for method in SUPPORTED_HTTP_METHODS:
        operation = path_item.get(method)

        if isinstance(operation, dict):
            strip_servers_from_operation(
                operation
            )


def strip_servers_from_callback(
    callback,
):
    if not isinstance(callback, dict):
        return

    if "$ref" in callback:
        return

    for expression, path_item in (
        callback.items()
    ):
        if str(expression).startswith("x-"):
            continue

        strip_servers_from_path_item(
            path_item
        )


def strip_servers_from_slice(
    slice_document,
):
    paths = slice_document.get(
        "paths",
        {},
    )

    if isinstance(paths, dict):
        for path_item in paths.values():
            strip_servers_from_path_item(
                path_item
            )

    components = slice_document.get(
        "components",
        {},
    )

    if not isinstance(components, dict):
        return

    callbacks = components.get("callbacks")

    if isinstance(callbacks, dict):
        for callback in callbacks.values():
            strip_servers_from_callback(
                callback
            )

    path_items = components.get(
        "pathItems"
    )

    if isinstance(path_items, dict):
        for path_item in path_items.values():
            strip_servers_from_path_item(
                path_item
            )


def add_prefix_mapping(
    mappings,
    slice_pointer,
    source_pointer,
    mapping_type="subtree_prefix",
):
    record = {
        "slice_pointer": slice_pointer,
        "source_pointer": source_pointer,
        "mapping_type": mapping_type,
    }

    if record not in mappings:
        mappings.append(record)


def build_operation_slice(
    source_document,
    operation_record,
):
    diagnostics = []
    mappings = []

    path_template = operation_record["path"]
    method = operation_record["method"]

    source_path_pointer = (
        operation_record[
            "source_path_pointer"
        ]
    )

    source_operation_pointer = (
        operation_record[
            "source_operation_pointer"
        ]
    )

    paths = source_document.get("paths")

    if not isinstance(paths, dict):
        raise TypeError(
            "Source paths must be an object."
        )

    path_item = paths[path_template]
    operation = path_item[method]

    if not isinstance(path_item, dict):
        raise TypeError(
            "Source Path Item must be an object."
        )

    if not isinstance(operation, dict):
        raise TypeError(
            "Source Operation must be an object."
        )

    effective_parameters, parameter_sources = (
        merge_effective_parameters(
            source_document,
            path_item,
            operation,
            source_path_pointer,
            source_operation_pointer,
        )
    )

    sliced_operation = deepcopy(operation)

    sliced_operation.pop("servers", None)

    if effective_parameters:
        sliced_operation[
            "parameters"
        ] = effective_parameters
    else:
        sliced_operation.pop(
            "parameters",
            None,
        )

    if "security" not in sliced_operation:
        if "security" in source_document:
            sliced_operation["security"] = deepcopy(
                source_document["security"]
            )

            add_prefix_mapping(
                mappings,
                (
                    source_operation_pointer
                    + "/security"
                ),
                "/security",
                "inherited_root_security",
            )

    sliced_path_item = {}

    for key, value in path_item.items():
        if key in {
            "summary",
            "description",
        } or str(key).startswith("x-"):
            sliced_path_item[key] = deepcopy(
                value
            )

            add_prefix_mapping(
                mappings,
                (
                    source_path_pointer
                    + "/"
                    + json_pointer_escape(key)
                ),
                (
                    source_path_pointer
                    + "/"
                    + json_pointer_escape(key)
                ),
            )

    sliced_path_item[method] = (
        sliced_operation
    )

    slice_document = {
        "openapi": source_document["openapi"],
        "info": deepcopy(
            STAGE_02_POLICY[
                "slice_policy"
            ]["fixed_info"]
        ),
        "paths": {
            path_template: sliced_path_item,
        },
    }

    add_prefix_mapping(
        mappings,
        "/openapi",
        "/openapi",
        "exact",
    )

    add_prefix_mapping(
        mappings,
        "/info",
        None,
        "synthetic_fixed_info",
    )

    add_prefix_mapping(
        mappings,
        source_operation_pointer,
        source_operation_pointer,
    )

    for index, parameter_source in enumerate(
        parameter_sources
    ):
        add_prefix_mapping(
            mappings,
            (
                source_operation_pointer
                + f"/parameters/{index}"
            ),
            parameter_source,
        )

    if (
        "security" in operation
        and "security" in sliced_operation
    ):
        add_prefix_mapping(
            mappings,
            (
                source_operation_pointer
                + "/security"
            ),
            (
                source_operation_pointer
                + "/security"
            ),
        )

    if (
        "jsonSchemaDialect"
        in source_document
    ):
        slice_document[
            "jsonSchemaDialect"
        ] = deepcopy(
            source_document[
                "jsonSchemaDialect"
            ]
        )

        add_prefix_mapping(
            mappings,
            "/jsonSchemaDialect",
            "/jsonSchemaDialect",
            "exact",
        )

    operation_tags = operation.get(
        "tags",
        [],
    )

    if isinstance(operation_tags, list):
        used_tag_names = {
            tag
            for tag in operation_tags
            if isinstance(tag, str)
        }
    else:
        used_tag_names = set()

    source_tags = source_document.get(
        "tags",
        [],
    )

    retained_tags = []

    if isinstance(source_tags, list):
        for source_tag_index, tag_object in (
            enumerate(source_tags)
        ):
            if not isinstance(
                tag_object,
                dict,
            ):
                continue

            tag_name = tag_object.get("name")

            if tag_name not in used_tag_names:
                continue

            output_tag_index = len(
                retained_tags
            )

            retained_tags.append(
                deepcopy(tag_object)
            )

            add_prefix_mapping(
                mappings,
                f"/tags/{output_tag_index}",
                f"/tags/{source_tag_index}",
            )

    if retained_tags:
        slice_document["tags"] = (
            retained_tags
        )

    effective_security = (
        sliced_operation.get(
            "security",
            [],
        )
    )

    required_security_scheme_names = set()

    if isinstance(effective_security, list):
        for requirement in effective_security:
            if not isinstance(
                requirement,
                dict,
            ):
                continue

            required_security_scheme_names.update(
                str(name)
                for name in requirement
            )

    source_components = source_document.get(
        "components",
        {},
    )

    if not isinstance(source_components, dict):
        source_components = {}

    for security_scheme_name in sorted(
        required_security_scheme_names
    ):
        source_security_schemes = (
            source_components.get(
                "securitySchemes",
                {},
            )
        )

        if not isinstance(
            source_security_schemes,
            dict,
        ):
            continue

        if (
            security_scheme_name
            not in source_security_schemes
        ):
            diagnostics.append(
                {
                    "type": (
                        "MISSING_SECURITY_SCHEME"
                    ),
                    "name":
                        security_scheme_name,
                }
            )
            continue

        slice_document.setdefault(
            "components",
            {},
        ).setdefault(
            "securitySchemes",
            {},
        )[security_scheme_name] = deepcopy(
            source_security_schemes[
                security_scheme_name
            ]
        )

        component_pointer = (
            "/components/securitySchemes/"
            + json_pointer_escape(
                security_scheme_name
            )
        )

        add_prefix_mapping(
            mappings,
            component_pointer,
            component_pointer,
        )

    copied_component_roots = set()

    while True:
        references = (
            collect_dependency_references(
                slice_document
            )
        )

        added_component = False

        for reference in sorted(references):
            if not reference.startswith("#"):
                diagnostics.append(
                    {
                        "type": (
                            "EXTERNAL_REFERENCE"
                        ),
                        "reference": reference,
                    }
                )
                continue

            try:
                resolve_local_reference(
                    slice_document,
                    reference,
                )
                continue
            except Exception:
                pass

            tokens = (
                tokens_from_local_reference(
                    reference
                )
            )

            if (
                tokens is None
                or len(tokens) < 3
                or tokens[0] != "components"
            ):
                diagnostics.append(
                    {
                        "type": (
                            "UNSUPPORTED_LOCAL_"
                            "REFERENCE_TARGET"
                        ),
                        "reference": reference,
                    }
                )
                continue

            component_root_tokens = (
                tokens[:3]
            )

            component_root_pointer = (
                pointer_from_tokens(
                    component_root_tokens
                )
            )

            if (
                component_root_pointer
                in copied_component_roots
            ):
                continue

            try:
                source_component = (
                    resolve_pointer_tokens(
                        source_document,
                        component_root_tokens,
                    )
                )
            except Exception as error:
                diagnostics.append(
                    {
                        "type": (
                            "MISSING_SOURCE_"
                            "REFERENCE_TARGET"
                        ),
                        "reference": reference,
                        "diagnostic": repr(error),
                    }
                )
                continue

            section_name = (
                component_root_tokens[1]
            )

            component_name = (
                component_root_tokens[2]
            )

            slice_document.setdefault(
                "components",
                {},
            ).setdefault(
                section_name,
                {},
            )[component_name] = deepcopy(
                source_component
            )

            copied_component_roots.add(
                component_root_pointer
            )

            add_prefix_mapping(
                mappings,
                component_root_pointer,
                component_root_pointer,
            )

            added_component = True

        if not added_component:
            break

    strip_servers_from_slice(
        slice_document
    )

    unresolved_references = []

    for reference in sorted(
        collect_dependency_references(
            slice_document
        )
    ):
        if not reference.startswith("#"):
            unresolved_references.append(
                reference
            )
            continue

        try:
            resolve_local_reference(
                slice_document,
                reference,
            )
        except Exception:
            unresolved_references.append(
                reference
            )

    for reference in unresolved_references:
        diagnostics.append(
            {
                "type": (
                    "UNRESOLVED_SLICE_REFERENCE"
                ),
                "reference": reference,
            }
        )

    mappings.sort(
        key=lambda record: (
            record["slice_pointer"],
            str(record["source_pointer"]),
            record["mapping_type"],
        )
    )

    return {
        "slice_document": slice_document,
        "pointer_mappings": mappings,
        "diagnostics": diagnostics,
    }


core_fixture = {
    "openapi": "3.0.3",
    "info": {
        "title": "Source API prose",
        "version": "9.9.9",
        "description": (
            "API-wide prose must not enter "
            "the operation slice."
        ),
    },
    "servers": [
        {
            "url": "https://unstable.example.test"
        }
    ],
    "security": [
        {
            "ApiKey": []
        }
    ],
    "tags": [
        {
            "name": "users",
            "description": (
                "Operations concerning users."
            ),
        },
        {
            "name": "unused",
            "description": "Must be omitted.",
        },
    ],
    "paths": {
        "/users/{id}": {
            "summary": "User path",
            "description": (
                "Path-level retained prose."
            ),
            "servers": [
                {
                    "url": (
                        "https://path.example.test"
                    )
                }
            ],
            "parameters": [
                {
                    "name": "id",
                    "in": "path",
                    "required": True,
                    "description": "User identifier.",
                    "schema": {
                        "type": "string"
                    },
                },
                {
                    "name": "verbose",
                    "in": "query",
                    "description": (
                        "Path-level verbose flag."
                    ),
                    "schema": {
                        "type": "boolean"
                    },
                },
            ],
            "get": {
                "operationId": "getUser",
                "summary": "Get one user",
                "description": (
                    "Returns the requested user."
                ),
                "tags": [
                    "users"
                ],
                "servers": [
                    {
                        "url": (
                            "https://operation."
                            "example.test"
                        )
                    }
                ],
                "parameters": [
                    {
                        "name": "verbose",
                        "in": "query",
                        "description": (
                            "Operation override."
                        ),
                        "schema": {
                            "type": "boolean"
                        },
                    },
                    {
                        "$ref": (
                            "#/components/"
                            "parameters/Limit"
                        )
                    },
                ],
                "requestBody": {
                    "description": (
                        "Optional user update."
                    ),
                    "content": {
                        "application/json": {
                            "schema": {
                                "$ref": (
                                    "#/components/"
                                    "schemas/User"
                                )
                            }
                        }
                    },
                },
                "responses": {
                    "200": {
                        "$ref": (
                            "#/components/"
                            "responses/UserResponse"
                        )
                    }
                },
            },
        }
    },
    "components": {
        "parameters": {
            "Limit": {
                "name": "limit",
                "in": "query",
                "description": "Result limit.",
                "schema": {
                    "type": "integer",
                    "minimum": 1,
                },
            }
        },
        "responses": {
            "UserResponse": {
                "description": "A user response.",
                "content": {
                    "application/json": {
                        "schema": {
                            "$ref": (
                                "#/components/"
                                "schemas/User"
                            )
                        }
                    }
                },
            }
        },
        "schemas": {
            "User": {
                "type": "object",
                "description": "A user.",
                "properties": {
                    "address": {
                        "$ref": (
                            "#/components/"
                            "schemas/Address"
                        )
                    }
                },
            },
            "Address": {
                "type": "object",
                "description": "An address.",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name.",
                    }
                },
            },
            "Unused": {
                "type": "object",
                "description": (
                    "Must not enter the slice."
                ),
            },
        },
        "securitySchemes": {
            "ApiKey": {
                "type": "apiKey",
                "in": "header",
                "name": "X-API-Key",
                "description": "API key.",
            }
        },
    },
}

core_operation_record = {
    "op_id": "fixture_core",
    "path": "/users/{id}",
    "method": "get",
    "source_path_pointer": (
        "/paths/~1users~1{id}"
    ),
    "source_operation_pointer": (
        "/paths/~1users~1{id}/get"
    ),
}

oas31_fixture = {
    "openapi": "3.1.0",
    "jsonSchemaDialect": (
        "https://json-schema.org/"
        "draft/2020-12/schema"
    ),
    "info": {
        "title": "OAS 3.1 source",
        "version": "1.0.0",
    },
    "security": [
        {
            "Basic": []
        }
    ],
    "paths": {
        "/status": {
            "get": {
                "operationId": "status",
                "security": [],
                "responses": {
                    "200": {
                        "description": "Current status.",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/Node"
                                    )
                                }
                            }
                        },
                    }
                },
            }
        }
    },
    "components": {
        "schemas": {
            "Node": {
                "type": "object",
                "$defs": {
                    "Child": {
                        "type": "string",
                        "description": (
                            "Nested child value."
                        ),
                    }
                },
                "properties": {
                    "child": {
                        "$ref": (
                            "#/components/schemas/"
                            "Node/$defs/Child"
                        )
                    }
                },
            },
            "Unused31": {
                "type": "string"
            },
        },
        "securitySchemes": {
            "Basic": {
                "type": "http",
                "scheme": "basic",
            }
        },
    },
}

oas31_operation_record = {
    "op_id": "fixture_oas31",
    "path": "/status",
    "method": "get",
    "source_path_pointer": (
        "/paths/~1status"
    ),
    "source_operation_pointer": (
        "/paths/~1status/get"
    ),
}

fixture_cases = [
    (
        "core_semantics",
        core_fixture,
        core_operation_record,
    ),
    (
        "oas31_nested_schema",
        oas31_fixture,
        oas31_operation_record,
    ),
]

qualification_start = time.perf_counter()
fixture_results = []
failures = []
check_count = 0
passed_check_count = 0


def record_check(
    fixture_name,
    check_name,
    passed,
    observed=None,
):
    global check_count
    global passed_check_count

    check_count += 1

    if passed:
        passed_check_count += 1
    else:
        failures.append(
            {
                "fixture": fixture_name,
                "check": check_name,
                "observed": observed,
            }
        )


for (
    fixture_name,
    fixture_document,
    fixture_operation,
) in fixture_cases:
    built = build_operation_slice(
        fixture_document,
        fixture_operation,
    )

    slice_document = built[
        "slice_document"
    ]

    pointer_mappings = built[
        "pointer_mappings"
    ]

    diagnostics = built["diagnostics"]

    validation_diagnostic = None

    try:
        validate(slice_document)
        validation_pass = True
    except Exception as error:
        validation_pass = False
        validation_diagnostic = repr(error)

    record_check(
        fixture_name,
        "openapi_validation",
        validation_pass,
        validation_diagnostic,
    )

    record_check(
        fixture_name,
        "no_slice_diagnostics",
        diagnostics == [],
        diagnostics,
    )

    record_check(
        fixture_name,
        "fixed_info_block",
        slice_document.get("info")
        == {
            "title": "Operation slice",
            "version": "1.0.0",
        },
        slice_document.get("info"),
    )

    record_check(
        fixture_name,
        "single_target_path",
        list(
            slice_document.get(
                "paths",
                {}
            )
        )
        == [fixture_operation["path"]],
        list(
            slice_document.get(
                "paths",
                {}
            )
        ),
    )

    record_check(
        fixture_name,
        "pointer_map_present",
        len(pointer_mappings) > 0,
        len(pointer_mappings),
    )

    slice_path_item = (
        slice_document["paths"][
            fixture_operation["path"]
        ]
    )

    slice_operation = slice_path_item[
        fixture_operation["method"]
    ]

    record_check(
        fixture_name,
        "root_servers_removed",
        "servers" not in slice_document,
        slice_document.get("servers"),
    )

    record_check(
        fixture_name,
        "path_servers_removed",
        "servers" not in slice_path_item,
        slice_path_item.get("servers"),
    )

    record_check(
        fixture_name,
        "operation_servers_removed",
        "servers" not in slice_operation,
        slice_operation.get("servers"),
    )

    if fixture_name == "core_semantics":
        effective_parameters = (
            slice_operation.get(
                "parameters",
                []
            )
        )

        effective_identities = [
            parameter_identity(
                parameter,
                slice_document,
            )
            for parameter
            in effective_parameters
        ]

        record_check(
            fixture_name,
            "effective_parameter_override",
            effective_identities
            == [
                ("id", "path"),
                ("verbose", "query"),
                ("limit", "query"),
            ],
            effective_identities,
        )

        record_check(
            fixture_name,
            "security_inherited",
            slice_operation.get("security")
            == [
                {
                    "ApiKey": []
                }
            ],
            slice_operation.get("security"),
        )

        retained_schema_names = set(
            slice_document.get(
                "components",
                {},
            ).get(
                "schemas",
                {},
            )
        )

        record_check(
            fixture_name,
            "reachable_schemas_only",
            retained_schema_names
            == {
                "User",
                "Address",
            },
            sorted(retained_schema_names),
        )

        retained_tag_names = [
            tag.get("name")
            for tag in slice_document.get(
                "tags",
                []
            )
        ]

        record_check(
            fixture_name,
            "used_tag_object_only",
            retained_tag_names == ["users"],
            retained_tag_names,
        )

    if fixture_name == "oas31_nested_schema":
        record_check(
            fixture_name,
            "json_schema_dialect_retained",
            slice_document.get(
                "jsonSchemaDialect"
            )
            == oas31_fixture[
                "jsonSchemaDialect"
            ],
            slice_document.get(
                "jsonSchemaDialect"
            ),
        )

        record_check(
            fixture_name,
            "explicit_empty_security_preserved",
            slice_operation.get("security")
            == [],
            slice_operation.get("security"),
        )

        security_schemes = (
            slice_document.get(
                "components",
                {},
            ).get(
                "securitySchemes",
                {},
            )
        )

        record_check(
            fixture_name,
            "unused_inherited_security_scheme_omitted",
            "Basic" not in security_schemes,
            sorted(security_schemes),
        )

        retained_schema_names = set(
            slice_document.get(
                "components",
                {},
            ).get(
                "schemas",
                {},
            )
        )

        record_check(
            fixture_name,
            "oas31_reachable_schema_only",
            retained_schema_names == {"Node"},
            sorted(retained_schema_names),
        )

    slice_path = (
        SLICE_QUALIFICATION_ROOT
        / f"{fixture_name}.slice.json"
    )

    mapping_path = (
        SLICE_QUALIFICATION_ROOT
        / f"{fixture_name}.pointer_map.json"
    )

    write_json_atomic(
        slice_path,
        slice_document,
    )

    write_json_atomic(
        mapping_path,
        {
            "op_id": fixture_operation[
                "op_id"
            ],
            "mappings": pointer_mappings,
            "diagnostics": diagnostics,
        },
    )

    fixture_results.append(
        {
            "fixture": fixture_name,
            "slice_path": str(slice_path),
            "slice_sha256": sha256_file(
                slice_path
            ),
            "pointer_map_path": str(
                mapping_path
            ),
            "pointer_map_sha256":
                sha256_file(mapping_path),
            "validation_pass":
                validation_pass,
            "diagnostics": diagnostics,
        }
    )

qualification_report = {
    "policy_version": (
        STAGE_02_POLICY["policy_version"]
    ),
    "fixture_count": len(fixture_cases),
    "check_count": check_count,
    "passed_check_count":
        passed_check_count,
    "failed_check_count": len(failures),
    "failures": failures,
    "fixture_results": fixture_results,
    "qualification_decision": (
        "QUALIFIED_FOR_CORPUS_SLICING"
        if not failures
        else "NOT_QUALIFIED"
    ),
    "duration_seconds": round(
        time.perf_counter()
        - qualification_start,
        3,
    ),
    "status": (
        "PASS"
        if not failures
        else "BLOCKED"
    ),
}

write_json_atomic(
    SLICE_QUALIFICATION_REPORT_PATH,
    qualification_report,
)

output = dict(qualification_report)

output["report_path"] = str(
    SLICE_QUALIFICATION_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    SLICE_QUALIFICATION_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if failures:
    raise RuntimeError(
        "L0 slice-builder qualification failed."
    )

print(
    "\nNOTEBOOK 02 — CELL 7: PASS"
)

{
  "policy_version": "1.0",
  "fixture_count": 2,
  "check_count": 24,
  "passed_check_count": 24,
  "failed_check_count": 0,
  "failures": [],
  "fixture_results": [
    {
      "fixture": "core_semantics",
      "slice_path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/development_examples/slice_qualification/core_semantics.slice.json",
      "slice_sha256": "8b89bcf1e15ed6488b750079545347447d0ab637cfbf2063d2aab956a49e1b50",
      "pointer_map_path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/development_examples/slice_qualification/core_semantics.pointer_map.json",
      "pointer_map_sha256": "d0201251ccf6f8fd2fd12e582ec2b28b0002c1f80ed41e54c1c81f3c34f3c195",
      "validation_pass": true,
      "diagnostics": []
    },
    {
      "fixture": "oas31_nested_schema",
      "slice_path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/development_examples/slice_qualification/oas31_nested_schema.slice.json",
      "slice_sha256":

In [8]:
# NOTEBOOK 02 — CELL 8 (CORRECTED)
# Amend the slicer for valid cross-path references, qualify the
# amendment, and rerun the 2,000-operation corpus preflight.

from collections import Counter, defaultdict
from copy import deepcopy
from pathlib import Path
import hashlib
import json
import math
import os
import shutil
import statistics
import time


PREFLIGHT_TARGET_COUNT = 2000

PREFLIGHT_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "data"
    / "checkpoints"
    / "slice_preflight"
)

PREFLIGHT_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_corpus_preflight.json"
)

REFERENCE_AMENDMENT_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_reference_relocation_amendment.json"
)

REFERENCE_QUALIFICATION_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_reference_relocation_qualification.json"
)

PREFLIGHT_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# Preserve the original qualified builder across reruns.
if (
    "BUILD_OPERATION_SLICE_COMPONENTS_ONLY"
    not in globals()
):
    BUILD_OPERATION_SLICE_COMPONENTS_ONLY = (
        build_operation_slice
    )


def collect_dependency_references(value):
    """
    Collect actual OpenAPI/JSON Schema references while avoiding
    arbitrary example payload values. Unlike the original walker,
    this does not mistake a schema property named `value` for an
    Example Object.
    """
    references = set()

    def walk(node, example_object=False):
        if isinstance(node, dict):
            for reference_keyword in (
                "$ref",
                "$dynamicRef",
                "$recursiveRef",
            ):
                reference = node.get(
                    reference_keyword
                )

                if isinstance(reference, str):
                    references.add(reference)

            discriminator = node.get(
                "discriminator"
            )

            if isinstance(discriminator, dict):
                mapping = discriminator.get(
                    "mapping"
                )

                if isinstance(mapping, dict):
                    for reference in (
                        mapping.values()
                    ):
                        if (
                            isinstance(
                                reference,
                                str,
                            )
                            and reference.startswith(
                                "#/"
                            )
                        ):
                            references.add(
                                reference
                            )

            for key, child in node.items():
                if key == "example":
                    continue

                if (
                    example_object
                    and key == "value"
                ):
                    continue

                if key == "examples":
                    if isinstance(child, list):
                        continue

                    if isinstance(child, dict):
                        for example_child in (
                            child.values()
                        ):
                            walk(
                                example_child,
                                example_object=True,
                            )
                    continue

                walk(
                    child,
                    example_object=False,
                )

        elif isinstance(node, list):
            for child in node:
                walk(
                    child,
                    example_object=False,
                )

    walk(value)

    return references


def iterate_reference_locations(value):
    """
    Yield mutable containers containing genuine reference strings.
    Each result includes the slice pointer of the Reference Object.
    """
    locations = []

    def walk(
        node,
        tokens,
        example_object=False,
    ):
        if isinstance(node, dict):
            for reference_keyword in (
                "$ref",
                "$dynamicRef",
                "$recursiveRef",
            ):
                reference = node.get(
                    reference_keyword
                )

                if isinstance(reference, str):
                    locations.append(
                        {
                            "container": node,
                            "key": reference_keyword,
                            "reference": reference,
                            "location_tokens": list(
                                tokens
                            ),
                            "reference_pointer": (
                                pointer_from_tokens(
                                    [
                                        *tokens,
                                        reference_keyword,
                                    ]
                                )
                            ),
                        }
                    )

            discriminator = node.get(
                "discriminator"
            )

            if isinstance(discriminator, dict):
                mapping = discriminator.get(
                    "mapping"
                )

                if isinstance(mapping, dict):
                    for mapping_name, reference in (
                        mapping.items()
                    ):
                        if (
                            isinstance(
                                reference,
                                str,
                            )
                            and reference.startswith(
                                "#/"
                            )
                        ):
                            locations.append(
                                {
                                    "container":
                                        mapping,
                                    "key":
                                        mapping_name,
                                    "reference":
                                        reference,
                                    "location_tokens":
                                        [
                                            *tokens,
                                            "discriminator",
                                            "mapping",
                                            mapping_name,
                                        ],
                                    "reference_pointer":
                                        pointer_from_tokens(
                                            [
                                                *tokens,
                                                "discriminator",
                                                "mapping",
                                                mapping_name,
                                            ]
                                        ),
                                }
                            )

            for key, child in list(
                node.items()
            ):
                if key == "example":
                    continue

                if (
                    example_object
                    and key == "value"
                ):
                    continue

                if key == "examples":
                    if isinstance(child, list):
                        continue

                    if isinstance(child, dict):
                        for example_name, example_child in (
                            list(child.items())
                        ):
                            walk(
                                example_child,
                                [
                                    *tokens,
                                    key,
                                    example_name,
                                ],
                                example_object=True,
                            )
                    continue

                walk(
                    child,
                    [
                        *tokens,
                        key,
                    ],
                    example_object=False,
                )

        elif isinstance(node, list):
            for index, child in enumerate(node):
                walk(
                    child,
                    [
                        *tokens,
                        str(index),
                    ],
                    example_object=False,
                )

    walk(value, [])

    return locations


def infer_component_section(
    location_tokens,
    target_value,
):
    tokens = [
        str(token)
        for token in location_tokens
    ]

    schema_markers = {
        "schema",
        "schemas",
        "properties",
        "items",
        "prefixItems",
        "allOf",
        "anyOf",
        "oneOf",
        "not",
        "contains",
        "additionalProperties",
        "unevaluatedProperties",
    }

    if any(
        token in schema_markers
        for token in tokens
    ):
        return "schemas"

    if len(tokens) >= 2:
        parent_key = tokens[-2]

        section_by_parent = {
            "parameters": "parameters",
            "responses": "responses",
            "headers": "headers",
            "callbacks": "callbacks",
            "links": "links",
            "examples": "examples",
        }

        if parent_key in section_by_parent:
            return section_by_parent[
                parent_key
            ]

    if tokens and tokens[-1] == "requestBody":
        return "requestBodies"

    if isinstance(target_value, dict):
        if {
            "name",
            "in",
        }.issubset(target_value):
            return "parameters"

        target_type = target_value.get("type")

        if target_type in {
            "apiKey",
            "http",
            "oauth2",
            "openIdConnect",
            "mutualTLS",
        }:
            return "securitySchemes"

    return "schemas"


def deterministic_relocated_name(
    section_name,
    original_reference,
):
    identity = (
        section_name
        + "\0"
        + original_reference
    ).encode("utf-8")

    return (
        "stage02_relocated_"
        + hashlib.sha256(identity)
        .hexdigest()[:24]
    )


def deduplicate_diagnostics(diagnostics):
    unique = {}

    for diagnostic in diagnostics:
        key = json.dumps(
            diagnostic,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
            default=str,
        )

        unique[key] = diagnostic

    return [
        unique[key]
        for key in sorted(unique)
    ]


def materialize_and_relocate_references(
    source_document,
    slice_document,
    pointer_mappings,
):
    diagnostics = []
    relocations = []

    maximum_passes = 1000

    for pass_index in range(maximum_passes):
        changed = False

        locations = iterate_reference_locations(
            slice_document
        )

        for location in locations:
            reference = location["reference"]

            if not reference.startswith("#"):
                diagnostics.append(
                    {
                        "type": (
                            "EXTERNAL_REFERENCE"
                        ),
                        "reference": reference,
                    }
                )
                continue

            try:
                resolve_local_reference(
                    slice_document,
                    reference,
                )
                continue
            except Exception:
                pass

            source_tokens = (
                tokens_from_local_reference(
                    reference
                )
            )

            if source_tokens is None:
                diagnostics.append(
                    {
                        "type": (
                            "INVALID_LOCAL_REFERENCE"
                        ),
                        "reference": reference,
                    }
                )
                continue

            try:
                source_target = (
                    resolve_pointer_tokens(
                        source_document,
                        source_tokens,
                    )
                )
            except Exception as error:
                diagnostics.append(
                    {
                        "type": (
                            "MISSING_SOURCE_REFERENCE_"
                            "TARGET"
                        ),
                        "reference": reference,
                        "diagnostic": repr(error),
                    }
                )
                continue

            # Preserve ordinary component references in their
            # original component namespace.
            if (
                len(source_tokens) >= 3
                and source_tokens[0]
                == "components"
            ):
                component_root_tokens = (
                    source_tokens[:3]
                )

                section_name = (
                    component_root_tokens[1]
                )

                component_name = (
                    component_root_tokens[2]
                )

                component_root_pointer = (
                    pointer_from_tokens(
                        component_root_tokens
                    )
                )

                try:
                    source_component = (
                        resolve_pointer_tokens(
                            source_document,
                            component_root_tokens,
                        )
                    )
                except Exception as error:
                    diagnostics.append(
                        {
                            "type": (
                                "MISSING_COMPONENT_ROOT"
                            ),
                            "reference": reference,
                            "diagnostic":
                                repr(error),
                        }
                    )
                    continue

                component_section = (
                    slice_document.setdefault(
                        "components",
                        {},
                    ).setdefault(
                        section_name,
                        {},
                    )
                )

                if (
                    component_name
                    not in component_section
                ):
                    component_section[
                        component_name
                    ] = deepcopy(
                        source_component
                    )

                    add_prefix_mapping(
                        pointer_mappings,
                        component_root_pointer,
                        component_root_pointer,
                    )

                    changed = True

                continue

            # A valid reference outside components is relocated
            # into a deterministic typed component.
            section_name = (
                infer_component_section(
                    location[
                        "location_tokens"
                    ],
                    source_target,
                )
            )

            component_name = (
                deterministic_relocated_name(
                    section_name,
                    reference,
                )
            )

            synthetic_pointer = (
                "/components/"
                + json_pointer_escape(
                    section_name
                )
                + "/"
                + json_pointer_escape(
                    component_name
                )
            )

            rewritten_reference = (
                "#"
                + synthetic_pointer
            )

            component_section = (
                slice_document.setdefault(
                    "components",
                    {},
                ).setdefault(
                    section_name,
                    {},
                )
            )

            if (
                component_name
                not in component_section
            ):
                component_section[
                    component_name
                ] = deepcopy(
                    source_target
                )

                add_prefix_mapping(
                    pointer_mappings,
                    synthetic_pointer,
                    pointer_from_tokens(
                        source_tokens
                    ),
                    (
                        "relocated_local_reference_"
                        "target"
                    ),
                )

            location["container"][
                location["key"]
            ] = rewritten_reference

            relocations.append(
                {
                    "slice_reference_pointer":
                        location[
                            "reference_pointer"
                        ],
                    "original_reference":
                        reference,
                    "rewritten_reference":
                        rewritten_reference,
                    "component_section":
                        section_name,
                    "source_target_pointer":
                        pointer_from_tokens(
                            source_tokens
                        ),
                }
            )

            changed = True

        if not changed:
            break

    else:
        diagnostics.append(
            {
                "type": (
                    "REFERENCE_RELOCATION_PASS_"
                    "LIMIT_EXCEEDED"
                ),
                "maximum_passes":
                    maximum_passes,
            }
        )

    strip_servers_from_slice(
        slice_document
    )

    for reference in sorted(
        collect_dependency_references(
            slice_document
        )
    ):
        if not reference.startswith("#"):
            diagnostics.append(
                {
                    "type": (
                        "EXTERNAL_REFERENCE"
                    ),
                    "reference": reference,
                }
            )
            continue

        try:
            resolve_local_reference(
                slice_document,
                reference,
            )
        except Exception as error:
            diagnostics.append(
                {
                    "type": (
                        "UNRESOLVED_SLICE_REFERENCE"
                    ),
                    "reference": reference,
                    "diagnostic": repr(error),
                }
            )

    pointer_mappings.sort(
        key=lambda record: (
            record["slice_pointer"],
            str(record["source_pointer"]),
            record["mapping_type"],
        )
    )

    relocation_map = {}

    for relocation in relocations:
        key = json.dumps(
            relocation,
            sort_keys=True,
            separators=(",", ":"),
        )

        relocation_map[key] = relocation

    return {
        "diagnostics":
            deduplicate_diagnostics(
                diagnostics
            ),
        "relocations": [
            relocation_map[key]
            for key in sorted(relocation_map)
        ],
    }


def build_operation_slice(
    source_document,
    operation_record,
):
    initial = (
        BUILD_OPERATION_SLICE_COMPONENTS_ONLY(
            source_document,
            operation_record,
        )
    )

    slice_document = initial[
        "slice_document"
    ]

    pointer_mappings = initial[
        "pointer_mappings"
    ]

    retained_initial_diagnostics = [
        diagnostic
        for diagnostic in initial[
            "diagnostics"
        ]
        if diagnostic.get("type")
        not in {
            "UNSUPPORTED_LOCAL_REFERENCE_TARGET",
            "UNRESOLVED_SLICE_REFERENCE",
        }
    ]

    relocation_result = (
        materialize_and_relocate_references(
            source_document,
            slice_document,
            pointer_mappings,
        )
    )

    diagnostics = (
        retained_initial_diagnostics
        + relocation_result[
            "diagnostics"
        ]
    )

    return {
        "slice_document": slice_document,
        "pointer_mappings":
            pointer_mappings,
        "reference_relocations":
            relocation_result[
                "relocations"
            ],
        "diagnostics":
            deduplicate_diagnostics(
                diagnostics
            ),
    }


# Record the prospective policy amendment before the full run.
previous_policy_sha256 = sha256_file(
    STAGE_02_POLICY_PATH
)

STAGE_02_POLICY["policy_version"] = "1.1"

STAGE_02_POLICY[
    "slice_policy"
][
    "noncomponent_local_reference_policy"
] = {
    "action": (
        "Relocate the referenced target into a "
        "deterministically named typed component "
        "and rewrite the slice reference."
    ),
    "preserve_target_content": True,
    "preserve_source_pointer_mapping": True,
    "deduplicate_same_target_per_component_type":
        True,
    "component_name_namespace":
        "stage02_relocated_<sha256-prefix>",
    "silent_patching_allowed": False,
    "relocations_must_be_audited": True,
}

write_yaml_atomic(
    STAGE_02_POLICY_PATH,
    STAGE_02_POLICY,
)

amendment_report = {
    "status": "PASS",
    "amendment": (
        "NONCOMPONENT_LOCAL_REFERENCE_RELOCATION"
    ),
    "policy_version": "1.1",
    "prospective": True,
    "full_corpus_slicing_started": False,
    "development_preflight_outputs_are_not_"
    "official_outputs": True,
    "reason": (
        "Valid eligible specifications contain "
        "local references into other paths. A "
        "single-operation portable slice cannot "
        "retain unrelated paths, so each target is "
        "copied once into a deterministic typed "
        "component and the reference is rewritten."
    ),
    "also_corrected": (
        "The dependency walker no longer treats "
        "a schema property named `value` as an "
        "Example Object value."
    ),
    "previous_policy_sha256":
        previous_policy_sha256,
    "updated_policy_sha256":
        sha256_file(STAGE_02_POLICY_PATH),
}

write_json_atomic(
    REFERENCE_AMENDMENT_REPORT_PATH,
    amendment_report,
)


# Qualify the new behavior on a controlled cross-path fixture.
relocation_fixture = {
    "openapi": "3.0.3",
    "info": {
        "title": "Relocation fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/shared": {
            "get": {
                "parameters": [
                    {
                        "name": "shared-id",
                        "in": "query",
                        "schema": {
                            "type": "string"
                        },
                    }
                ],
                "responses": {
                    "default": {
                        "description": (
                            "Shared response."
                        )
                    }
                },
            }
        },
        "/target": {
            "get": {
                "parameters": [
                    {
                        "$ref": (
                            "#/paths/~1shared/get/"
                            "parameters/0"
                        )
                    }
                ],
                "responses": {
                    "200": {
                        "$ref": (
                            "#/paths/~1shared/get/"
                            "responses/default"
                        )
                    },
                    "201": {
                        "description": "Created.",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/Root"
                                    )
                                }
                            }
                        },
                    },
                },
            }
        },
    },
    "components": {
        "schemas": {
            "Root": {
                "type": "object",
                "properties": {
                    "value": {
                        "$ref": (
                            "#/components/"
                            "schemas/Child"
                        )
                    }
                },
            },
            "Child": {
                "type": "string",
                "description": "Child value.",
            },
        }
    },
}

relocation_operation = {
    "op_id": "fixture_relocation",
    "path": "/target",
    "method": "get",
    "source_path_pointer":
        "/paths/~1target",
    "source_operation_pointer":
        "/paths/~1target/get",
}

relocation_built = build_operation_slice(
    relocation_fixture,
    relocation_operation,
)

relocation_slice = relocation_built[
    "slice_document"
]

relocation_validation_diagnostic = None

try:
    validate(relocation_slice)
    relocation_validation_pass = True
except Exception as error:
    relocation_validation_pass = False
    relocation_validation_diagnostic = repr(
        error
    )

relocated_components = (
    relocation_slice.get(
        "components",
        {},
    )
)

relocation_checks = {
    "validation_pass":
        relocation_validation_pass,
    "no_diagnostics":
        relocation_built[
            "diagnostics"
        ]
        == [],
    "two_reference_relocations":
        len(
            relocation_built[
                "reference_relocations"
            ]
        )
        == 2,
    "parameter_component_created":
        len(
            relocated_components.get(
                "parameters",
                {},
            )
        )
        == 1,
    "response_component_created":
        len(
            relocated_components.get(
                "responses",
                {},
            )
        )
        == 1,
    "nested_component_dependency_found":
        "Child"
        in relocated_components.get(
            "schemas",
            {},
        ),
    "unrelated_source_path_omitted":
        "/shared"
        not in relocation_slice["paths"],
}

reference_qualification_status = (
    "PASS"
    if all(relocation_checks.values())
    else "BLOCKED"
)

reference_qualification_report = {
    "status":
        reference_qualification_status,
    "checks": relocation_checks,
    "validation_diagnostic":
        relocation_validation_diagnostic,
    "diagnostics":
        relocation_built["diagnostics"],
    "reference_relocations":
        relocation_built[
            "reference_relocations"
        ],
}

write_json_atomic(
    REFERENCE_QUALIFICATION_REPORT_PATH,
    reference_qualification_report,
)

if reference_qualification_status != "PASS":
    print(
        json.dumps(
            reference_qualification_report,
            indent=2,
        )
    )

    raise RuntimeError(
        "Reference relocation qualification "
        "failed."
    )


# Remove failed-development checkpoint records.
for stale_checkpoint in (
    PREFLIGHT_CHECKPOINT_DIR.glob(
        "preflight_*.jsonl"
    )
):
    stale_checkpoint.unlink()


operations_by_spec = defaultdict(list)

for operation_record in operation_records:
    operations_by_spec[
        operation_record["spec_id"]
    ].append(operation_record)

selected_operation_ids = set()

for spec_id in sorted(operations_by_spec):
    if operations_by_spec[spec_id]:
        selected_operation_ids.add(
            operations_by_spec[
                spec_id
            ][0]["op_id"]
        )

remaining_needed = max(
    0,
    PREFLIGHT_TARGET_COUNT
    - len(selected_operation_ids),
)

if remaining_needed:
    spacing = (
        len(operation_records)
        / remaining_needed
    )

    for index in range(remaining_needed):
        selected_index = min(
            int(
                math.floor(
                    index * spacing
                    + spacing / 2
                )
            ),
            len(operation_records) - 1,
        )

        selected_operation_ids.add(
            operation_records[
                selected_index
            ]["op_id"]
        )

if (
    len(selected_operation_ids)
    < PREFLIGHT_TARGET_COUNT
):
    for operation_record in operation_records:
        selected_operation_ids.add(
            operation_record["op_id"]
        )

        if (
            len(selected_operation_ids)
            >= PREFLIGHT_TARGET_COUNT
        ):
            break

selected_operations = [
    operation_record
    for operation_record in operation_records
    if operation_record["op_id"]
    in selected_operation_ids
]

selected_operations.sort(
    key=lambda record:
        record["global_operation_index"]
)

specification_record_map = {
    record["spec_id"]: record
    for record in records
}

preflight_start = time.perf_counter()

preflight_records = []
checkpoint_buffer = []
problem_records = []
slice_sizes = []
mapping_counts = []
relocation_counts = []

method_counts = Counter()
source_class_counts = Counter()
version_counts = Counter()

current_spec_id = None
current_document = None

for sample_index, operation_record in enumerate(
    selected_operations,
    start=1,
):
    operation_start = time.perf_counter()

    spec_id = operation_record["spec_id"]
    op_id = operation_record["op_id"]

    if spec_id != current_spec_id:
        spec_record = (
            specification_record_map[
                spec_id
            ]
        )

        canonical_path = (
            STAGE_01_PROJECT_ROOT
            / spec_record[
                "canonical_relative_path"
            ]
        )

        current_document = json.loads(
            canonical_path.read_text(
                encoding="utf-8"
            )
        )

        current_spec_id = spec_id

    try:
        built = build_operation_slice(
            current_document,
            operation_record,
        )

        slice_document = built[
            "slice_document"
        ]

        serialized_slice = (
            canonical_json_bytes(
                slice_document
            )
        )

        slice_path = (
            PROJECT_ROOT
            / "data"
            / "slices"
            / "L0"
            / f"{op_id}.json"
        )

        write_bytes_atomic(
            slice_path,
            serialized_slice,
        )

        validation_status = "PASS"
        validation_diagnostic = None

        try:
            validate(slice_document)
        except Exception as error:
            validation_status = "FAIL"
            validation_diagnostic = repr(
                error
            )

        status = (
            "PASS"
            if (
                not built["diagnostics"]
                and validation_status
                == "PASS"
            )
            else "FAIL"
        )

        checkpoint_record = {
            "op_id": op_id,
            "global_operation_index":
                operation_record[
                    "global_operation_index"
                ],
            "spec_id": spec_id,
            "path": operation_record["path"],
            "method":
                operation_record["method"],
            "source_class":
                operation_record[
                    "source_class"
                ],
            "canonical_openapi_version":
                operation_record[
                    "canonical_openapi_version"
                ],
            "slice_relative_path": (
                slice_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "slice_bytes": len(
                serialized_slice
            ),
            "slice_sha256":
                hashlib.sha256(
                    serialized_slice
                ).hexdigest(),
            "pointer_mapping_count": len(
                built["pointer_mappings"]
            ),
            "pointer_mappings":
                built["pointer_mappings"],
            "reference_relocation_count": len(
                built[
                    "reference_relocations"
                ]
            ),
            "reference_relocations":
                built[
                    "reference_relocations"
                ],
            "diagnostics":
                built["diagnostics"],
            "validation_status":
                validation_status,
            "validation_diagnostic":
                validation_diagnostic,
            "status": status,
            "duration_seconds": round(
                time.perf_counter()
                - operation_start,
                6,
            ),
        }

    except Exception as error:
        checkpoint_record = {
            "op_id": op_id,
            "global_operation_index":
                operation_record[
                    "global_operation_index"
                ],
            "spec_id": spec_id,
            "path": operation_record["path"],
            "method":
                operation_record["method"],
            "source_class":
                operation_record[
                    "source_class"
                ],
            "canonical_openapi_version":
                operation_record[
                    "canonical_openapi_version"
                ],
            "slice_relative_path": None,
            "slice_bytes": None,
            "slice_sha256": None,
            "pointer_mapping_count": 0,
            "pointer_mappings": [],
            "reference_relocation_count": 0,
            "reference_relocations": [],
            "diagnostics": [
                {
                    "type": (
                        "SLICE_BUILD_EXCEPTION"
                    ),
                    "diagnostic": repr(error),
                }
            ],
            "validation_status": "NOT_RUN",
            "validation_diagnostic": None,
            "status": "FAIL",
            "duration_seconds": round(
                time.perf_counter()
                - operation_start,
                6,
            ),
        }

    preflight_records.append(
        checkpoint_record
    )

    checkpoint_buffer.append(
        checkpoint_record
    )

    if checkpoint_record["status"] == "PASS":
        slice_sizes.append(
            checkpoint_record["slice_bytes"]
        )

        mapping_counts.append(
            checkpoint_record[
                "pointer_mapping_count"
            ]
        )

        relocation_counts.append(
            checkpoint_record[
                "reference_relocation_count"
            ]
        )
    else:
        problem_records.append(
            checkpoint_record
        )

    method_counts[
        operation_record["method"]
    ] += 1

    source_class_counts[
        operation_record["source_class"]
    ] += 1

    version_counts[
        operation_record[
            "canonical_openapi_version"
        ]
    ] += 1

    if (
        len(checkpoint_buffer) == 250
        or sample_index
        == len(selected_operations)
    ):
        checkpoint_number = (
            (sample_index - 1) // 250
        )

        checkpoint_path = (
            PREFLIGHT_CHECKPOINT_DIR
            / (
                "preflight_"
                f"{checkpoint_number:03d}.jsonl"
            )
        )

        write_jsonl_atomic(
            checkpoint_path,
            checkpoint_buffer,
        )

        checkpoint_buffer = []

    if (
        sample_index % 100 == 0
        or sample_index
        == len(selected_operations)
    ):
        print(
            json.dumps(
                {
                    "sample_processed":
                        sample_index,
                    "sample_total":
                        len(
                            selected_operations
                        ),
                    "slice_pass_count":
                        sample_index
                        - len(problem_records),
                    "problem_count":
                        len(problem_records),
                    "reference_relocation_count":
                        sum(relocation_counts),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - preflight_start,
                        1,
                    ),
                }
            )
        )

if not slice_sizes:
    raise RuntimeError(
        "No preflight slices passed."
    )

sorted_slice_sizes = sorted(slice_sizes)

mean_slice_bytes = statistics.fmean(
    slice_sizes
)

p95_index = min(
    math.ceil(
        0.95 * len(sorted_slice_sizes)
    )
    - 1,
    len(sorted_slice_sizes) - 1,
)

projected_slice_bytes = int(
    math.ceil(
        mean_slice_bytes
        * EXPECTED_OPERATION_COUNT
    )
)

disk_usage = shutil.disk_usage(
    "/kaggle/working"
)

additional_reserve = int(
    projected_slice_bytes * 0.25
    + 2 * (1024 ** 3)
)

estimated_total_requirement = (
    projected_slice_bytes
    + additional_reserve
)

storage_feasible = (
    estimated_total_requirement
    <= disk_usage.free
)

checkpoint_paths = sorted(
    PREFLIGHT_CHECKPOINT_DIR.glob(
        "preflight_*.jsonl"
    ),
    key=lambda path: path.name,
)

preflight_report = {
    "policy_version": "1.1",
    "reference_relocation_qualification":
        reference_qualification_status,
    "selection_policy": (
        "First operation from every "
        "operation-bearing specification, then "
        "evenly spaced operations to the target."
    ),
    "target_sample_count":
        PREFLIGHT_TARGET_COUNT,
    "actual_sample_count":
        len(selected_operations),
    "covered_specification_count": len(
        {
            record["spec_id"]
            for record in selected_operations
        }
    ),
    "slice_pass_count": (
        len(preflight_records)
        - len(problem_records)
    ),
    "slice_failure_count":
        len(problem_records),
    "reference_relocation_count":
        sum(relocation_counts),
    "operations_with_reference_relocations":
        sum(
            count > 0
            for count in relocation_counts
        ),
    "method_counts": dict(
        sorted(method_counts.items())
    ),
    "source_class_counts": dict(
        sorted(source_class_counts.items())
    ),
    "canonical_version_counts": dict(
        sorted(version_counts.items())
    ),
    "slice_size_statistics": {
        "minimum_bytes": min(slice_sizes),
        "mean_bytes": round(
            mean_slice_bytes,
            3,
        ),
        "median_bytes":
            statistics.median(slice_sizes),
        "p95_bytes":
            sorted_slice_sizes[p95_index],
        "maximum_bytes": max(slice_sizes),
    },
    "pointer_mapping_statistics": {
        "minimum": min(mapping_counts),
        "mean": round(
            statistics.fmean(
                mapping_counts
            ),
            3,
        ),
        "median":
            statistics.median(
                mapping_counts
            ),
        "maximum": max(mapping_counts),
    },
    "storage_projection": {
        "projected_full_slice_bytes":
            projected_slice_bytes,
        "projected_full_slice_gib": round(
            projected_slice_bytes
            / (1024 ** 3),
            3,
        ),
        "estimated_total_requirement_bytes":
            estimated_total_requirement,
        "estimated_total_requirement_gib":
            round(
                estimated_total_requirement
                / (1024 ** 3),
                3,
            ),
        "current_free_bytes":
            disk_usage.free,
        "current_free_gib": round(
            disk_usage.free
            / (1024 ** 3),
            3,
        ),
        "storage_feasible":
            storage_feasible,
    },
    "checkpoint_count":
        len(checkpoint_paths),
    "checkpoint_hashes": [
        {
            "path": str(path),
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in checkpoint_paths
    ],
    "problem_examples": [
        {
            "op_id": record["op_id"],
            "spec_id": record["spec_id"],
            "path": record["path"],
            "method": record["method"],
            "diagnostics":
                record["diagnostics"],
            "validation_status":
                record["validation_status"],
            "validation_diagnostic":
                record[
                    "validation_diagnostic"
                ],
        }
        for record in problem_records[:25]
    ],
    "duration_seconds": round(
        time.perf_counter()
        - preflight_start,
        3,
    ),
    "status": (
        "PASS"
        if (
            not problem_records
            and storage_feasible
        )
        else "BLOCKED"
    ),
}

write_json_atomic(
    PREFLIGHT_REPORT_PATH,
    preflight_report,
)

output = dict(preflight_report)

output["amendment_report_path"] = str(
    REFERENCE_AMENDMENT_REPORT_PATH
)

output["amendment_report_sha256"] = (
    sha256_file(
        REFERENCE_AMENDMENT_REPORT_PATH
    )
)

output[
    "reference_qualification_report_path"
] = str(
    REFERENCE_QUALIFICATION_REPORT_PATH
)

output[
    "reference_qualification_report_sha256"
] = sha256_file(
    REFERENCE_QUALIFICATION_REPORT_PATH
)

output["report_path"] = str(
    PREFLIGHT_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    PREFLIGHT_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if problem_records:
    raise RuntimeError(
        "One or more corrected preflight "
        "slices failed."
    )

if not storage_feasible:
    raise RuntimeError(
        "Projected Stage 02 storage exceeds "
        "the safe available-space limit."
    )

print(
    "\nNOTEBOOK 02 — CELL 8: PASS"
)

{"sample_processed": 100, "sample_total": 2000, "slice_pass_count": 100, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 5.7}


/usr/local/lib/python3.13/dist-packages/openapi_schema_validator/_regex.py:25: FutureWarning: Possible nested set at position 1
  re.compile(pattern)


{"sample_processed": 200, "sample_total": 2000, "slice_pass_count": 200, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 9.0}
{"sample_processed": 300, "sample_total": 2000, "slice_pass_count": 300, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 12.0}
{"sample_processed": 400, "sample_total": 2000, "slice_pass_count": 400, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 15.0}
{"sample_processed": 500, "sample_total": 2000, "slice_pass_count": 500, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 17.7}
{"sample_processed": 600, "sample_total": 2000, "slice_pass_count": 600, "problem_count": 0, "reference_relocation_count": 0, "elapsed_seconds": 20.8}
{"sample_processed": 700, "sample_total": 2000, "slice_pass_count": 700, "problem_count": 0, "reference_relocation_count": 1, "elapsed_seconds": 22.2}
{"sample_processed": 800, "sample_total": 2000, "slice_pass_count": 800, "problem_count": 0, "r

In [9]:
# NOTEBOOK 02 — CELL 9 (CORRECTED)
# Correct context-aware reference discovery, explicitly classify one
# incompatible source reference, and resume the full slicing run.

from collections import Counter
from pathlib import Path
import hashlib
import json
import os
import shutil
import statistics
import time
import warnings


warnings.filterwarnings(
    "once",
    category=FutureWarning,
)

CHECKPOINT_SIZE = 250

FULL_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "data"
    / "checkpoints"
    / "slice_full"
)

SLICE_RECORDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "slice_records.jsonl"
)

POINTER_MAPS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "slice_pointer_maps.jsonl"
)

REFERENCE_RELOCATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "reference_relocations.jsonl"
)

FULL_SLICE_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "full_slice_construction.json"
)

REFERENCE_POLICY_V12_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_reference_policy_v1_2.json"
)

REFERENCE_POLICY_V12_QUALIFICATION_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "slice_reference_policy_v1_2_qualification.json"
)

FULL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def iterate_reference_locations(value):
    """
    Context-aware traversal of genuine OpenAPI and JSON Schema
    reference keywords.

    Arbitrary Example Object values and Schema `example`/`examples`
    payloads are not traversed. Schema properties literally named
    `example`, `examples`, or `value` remain ordinary schemas.
    Discriminator mapping strings are retained but are not interpreted
    as JSON Reference Objects.
    """
    locations = []

    schema_single_keys = {
        "items",
        "contains",
        "not",
        "additionalProperties",
        "unevaluatedProperties",
        "propertyNames",
        "if",
        "then",
        "else",
    }

    schema_array_keys = {
        "allOf",
        "anyOf",
        "oneOf",
        "prefixItems",
    }

    def walk(node, tokens, context):
        if context == "example_payload":
            return

        if isinstance(node, dict):
            for reference_keyword in (
                "$ref",
                "$dynamicRef",
                "$recursiveRef",
            ):
                reference = node.get(
                    reference_keyword
                )

                if isinstance(reference, str):
                    locations.append(
                        {
                            "container": node,
                            "key": reference_keyword,
                            "reference": reference,
                            "location_tokens": list(
                                tokens
                            ),
                            "reference_pointer":
                                pointer_from_tokens(
                                    [
                                        *tokens,
                                        reference_keyword,
                                    ]
                                ),
                        }
                    )

            if context == "schema_map":
                for property_name, schema in (
                    node.items()
                ):
                    walk(
                        schema,
                        [
                            *tokens,
                            property_name,
                        ],
                        "schema",
                    )
                return

            if context == "example_map":
                for example_name, example_object in (
                    node.items()
                ):
                    walk(
                        example_object,
                        [
                            *tokens,
                            example_name,
                        ],
                        "example_object",
                    )
                return

            for key, child in list(
                node.items()
            ):
                child_tokens = [
                    *tokens,
                    key,
                ]

                if (
                    context == "example_object"
                    and key == "value"
                ):
                    continue

                if (
                    context == "schema"
                    and key in {
                        "example",
                        "examples",
                    }
                ):
                    continue

                if (
                    context != "schema"
                    and key == "example"
                ):
                    continue

                if key == "properties":
                    walk(
                        child,
                        child_tokens,
                        "schema_map",
                    )
                    continue

                if key == "schemas":
                    walk(
                        child,
                        child_tokens,
                        "schema_map",
                    )
                    continue

                if key == "schema":
                    walk(
                        child,
                        child_tokens,
                        "schema",
                    )
                    continue

                if (
                    context == "schema"
                    and key in schema_single_keys
                ):
                    walk(
                        child,
                        child_tokens,
                        "schema",
                    )
                    continue

                if (
                    context == "schema"
                    and key in schema_array_keys
                ):
                    if isinstance(child, list):
                        for index, schema in (
                            enumerate(child)
                        ):
                            walk(
                                schema,
                                [
                                    *child_tokens,
                                    str(index),
                                ],
                                "schema",
                            )
                    continue

                if (
                    context != "schema"
                    and key == "examples"
                ):
                    if isinstance(child, dict):
                        walk(
                            child,
                            child_tokens,
                            "example_map",
                        )
                    continue

                walk(
                    child,
                    child_tokens,
                    "generic",
                )

        elif isinstance(node, list):
            for index, child in enumerate(node):
                walk(
                    child,
                    [
                        *tokens,
                        str(index),
                    ],
                    context,
                )

    walk(value, [], "generic")

    return locations


def collect_dependency_references(value):
    return {
        location["reference"]
        for location in (
            iterate_reference_locations(value)
        )
    }


if (
    "MATERIALIZE_REFERENCE_POLICY_V11"
    not in globals()
):
    MATERIALIZE_REFERENCE_POLICY_V11 = (
        materialize_and_relocate_references
    )


def looks_like_parameter_object(value):
    return (
        isinstance(value, dict)
        and isinstance(
            value.get("name"),
            str,
        )
        and isinstance(
            value.get("in"),
            str,
        )
        and "schema" in value
    )


def materialize_and_relocate_references(
    source_document,
    slice_document,
    pointer_mappings,
):
    result = (
        MATERIALIZE_REFERENCE_POLICY_V11(
            source_document,
            slice_document,
            pointer_mappings,
        )
    )

    retained_relocations = []
    mismatch_diagnostics = []

    for relocation in result["relocations"]:
        original_reference = relocation[
            "original_reference"
        ]

        source_tokens = (
            tokens_from_local_reference(
                original_reference
            )
        )

        source_target = None

        if source_tokens is not None:
            try:
                source_target = (
                    resolve_pointer_tokens(
                        source_document,
                        source_tokens,
                    )
                )
            except Exception:
                source_target = None

        incompatible_schema_target = (
            relocation[
                "component_section"
            ]
            == "schemas"
            and looks_like_parameter_object(
                source_target
            )
        )

        if not incompatible_schema_target:
            retained_relocations.append(
                relocation
            )
            continue

        rewritten_reference = relocation[
            "rewritten_reference"
        ]

        for location in (
            iterate_reference_locations(
                slice_document
            )
        ):
            if (
                location["reference"]
                == rewritten_reference
            ):
                location["container"][
                    location["key"]
                ] = original_reference

        rewritten_tokens = (
            tokens_from_local_reference(
                rewritten_reference
            )
        )

        if (
            rewritten_tokens is not None
            and len(rewritten_tokens) >= 3
            and rewritten_tokens[0]
            == "components"
        ):
            section_name = (
                rewritten_tokens[1]
            )
            component_name = (
                rewritten_tokens[2]
            )

            components = (
                slice_document.get(
                    "components",
                    {},
                )
            )

            section = components.get(
                section_name,
                {},
            )

            if isinstance(section, dict):
                section.pop(
                    component_name,
                    None,
                )

                if not section:
                    components.pop(
                        section_name,
                        None,
                    )

            if not components:
                slice_document.pop(
                    "components",
                    None,
                )

            synthetic_pointer = (
                pointer_from_tokens(
                    rewritten_tokens[:3]
                )
            )

            pointer_mappings[:] = [
                mapping
                for mapping in pointer_mappings
                if mapping.get(
                    "slice_pointer"
                )
                != synthetic_pointer
            ]

        mismatch_diagnostics.append(
            {
                "type": (
                    "REFERENCE_TARGET_TYPE_"
                    "MISMATCH"
                ),
                "reference":
                    original_reference,
                "reference_context":
                    "schema",
                "observed_target_type":
                    "parameter_object",
                "source_target_pointer":
                    relocation[
                        "source_target_pointer"
                    ],
                "policy": (
                    "Explicitly exclude; do not "
                    "silently substitute the "
                    "parameter's nested schema."
                ),
            }
        )

    return {
        "diagnostics":
            deduplicate_diagnostics(
                [
                    *result["diagnostics"],
                    *mismatch_diagnostics,
                ]
            ),
        "relocations":
            retained_relocations,
    }


previous_policy_sha256 = sha256_file(
    STAGE_02_POLICY_PATH
)

STAGE_02_POLICY["policy_version"] = "1.2"

STAGE_02_POLICY[
    "slice_policy"
][
    "reference_discovery_policy"
] = {
    "schema_property_names_are_not_keywords":
        True,
    "schema_example_payloads_are_not_traversed":
        True,
    "example_object_values_are_not_traversed":
        True,
    "discriminator_mapping_values_are_not_"
    "json_reference_objects": True,
    "typed_reference_target_mismatch": (
        "explicit_operation_exclusion"
    ),
    "silent_schema_extraction_from_parameter":
        False,
}

write_yaml_atomic(
    STAGE_02_POLICY_PATH,
    STAGE_02_POLICY,
)

policy_v12_report = {
    "status": "PASS",
    "policy_version": "1.2",
    "prospective_before_grouping_and_sampling":
        True,
    "previous_policy_sha256":
        previous_policy_sha256,
    "updated_policy_sha256":
        sha256_file(
            STAGE_02_POLICY_PATH
        ),
    "observed_failure_categories": {
        "schema_property_named_examples": 33,
        "discriminator_mapping_not_reference": 2,
        "schema_reference_targets_parameter": 1,
    },
    "decisions": {
        "schema_property_named_examples": (
            "Correct dependency-walker context."
        ),
        "discriminator_mapping_not_reference": (
            "Preserve mapping string unchanged "
            "without resolving it as a $ref."
        ),
        "schema_reference_targets_parameter": (
            "Exclude the affected operation; "
            "do not infer a source repair."
        ),
    },
}

write_json_atomic(
    REFERENCE_POLICY_V12_REPORT_PATH,
    policy_v12_report,
)


# Qualification fixture for all three corrected cases.
policy_v12_fixture = {
    "openapi": "3.0.3",
    "info": {
        "title": "Reference policy fixture",
        "version": "1.0.0",
    },
    "paths": {
        "/shared": {
            "get": {
                "parameters": [
                    {
                        "name": "field",
                        "in": "query",
                        "schema": {
                            "type": "string"
                        },
                    }
                ],
                "responses": {
                    "200": {
                        "description": "Shared."
                    }
                },
            }
        },
        "/target": {
            "get": {
                "responses": {
                    "200": {
                        "description": "Target.",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/Root"
                                    )
                                }
                            }
                        },
                    }
                }
            }
        },
        "/mismatch": {
            "get": {
                "responses": {
                    "200": {
                        "description": "Mismatch.",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "items": {
                                        "$ref": (
                                            "#/paths/~1shared/"
                                            "get/parameters/0"
                                        )
                                    },
                                    "type": "array",
                                }
                            }
                        },
                    }
                }
            }
        },
    },
    "components": {
        "schemas": {
            "Root": {
                "type": "object",
                "properties": {
                    "examples": {
                        "$ref": (
                            "#/components/"
                            "schemas/Child"
                        )
                    },
                    "animal": {
                        "type": "object",
                        "discriminator": {
                            "propertyName": "kind",
                            "mapping": {
                                "dog": "#/dog_alias"
                            },
                        },
                    },
                },
            },
            "Child": {
                "type": "string"
            },
        }
    },
}

target_fixture_record = {
    "op_id": "fixture_policy_v12_target",
    "path": "/target",
    "method": "get",
    "source_path_pointer":
        "/paths/~1target",
    "source_operation_pointer":
        "/paths/~1target/get",
}

mismatch_fixture_record = {
    "op_id": "fixture_policy_v12_mismatch",
    "path": "/mismatch",
    "method": "get",
    "source_path_pointer":
        "/paths/~1mismatch",
    "source_operation_pointer":
        "/paths/~1mismatch/get",
}

target_fixture_built = (
    build_operation_slice(
        policy_v12_fixture,
        target_fixture_record,
    )
)

mismatch_fixture_built = (
    build_operation_slice(
        policy_v12_fixture,
        mismatch_fixture_record,
    )
)

try:
    validate(
        target_fixture_built[
            "slice_document"
        ]
    )
    target_fixture_validation = True
except Exception:
    target_fixture_validation = False

try:
    validate(
        mismatch_fixture_built[
            "slice_document"
        ]
    )
    mismatch_fixture_validation = True
except Exception:
    mismatch_fixture_validation = False

target_components = (
    target_fixture_built[
        "slice_document"
    ]
    .get("components", {})
    .get("schemas", {})
)

qualification_checks = {
    "schema_property_named_examples_traversed":
        "Child" in target_components,
    "discriminator_mapping_preserved":
        (
            target_components["Root"]
            ["properties"]["animal"]
            ["discriminator"]["mapping"]["dog"]
            == "#/dog_alias"
        ),
    "discriminator_mapping_not_diagnostic":
        target_fixture_built[
            "diagnostics"
        ]
        == [],
    "normal_target_slice_valid":
        target_fixture_validation,
    "typed_mismatch_detected":
        any(
            diagnostic.get("type")
            == "REFERENCE_TARGET_TYPE_MISMATCH"
            for diagnostic
            in mismatch_fixture_built[
                "diagnostics"
            ]
        ),
    "typed_mismatch_not_silently_repaired":
        not mismatch_fixture_validation,
}

qualification_status = (
    "PASS"
    if all(
        qualification_checks.values()
    )
    else "BLOCKED"
)

write_json_atomic(
    REFERENCE_POLICY_V12_QUALIFICATION_PATH,
    {
        "status": qualification_status,
        "checks": qualification_checks,
        "target_diagnostics":
            target_fixture_built[
                "diagnostics"
            ],
        "mismatch_diagnostics":
            mismatch_fixture_built[
                "diagnostics"
            ],
    },
)

if qualification_status != "PASS":
    raise RuntimeError(
        "Reference policy v1.2 "
        "qualification failed."
    )


def load_jsonl(path):
    rows = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                rows.append(json.loads(line))
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}, "
                    f"line {line_number}: {error}"
                ) from error

    return rows


def record_uses_old_discriminator_relocation(
    record,
):
    return any(
        "/discriminator/mapping/"
        in relocation.get(
            "slice_reference_pointer",
            "",
        )
        for relocation in record.get(
            "reference_relocations",
            []
        )
    )


def verify_slice_record(
    record,
    expected_operation,
):
    if (
        record.get("op_id")
        != expected_operation["op_id"]
    ):
        return False

    if record_uses_old_discriminator_relocation(
        record
    ):
        return False

    status = record.get("status")

    if status not in {
        "PASS",
        (
            "EXCLUDED_REFERENCE_TARGET_"
            "TYPE_MISMATCH"
        ),
    }:
        return False

    relative_path = record.get(
        "slice_relative_path"
    )

    if not isinstance(relative_path, str):
        return False

    slice_path = (
        PROJECT_ROOT / relative_path
    )

    if not slice_path.is_file():
        return False

    if (
        slice_path.stat().st_size
        != record.get("slice_bytes")
    ):
        return False

    if (
        sha256_file(slice_path)
        != record.get("slice_sha256")
    ):
        return False

    if status == "PASS":
        return (
            record.get("validation_status")
            == "PASS"
            and not record.get("diagnostics")
        )

    diagnostic_types = {
        diagnostic.get("type")
        for diagnostic in record.get(
            "diagnostics",
            []
        )
    }

    return diagnostic_types == {
        "REFERENCE_TARGET_TYPE_MISMATCH"
    }


def count_target_operations(
    slice_document,
):
    count = 0

    for path_item in (
        slice_document.get(
            "paths",
            {}
        ).values()
    ):
        if not isinstance(path_item, dict):
            continue

        count += sum(
            method in path_item
            and isinstance(
                path_item[method],
                dict,
            )
            for method in (
                SUPPORTED_HTTP_METHODS
            )
        )

    return count


def build_full_slice_record(
    operation_record,
    source_document,
):
    operation_start = time.perf_counter()
    op_id = operation_record["op_id"]

    try:
        built = build_operation_slice(
            source_document,
            operation_record,
        )

        slice_document = built[
            "slice_document"
        ]

        diagnostics = list(
            built["diagnostics"]
        )

        target_operation_count = (
            count_target_operations(
                slice_document
            )
        )

        if target_operation_count != 1:
            diagnostics.append(
                {
                    "type": (
                        "SLICE_TARGET_OPERATION_"
                        "COUNT_MISMATCH"
                    ),
                    "expected": 1,
                    "observed":
                        target_operation_count,
                }
            )

        serialized_slice = (
            canonical_json_bytes(
                slice_document
            )
        )

        slice_path = (
            PROJECT_ROOT
            / "data"
            / "slices"
            / "L0"
            / f"{op_id}.json"
        )

        write_bytes_atomic(
            slice_path,
            serialized_slice,
        )

        validation_status = "PASS"
        validation_diagnostic = None

        try:
            validate(slice_document)
        except Exception as error:
            validation_status = "FAIL"
            validation_diagnostic = repr(
                error
            )

        diagnostic_types = {
            diagnostic.get("type")
            for diagnostic in diagnostics
        }

        if (
            not diagnostics
            and validation_status == "PASS"
        ):
            status = "PASS"

        elif diagnostic_types == {
            "REFERENCE_TARGET_TYPE_MISMATCH"
        }:
            status = (
                "EXCLUDED_REFERENCE_TARGET_"
                "TYPE_MISMATCH"
            )

        else:
            status = "FAIL"

        return {
            "op_id": op_id,
            "global_operation_index":
                operation_record[
                    "global_operation_index"
                ],
            "spec_id":
                operation_record["spec_id"],
            "path":
                operation_record["path"],
            "method":
                operation_record["method"],
            "source_class":
                operation_record[
                    "source_class"
                ],
            "source_dialect":
                operation_record[
                    "source_dialect"
                ],
            "canonical_openapi_version":
                operation_record[
                    "canonical_openapi_version"
                ],
            "slice_relative_path": (
                slice_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "slice_bytes":
                len(serialized_slice),
            "slice_sha256":
                hashlib.sha256(
                    serialized_slice
                ).hexdigest(),
            "pointer_mapping_count":
                len(
                    built[
                        "pointer_mappings"
                    ]
                ),
            "pointer_mappings":
                built[
                    "pointer_mappings"
                ],
            "reference_relocation_count":
                len(
                    built[
                        "reference_relocations"
                    ]
                ),
            "reference_relocations":
                built[
                    "reference_relocations"
                ],
            "diagnostics": diagnostics,
            "validation_status":
                validation_status,
            "validation_diagnostic":
                validation_diagnostic,
            "status": status,
            "duration_seconds": round(
                time.perf_counter()
                - operation_start,
                6,
            ),
        }

    except Exception as error:
        return {
            "op_id": op_id,
            "global_operation_index":
                operation_record[
                    "global_operation_index"
                ],
            "spec_id":
                operation_record["spec_id"],
            "path":
                operation_record["path"],
            "method":
                operation_record["method"],
            "source_class":
                operation_record[
                    "source_class"
                ],
            "source_dialect":
                operation_record[
                    "source_dialect"
                ],
            "canonical_openapi_version":
                operation_record[
                    "canonical_openapi_version"
                ],
            "slice_relative_path": None,
            "slice_bytes": None,
            "slice_sha256": None,
            "pointer_mapping_count": 0,
            "pointer_mappings": [],
            "reference_relocation_count": 0,
            "reference_relocations": [],
            "diagnostics": [
                {
                    "type": (
                        "SLICE_BUILD_EXCEPTION"
                    ),
                    "diagnostic": repr(error),
                }
            ],
            "validation_status": "NOT_RUN",
            "validation_diagnostic": None,
            "status": "FAIL",
            "duration_seconds": round(
                time.perf_counter()
                - operation_start,
                6,
            ),
        }


preflight_record_map = {}

for checkpoint_path in sorted(
    PREFLIGHT_CHECKPOINT_DIR.glob(
        "preflight_*.jsonl"
    ),
    key=lambda path: path.name,
):
    for record in load_jsonl(
        checkpoint_path
    ):
        preflight_record_map[
            record["op_id"]
        ] = record

operation_record_map = {
    record["op_id"]: record
    for record in operation_records
}

valid_preflight_records = {
    op_id: record
    for op_id, record
    in preflight_record_map.items()
    if (
        op_id in operation_record_map
        and verify_slice_record(
            record,
            operation_record_map[op_id],
        )
    )
}

specification_record_map = {
    record["spec_id"]: record
    for record in records
}

run_start = time.perf_counter()

all_records = []
unexpected_failures = []

full_checkpoint_reused = 0
preflight_reused = 0
rebuilt = 0

current_spec_id = None
current_document = None

total_chunks = (
    len(operation_records)
    + CHECKPOINT_SIZE
    - 1
) // CHECKPOINT_SIZE

for chunk_index in range(total_chunks):
    start_index = (
        chunk_index * CHECKPOINT_SIZE
    )

    end_index = min(
        start_index + CHECKPOINT_SIZE,
        len(operation_records),
    )

    expected_operations = (
        operation_records[
            start_index:end_index
        ]
    )

    first_index = expected_operations[
        0
    ]["global_operation_index"]

    last_index = expected_operations[
        -1
    ]["global_operation_index"]

    checkpoint_path = (
        FULL_CHECKPOINT_DIR
        / (
            f"slice_{first_index:06d}_"
            f"{last_index:06d}.jsonl"
        )
    )

    chunk_records = None

    if checkpoint_path.is_file():
        candidates = load_jsonl(
            checkpoint_path
        )

        candidates_valid = (
            len(candidates)
            == len(expected_operations)
        )

        if candidates_valid:
            for candidate, expected in zip(
                candidates,
                expected_operations,
            ):
                if not verify_slice_record(
                    candidate,
                    expected,
                ):
                    candidates_valid = False
                    break

        if candidates_valid:
            chunk_records = candidates
            full_checkpoint_reused += len(
                candidates
            )

    if chunk_records is None:
        chunk_records = []

        for operation_record in (
            expected_operations
        ):
            op_id = operation_record["op_id"]

            preflight_record = (
                valid_preflight_records.get(
                    op_id
                )
            )

            if preflight_record is not None:
                chunk_records.append(
                    preflight_record
                )
                preflight_reused += 1
                continue

            spec_id = operation_record[
                "spec_id"
            ]

            if spec_id != current_spec_id:
                spec_record = (
                    specification_record_map[
                        spec_id
                    ]
                )

                canonical_path = (
                    STAGE_01_PROJECT_ROOT
                    / spec_record[
                        "canonical_relative_path"
                    ]
                )

                current_document = json.loads(
                    canonical_path.read_text(
                        encoding="utf-8"
                    )
                )

                current_spec_id = spec_id

            chunk_records.append(
                build_full_slice_record(
                    operation_record,
                    current_document,
                )
            )

            rebuilt += 1

        write_jsonl_atomic(
            checkpoint_path,
            chunk_records,
        )

    all_records.extend(chunk_records)

    for record in chunk_records:
        if record["status"] == "FAIL":
            unexpected_failures.append(
                record
            )

    processed = len(all_records)

    if (
        processed % 1000 < CHECKPOINT_SIZE
        or processed
        == len(operation_records)
    ):
        status_counts = Counter(
            record["status"]
            for record in all_records
        )

        print(
            json.dumps(
                {
                    "processed": processed,
                    "total":
                        len(operation_records),
                    "full_checkpoint_reused":
                        full_checkpoint_reused,
                    "preflight_reused":
                        preflight_reused,
                    "rebuilt": rebuilt,
                    "status_counts":
                        dict(status_counts),
                    "unexpected_failure_count":
                        len(
                            unexpected_failures
                        ),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - run_start,
                        1,
                    ),
                }
            )
        )

all_records.sort(
    key=lambda record:
        record["global_operation_index"]
)

expected_ids = [
    record["op_id"]
    for record in operation_records
]

observed_ids = [
    record["op_id"]
    for record in all_records
]

if observed_ids != expected_ids:
    raise RuntimeError(
        "Corrected slice records do not align "
        "with the frozen operation inventory."
    )

status_counts = Counter(
    record["status"]
    for record in all_records
)

excluded_records = [
    record
    for record in all_records
    if record["status"].startswith(
        "EXCLUDED_"
    )
]

pass_records = [
    record
    for record in all_records
    if record["status"] == "PASS"
]

compact_records = []
pointer_map_records = []
relocation_records = []

for record in all_records:
    compact_records.append(
        {
            key: value
            for key, value in record.items()
            if key
            not in {
                "pointer_mappings",
                "reference_relocations",
                "duration_seconds",
            }
        }
    )

    pointer_map_records.append(
        {
            "op_id": record["op_id"],
            "spec_id": record["spec_id"],
            "slice_relative_path":
                record[
                    "slice_relative_path"
                ],
            "mapping_count":
                record[
                    "pointer_mapping_count"
                ],
            "mappings":
                record[
                    "pointer_mappings"
                ],
        }
    )

    for relocation in record[
        "reference_relocations"
    ]:
        relocation_records.append(
            {
                "op_id": record["op_id"],
                "spec_id": record["spec_id"],
                "path": record["path"],
                "method": record["method"],
                **relocation,
            }
        )

write_jsonl_atomic(
    SLICE_RECORDS_PATH,
    compact_records,
)

write_jsonl_atomic(
    POINTER_MAPS_PATH,
    pointer_map_records,
)

write_jsonl_atomic(
    REFERENCE_RELOCATIONS_PATH,
    relocation_records,
)

all_slice_bytes = [
    int(record["slice_bytes"])
    for record in all_records
    if record["slice_bytes"] is not None
]

usable_source_counts = Counter(
    record["source_class"]
    for record in pass_records
)

full_report = {
    "policy_version": "1.2",
    "operation_count":
        len(operation_records),
    "accounted_operation_count":
        len(all_records),
    "usable_valid_slice_count":
        len(pass_records),
    "explicit_exclusion_count":
        len(excluded_records),
    "unexpected_failure_count":
        len(unexpected_failures),
    "status_counts": dict(
        sorted(status_counts.items())
    ),
    "validation_pass_count":
        sum(
            record["validation_status"]
            == "PASS"
            for record in all_records
        ),
    "usable_source_class_counts": dict(
        sorted(
            usable_source_counts.items()
        )
    ),
    "explicit_exclusions": [
        {
            "op_id": record["op_id"],
            "spec_id": record["spec_id"],
            "path": record["path"],
            "method": record["method"],
            "status": record["status"],
            "diagnostics":
                record["diagnostics"],
        }
        for record in excluded_records
    ],
    "full_checkpoint_reused_count":
        full_checkpoint_reused,
    "preflight_reused_count":
        preflight_reused,
    "rebuilt_count": rebuilt,
    "checkpoint_count": len(
        list(
            FULL_CHECKPOINT_DIR.glob(
                "slice_*.jsonl"
            )
        )
    ),
    "total_slice_bytes":
        sum(all_slice_bytes),
    "total_slice_gib": round(
        sum(all_slice_bytes)
        / (1024 ** 3),
        3,
    ),
    "mean_slice_bytes": round(
        statistics.fmean(
            all_slice_bytes
        ),
        3,
    ),
    "median_slice_bytes":
        statistics.median(
            all_slice_bytes
        ),
    "maximum_slice_bytes":
        max(all_slice_bytes),
    "operation_with_relocation_count":
        sum(
            record[
                "reference_relocation_count"
            ]
            > 0
            for record in all_records
        ),
    "reference_relocation_count":
        len(relocation_records),
    "slice_records": {
        "path": str(SLICE_RECORDS_PATH),
        "bytes":
            SLICE_RECORDS_PATH.stat().st_size,
        "sha256":
            sha256_file(
                SLICE_RECORDS_PATH
            ),
    },
    "pointer_maps": {
        "path": str(POINTER_MAPS_PATH),
        "bytes":
            POINTER_MAPS_PATH.stat().st_size,
        "sha256":
            sha256_file(
                POINTER_MAPS_PATH
            ),
    },
    "reference_relocations": {
        "path": str(
            REFERENCE_RELOCATIONS_PATH
        ),
        "bytes": (
            REFERENCE_RELOCATIONS_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            REFERENCE_RELOCATIONS_PATH
        ),
    },
    "unexpected_failure_examples": [
        {
            "op_id": record["op_id"],
            "spec_id": record["spec_id"],
            "path": record["path"],
            "method": record["method"],
            "diagnostics":
                record["diagnostics"],
            "validation_status":
                record["validation_status"],
            "validation_diagnostic":
                record[
                    "validation_diagnostic"
                ],
        }
        for record
        in unexpected_failures[:25]
    ],
    "duration_seconds": round(
        time.perf_counter()
        - run_start,
        3,
    ),
    "status": (
        "PASS"
        if (
            len(all_records)
            == EXPECTED_OPERATION_COUNT
            and not unexpected_failures
            and len(excluded_records) == 1
        )
        else "BLOCKED"
    ),
}

write_json_atomic(
    FULL_SLICE_REPORT_PATH,
    full_report,
)

output = dict(full_report)

output["reference_policy_report_path"] = (
    str(
        REFERENCE_POLICY_V12_REPORT_PATH
    )
)

output["reference_policy_report_sha256"] = (
    sha256_file(
        REFERENCE_POLICY_V12_REPORT_PATH
    )
)

output["report_path"] = str(
    FULL_SLICE_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    FULL_SLICE_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if full_report["status"] != "PASS":
    raise RuntimeError(
        "Corrected full-corpus slicing did "
        "not reach the expected disposition."
    )

print(
    "\nNOTEBOOK 02 — CELL 9: PASS"
)

{"processed": 1000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 55, "rebuilt": 945, "status_counts": {"PASS": 1000}, "unexpected_failure_count": 0, "elapsed_seconds": 92.7}
{"processed": 2000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 100, "rebuilt": 1900, "status_counts": {"PASS": 2000}, "unexpected_failure_count": 0, "elapsed_seconds": 141.9}


/usr/local/lib/python3.13/dist-packages/openapi_schema_validator/_regex.py:25: FutureWarning: Possible nested set at position 1
  re.compile(pattern)


{"processed": 3000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 141, "rebuilt": 2859, "status_counts": {"PASS": 3000}, "unexpected_failure_count": 0, "elapsed_seconds": 167.0}
{"processed": 4000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 163, "rebuilt": 3837, "status_counts": {"PASS": 4000}, "unexpected_failure_count": 0, "elapsed_seconds": 192.9}
{"processed": 5000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 193, "rebuilt": 4807, "status_counts": {"PASS": 5000}, "unexpected_failure_count": 0, "elapsed_seconds": 221.5}
{"processed": 6000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 201, "rebuilt": 5799, "status_counts": {"PASS": 6000}, "unexpected_failure_count": 0, "elapsed_seconds": 255.6}
{"processed": 7000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 214, "rebuilt": 6786, "status_counts": {"PASS": 7000}, "unexpected_failure_count": 0, "elapsed_seconds": 287.2}
{"processed": 8

/usr/local/lib/python3.13/dist-packages/openapi_schema_validator/_regex.py:25: FutureWarning: Possible nested set at position 122
  re.compile(pattern)


{"processed": 9000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 277, "rebuilt": 8723, "status_counts": {"PASS": 9000}, "unexpected_failure_count": 0, "elapsed_seconds": 356.7}
{"processed": 10000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 304, "rebuilt": 9696, "status_counts": {"PASS": 10000}, "unexpected_failure_count": 0, "elapsed_seconds": 387.0}
{"processed": 11000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 325, "rebuilt": 10675, "status_counts": {"PASS": 11000}, "unexpected_failure_count": 0, "elapsed_seconds": 430.5}
{"processed": 12000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 356, "rebuilt": 11644, "status_counts": {"PASS": 12000}, "unexpected_failure_count": 0, "elapsed_seconds": 465.7}
{"processed": 13000, "total": 58585, "full_checkpoint_reused": 0, "preflight_reused": 389, "rebuilt": 12611, "status_counts": {"PASS": 13000}, "unexpected_failure_count": 0, "elapsed_seconds": 489.7}
{"pr

In [10]:
# NOTEBOOK 02 — CELL 10
# Load and qualify the exact frozen tokenizer used for prose counting.

from pathlib import Path
import hashlib
import json
import time

from transformers import AutoTokenizer


MODEL_ID = (
    STAGE_00_EXTERNAL_MANIFEST["model_id"]
)

MODEL_REVISION = (
    STAGE_00_EXTERNAL_MANIFEST[
        "model_revision"
    ]
)

TOKENIZER_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "runtime"
    / "tokenizer_qualification.json"
)


def obtain_optional_hf_token():
    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )

        client = UserSecretsClient()

        token = client.get_secret(
            "HF_TOKEN"
        )

        if (
            isinstance(token, str)
            and token.strip()
        ):
            return token

    except Exception:
        pass

    return None


hf_token = obtain_optional_hf_token()

load_start = time.perf_counter()

TOKENIZER = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=hf_token,
    trust_remote_code=False,
    local_files_only=False,
)

load_duration = (
    time.perf_counter() - load_start
)

qualification_texts = [
    "",
    "Create a new user.",
    (
        "Returns the requested resource when "
        "the identifier is valid."
    ),
    (
        "The maximum number of records to "
        "return per page."
    ),
    "短い説明",
    "Whitespace   must\nremain\tunchanged.",
]

qualification_results = []
problems = []

for text in qualification_texts:
    first_ids = TOKENIZER.encode(
        text,
        add_special_tokens=False,
    )

    second_ids = TOKENIZER.encode(
        text,
        add_special_tokens=False,
    )

    deterministic = (
        first_ids == second_ids
    )

    if not deterministic:
        problems.append(
            {
                "problem": (
                    "NONDETERMINISTIC_TOKENIZATION"
                ),
                "text": text,
            }
        )

    encoded_ids_bytes = json.dumps(
        first_ids,
        separators=(",", ":"),
    ).encode("utf-8")

    qualification_results.append(
        {
            "text_sha256": hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest(),
            "character_count": len(text),
            "token_count": len(first_ids),
            "token_ids_sha256":
                hashlib.sha256(
                    encoded_ids_bytes
                ).hexdigest(),
            "deterministic": deterministic,
        }
    )

empty_ids = TOKENIZER.encode(
    "",
    add_special_tokens=False,
)

if empty_ids != []:
    problems.append(
        {
            "problem": (
                "EMPTY_STRING_TOKENIZATION_"
                "NOT_EMPTY"
            ),
            "observed_ids": empty_ids,
        }
    )

tokenizer_length = len(TOKENIZER)

if tokenizer_length != 151665:
    problems.append(
        {
            "problem": (
                "TOKENIZER_VOCABULARY_SIZE_"
                "MISMATCH"
            ),
            "expected": 151665,
            "observed": tokenizer_length,
        }
    )

if (
    TOKENIZER.__class__.__name__
    != "Qwen2Tokenizer"
):
    problems.append(
        {
            "problem": (
                "TOKENIZER_CLASS_MISMATCH"
            ),
            "expected": "Qwen2Tokenizer",
            "observed": (
                TOKENIZER
                .__class__
                .__name__
            ),
        }
    )

tokenizer_report = {
    "model_id": MODEL_ID,
    "requested_revision":
        MODEL_REVISION,
    "tokenizer_class": (
        TOKENIZER.__class__.__name__
    ),
    "tokenizer_length":
        tokenizer_length,
    "vocab_size": getattr(
        TOKENIZER,
        "vocab_size",
        None,
    ),
    "model_max_length": getattr(
        TOKENIZER,
        "model_max_length",
        None,
    ),
    "bos_token_id": (
        TOKENIZER.bos_token_id
    ),
    "eos_token_id": (
        TOKENIZER.eos_token_id
    ),
    "pad_token_id": (
        TOKENIZER.pad_token_id
    ),
    "add_special_tokens_for_measurement":
        False,
    "empty_string_token_ids":
        empty_ids,
    "qualification_results":
        qualification_results,
    "hf_token_available":
        hf_token is not None,
    "accelerator_used": False,
    "load_duration_seconds": round(
        load_duration,
        3,
    ),
    "problems": problems,
    "status": (
        "PASS"
        if not problems
        else "BLOCKED"
    ),
}

write_json_atomic(
    TOKENIZER_REPORT_PATH,
    tokenizer_report,
)

output = dict(tokenizer_report)

output["report_path"] = str(
    TOKENIZER_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    TOKENIZER_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if problems:
    raise RuntimeError(
        "Frozen tokenizer qualification failed."
    )

print(
    "\nNOTEBOOK 02 — CELL 10: PASS"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{
  "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "requested_revision": "c03e6d358207e414f1eca0bb1891e29f1db0e242",
  "tokenizer_class": "Qwen2Tokenizer",
  "tokenizer_length": 151665,
  "vocab_size": 151643,
  "model_max_length": 32768,
  "bos_token_id": null,
  "eos_token_id": 151645,
  "pad_token_id": 151643,
  "add_special_tokens_for_measurement": false,
  "empty_string_token_ids": [],
  "qualification_results": [
    {
      "text_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
      "character_count": 0,
      "token_count": 0,
      "token_ids_sha256": "4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945",
      "deterministic": true
    },
    {
      "text_sha256": "89083afea10287e0cc9626e3f181b32f4c787c0d8afdb9f682dad3a96e2f623c",
      "character_count": 18,
      "token_count": 5,
      "token_ids_sha256": "0aaa195a4cdc0122a565d5e851d387568a29d5058bc7b35394aa62ecb24c97ed",
      "deterministic": true
    },
    {
      "text_sha25

In [11]:
# NOTEBOOK 02 — CELL 11
# Implement and qualify the schema-aware prose annotation walker.

from pathlib import Path
import hashlib
import json
import time


PROSE_WALKER_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "prose_walker_qualification.json"
)

PROSE_WALKER_FIXTURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "development_examples"
    / "prose_walker_fixture.json"
)


def append_pointer(pointer, token):
    escaped = json_pointer_escape(token)

    if pointer == "":
        return "/" + escaped

    return pointer + "/" + escaped


def source_pointer_for_slice_pointer(
    slice_pointer,
    pointer_mappings,
):
    candidates = []

    for mapping in pointer_mappings:
        mapped_slice_pointer = mapping.get(
            "slice_pointer"
        )

        if not isinstance(
            mapped_slice_pointer,
            str,
        ):
            continue

        exact_match = (
            slice_pointer
            == mapped_slice_pointer
        )

        descendant_match = (
            mapped_slice_pointer == ""
            or slice_pointer.startswith(
                mapped_slice_pointer + "/"
            )
        )

        if exact_match or descendant_match:
            candidates.append(mapping)

    if not candidates:
        return None

    selected = max(
        candidates,
        key=lambda mapping: len(
            mapping["slice_pointer"]
        ),
    )

    source_prefix = selected.get(
        "source_pointer"
    )

    if source_prefix is None:
        return None

    slice_prefix = selected[
        "slice_pointer"
    ]

    suffix = slice_pointer[
        len(slice_prefix):
    ]

    if source_prefix == "":
        return suffix or ""

    return source_prefix + suffix


def walk_prose_annotations(
    slice_document,
    pointer_mappings,
    tokenizer,
):
    rows = []
    diagnostics = []

    openapi_version = str(
        slice_document.get(
            "openapi",
            ""
        )
    )

    reference_object_annotations = (
        openapi_version.startswith("3.1")
    )

    def emit(
        object_value,
        field_name,
        object_kind,
        object_pointer,
    ):
        if field_name not in object_value:
            return

        field_value = object_value[
            field_name
        ]

        slice_pointer = append_pointer(
            object_pointer,
            field_name,
        )

        source_pointer = (
            source_pointer_for_slice_pointer(
                slice_pointer,
                pointer_mappings,
            )
        )

        if not isinstance(
            field_value,
            str,
        ):
            diagnostics.append(
                {
                    "type": (
                        "NONSTRING_PROSE_ANNOTATION"
                    ),
                    "slice_pointer":
                        slice_pointer,
                    "source_pointer":
                        source_pointer,
                    "object_kind":
                        object_kind,
                    "field_name":
                        field_name,
                    "observed_type":
                        type(
                            field_value
                        ).__name__,
                }
            )

            rows.append(
                {
                    "slice_pointer":
                        slice_pointer,
                    "source_pointer":
                        source_pointer,
                    "object_kind":
                        object_kind,
                    "field_name":
                        field_name,
                    "text_sha256": None,
                    "char_count": None,
                    "token_count": None,
                    "included_in_grouping":
                        False,
                    "diagnostic": (
                        "NONSTRING_PROSE_ANNOTATION"
                    ),
                }
            )
            return

        token_ids = tokenizer.encode(
            field_value,
            add_special_tokens=False,
        )

        rows.append(
            {
                "slice_pointer":
                    slice_pointer,
                "source_pointer":
                    source_pointer,
                "object_kind":
                    object_kind,
                "field_name":
                    field_name,
                "text_sha256":
                    hashlib.sha256(
                        field_value.encode(
                            "utf-8"
                        )
                    ).hexdigest(),
                "char_count":
                    len(field_value),
                "token_count":
                    len(token_ids),
                "included_in_grouping":
                    True,
                "diagnostic": None,
            }
        )

    def walk_reference_object(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return True

        if "$ref" not in value:
            return False

        if reference_object_annotations:
            emit(
                value,
                "summary",
                "reference",
                pointer,
            )

            emit(
                value,
                "description",
                "reference",
                pointer,
            )

        return True

    def walk_external_docs(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        emit(
            value,
            "description",
            "external_documentation",
            pointer,
        )

    def walk_example(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "summary",
            "example",
            pointer,
        )

        emit(
            value,
            "description",
            "example",
            pointer,
        )

        # Example `value` is arbitrary payload data
        # and is deliberately not traversed.

    def walk_examples_map(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        for name in sorted(value):
            walk_example(
                value[name],
                append_pointer(
                    pointer,
                    name,
                ),
            )

    def walk_schema_map(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        for name in sorted(value):
            walk_schema(
                value[name],
                append_pointer(
                    pointer,
                    name,
                ),
            )

    def walk_schema(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        emit(
            value,
            "description",
            "schema",
            pointer,
        )

        schema_map_fields = [
            "properties",
            "patternProperties",
            "dependentSchemas",
            "$defs",
            "definitions",
        ]

        for field_name in schema_map_fields:
            child = value.get(
                field_name
            )

            if isinstance(child, dict):
                walk_schema_map(
                    child,
                    append_pointer(
                        pointer,
                        field_name,
                    ),
                )

        schema_single_fields = [
            "items",
            "contains",
            "not",
            "additionalProperties",
            "unevaluatedProperties",
            "propertyNames",
            "if",
            "then",
            "else",
        ]

        for field_name in schema_single_fields:
            child = value.get(
                field_name
            )

            if isinstance(child, dict):
                walk_schema(
                    child,
                    append_pointer(
                        pointer,
                        field_name,
                    ),
                )

        schema_array_fields = [
            "allOf",
            "anyOf",
            "oneOf",
            "prefixItems",
        ]

        for field_name in schema_array_fields:
            children = value.get(
                field_name
            )

            if not isinstance(children, list):
                continue

            for index, child in enumerate(
                children
            ):
                walk_schema(
                    child,
                    append_pointer(
                        append_pointer(
                            pointer,
                            field_name,
                        ),
                        str(index),
                    ),
                )

        # title, enum, default, examples, example,
        # const, pattern, and extensions are not prose
        # fields selected by this protocol.

    def walk_encoding(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        headers = value.get("headers")

        if isinstance(headers, dict):
            for name in sorted(headers):
                walk_header(
                    headers[name],
                    append_pointer(
                        append_pointer(
                            pointer,
                            "headers",
                        ),
                        name,
                    ),
                )

    def walk_media_type(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        schema = value.get("schema")

        if isinstance(schema, dict):
            walk_schema(
                schema,
                append_pointer(
                    pointer,
                    "schema",
                ),
            )

        examples = value.get("examples")

        if isinstance(examples, dict):
            walk_examples_map(
                examples,
                append_pointer(
                    pointer,
                    "examples",
                ),
            )

        encoding = value.get("encoding")

        if isinstance(encoding, dict):
            for name in sorted(encoding):
                walk_encoding(
                    encoding[name],
                    append_pointer(
                        append_pointer(
                            pointer,
                            "encoding",
                        ),
                        name,
                    ),
                )

        # Singular `example` is arbitrary payload data.

    def walk_content(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        for media_type in sorted(value):
            walk_media_type(
                value[media_type],
                append_pointer(
                    pointer,
                    media_type,
                ),
            )

    def walk_parameter(
        value,
        pointer,
        object_kind="parameter",
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "description",
            object_kind,
            pointer,
        )

        schema = value.get("schema")

        if isinstance(schema, dict):
            walk_schema(
                schema,
                append_pointer(
                    pointer,
                    "schema",
                ),
            )

        content = value.get("content")

        if isinstance(content, dict):
            walk_content(
                content,
                append_pointer(
                    pointer,
                    "content",
                ),
            )

        examples = value.get("examples")

        if isinstance(examples, dict):
            walk_examples_map(
                examples,
                append_pointer(
                    pointer,
                    "examples",
                ),
            )

    def walk_header(value, pointer):
        walk_parameter(
            value,
            pointer,
            object_kind="header",
        )

    def walk_request_body(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "description",
            "request_body",
            pointer,
        )

        content = value.get("content")

        if isinstance(content, dict):
            walk_content(
                content,
                append_pointer(
                    pointer,
                    "content",
                ),
            )

    def walk_link(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "description",
            "link",
            pointer,
        )

        # Link parameters and requestBody contain
        # runtime expressions or arbitrary values.

    def walk_response(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "description",
            "response",
            pointer,
        )

        headers = value.get("headers")

        if isinstance(headers, dict):
            for name in sorted(headers):
                walk_header(
                    headers[name],
                    append_pointer(
                        append_pointer(
                            pointer,
                            "headers",
                        ),
                        name,
                    ),
                )

        content = value.get("content")

        if isinstance(content, dict):
            walk_content(
                content,
                append_pointer(
                    pointer,
                    "content",
                ),
            )

        links = value.get("links")

        if isinstance(links, dict):
            for name in sorted(links):
                walk_link(
                    links[name],
                    append_pointer(
                        append_pointer(
                            pointer,
                            "links",
                        ),
                        name,
                    ),
                )

    def walk_responses(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        for status_code in sorted(value):
            walk_response(
                value[status_code],
                append_pointer(
                    pointer,
                    status_code,
                ),
            )

    def walk_security_scheme(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        emit(
            value,
            "description",
            "security_scheme",
            pointer,
        )

        # OAuth scope map values are deliberately excluded.

    def walk_callback(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        if walk_reference_object(
            value,
            pointer,
        ):
            return

        for expression in sorted(value):
            if str(expression).startswith("x-"):
                continue

            walk_path_item(
                value[expression],
                append_pointer(
                    pointer,
                    expression,
                ),
            )

    def walk_operation(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        emit(
            value,
            "summary",
            "operation",
            pointer,
        )

        emit(
            value,
            "description",
            "operation",
            pointer,
        )

        parameters = value.get(
            "parameters"
        )

        if isinstance(parameters, list):
            for index, parameter in enumerate(
                parameters
            ):
                walk_parameter(
                    parameter,
                    append_pointer(
                        append_pointer(
                            pointer,
                            "parameters",
                        ),
                        str(index),
                    ),
                )

        request_body = value.get(
            "requestBody"
        )

        if isinstance(request_body, dict):
            walk_request_body(
                request_body,
                append_pointer(
                    pointer,
                    "requestBody",
                ),
            )

        responses = value.get("responses")

        if isinstance(responses, dict):
            walk_responses(
                responses,
                append_pointer(
                    pointer,
                    "responses",
                ),
            )

        callbacks = value.get("callbacks")

        if isinstance(callbacks, dict):
            for name in sorted(callbacks):
                walk_callback(
                    callbacks[name],
                    append_pointer(
                        append_pointer(
                            pointer,
                            "callbacks",
                        ),
                        name,
                    ),
                )

        external_docs = value.get(
            "externalDocs"
        )

        if isinstance(external_docs, dict):
            walk_external_docs(
                external_docs,
                append_pointer(
                    pointer,
                    "externalDocs",
                ),
            )

        # x-* extensions are deliberately not traversed.

    def walk_path_item(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        emit(
            value,
            "summary",
            "path_item",
            pointer,
        )

        emit(
            value,
            "description",
            "path_item",
            pointer,
        )

        parameters = value.get(
            "parameters"
        )

        if isinstance(parameters, list):
            for index, parameter in enumerate(
                parameters
            ):
                walk_parameter(
                    parameter,
                    append_pointer(
                        append_pointer(
                            pointer,
                            "parameters",
                        ),
                        str(index),
                    ),
                )

        for method in SUPPORTED_HTTP_METHODS:
            operation = value.get(method)

            if isinstance(operation, dict):
                walk_operation(
                    operation,
                    append_pointer(
                        pointer,
                        method,
                    ),
                )

    def walk_tag(value, pointer):
        if not isinstance(value, dict):
            return

        emit(
            value,
            "description",
            "tag",
            pointer,
        )

        external_docs = value.get(
            "externalDocs"
        )

        if isinstance(external_docs, dict):
            walk_external_docs(
                external_docs,
                append_pointer(
                    pointer,
                    "externalDocs",
                ),
            )

    def walk_components(
        value,
        pointer,
    ):
        if not isinstance(value, dict):
            return

        handlers = {
            "schemas": walk_schema,
            "responses": walk_response,
            "parameters": walk_parameter,
            "examples": walk_example,
            "requestBodies":
                walk_request_body,
            "headers": walk_header,
            "securitySchemes":
                walk_security_scheme,
            "links": walk_link,
            "callbacks": walk_callback,
            "pathItems": walk_path_item,
        }

        for section_name, handler in (
            handlers.items()
        ):
            section = value.get(
                section_name
            )

            if not isinstance(section, dict):
                continue

            section_pointer = append_pointer(
                pointer,
                section_name,
            )

            for name in sorted(section):
                handler(
                    section[name],
                    append_pointer(
                        section_pointer,
                        name,
                    ),
                )

    info = slice_document.get("info")

    if isinstance(info, dict):
        emit(
            info,
            "description",
            "info",
            "/info",
        )

    tags = slice_document.get("tags")

    if isinstance(tags, list):
        for index, tag in enumerate(tags):
            walk_tag(
                tag,
                f"/tags/{index}",
            )

    paths = slice_document.get("paths")

    if isinstance(paths, dict):
        for path_name in sorted(paths):
            walk_path_item(
                paths[path_name],
                append_pointer(
                    "/paths",
                    path_name,
                ),
            )

    components = slice_document.get(
        "components"
    )

    if isinstance(components, dict):
        walk_components(
            components,
            "/components",
        )

    rows.sort(
        key=lambda row: (
            row["slice_pointer"],
            row["object_kind"],
            row["field_name"],
        )
    )

    for annotation_index, row in enumerate(
        rows,
        start=1,
    ):
        row["annotation_index"] = (
            annotation_index
        )

    return {
        "rows": rows,
        "diagnostics": diagnostics,
        "prose_tokens_L0": sum(
            int(row["token_count"])
            for row in rows
            if (
                row[
                    "included_in_grouping"
                ]
                and row["token_count"]
                is not None
            )
        ),
    }


fixture = {
    "openapi": "3.0.3",
    "info": {
        "title": "Excluded title",
        "version": "1.0.0",
        "description": "Information prose.",
    },
    "tags": [
        {
            "name": "pets",
            "description": "Repeated prose.",
        }
    ],
    "paths": {
        "/pets": {
            "summary": "Pet path summary.",
            "description": "Pet path description.",
            "get": {
                "summary": "List pets.",
                "description": "Repeated prose.",
                "parameters": [
                    {
                        "name": "limit",
                        "in": "query",
                        "description": "Result limit.",
                        "schema": {
                            "type": "integer"
                        },
                    }
                ],
                "requestBody": {
                    "description": "Request prose.",
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "description": (
                                    "Inline schema."
                                ),
                                "properties": {
                                    "description": {
                                        "type": "string",
                                        "description": (
                                            "Property schema prose."
                                        ),
                                    }
                                },
                            }
                        }
                    },
                },
                "responses": {
                    "200": {
                        "description": "Success.",
                        "headers": {
                            "X-Rate": {
                                "description": (
                                    "Rate header."
                                ),
                                "schema": {
                                    "type": "integer"
                                },
                            }
                        },
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/Shared"
                                    )
                                },
                                "examples": {
                                    "sample": {
                                        "summary": (
                                            "Example summary."
                                        ),
                                        "description": (
                                            "Example prose."
                                        ),
                                        "value": {
                                            "summary": (
                                                "Payload summary "
                                                "must be excluded."
                                            ),
                                            "description": (
                                                "Payload description "
                                                "must be excluded."
                                            ),
                                        },
                                    }
                                },
                            }
                        },
                        "links": {
                            "next": {
                                "description": (
                                    "Next-page link."
                                ),
                                "operationId": "nextPage",
                            }
                        },
                    },
                    "201": {
                        "description": "Created.",
                        "content": {
                            "application/json": {
                                "schema": {
                                    "$ref": (
                                        "#/components/"
                                        "schemas/Shared"
                                    )
                                }
                            }
                        },
                    },
                },
                "externalDocs": {
                    "description": (
                        "Operation documentation."
                    ),
                    "url": "https://example.test",
                },
                "x-extension": {
                    "summary": (
                        "Extension summary excluded."
                    ),
                    "description": (
                        "Extension description "
                        "excluded."
                    ),
                },
            },
        }
    },
    "components": {
        "schemas": {
            "Shared": {
                "title": "Excluded schema title",
                "description": "Shared schema.",
                "type": "object",
                "properties": {
                    "child": {
                        "type": "string",
                        "description": (
                            "Child schema."
                        ),
                        "enum": [
                            "description",
                            "summary",
                        ],
                        "example": {
                            "description": (
                                "Schema example payload "
                                "excluded."
                            )
                        },
                    }
                },
            },
            "Invalid": {
                "type": "string",
                "description": 123,
            },
        },
        "examples": {
            "Reusable": {
                "summary": "Reusable example.",
                "description": (
                    "Reusable example prose."
                ),
                "value": {
                    "description": (
                        "Reusable payload excluded."
                    )
                },
            }
        },
        "securitySchemes": {
            "OAuth": {
                "type": "oauth2",
                "description": (
                    "Security scheme prose."
                ),
                "flows": {
                    "authorizationCode": {
                        "authorizationUrl": (
                            "https://example.test/auth"
                        ),
                        "tokenUrl": (
                            "https://example.test/token"
                        ),
                        "scopes": {
                            "read": (
                                "Scope description excluded."
                            )
                        },
                    }
                },
            }
        },
    },
}

identity_mapping = [
    {
        "slice_pointer": "",
        "source_pointer": "",
        "mapping_type": (
            "qualification_identity"
        ),
    }
]

qualification = walk_prose_annotations(
    fixture,
    identity_mapping,
    TOKENIZER,
)

expected_valid_pointers = {
    "/info/description",
    "/tags/0/description",
    "/paths/~1pets/summary",
    "/paths/~1pets/description",
    "/paths/~1pets/get/summary",
    "/paths/~1pets/get/description",
    (
        "/paths/~1pets/get/parameters/0/"
        "description"
    ),
    (
        "/paths/~1pets/get/requestBody/"
        "description"
    ),
    (
        "/paths/~1pets/get/requestBody/"
        "content/application~1json/schema/"
        "description"
    ),
    (
        "/paths/~1pets/get/requestBody/"
        "content/application~1json/schema/"
        "properties/description/description"
    ),
    (
        "/paths/~1pets/get/responses/200/"
        "description"
    ),
    (
        "/paths/~1pets/get/responses/200/"
        "headers/X-Rate/description"
    ),
    (
        "/paths/~1pets/get/responses/200/"
        "content/application~1json/examples/"
        "sample/summary"
    ),
    (
        "/paths/~1pets/get/responses/200/"
        "content/application~1json/examples/"
        "sample/description"
    ),
    (
        "/paths/~1pets/get/responses/200/"
        "links/next/description"
    ),
    (
        "/paths/~1pets/get/responses/201/"
        "description"
    ),
    (
        "/paths/~1pets/get/externalDocs/"
        "description"
    ),
    (
        "/components/schemas/Shared/"
        "description"
    ),
    (
        "/components/schemas/Shared/"
        "properties/child/description"
    ),
    (
        "/components/examples/Reusable/"
        "summary"
    ),
    (
        "/components/examples/Reusable/"
        "description"
    ),
    (
        "/components/securitySchemes/OAuth/"
        "description"
    ),
}

expected_invalid_pointer = (
    "/components/schemas/Invalid/"
    "description"
)

observed_valid_pointers = {
    row["slice_pointer"]
    for row in qualification["rows"]
    if row["included_in_grouping"]
}

observed_invalid_pointers = {
    row["slice_pointer"]
    for row in qualification["rows"]
    if not row["included_in_grouping"]
}

repeated_hash = hashlib.sha256(
    "Repeated prose.".encode("utf-8")
).hexdigest()

repeated_occurrence_count = sum(
    row["text_sha256"] == repeated_hash
    for row in qualification["rows"]
)

excluded_pointer_fragments = [
    "x-extension",
    "/value/",
    "/enum/",
    "/title",
    "/scopes/",
    "/example/",
]

excluded_content_absent = all(
    not any(
        fragment in row["slice_pointer"]
        for fragment
        in excluded_pointer_fragments
    )
    for row in qualification["rows"]
)

qualification_checks = {
    "valid_pointer_set_matches": (
        observed_valid_pointers
        == expected_valid_pointers
    ),
    "invalid_pointer_recorded": (
        observed_invalid_pointers
        == {expected_invalid_pointer}
    ),
    "identical_text_counted_twice": (
        repeated_occurrence_count == 2
    ),
    "excluded_content_absent": (
        excluded_content_absent
    ),
    "one_nonstring_diagnostic": (
        len(qualification["diagnostics"])
        == 1
        and qualification[
            "diagnostics"
        ][0]["type"]
        == "NONSTRING_PROSE_ANNOTATION"
    ),
    "row_token_sum_matches_score": (
        qualification[
            "prose_tokens_L0"
        ]
        == sum(
            row["token_count"]
            for row in qualification[
                "rows"
            ]
            if row[
                "included_in_grouping"
            ]
        )
    ),
}

qualification_status = (
    "PASS"
    if all(
        qualification_checks.values()
    )
    else "BLOCKED"
)

write_json_atomic(
    PROSE_WALKER_FIXTURE_PATH,
    fixture,
)

walker_report = {
    "status": qualification_status,
    "policy_version": "1.2",
    "checks": qualification_checks,
    "expected_valid_annotation_count": (
        len(expected_valid_pointers)
    ),
    "observed_valid_annotation_count": (
        len(observed_valid_pointers)
    ),
    "observed_invalid_annotation_count": (
        len(observed_invalid_pointers)
    ),
    "prose_tokens_L0": (
        qualification[
            "prose_tokens_L0"
        ]
    ),
    "diagnostics": (
        qualification["diagnostics"]
    ),
    "fixture_path": str(
        PROSE_WALKER_FIXTURE_PATH
    ),
    "fixture_sha256": sha256_file(
        PROSE_WALKER_FIXTURE_PATH
    ),
}

write_json_atomic(
    PROSE_WALKER_REPORT_PATH,
    walker_report,
)

output = dict(walker_report)

output["report_path"] = str(
    PROSE_WALKER_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    PROSE_WALKER_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if qualification_status != "PASS":
    raise RuntimeError(
        "Prose annotation walker qualification "
        "failed."
    )

print(
    "\nNOTEBOOK 02 — CELL 11: PASS"
)

{
  "status": "PASS",
  "policy_version": "1.2",
  "checks": {
    "valid_pointer_set_matches": true,
    "invalid_pointer_recorded": true,
    "identical_text_counted_twice": true,
    "excluded_content_absent": true,
    "one_nonstring_diagnostic": true,
    "row_token_sum_matches_score": true
  },
  "expected_valid_annotation_count": 22,
  "observed_valid_annotation_count": 22,
  "observed_invalid_annotation_count": 1,
  "prose_tokens_L0": 70,
  "diagnostics": [
    {
      "type": "NONSTRING_PROSE_ANNOTATION",
      "slice_pointer": "/components/schemas/Invalid/description",
      "source_pointer": "/components/schemas/Invalid/description",
      "object_kind": "schema",
      "field_name": "description",
      "observed_type": "int"
    }
  ],
  "fixture_path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/development_examples/prose_walker_fixture.json",
  "fixture_sha256": "d751da3e42901cb8808c9424342763f9a919ad2a53799efe53ac8d8175b9b8fd",
  "report_path": "/kag

In [12]:
# NOTEBOOK 02 — CELL 12
# Measure schema-defined prose annotations for all usable L0 slices.

from collections import Counter
from pathlib import Path
import csv
import hashlib
import json
import os
import shutil
import statistics
import time


PROSE_CHECKPOINT_SIZE = 250

PROSE_CHECKPOINT_DIR = (
    PROJECT_ROOT
    / "data"
    / "checkpoints"
    / "prose_measurement"
)

PROSE_FIELDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "prose_fields.csv"
)

PROSE_MEASUREMENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "audits"
    / "prose_measurements.jsonl"
)

PROSE_MEASUREMENT_REPORT_PATH = (
    PROJECT_ROOT
    / "reports"
    / "audit"
    / "prose_measurement.json"
)

PROSE_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROSE_FIELD_COLUMNS = [
    "op_id",
    "annotation_index",
    "slice_pointer",
    "source_pointer",
    "object_kind",
    "field_name",
    "text_sha256",
    "char_count",
    "token_count",
    "included_in_grouping",
    "diagnostic",
]


def checkpoint_identity_hash(
    slice_records,
    pointer_map_records,
):
    identity = []

    for slice_record, pointer_record in zip(
        slice_records,
        pointer_map_records,
    ):
        pointer_hash = hashlib.sha256(
            canonical_json_bytes(
                pointer_record
            )
        ).hexdigest()

        identity.append(
            {
                "op_id":
                    slice_record["op_id"],
                "status":
                    slice_record["status"],
                "slice_sha256":
                    slice_record.get(
                        "slice_sha256"
                    ),
                "pointer_map_sha256":
                    pointer_hash,
            }
        )

    return hashlib.sha256(
        canonical_json_bytes(identity)
    ).hexdigest()


def write_prose_csv_part(
    path,
    rows,
):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".temporary"
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=PROSE_FIELD_COLUMNS,
            extrasaction="raise",
            lineterminator="\n",
        )

        writer.writeheader()
        writer.writerows(rows)

    os.replace(
        temporary_path,
        path,
    )


def verify_prose_checkpoint(
    manifest_path,
    expected_identity_sha256,
    expected_operation_count,
):
    if not manifest_path.is_file():
        return None

    try:
        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return None

    if (
        manifest.get(
            "input_identity_sha256"
        )
        != expected_identity_sha256
    ):
        return None

    if (
        manifest.get("operation_count")
        != expected_operation_count
    ):
        return None

    operation_path = Path(
        manifest.get(
            "operation_records_path",
            "",
        )
    )

    prose_path = Path(
        manifest.get(
            "prose_rows_path",
            "",
        )
    )

    if (
        not operation_path.is_file()
        or not prose_path.is_file()
    ):
        return None

    if (
        sha256_file(operation_path)
        != manifest.get(
            "operation_records_sha256"
        )
    ):
        return None

    if (
        sha256_file(prose_path)
        != manifest.get(
            "prose_rows_sha256"
        )
    ):
        return None

    operation_rows = load_jsonl(
        operation_path
    )

    if (
        len(operation_rows)
        != expected_operation_count
    ):
        return None

    return {
        "manifest": manifest,
        "operation_rows":
            operation_rows,
    }


measurement_start = time.perf_counter()

slice_handle = SLICE_RECORDS_PATH.open(
    "r",
    encoding="utf-8",
)

pointer_handle = POINTER_MAPS_PATH.open(
    "r",
    encoding="utf-8",
)

checkpoint_manifests = []
all_operation_measurements = []

processed_count = 0
checkpoint_reused_count = 0
newly_measured_count = 0
unexpected_failure_count = 0

try:
    chunk_index = 0

    while True:
        slice_chunk = []
        pointer_chunk = []

        for _ in range(PROSE_CHECKPOINT_SIZE):
            slice_line = slice_handle.readline()
            pointer_line = (
                pointer_handle.readline()
            )

            if not slice_line:
                if pointer_line:
                    raise RuntimeError(
                        "Pointer-map file contains "
                        "more records than the "
                        "slice-record file."
                    )
                break

            if not pointer_line:
                raise RuntimeError(
                    "Pointer-map file ended before "
                    "the slice-record file."
                )

            slice_record = json.loads(
                slice_line
            )

            pointer_record = json.loads(
                pointer_line
            )

            if (
                slice_record["op_id"]
                != pointer_record["op_id"]
            ):
                raise RuntimeError(
                    "Slice and pointer-map records "
                    "are misaligned: "
                    f"{slice_record['op_id']} versus "
                    f"{pointer_record['op_id']}."
                )

            slice_chunk.append(
                slice_record
            )

            pointer_chunk.append(
                pointer_record
            )

        if not slice_chunk:
            break

        first_global_index = (
            operation_record_map[
                slice_chunk[0]["op_id"]
            ]["global_operation_index"]
        )

        last_global_index = (
            operation_record_map[
                slice_chunk[-1]["op_id"]
            ]["global_operation_index"]
        )

        checkpoint_stem = (
            f"prose_{first_global_index:06d}_"
            f"{last_global_index:06d}"
        )

        prose_part_path = (
            PROSE_CHECKPOINT_DIR
            / f"{checkpoint_stem}.csv"
        )

        operation_part_path = (
            PROSE_CHECKPOINT_DIR
            / f"{checkpoint_stem}.jsonl"
        )

        manifest_path = (
            PROSE_CHECKPOINT_DIR
            / f"{checkpoint_stem}.manifest.json"
        )

        identity_sha256 = (
            checkpoint_identity_hash(
                slice_chunk,
                pointer_chunk,
            )
        )

        verified_checkpoint = (
            verify_prose_checkpoint(
                manifest_path,
                identity_sha256,
                len(slice_chunk),
            )
        )

        if verified_checkpoint is not None:
            operation_measurements = (
                verified_checkpoint[
                    "operation_rows"
                ]
            )

            checkpoint_manifest = (
                verified_checkpoint[
                    "manifest"
                ]
            )

            checkpoint_reused_count += len(
                slice_chunk
            )

        else:
            prose_rows = []
            operation_measurements = []

            for (
                slice_record,
                pointer_record,
            ) in zip(
                slice_chunk,
                pointer_chunk,
            ):
                operation_start = (
                    time.perf_counter()
                )

                op_id = slice_record["op_id"]
                slice_status = (
                    slice_record["status"]
                )

                if slice_status != "PASS":
                    operation_measurements.append(
                        {
                            "op_id": op_id,
                            "spec_id":
                                slice_record[
                                    "spec_id"
                                ],
                            "slice_status":
                                slice_status,
                            "measurement_status":
                                (
                                    "NOT_APPLICABLE_"
                                    "EXCLUDED_SLICE"
                                ),
                            "annotation_count": 0,
                            "included_annotation_count":
                                0,
                            "invalid_annotation_count":
                                0,
                            "summary_count": 0,
                            "description_count": 0,
                            "total_character_count":
                                None,
                            "prose_tokens_L0": None,
                            "object_kind_counts": {},
                            "field_name_counts": {},
                            "missing_source_pointer_count":
                                0,
                            "diagnostics":
                                slice_record.get(
                                    "diagnostics",
                                    [],
                                ),
                            "slice_sha256":
                                slice_record.get(
                                    "slice_sha256"
                                ),
                            "duration_seconds": round(
                                time.perf_counter()
                                - operation_start,
                                6,
                            ),
                        }
                    )
                    continue

                slice_path = (
                    PROJECT_ROOT
                    / slice_record[
                        "slice_relative_path"
                    ]
                )

                if (
                    sha256_file(slice_path)
                    != slice_record[
                        "slice_sha256"
                    ]
                ):
                    raise RuntimeError(
                        "Slice hash mismatch before "
                        f"prose measurement: {op_id}"
                    )

                slice_document = json.loads(
                    slice_path.read_text(
                        encoding="utf-8"
                    )
                )

                measured = (
                    walk_prose_annotations(
                        slice_document,
                        pointer_record[
                            "mappings"
                        ],
                        TOKENIZER,
                    )
                )

                missing_source_pointers = [
                    row
                    for row in measured["rows"]
                    if (
                        row[
                            "included_in_grouping"
                        ]
                        and row[
                            "source_pointer"
                        ]
                        is None
                    )
                ]

                diagnostics = list(
                    measured["diagnostics"]
                )

                for row in (
                    missing_source_pointers
                ):
                    diagnostics.append(
                        {
                            "type": (
                                "MISSING_SOURCE_POINTER_"
                                "MAPPING"
                            ),
                            "slice_pointer":
                                row[
                                    "slice_pointer"
                                ],
                        }
                    )

                measurement_status = (
                    "PASS"
                    if not diagnostics
                    else "FAIL"
                )

                if measurement_status == "FAIL":
                    unexpected_failure_count += 1

                object_kind_counts = Counter(
                    row["object_kind"]
                    for row in measured["rows"]
                    if row[
                        "included_in_grouping"
                    ]
                )

                field_name_counts = Counter(
                    row["field_name"]
                    for row in measured["rows"]
                    if row[
                        "included_in_grouping"
                    ]
                )

                for row in measured["rows"]:
                    prose_rows.append(
                        {
                            "op_id": op_id,
                            "annotation_index":
                                row[
                                    "annotation_index"
                                ],
                            "slice_pointer":
                                row[
                                    "slice_pointer"
                                ],
                            "source_pointer":
                                row[
                                    "source_pointer"
                                ],
                            "object_kind":
                                row[
                                    "object_kind"
                                ],
                            "field_name":
                                row[
                                    "field_name"
                                ],
                            "text_sha256":
                                row[
                                    "text_sha256"
                                ],
                            "char_count":
                                row[
                                    "char_count"
                                ],
                            "token_count":
                                row[
                                    "token_count"
                                ],
                            "included_in_grouping":
                                row[
                                    "included_in_grouping"
                                ],
                            "diagnostic":
                                row[
                                    "diagnostic"
                                ],
                        }
                    )

                operation_measurements.append(
                    {
                        "op_id": op_id,
                        "spec_id":
                            slice_record[
                                "spec_id"
                            ],
                        "slice_status":
                            slice_status,
                        "measurement_status":
                            measurement_status,
                        "annotation_count":
                            len(
                                measured["rows"]
                            ),
                        "included_annotation_count":
                            sum(
                                row[
                                    "included_in_grouping"
                                ]
                                for row
                                in measured["rows"]
                            ),
                        "invalid_annotation_count":
                            sum(
                                not row[
                                    "included_in_grouping"
                                ]
                                for row
                                in measured["rows"]
                            ),
                        "summary_count":
                            field_name_counts.get(
                                "summary",
                                0,
                            ),
                        "description_count":
                            field_name_counts.get(
                                "description",
                                0,
                            ),
                        "total_character_count":
                            sum(
                                int(
                                    row[
                                        "char_count"
                                    ]
                                )
                                for row
                                in measured["rows"]
                                if (
                                    row[
                                        "included_in_grouping"
                                    ]
                                    and row[
                                        "char_count"
                                    ]
                                    is not None
                                )
                            ),
                        "prose_tokens_L0":
                            measured[
                                "prose_tokens_L0"
                            ],
                        "object_kind_counts":
                            dict(
                                sorted(
                                    object_kind_counts
                                    .items()
                                )
                            ),
                        "field_name_counts":
                            dict(
                                sorted(
                                    field_name_counts
                                    .items()
                                )
                            ),
                        "missing_source_pointer_count":
                            len(
                                missing_source_pointers
                            ),
                        "diagnostics":
                            diagnostics,
                        "slice_sha256":
                            slice_record[
                                "slice_sha256"
                            ],
                        "duration_seconds": round(
                            time.perf_counter()
                            - operation_start,
                            6,
                        ),
                    }
                )

                newly_measured_count += 1

            write_prose_csv_part(
                prose_part_path,
                prose_rows,
            )

            write_jsonl_atomic(
                operation_part_path,
                operation_measurements,
            )

            checkpoint_manifest = {
                "checkpoint_index":
                    chunk_index,
                "first_global_operation_index":
                    first_global_index,
                "last_global_operation_index":
                    last_global_index,
                "input_identity_sha256":
                    identity_sha256,
                "operation_count":
                    len(slice_chunk),
                "prose_row_count":
                    len(prose_rows),
                "operation_records_path":
                    str(operation_part_path),
                "operation_records_sha256":
                    sha256_file(
                        operation_part_path
                    ),
                "prose_rows_path":
                    str(prose_part_path),
                "prose_rows_sha256":
                    sha256_file(
                        prose_part_path
                    ),
                "status": (
                    "PASS"
                    if all(
                        row[
                            "measurement_status"
                        ]
                        in {
                            "PASS",
                            (
                                "NOT_APPLICABLE_"
                                "EXCLUDED_SLICE"
                            ),
                        }
                        for row in (
                            operation_measurements
                        )
                    )
                    else "BLOCKED"
                ),
            }

            write_json_atomic(
                manifest_path,
                checkpoint_manifest,
            )

        checkpoint_manifests.append(
            {
                **checkpoint_manifest,
                "manifest_path":
                    str(manifest_path),
                "manifest_sha256":
                    sha256_file(
                        manifest_path
                    ),
            }
        )

        all_operation_measurements.extend(
            operation_measurements
        )

        processed_count += len(
            slice_chunk
        )

        if (
            processed_count % 1000
            < PROSE_CHECKPOINT_SIZE
            or processed_count
            == EXPECTED_OPERATION_COUNT
        ):
            measured_pass_count = sum(
                row[
                    "measurement_status"
                ]
                == "PASS"
                for row in (
                    all_operation_measurements
                )
            )

            zero_prose_count = sum(
                row["prose_tokens_L0"] == 0
                for row in (
                    all_operation_measurements
                )
                if row[
                    "measurement_status"
                ]
                == "PASS"
            )

            disk_usage = shutil.disk_usage(
                "/kaggle/working"
            )

            print(
                json.dumps(
                    {
                        "processed":
                            processed_count,
                        "total":
                            EXPECTED_OPERATION_COUNT,
                        "measurement_pass_count":
                            measured_pass_count,
                        "zero_prose_count":
                            zero_prose_count,
                        "checkpoint_reused_count":
                            checkpoint_reused_count,
                        "newly_measured_count":
                            newly_measured_count,
                        "unexpected_failure_count":
                            unexpected_failure_count,
                        "free_storage_gib": round(
                            disk_usage.free
                            / (1024 ** 3),
                            3,
                        ),
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - measurement_start,
                            1,
                        ),
                    }
                )
            )

        chunk_index += 1

    if pointer_handle.readline():
        raise RuntimeError(
            "Pointer-map file contains trailing "
            "records."
        )

finally:
    slice_handle.close()
    pointer_handle.close()

if (
    len(all_operation_measurements)
    != EXPECTED_OPERATION_COUNT
):
    raise RuntimeError(
        "Prose measurement operation count "
        "does not match the frozen inventory."
    )

observed_ids = [
    row["op_id"]
    for row in all_operation_measurements
]

expected_ids = [
    row["op_id"]
    for row in operation_records
]

if observed_ids != expected_ids:
    raise RuntimeError(
        "Prose measurement records are not "
        "aligned with the operation inventory."
    )


# Consolidate operation-level measurement records.
write_jsonl_atomic(
    PROSE_MEASUREMENTS_PATH,
    all_operation_measurements,
)


# Consolidate CSV checkpoint parts without loading all rows.
temporary_prose_fields_path = (
    PROSE_FIELDS_PATH.with_name(
        PROSE_FIELDS_PATH.name
        + ".temporary"
    )
)

with temporary_prose_fields_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as output_handle:
    wrote_header = False

    for checkpoint_manifest in (
        checkpoint_manifests
    ):
        part_path = Path(
            checkpoint_manifest[
                "prose_rows_path"
            ]
        )

        with part_path.open(
            "r",
            encoding="utf-8",
            newline="",
        ) as input_handle:
            header = input_handle.readline()

            if not wrote_header:
                output_handle.write(header)
                wrote_header = True

            for line in input_handle:
                output_handle.write(line)

os.replace(
    temporary_prose_fields_path,
    PROSE_FIELDS_PATH,
)

status_counts = Counter(
    row["measurement_status"]
    for row in all_operation_measurements
)

pass_measurements = [
    row
    for row in all_operation_measurements
    if row["measurement_status"] == "PASS"
]

token_scores = [
    int(row["prose_tokens_L0"])
    for row in pass_measurements
]

annotation_count_total = sum(
    int(row["included_annotation_count"])
    for row in pass_measurements
)

summary_count_total = sum(
    int(row["summary_count"])
    for row in pass_measurements
)

description_count_total = sum(
    int(row["description_count"])
    for row in pass_measurements
)

invalid_annotation_total = sum(
    int(row["invalid_annotation_count"])
    for row in pass_measurements
)

missing_source_pointer_total = sum(
    int(
        row[
            "missing_source_pointer_count"
        ]
    )
    for row in pass_measurements
)

unexpected_failures = [
    row
    for row in all_operation_measurements
    if row["measurement_status"]
    not in {
        "PASS",
        (
            "NOT_APPLICABLE_"
            "EXCLUDED_SLICE"
        ),
    }
]

measurement_report = {
    "policy_version": "1.2",
    "tokenizer_model_id": MODEL_ID,
    "tokenizer_revision":
        MODEL_REVISION,
    "operation_count":
        len(all_operation_measurements),
    "measured_operation_count":
        len(pass_measurements),
    "excluded_operation_count":
        status_counts.get(
            (
                "NOT_APPLICABLE_"
                "EXCLUDED_SLICE"
            ),
            0,
        ),
    "unexpected_failure_count":
        len(unexpected_failures),
    "measurement_status_counts": dict(
        sorted(status_counts.items())
    ),
    "included_annotation_count":
        annotation_count_total,
    "summary_annotation_count":
        summary_count_total,
    "description_annotation_count":
        description_count_total,
    "invalid_annotation_count":
        invalid_annotation_total,
    "missing_source_pointer_count":
        missing_source_pointer_total,
    "zero_prose_operation_count":
        sum(
            score == 0
            for score in token_scores
        ),
    "prose_token_statistics": {
        "minimum": min(token_scores),
        "mean": round(
            statistics.fmean(
                token_scores
            ),
            3,
        ),
        "median":
            statistics.median(
                token_scores
            ),
        "maximum": max(token_scores),
        "total": sum(token_scores),
    },
    "checkpoint_count":
        len(checkpoint_manifests),
    "checkpoint_reused_count":
        checkpoint_reused_count,
    "newly_measured_count":
        newly_measured_count,
    "prose_fields": {
        "path": str(PROSE_FIELDS_PATH),
        "bytes":
            PROSE_FIELDS_PATH.stat().st_size,
        "sha256":
            sha256_file(
                PROSE_FIELDS_PATH
            ),
    },
    "prose_measurements": {
        "path": str(
            PROSE_MEASUREMENTS_PATH
        ),
        "bytes": (
            PROSE_MEASUREMENTS_PATH
            .stat()
            .st_size
        ),
        "sha256": sha256_file(
            PROSE_MEASUREMENTS_PATH
        ),
    },
    "failure_examples": [
        {
            "op_id": row["op_id"],
            "spec_id": row["spec_id"],
            "diagnostics":
                row["diagnostics"],
        }
        for row
        in unexpected_failures[:25]
    ],
    "duration_seconds": round(
        time.perf_counter()
        - measurement_start,
        3,
    ),
    "status": (
        "PASS"
        if (
            len(pass_measurements)
            == 58584
            and len(unexpected_failures)
            == 0
            and invalid_annotation_total == 0
            and missing_source_pointer_total
            == 0
        )
        else "BLOCKED"
    ),
}

write_json_atomic(
    PROSE_MEASUREMENT_REPORT_PATH,
    measurement_report,
)

output = dict(measurement_report)

output["report_path"] = str(
    PROSE_MEASUREMENT_REPORT_PATH
)

output["report_sha256"] = sha256_file(
    PROSE_MEASUREMENT_REPORT_PATH
)

print(
    json.dumps(
        output,
        indent=2,
    )
)

if measurement_report["status"] != "PASS":
    raise RuntimeError(
        "Full-corpus prose measurement failed."
    )

print(
    "\nNOTEBOOK 02 — CELL 12: PASS"
)

{"processed": 1000, "total": 58585, "measurement_pass_count": 1000, "zero_prose_count": 0, "checkpoint_reused_count": 0, "newly_measured_count": 1000, "unexpected_failure_count": 0, "free_storage_gib": 17.888, "elapsed_seconds": 14.4}
{"processed": 2000, "total": 58585, "measurement_pass_count": 2000, "zero_prose_count": 0, "checkpoint_reused_count": 0, "newly_measured_count": 2000, "unexpected_failure_count": 0, "free_storage_gib": 17.87, "elapsed_seconds": 23.5}
{"processed": 3000, "total": 58585, "measurement_pass_count": 3000, "zero_prose_count": 0, "checkpoint_reused_count": 0, "newly_measured_count": 3000, "unexpected_failure_count": 0, "free_storage_gib": 17.861, "elapsed_seconds": 28.7}
{"processed": 4000, "total": 58585, "measurement_pass_count": 4000, "zero_prose_count": 0, "checkpoint_reused_count": 0, "newly_measured_count": 4000, "unexpected_failure_count": 0, "free_storage_gib": 17.854, "elapsed_seconds": 33.3}
{"processed": 5000, "total": 58585, "measurement_pass_count":

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (131871 > 32768). Running this sequence through the model will result in indexing errors


{"processed": 40000, "total": 58585, "measurement_pass_count": 39999, "zero_prose_count": 24, "checkpoint_reused_count": 0, "newly_measured_count": 39999, "unexpected_failure_count": 0, "free_storage_gib": 17.437, "elapsed_seconds": 289.9}
{"processed": 41000, "total": 58585, "measurement_pass_count": 40999, "zero_prose_count": 24, "checkpoint_reused_count": 0, "newly_measured_count": 40999, "unexpected_failure_count": 0, "free_storage_gib": 17.428, "elapsed_seconds": 293.2}
{"processed": 42000, "total": 58585, "measurement_pass_count": 41999, "zero_prose_count": 24, "checkpoint_reused_count": 0, "newly_measured_count": 41999, "unexpected_failure_count": 0, "free_storage_gib": 17.423, "elapsed_seconds": 295.2}
{"processed": 43000, "total": 58585, "measurement_pass_count": 42999, "zero_prose_count": 24, "checkpoint_reused_count": 0, "newly_measured_count": 42999, "unexpected_failure_count": 0, "free_storage_gib": 17.417, "elapsed_seconds": 298.6}
{"processed": 44000, "total": 58585, "me

In [13]:
# Notebook 02 — Cell 13
# Render every L0 prompt exactly as qualified in Notebook 00,
# count complete prompt tokens, classify input-limit eligibility,
# and preserve exact rendered prompts plus token IDs in compressed
# resumable checkpoint chunks.

from pathlib import Path
from collections import Counter
from statistics import mean, median
import gzip
import hashlib
import json
import os
import shutil
import time

import yaml


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL13 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

SLICE_RECORDS_PATH_CELL13 = (
    PROJECT_ROOT_CELL13
    / "data/audits/slice_records.jsonl"
)

FULL_SLICE_REPORT_PATH_CELL13 = (
    PROJECT_ROOT_CELL13
    / "reports/audit/full_slice_construction.json"
)

PROMPT_MEASUREMENTS_PATH_CELL13 = (
    PROJECT_ROOT_CELL13
    / "data/audits/rendered_prompt_measurements.jsonl"
)

PAYLOAD_INDEX_PATH_CELL13 = (
    PROJECT_ROOT_CELL13
    / "data/audits/rendered_prompt_payload_index.jsonl"
)

CHECKPOINT_ROOT_CELL13 = (
    PROJECT_ROOT_CELL13
    / "data/checkpoints/rendered_prompt_measurement_v1"
)

PAYLOAD_ROOT_CELL13 = (
    PROJECT_ROOT_CELL13
    / "data/audits/rendered_prompt_payload_chunks"
)

REPORT_PATH_CELL13 = (
    PROJECT_ROOT_CELL13
    / "reports/audit/rendered_prompt_measurement.json"
)

UPSTREAM_CONFIG_ROOT_CELL13 = (
    PROJECT_ROOT_CELL13
    / "config/upstream"
)

for directory in [
    CHECKPOINT_ROOT_CELL13,
    PAYLOAD_ROOT_CELL13,
    REPORT_PATH_CELL13.parent,
    PROMPT_MEASUREMENTS_PATH_CELL13.parent,
    UPSTREAM_CONFIG_ROOT_CELL13,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Basic deterministic helpers
# ---------------------------------------------------------------------

def canonical_json_cell13(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )


def sha256_bytes_cell13(value):
    return hashlib.sha256(value).hexdigest()


def sha256_text_cell13(value):
    return sha256_bytes_cell13(
        value.encode("utf-8")
    )


def sha256_file_cell13(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def write_text_atomic_cell13(path, text):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_json_atomic_cell13(path, value):
    write_text_atomic_cell13(
        path,
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
    )


def read_jsonl_cell13(path):
    records = []

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                records.append(
                    json.loads(line)
                )
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}, "
                    f"line {line_number}: {error!r}"
                ) from error

    return records


def compact_token_ids_cell13(token_ids):
    return json.dumps(
        [int(value) for value in token_ids],
        ensure_ascii=False,
        separators=(",", ":"),
    )


def token_ids_sha256_cell13(token_ids):
    return sha256_text_cell13(
        compact_token_ids_cell13(token_ids)
    )


# ---------------------------------------------------------------------
# Locate the frozen Stage 00 project
# ---------------------------------------------------------------------

stage_00_project_candidates_cell13 = [
    STAGE_00_PROJECT_ROOT,
    Path(
        "/kaggle/input/prose-operation-stage-00-runtime/"
        "stage_00_protocol_runtime_v1/"
        "prose_operation_experiment"
    ),
]

if (
    "STAGE_00_PROJECT_ROOT" in globals()
    and globals()["STAGE_00_PROJECT_ROOT"] is not None
):
    stage_00_project_candidates_cell13.insert(
        0,
        Path(globals()["STAGE_00_PROJECT_ROOT"]),
    )

STAGE_00_PROJECT_ROOT_CELL13 = next(
    (
        path
        for path in stage_00_project_candidates_cell13
        if path.exists()
    ),
    None,
)

if STAGE_00_PROJECT_ROOT_CELL13 is None:
    raise RuntimeError(
        "Could not locate the expanded Stage 00 project."
    )


STUDY_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "config/study.yaml"
)

SYSTEM_PROMPT_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "config/system_prompt.txt"
)

USER_PROMPT_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "config/user_prompt.txt"
)

DEVELOPMENT_FIXTURE_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "data/development_examples/users.json"
)

PROMPT_QUALIFICATION_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "reports/runtime/prompt_reproducibility.json"
)

DEVELOPMENT_TOKEN_IDS_PATH_CELL13 = (
    STAGE_00_PROJECT_ROOT_CELL13
    / "reports/runtime/development_prompt_token_ids.json"
)

required_paths_cell13 = [
    STUDY_PATH_CELL13,
    SYSTEM_PROMPT_PATH_CELL13,
    USER_PROMPT_PATH_CELL13,
    DEVELOPMENT_FIXTURE_PATH_CELL13,
    PROMPT_QUALIFICATION_PATH_CELL13,
    DEVELOPMENT_TOKEN_IDS_PATH_CELL13,
    SLICE_RECORDS_PATH_CELL13,
    FULL_SLICE_REPORT_PATH_CELL13,
]

missing_paths_cell13 = [
    str(path)
    for path in required_paths_cell13
    if not path.is_file()
]

if missing_paths_cell13:
    raise RuntimeError(
        "Missing required Cell 13 inputs:\n"
        + "\n".join(missing_paths_cell13)
    )


# ---------------------------------------------------------------------
# Load frozen prompt configuration
# ---------------------------------------------------------------------

study_cell13 = yaml.safe_load(
    STUDY_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

system_prompt_cell13 = (
    SYSTEM_PROMPT_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

user_prompt_template_cell13 = (
    USER_PROMPT_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

prompt_qualification_cell13 = json.loads(
    PROMPT_QUALIFICATION_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

development_expected_ids_cell13 = json.loads(
    DEVELOPMENT_TOKEN_IDS_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

STUDY_ID_CELL13 = str(
    study_cell13["study_id"]
)

MODEL_ID_CELL13 = str(
    study_cell13["model_id"]
)

TOKENIZER_REVISION_CELL13 = str(
    study_cell13["tokenizer_revision"]
)

INPUT_TOKEN_LIMIT_CELL13 = int(
    study_cell13["max_rendered_input_tokens"]
)

if user_prompt_template_cell13.count(
    "{BOUNDARY}"
) != 2:
    raise RuntimeError(
        "The frozen user template must contain "
        "exactly two {BOUNDARY} placeholders."
    )

if user_prompt_template_cell13.count(
    "{CANONICAL_OPENAPI_JSON}"
) != 1:
    raise RuntimeError(
        "The frozen user template must contain "
        "exactly one canonical JSON placeholder."
    )


# Freeze exact upstream prompt inputs into Stage 02.
frozen_prompt_inputs_cell13 = {
    "stage_00_study_for_prompt.yaml":
        STUDY_PATH_CELL13,
    "stage_00_system_prompt.txt":
        SYSTEM_PROMPT_PATH_CELL13,
    "stage_00_user_prompt.txt":
        USER_PROMPT_PATH_CELL13,
    "stage_00_prompt_reproducibility.json":
        PROMPT_QUALIFICATION_PATH_CELL13,
}

for filename, source_path in (
    frozen_prompt_inputs_cell13.items()
):
    destination_path = (
        UPSTREAM_CONFIG_ROOT_CELL13
        / filename
    )

    destination_path.write_bytes(
        source_path.read_bytes()
    )


# ---------------------------------------------------------------------
# Reuse the tokenizer qualified by Cell 10, or reload it if needed
# ---------------------------------------------------------------------

tokenizer_cell13 = None

if (
    "TOKENIZER" in globals()
    and hasattr(
        globals()["TOKENIZER"],
        "apply_chat_template",
    )
):
    tokenizer_cell13 = globals()["TOKENIZER"]

elif (
    "tokenizer" in globals()
    and hasattr(
        globals()["tokenizer"],
        "apply_chat_template",
    )
):
    tokenizer_cell13 = globals()["tokenizer"]

else:
    from transformers import AutoTokenizer

    try:
        tokenizer_cell13 = (
            AutoTokenizer.from_pretrained(
                MODEL_ID_CELL13,
                revision=TOKENIZER_REVISION_CELL13,
                local_files_only=True,
                trust_remote_code=False,
            )
        )
    except Exception:
        tokenizer_cell13 = (
            AutoTokenizer.from_pretrained(
                MODEL_ID_CELL13,
                revision=TOKENIZER_REVISION_CELL13,
                trust_remote_code=False,
            )
        )

if tokenizer_cell13.chat_template is None:
    raise RuntimeError(
        "The frozen tokenizer has no chat template."
    )


# ---------------------------------------------------------------------
# Exact rendering functions
# ---------------------------------------------------------------------

def boundary_cell13(operation_id):
    material = (
        f"{STUDY_ID_CELL13}|{operation_id}"
    )

    return sha256_text_cell13(material)[:24]


def render_prompt_cell13(
    operation_id,
    canonical_openapi_json,
):
    boundary = boundary_cell13(operation_id)

    if boundary in canonical_openapi_json:
        raise RuntimeError(
            "Generated boundary occurs inside canonical "
            f"JSON for {operation_id}."
        )

    user_prompt = (
        user_prompt_template_cell13
        .replace(
            "{BOUNDARY}",
            boundary,
        )
        .replace(
            "{CANONICAL_OPENAPI_JSON}",
            canonical_openapi_json,
        )
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt_cell13,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    rendered_prompt = (
        tokenizer_cell13.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    encoded = tokenizer_cell13(
        rendered_prompt,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded["input_ids"]

    if (
        token_ids
        and isinstance(
            token_ids[0],
            (list, tuple),
        )
    ):
        if len(token_ids) != 1:
            raise RuntimeError(
                "Unexpected batched tokenizer result."
            )

        token_ids = token_ids[0]

    token_ids = [
        int(value)
        for value in token_ids
    ]

    return {
        "boundary": boundary,
        "rendered_prompt": rendered_prompt,
        "token_ids": token_ids,
    }


# ---------------------------------------------------------------------
# Reproduce Notebook 00's qualified development prompt exactly
# ---------------------------------------------------------------------

development_fixture_cell13 = json.loads(
    DEVELOPMENT_FIXTURE_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

development_canonical_json_cell13 = (
    canonical_json_cell13(
        development_fixture_cell13
    )
)

development_operation_id_cell13 = str(
    prompt_qualification_cell13[
        "operation_id"
    ]
)

development_render_cell13 = (
    render_prompt_cell13(
        development_operation_id_cell13,
        development_canonical_json_cell13,
    )
)

development_rendered_sha_cell13 = (
    sha256_text_cell13(
        development_render_cell13[
            "rendered_prompt"
        ]
    )
)

development_token_ids_sha_cell13 = (
    token_ids_sha256_cell13(
        development_render_cell13[
            "token_ids"
        ]
    )
)

development_checks_cell13 = {
    "boundary_matches": (
        development_render_cell13["boundary"]
        == prompt_qualification_cell13["boundary"]
    ),
    "rendered_prompt_sha256_matches": (
        development_rendered_sha_cell13
        == prompt_qualification_cell13[
            "rendered_prompt_sha256"
        ]
    ),
    "token_ids_sha256_matches": (
        development_token_ids_sha_cell13
        == prompt_qualification_cell13[
            "token_ids_sha256"
        ]
    ),
    "token_count_matches": (
        len(
            development_render_cell13[
                "token_ids"
            ]
        )
        == int(
            prompt_qualification_cell13[
                "rendered_prompt_tokens"
            ]
        )
    ),
    "token_ids_match_saved_record": (
        development_render_cell13["token_ids"]
        == [
            int(value)
            for value
            in development_expected_ids_cell13
        ]
    ),
}

if not all(
    development_checks_cell13.values()
):
    raise RuntimeError(
        "Frozen prompt renderer does not reproduce "
        "Notebook 00 exactly:\n"
        + json.dumps(
            development_checks_cell13,
            indent=2,
        )
    )


# ---------------------------------------------------------------------
# Load slice records and construction gate
# ---------------------------------------------------------------------

slice_records_cell13 = read_jsonl_cell13(
    SLICE_RECORDS_PATH_CELL13
)

full_slice_report_cell13 = json.loads(
    FULL_SLICE_REPORT_PATH_CELL13.read_text(
        encoding="utf-8"
    )
)

expected_operation_count_cell13 = int(
    full_slice_report_cell13[
        "operation_count"
    ]
)

expected_usable_count_cell13 = int(
    full_slice_report_cell13[
        "usable_valid_slice_count"
    ]
)

expected_exclusion_count_cell13 = int(
    full_slice_report_cell13[
        "explicit_exclusion_count"
    ]
)

if (
    len(slice_records_cell13)
    != expected_operation_count_cell13
):
    raise RuntimeError(
        "Slice-record count does not match the "
        "full slice report."
    )

operation_ids_cell13 = [
    str(record["op_id"])
    for record in slice_records_cell13
]

if len(set(operation_ids_cell13)) != len(
    operation_ids_cell13
):
    raise RuntimeError(
        "Duplicate op_id values in slice records."
    )


# ---------------------------------------------------------------------
# Resolve slice paths robustly
# ---------------------------------------------------------------------

slice_path_keys_cell13 = [
    "slice_path",
    "slice_file",
    "slice_absolute_path",
    "slice_relative_path",
    "output_path",
]

slice_path_index_cell13 = None


def build_slice_path_index_cell13():
    index = {}

    for path in (
        PROJECT_ROOT_CELL13 / "data"
    ).rglob("*.json"):
        filename = path.name

        if not filename.startswith("op_"):
            continue

        operation_id = filename.split(".", 1)[0]

        if operation_id in index:
            raise RuntimeError(
                "Multiple candidate slice files for "
                f"{operation_id}."
            )

        index[operation_id] = path

    return index


def resolve_slice_path_cell13(record):
    global slice_path_index_cell13

    for key in slice_path_keys_cell13:
        value = record.get(key)

        if not value:
            continue

        candidate = Path(str(value))

        if not candidate.is_absolute():
            candidate = (
                PROJECT_ROOT_CELL13
                / candidate
            )

        if candidate.is_file():
            return candidate

    operation_id = str(record["op_id"])

    direct_candidates = [
        (
            PROJECT_ROOT_CELL13
            / "data/slices/l0"
            / f"{operation_id}.json"
        ),
        (
            PROJECT_ROOT_CELL13
            / "data/slices/L0"
            / f"{operation_id}.json"
        ),
        (
            PROJECT_ROOT_CELL13
            / "data/l0_slices"
            / f"{operation_id}.json"
        ),
    ]

    for candidate in direct_candidates:
        if candidate.is_file():
            return candidate

    if slice_path_index_cell13 is None:
        slice_path_index_cell13 = (
            build_slice_path_index_cell13()
        )

    candidate = slice_path_index_cell13.get(
        operation_id
    )

    if candidate is None:
        raise FileNotFoundError(
            f"No slice file found for {operation_id}."
        )

    return candidate


def recorded_slice_hash_cell13(record):
    for key in [
        "slice_sha256",
        "canonical_slice_sha256",
        "output_sha256",
    ]:
        value = record.get(key)

        if value:
            return str(value)

    return None


# ---------------------------------------------------------------------
# Renderer fingerprint
# ---------------------------------------------------------------------

renderer_configuration_cell13 = {
    "measurement_version": "1.0",
    "study_id": STUDY_ID_CELL13,
    "model_id": MODEL_ID_CELL13,
    "tokenizer_revision":
        TOKENIZER_REVISION_CELL13,
    "input_token_limit":
        INPUT_TOKEN_LIMIT_CELL13,
    "system_prompt_sha256":
        sha256_file_cell13(
            SYSTEM_PROMPT_PATH_CELL13
        ),
    "user_prompt_sha256":
        sha256_file_cell13(
            USER_PROMPT_PATH_CELL13
        ),
    "chat_template_sha256":
        sha256_text_cell13(
            tokenizer_cell13.chat_template
        ),
    "slice_records_sha256":
        sha256_file_cell13(
            SLICE_RECORDS_PATH_CELL13
        ),
    "add_generation_prompt": True,
    "add_special_tokens_for_measurement":
        False,
}

renderer_fingerprint_cell13 = (
    sha256_text_cell13(
        canonical_json_cell13(
            renderer_configuration_cell13
        )
    )
)


# ---------------------------------------------------------------------
# Resumable corpus measurement
# ---------------------------------------------------------------------

CHECKPOINT_SIZE_CELL13 = 250
start_time_cell13 = time.perf_counter()

all_results_cell13 = []
payload_index_cell13 = []

checkpoint_reused_count_cell13 = 0
newly_measured_count_cell13 = 0
processed_count_cell13 = 0

running_status_counts_cell13 = Counter()
running_within_limit_cell13 = 0
running_exceeds_limit_cell13 = 0
running_failure_count_cell13 = 0


def checkpoint_paths_cell13(chunk_number):
    stem = f"chunk_{chunk_number:05d}"

    return {
        "summary": (
            CHECKPOINT_ROOT_CELL13
            / f"{stem}.jsonl"
        ),
        "payload": (
            PAYLOAD_ROOT_CELL13
            / f"{stem}.jsonl.gz"
        ),
        "metadata": (
            CHECKPOINT_ROOT_CELL13
            / f"{stem}.metadata.json"
        ),
    }


def reusable_checkpoint_cell13(
    paths,
    expected_operation_ids,
):
    if not all(
        path.is_file()
        for path in paths.values()
    ):
        return None

    try:
        metadata = json.loads(
            paths["metadata"].read_text(
                encoding="utf-8"
            )
        )

        if (
            metadata.get("renderer_fingerprint")
            != renderer_fingerprint_cell13
        ):
            return None

        if metadata.get(
            "operation_ids_sha256"
        ) != sha256_text_cell13(
            canonical_json_cell13(
                expected_operation_ids
            )
        ):
            return None

        if (
            metadata.get("payload_sha256")
            != sha256_file_cell13(
                paths["payload"]
            )
        ):
            return None

        records = read_jsonl_cell13(
            paths["summary"]
        )

        if [
            record["op_id"]
            for record in records
        ] != expected_operation_ids:
            return None

        if any(
            record.get("status") == "FAILED"
            for record in records
        ):
            return None

        if any(
            record.get("renderer_fingerprint")
            != renderer_fingerprint_cell13
            for record in records
        ):
            return None

        return records, metadata

    except Exception:
        return None


chunk_total_cell13 = (
    len(slice_records_cell13)
    + CHECKPOINT_SIZE_CELL13
    - 1
) // CHECKPOINT_SIZE_CELL13

for chunk_number_cell13 in range(
    chunk_total_cell13
):
    chunk_start_cell13 = (
        chunk_number_cell13
        * CHECKPOINT_SIZE_CELL13
    )

    chunk_end_cell13 = min(
        chunk_start_cell13
        + CHECKPOINT_SIZE_CELL13,
        len(slice_records_cell13),
    )

    source_chunk_cell13 = (
        slice_records_cell13[
            chunk_start_cell13:
            chunk_end_cell13
        ]
    )

    expected_chunk_operation_ids_cell13 = [
        str(record["op_id"])
        for record in source_chunk_cell13
    ]

    paths_cell13 = checkpoint_paths_cell13(
        chunk_number_cell13
    )

    reusable_cell13 = reusable_checkpoint_cell13(
        paths_cell13,
        expected_chunk_operation_ids_cell13,
    )

    if reusable_cell13 is not None:
        (
            chunk_results_cell13,
            payload_metadata_cell13,
        ) = reusable_cell13

        checkpoint_reused_count_cell13 += len(
            chunk_results_cell13
        )

    else:
        chunk_results_cell13 = []
        payload_count_cell13 = 0

        temporary_payload_path_cell13 = (
            paths_cell13["payload"].with_name(
                paths_cell13["payload"].name
                + ".tmp"
            )
        )

        if temporary_payload_path_cell13.exists():
            temporary_payload_path_cell13.unlink()

        with gzip.open(
            temporary_payload_path_cell13,
            mode="wt",
            encoding="utf-8",
            newline="\n",
            compresslevel=6,
        ) as payload_handle_cell13:

            for record_cell13 in source_chunk_cell13:
                operation_id_cell13 = str(
                    record_cell13["op_id"]
                )

                source_status_cell13 = str(
                    record_cell13.get(
                        "status",
                        "UNKNOWN",
                    )
                )

                base_result_cell13 = {
                    "op_id": operation_id_cell13,
                    "spec_id": record_cell13.get(
                        "spec_id"
                    ),
                    "path": record_cell13.get(
                        "path"
                    ),
                    "method": record_cell13.get(
                        "method"
                    ),
                    "source_class":
                        record_cell13.get(
                            "source_class"
                        ),
                    "slice_status":
                        source_status_cell13,
                    "renderer_fingerprint":
                        renderer_fingerprint_cell13,
                }

                if source_status_cell13 != "PASS":
                    result_cell13 = {
                        **base_result_cell13,
                        "status":
                            "NOT_APPLICABLE_EXCLUDED_SLICE",
                        "diagnostic":
                            record_cell13.get(
                                "diagnostics"
                            ),
                        "boundary": None,
                        "slice_sha256":
                            recorded_slice_hash_cell13(
                                record_cell13
                            ),
                        "rendered_prompt_bytes":
                            None,
                        "rendered_prompt_tokens":
                            None,
                        "input_token_limit":
                            INPUT_TOKEN_LIMIT_CELL13,
                        "within_input_limit":
                            None,
                        "excess_tokens": None,
                        "rendered_prompt_sha256":
                            None,
                        "token_ids_sha256": None,
                    }

                    chunk_results_cell13.append(
                        result_cell13
                    )
                    continue

                try:
                    slice_path_cell13 = (
                        resolve_slice_path_cell13(
                            record_cell13
                        )
                    )

                    slice_object_cell13 = json.loads(
                        slice_path_cell13.read_text(
                            encoding="utf-8"
                        )
                    )

                    canonical_slice_cell13 = (
                        canonical_json_cell13(
                            slice_object_cell13
                        )
                    )

                    observed_slice_sha_cell13 = (
                        sha256_text_cell13(
                            canonical_slice_cell13
                        )
                    )

                    expected_slice_sha_cell13 = (
                        recorded_slice_hash_cell13(
                            record_cell13
                        )
                    )

                    if (
                        expected_slice_sha_cell13
                        and observed_slice_sha_cell13
                        != expected_slice_sha_cell13
                    ):
                        raise RuntimeError(
                            "Canonical slice hash mismatch: "
                            f"expected "
                            f"{expected_slice_sha_cell13}, "
                            f"observed "
                            f"{observed_slice_sha_cell13}."
                        )

                    render_cell13 = (
                        render_prompt_cell13(
                            operation_id_cell13,
                            canonical_slice_cell13,
                        )
                    )

                    rendered_prompt_cell13 = (
                        render_cell13[
                            "rendered_prompt"
                        ]
                    )

                    token_ids_cell13 = (
                        render_cell13[
                            "token_ids"
                        ]
                    )

                    prompt_token_count_cell13 = len(
                        token_ids_cell13
                    )

                    within_limit_cell13 = (
                        prompt_token_count_cell13
                        <= INPUT_TOKEN_LIMIT_CELL13
                    )

                    result_status_cell13 = (
                        "PASS_WITHIN_INPUT_LIMIT"
                        if within_limit_cell13
                        else
                        "EXCEEDS_RENDERED_INPUT_LIMIT"
                    )

                    rendered_prompt_sha_cell13 = (
                        sha256_text_cell13(
                            rendered_prompt_cell13
                        )
                    )

                    token_ids_hash_cell13 = (
                        token_ids_sha256_cell13(
                            token_ids_cell13
                        )
                    )

                    result_cell13 = {
                        **base_result_cell13,
                        "status":
                            result_status_cell13,
                        "diagnostic": None,
                        "boundary":
                            render_cell13["boundary"],
                        "slice_path": str(
                            slice_path_cell13
                        ),
                        "slice_sha256":
                            observed_slice_sha_cell13,
                        "rendered_prompt_bytes":
                            len(
                                rendered_prompt_cell13
                                .encode("utf-8")
                            ),
                        "rendered_prompt_tokens":
                            prompt_token_count_cell13,
                        "input_token_limit":
                            INPUT_TOKEN_LIMIT_CELL13,
                        "within_input_limit":
                            within_limit_cell13,
                        "excess_tokens": max(
                            0,
                            prompt_token_count_cell13
                            - INPUT_TOKEN_LIMIT_CELL13,
                        ),
                        "rendered_prompt_sha256":
                            rendered_prompt_sha_cell13,
                        "token_ids_sha256":
                            token_ids_hash_cell13,
                    }

                    payload_record_cell13 = {
                        "op_id":
                            operation_id_cell13,
                        "encoding": "utf-8",
                        "boundary":
                            render_cell13["boundary"],
                        "rendered_prompt":
                            rendered_prompt_cell13,
                        "rendered_prompt_sha256":
                            rendered_prompt_sha_cell13,
                        "token_ids":
                            token_ids_cell13,
                        "token_ids_sha256":
                            token_ids_hash_cell13,
                        "token_count":
                            prompt_token_count_cell13,
                    }

                    payload_handle_cell13.write(
                        json.dumps(
                            payload_record_cell13,
                            ensure_ascii=False,
                            separators=(",", ":"),
                            allow_nan=False,
                        )
                        + "\n"
                    )

                    payload_count_cell13 += 1
                    newly_measured_count_cell13 += 1

                except Exception as error:
                    result_cell13 = {
                        **base_result_cell13,
                        "status": "FAILED",
                        "diagnostic": repr(error),
                        "boundary": None,
                        "slice_sha256":
                            recorded_slice_hash_cell13(
                                record_cell13
                            ),
                        "rendered_prompt_bytes":
                            None,
                        "rendered_prompt_tokens":
                            None,
                        "input_token_limit":
                            INPUT_TOKEN_LIMIT_CELL13,
                        "within_input_limit":
                            None,
                        "excess_tokens": None,
                        "rendered_prompt_sha256":
                            None,
                        "token_ids_sha256": None,
                    }

                chunk_results_cell13.append(
                    result_cell13
                )

        os.replace(
            temporary_payload_path_cell13,
            paths_cell13["payload"],
        )

        payload_metadata_cell13 = {
            "chunk_number": chunk_number_cell13,
            "renderer_fingerprint":
                renderer_fingerprint_cell13,
            "operation_count": len(
                source_chunk_cell13
            ),
            "payload_record_count":
                payload_count_cell13,
            "operation_ids_sha256":
                sha256_text_cell13(
                    canonical_json_cell13(
                        expected_chunk_operation_ids_cell13
                    )
                ),
            "payload_path": str(
                paths_cell13["payload"]
            ),
            "payload_bytes":
                paths_cell13[
                    "payload"
                ].stat().st_size,
            "payload_sha256":
                sha256_file_cell13(
                    paths_cell13["payload"]
                ),
        }

        write_text_atomic_cell13(
            paths_cell13["summary"],
            "".join(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":"),
                    allow_nan=False,
                )
                + "\n"
                for record in chunk_results_cell13
            ),
        )

        write_json_atomic_cell13(
            paths_cell13["metadata"],
            payload_metadata_cell13,
        )

    all_results_cell13.extend(
        chunk_results_cell13
    )

    payload_index_cell13.append(
        payload_metadata_cell13
    )

    for result_cell13 in chunk_results_cell13:
        running_status_counts_cell13[
            result_cell13["status"]
        ] += 1

        if (
            result_cell13["status"]
            == "PASS_WITHIN_INPUT_LIMIT"
        ):
            running_within_limit_cell13 += 1

        elif (
            result_cell13["status"]
            == "EXCEEDS_RENDERED_INPUT_LIMIT"
        ):
            running_exceeds_limit_cell13 += 1

        elif result_cell13["status"] == "FAILED":
            running_failure_count_cell13 += 1

    processed_count_cell13 += len(
        chunk_results_cell13
    )

    if (
        processed_count_cell13 % 1000 == 0
        or processed_count_cell13
        == len(slice_records_cell13)
    ):
        elapsed_cell13 = (
            time.perf_counter()
            - start_time_cell13
        )

        free_storage_gib_cell13 = (
            shutil.disk_usage(
                PROJECT_ROOT_CELL13
            ).free
            / (1024 ** 3)
        )

        print(
            json.dumps(
                {
                    "processed":
                        processed_count_cell13,
                    "total":
                        len(slice_records_cell13),
                    "within_input_limit":
                        running_within_limit_cell13,
                    "exceeds_input_limit":
                        running_exceeds_limit_cell13,
                    "excluded":
                        running_status_counts_cell13[
                            "NOT_APPLICABLE_EXCLUDED_SLICE"
                        ],
                    "checkpoint_reused":
                        checkpoint_reused_count_cell13,
                    "newly_measured":
                        newly_measured_count_cell13,
                    "unexpected_failures":
                        running_failure_count_cell13,
                    "free_storage_gib": round(
                        free_storage_gib_cell13,
                        3,
                    ),
                    "elapsed_seconds": round(
                        elapsed_cell13,
                        1,
                    ),
                }
            )
        )


# ---------------------------------------------------------------------
# Write consolidated measurement and payload index
# ---------------------------------------------------------------------

if len(all_results_cell13) != len(
    slice_records_cell13
):
    raise RuntimeError(
        "Cell 13 did not account for every operation."
    )

write_text_atomic_cell13(
    PROMPT_MEASUREMENTS_PATH_CELL13,
    "".join(
        json.dumps(
            record,
            ensure_ascii=False,
            separators=(",", ":"),
            allow_nan=False,
        )
        + "\n"
        for record in all_results_cell13
    ),
)

write_text_atomic_cell13(
    PAYLOAD_INDEX_PATH_CELL13,
    "".join(
        json.dumps(
            record,
            ensure_ascii=False,
            separators=(",", ":"),
            allow_nan=False,
        )
        + "\n"
        for record in payload_index_cell13
    ),
)


# ---------------------------------------------------------------------
# Reconciliation and final report
# ---------------------------------------------------------------------

status_counts_cell13 = Counter(
    record["status"]
    for record in all_results_cell13
)

measured_records_cell13 = [
    record
    for record in all_results_cell13
    if record["status"] in {
        "PASS_WITHIN_INPUT_LIMIT",
        "EXCEEDS_RENDERED_INPUT_LIMIT",
    }
]

within_limit_records_cell13 = [
    record
    for record in measured_records_cell13
    if record["within_input_limit"] is True
]

over_limit_records_cell13 = [
    record
    for record in measured_records_cell13
    if record["within_input_limit"] is False
]

excluded_records_cell13 = [
    record
    for record in all_results_cell13
    if record["status"]
    == "NOT_APPLICABLE_EXCLUDED_SLICE"
]

failed_records_cell13 = [
    record
    for record in all_results_cell13
    if record["status"] == "FAILED"
]

prompt_token_counts_cell13 = [
    int(record["rendered_prompt_tokens"])
    for record in measured_records_cell13
]

if (
    len(measured_records_cell13)
    != expected_usable_count_cell13
):
    raise RuntimeError(
        "Measured-prompt count does not match the "
        "usable-slice count."
    )

if (
    len(excluded_records_cell13)
    != expected_exclusion_count_cell13
):
    raise RuntimeError(
        "Prompt exclusion count does not match the "
        "slice exclusion count."
    )

payload_record_count_cell13 = sum(
    int(record["payload_record_count"])
    for record in payload_index_cell13
)

payload_bytes_cell13 = sum(
    int(record["payload_bytes"])
    for record in payload_index_cell13
)

if (
    payload_record_count_cell13
    != len(measured_records_cell13)
):
    raise RuntimeError(
        "Exact prompt payload count does not match "
        "the measured-prompt count."
    )

duration_seconds_cell13 = (
    time.perf_counter()
    - start_time_cell13
)

report_cell13 = {
    "status": (
        "PASS"
        if not failed_records_cell13
        else "BLOCKED"
    ),
    "stage": "RENDERED_PROMPT_MEASUREMENT",
    "measurement_version": "1.0",
    "policy_version": "1.2",
    "renderer_configuration":
        renderer_configuration_cell13,
    "renderer_fingerprint":
        renderer_fingerprint_cell13,
    "development_prompt_reproduction": {
        "checks":
            development_checks_cell13,
        "operation_id":
            development_operation_id_cell13,
        "boundary":
            development_render_cell13[
                "boundary"
            ],
        "rendered_prompt_tokens": len(
            development_render_cell13[
                "token_ids"
            ]
        ),
        "rendered_prompt_sha256":
            development_rendered_sha_cell13,
        "token_ids_sha256":
            development_token_ids_sha_cell13,
    },
    "operation_count":
        len(all_results_cell13),
    "measured_operation_count":
        len(measured_records_cell13),
    "within_input_limit_count":
        len(within_limit_records_cell13),
    "exceeds_input_limit_count":
        len(over_limit_records_cell13),
    "excluded_slice_count":
        len(excluded_records_cell13),
    "unexpected_failure_count":
        len(failed_records_cell13),
    "status_counts": dict(
        sorted(status_counts_cell13.items())
    ),
    "input_token_limit":
        INPUT_TOKEN_LIMIT_CELL13,
    "rendered_prompt_token_statistics": {
        "minimum": min(
            prompt_token_counts_cell13
        ),
        "mean": round(
            mean(prompt_token_counts_cell13),
            3,
        ),
        "median": median(
            prompt_token_counts_cell13
        ),
        "maximum": max(
            prompt_token_counts_cell13
        ),
        "total": sum(
            prompt_token_counts_cell13
        ),
    },
    "largest_over_limit_examples": [
        {
            "op_id": record["op_id"],
            "spec_id": record.get("spec_id"),
            "method": record.get("method"),
            "path": record.get("path"),
            "rendered_prompt_tokens":
                record["rendered_prompt_tokens"],
            "excess_tokens":
                record["excess_tokens"],
        }
        for record in sorted(
            over_limit_records_cell13,
            key=lambda value: int(
                value["rendered_prompt_tokens"]
            ),
            reverse=True,
        )[:20]
    ],
    "checkpoint_count":
        len(payload_index_cell13),
    "checkpoint_reused_count":
        checkpoint_reused_count_cell13,
    "newly_measured_count":
        newly_measured_count_cell13,
    "exact_payload_storage": {
        "format":
            "gzip-compressed UTF-8 JSONL chunks",
        "contents": [
            "exact rendered prompt",
            "exact token ID sequence",
            "prompt and token hashes",
        ],
        "chunk_count":
            len(payload_index_cell13),
        "payload_record_count":
            payload_record_count_cell13,
        "compressed_bytes":
            payload_bytes_cell13,
        "index_path":
            str(PAYLOAD_INDEX_PATH_CELL13),
        "index_sha256":
            sha256_file_cell13(
                PAYLOAD_INDEX_PATH_CELL13
            ),
    },
    "measurements": {
        "path":
            str(PROMPT_MEASUREMENTS_PATH_CELL13),
        "bytes":
            PROMPT_MEASUREMENTS_PATH_CELL13
            .stat().st_size,
        "sha256":
            sha256_file_cell13(
                PROMPT_MEASUREMENTS_PATH_CELL13
            ),
    },
    "failure_examples":
        failed_records_cell13[:20],
    "duration_seconds": round(
        duration_seconds_cell13,
        3,
    ),
    "interpretation": (
        "Operations exceeding 5,120 rendered "
        "tokens are retained in the audit but "
        "are ineligible for experimental generation "
        "under the frozen input limit."
    ),
}

write_json_atomic_cell13(
    REPORT_PATH_CELL13,
    report_cell13,
)

report_cell13["report_path"] = str(
    REPORT_PATH_CELL13
)

report_cell13["report_sha256"] = (
    sha256_file_cell13(
        REPORT_PATH_CELL13
    )
)

print(
    json.dumps(
        report_cell13,
        indent=2,
        ensure_ascii=False,
    )
)

if failed_records_cell13:
    raise RuntimeError(
        "Rendered-prompt measurement had "
        f"{len(failed_records_cell13)} unexpected "
        "failure(s). Review the report."
    )

print("\nNOTEBOOK 02 — CELL 13: PASS")

{"processed": 1000, "total": 58585, "within_input_limit": 760, "exceeds_input_limit": 240, "excluded": 0, "checkpoint_reused": 0, "newly_measured": 1000, "unexpected_failures": 0, "free_storage_gib": 16.107, "elapsed_seconds": 21.9}
{"processed": 2000, "total": 58585, "within_input_limit": 1623, "exceeds_input_limit": 377, "excluded": 0, "checkpoint_reused": 0, "newly_measured": 2000, "unexpected_failures": 0, "free_storage_gib": 16.101, "elapsed_seconds": 33.4}
{"processed": 3000, "total": 58585, "within_input_limit": 2554, "exceeds_input_limit": 446, "excluded": 0, "checkpoint_reused": 0, "newly_measured": 3000, "unexpected_failures": 0, "free_storage_gib": 16.098, "elapsed_seconds": 41.1}
{"processed": 4000, "total": 58585, "within_input_limit": 3514, "exceeds_input_limit": 486, "excluded": 0, "checkpoint_reused": 0, "newly_measured": 4000, "unexpected_failures": 0, "free_storage_gib": 16.095, "elapsed_seconds": 48.2}
{"processed": 5000, "total": 58585, "within_input_limit": 4464, "

In [14]:
# Notebook 02 — Cell 14
# Build and reconcile the operation-level audit table.

from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import unquote
import csv
import hashlib
import json
import os
import time


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL14 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

OPERATION_INVENTORY_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/inventory/operations_enumerated.jsonl"
)

SLICE_RECORDS_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/slice_records.jsonl"
)

PROSE_MEASUREMENTS_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/prose_measurements.jsonl"
)

PROSE_FIELDS_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/prose_fields.csv"
)

PROMPT_MEASUREMENTS_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/rendered_prompt_measurements.jsonl"
)

REFERENCE_RELOCATIONS_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/reference_relocations.jsonl"
)

PROSE_REPORT_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "reports/audit/prose_measurement.json"
)

PROMPT_REPORT_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "reports/audit/rendered_prompt_measurement.json"
)

SLICE_REPORT_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "reports/audit/full_slice_construction.json"
)

OPERATIONS_CSV_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "data/audits/operations.csv"
)

REPORT_PATH_CELL14 = (
    PROJECT_ROOT_CELL14
    / "reports/audit/operation_audit_summary.json"
)

OPERATIONS_CSV_PATH_CELL14.parent.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_PATH_CELL14.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------

def sha256_file_cell14(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sha256_text_cell14(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def compact_json_cell14(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=False,
    )


def write_json_atomic_cell14(path, value):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def read_jsonl_map_cell14(path):
    records = {}

    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line_number, line in enumerate(
            handle,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except Exception as error:
                raise RuntimeError(
                    f"Invalid JSONL at {path}, "
                    f"line {line_number}: {error!r}"
                ) from error

            operation_id = record.get("op_id")

            if operation_id is None:
                raise RuntimeError(
                    f"Missing op_id at {path}, "
                    f"line {line_number}."
                )

            operation_id = str(operation_id)

            if operation_id in records:
                raise RuntimeError(
                    f"Duplicate op_id {operation_id} "
                    f"in {path}."
                )

            records[operation_id] = record

    return records


def first_value_cell14(*values):
    for value in values:
        if value is not None:
            return value

    return None


def find_nested_value_cell14(
    value,
    candidate_keys,
):
    if not isinstance(value, dict):
        return None

    for key in candidate_keys:
        if key in value and value[key] is not None:
            return value[key]

    for nested_value in value.values():
        if isinstance(nested_value, dict):
            result = find_nested_value_cell14(
                nested_value,
                candidate_keys,
            )

            if result is not None:
                return result

    return None


def integer_value_cell14(
    record,
    candidate_keys,
):
    value = find_nested_value_cell14(
        record,
        candidate_keys,
    )

    if value is None:
        return None

    try:
        return int(value)
    except Exception:
        return None


def boolean_csv_cell14(value):
    if value is None:
        return ""

    return "true" if bool(value) else "false"


# ---------------------------------------------------------------------
# Validate inputs
# ---------------------------------------------------------------------

required_paths_cell14 = [
    OPERATION_INVENTORY_PATH_CELL14,
    SLICE_RECORDS_PATH_CELL14,
    PROSE_MEASUREMENTS_PATH_CELL14,
    PROSE_FIELDS_PATH_CELL14,
    PROMPT_MEASUREMENTS_PATH_CELL14,
    PROSE_REPORT_PATH_CELL14,
    PROMPT_REPORT_PATH_CELL14,
    SLICE_REPORT_PATH_CELL14,
]

missing_paths_cell14 = [
    str(path)
    for path in required_paths_cell14
    if not path.is_file()
]

if missing_paths_cell14:
    raise RuntimeError(
        "Missing required Cell 14 inputs:\n"
        + "\n".join(missing_paths_cell14)
    )


# ---------------------------------------------------------------------
# Load operation-level inputs
# ---------------------------------------------------------------------

operation_map_cell14 = read_jsonl_map_cell14(
    OPERATION_INVENTORY_PATH_CELL14
)

slice_map_cell14 = read_jsonl_map_cell14(
    SLICE_RECORDS_PATH_CELL14
)

prose_map_cell14 = read_jsonl_map_cell14(
    PROSE_MEASUREMENTS_PATH_CELL14
)

prompt_map_cell14 = read_jsonl_map_cell14(
    PROMPT_MEASUREMENTS_PATH_CELL14
)

slice_report_cell14 = json.loads(
    SLICE_REPORT_PATH_CELL14.read_text(
        encoding="utf-8"
    )
)

prose_report_cell14 = json.loads(
    PROSE_REPORT_PATH_CELL14.read_text(
        encoding="utf-8"
    )
)

prompt_report_cell14 = json.loads(
    PROMPT_REPORT_PATH_CELL14.read_text(
        encoding="utf-8"
    )
)

expected_operation_count_cell14 = int(
    slice_report_cell14["operation_count"]
)

input_counts_cell14 = {
    "operation_inventory":
        len(operation_map_cell14),
    "slice_records":
        len(slice_map_cell14),
    "prose_measurements":
        len(prose_map_cell14),
    "prompt_measurements":
        len(prompt_map_cell14),
}

if any(
    value != expected_operation_count_cell14
    for value in input_counts_cell14.values()
):
    raise RuntimeError(
        "Operation-level input counts do not agree:\n"
        + json.dumps(
            input_counts_cell14,
            indent=2,
        )
    )

operation_id_sets_cell14 = [
    set(operation_map_cell14),
    set(slice_map_cell14),
    set(prose_map_cell14),
    set(prompt_map_cell14),
]

if not all(
    operation_id_sets_cell14[0] == value
    for value in operation_id_sets_cell14[1:]
):
    raise RuntimeError(
        "Operation ID sets differ across Cell 14 inputs."
    )

ordered_operation_ids_cell14 = list(
    operation_map_cell14
)


# ---------------------------------------------------------------------
# Correctly interpret prose-measurement status
# ---------------------------------------------------------------------

def get_prose_status_cell14(record):
    for key in [
        "measurement_status",
        "prose_measurement_status",
        "status",
    ]:
        value = record.get(key)

        if value is None:
            continue

        normalized = str(value).strip().upper()

        if normalized in {
            "PASS",
            "MEASUREMENT_PASS",
        }:
            return "PASS"

        return normalized

    measured_tokens = integer_value_cell14(
        record,
        [
            "prose_tokens_L0",
            "prose_token_count",
            "total_prose_tokens",
        ],
    )

    if measured_tokens is not None:
        return "PASS"

    return "UNKNOWN"


def prose_values_from_record_cell14(record):
    return {
        "summary_count":
            integer_value_cell14(
                record,
                [
                    "summary_annotation_count",
                    "summary_count",
                    "included_summary_count",
                ],
            ),
        "description_count":
            integer_value_cell14(
                record,
                [
                    "description_annotation_count",
                    "description_count",
                    "included_description_count",
                ],
            ),
        "annotation_count":
            integer_value_cell14(
                record,
                [
                    "included_annotation_count",
                    "annotation_count",
                    "prose_annotation_count",
                ],
            ),
        "prose_tokens":
            integer_value_cell14(
                record,
                [
                    "prose_tokens_L0",
                    "prose_token_count",
                    "total_prose_tokens",
                ],
            ),
    }


sample_usable_prose_cell14 = next(
    (
        record
        for record in prose_map_cell14.values()
        if get_prose_status_cell14(record)
        == "PASS"
    ),
    None,
)

if sample_usable_prose_cell14 is None:
    sample_keys_cell14 = sorted(
        next(iter(prose_map_cell14.values())).keys()
    )

    raise RuntimeError(
        "No usable prose-measurement record found. "
        f"Observed record keys: {sample_keys_cell14}"
    )

sample_prose_values_cell14 = (
    prose_values_from_record_cell14(
        sample_usable_prose_cell14
    )
)

use_prose_field_fallback_cell14 = any(
    value is None
    for value in sample_prose_values_cell14.values()
)


# ---------------------------------------------------------------------
# Aggregate prose_fields.csv only when needed
# ---------------------------------------------------------------------

prose_field_aggregates_cell14 = defaultdict(
    lambda: {
        "summary_count": 0,
        "description_count": 0,
        "annotation_count": 0,
        "prose_tokens": 0,
    }
)

if use_prose_field_fallback_cell14:
    print(
        "Aggregating prose_fields.csv because "
        "the compact measurement records do not "
        "contain every required count."
    )

    with PROSE_FIELDS_PATH_CELL14.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        required_columns_cell14 = {
            "op_id",
            "field_name",
            "token_count",
            "included_in_grouping",
        }

        missing_columns_cell14 = (
            required_columns_cell14
            - set(reader.fieldnames or [])
        )

        if missing_columns_cell14:
            raise RuntimeError(
                "prose_fields.csv is missing columns: "
                + ", ".join(
                    sorted(missing_columns_cell14)
                )
            )

        for row_number_cell14, row_cell14 in enumerate(
            reader,
            start=1,
        ):
            included_value_cell14 = str(
                row_cell14[
                    "included_in_grouping"
                ]
            ).strip().lower()

            if included_value_cell14 not in {
                "true",
                "1",
                "yes",
            }:
                continue

            operation_id_cell14 = str(
                row_cell14["op_id"]
            )

            field_name_cell14 = str(
                row_cell14["field_name"]
            )

            token_count_cell14 = int(
                row_cell14["token_count"]
            )

            aggregate_cell14 = (
                prose_field_aggregates_cell14[
                    operation_id_cell14
                ]
            )

            aggregate_cell14[
                "annotation_count"
            ] += 1

            aggregate_cell14[
                "prose_tokens"
            ] += token_count_cell14

            if field_name_cell14 == "summary":
                aggregate_cell14[
                    "summary_count"
                ] += 1

            elif field_name_cell14 == "description":
                aggregate_cell14[
                    "description_count"
                ] += 1

            if row_number_cell14 % 500000 == 0:
                print(
                    json.dumps(
                        {
                            "prose_field_rows_read":
                                row_number_cell14
                        }
                    )
                )


# ---------------------------------------------------------------------
# Reference relocation counts
# ---------------------------------------------------------------------

relocation_counts_cell14 = Counter()

if REFERENCE_RELOCATIONS_PATH_CELL14.is_file():
    with REFERENCE_RELOCATIONS_PATH_CELL14.open(
        "r",
        encoding="utf-8",
    ) as handle:
        for line in handle:
            if not line.strip():
                continue

            record = json.loads(line)
            operation_id = record.get("op_id")

            if operation_id is not None:
                relocation_counts_cell14[
                    str(operation_id)
                ] += 1


# ---------------------------------------------------------------------
# JSON Pointer and reference helpers
# ---------------------------------------------------------------------

def decode_pointer_token_cell14(token):
    return unquote(token).replace(
        "~1",
        "/",
    ).replace(
        "~0",
        "~",
    )


def resolve_local_reference_cell14(
    document,
    reference,
):
    if not isinstance(reference, str):
        return None

    if not reference.startswith("#"):
        return None

    fragment = unquote(reference[1:])

    if fragment == "":
        return document

    if not fragment.startswith("/"):
        return None

    current = document

    for raw_token in fragment.split("/")[1:]:
        token = decode_pointer_token_cell14(
            raw_token
        )

        if isinstance(current, dict):
            if token not in current:
                return None

            current = current[token]

        elif isinstance(current, list):
            try:
                index = int(token)
            except Exception:
                return None

            if not 0 <= index < len(current):
                return None

            current = current[index]

        else:
            return None

    return current


def dereference_cell14(document, value):
    current = value
    seen = set()

    for _ in range(50):
        if not isinstance(current, dict):
            return current

        reference = current.get("$ref")

        if not isinstance(reference, str):
            return current

        if reference in seen:
            return current

        seen.add(reference)

        target = resolve_local_reference_cell14(
            document,
            reference,
        )

        if target is None:
            return current

        current = target

    return current


def count_references_cell14(value):
    if isinstance(value, dict):
        count = (
            1
            if isinstance(value.get("$ref"), str)
            else 0
        )

        return count + sum(
            count_references_cell14(child)
            for child in value.values()
        )

    if isinstance(value, list):
        return sum(
            count_references_cell14(child)
            for child in value
        )

    return 0


# ---------------------------------------------------------------------
# Locate the operation inside each L0 slice
# ---------------------------------------------------------------------

HTTP_METHODS_CELL14 = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


def locate_operation_cell14(
    document,
    expected_path,
    expected_method,
):
    paths = document.get("paths", {})

    if not isinstance(paths, dict):
        raise RuntimeError(
            "Slice has no paths mapping."
        )

    expected_method = (
        str(expected_method).lower()
        if expected_method is not None
        else None
    )

    if (
        expected_path in paths
        and isinstance(paths[expected_path], dict)
        and expected_method
        in paths[expected_path]
    ):
        path_item = paths[expected_path]
        operation = path_item[expected_method]

        if not isinstance(operation, dict):
            raise RuntimeError(
                "Selected operation is not a mapping."
            )

        return (
            expected_path,
            expected_method,
            path_item,
            operation,
        )

    candidates = []

    for path_value, path_item in paths.items():
        if not isinstance(path_item, dict):
            continue

        for method, operation in path_item.items():
            method = str(method).lower()

            if (
                method in HTTP_METHODS_CELL14
                and isinstance(operation, dict)
            ):
                candidates.append(
                    (
                        path_value,
                        method,
                        path_item,
                        operation,
                    )
                )

    if len(candidates) != 1:
        raise RuntimeError(
            "Expected exactly one operation in the "
            f"L0 slice; observed {len(candidates)}."
        )

    return candidates[0]


# ---------------------------------------------------------------------
# Parameter and schema analysis
# ---------------------------------------------------------------------

def effective_parameters_cell14(
    document,
    path_item,
    operation,
):
    effective = {}

    for parameter_list in [
        path_item.get("parameters", []),
        operation.get("parameters", []),
    ]:
        if not isinstance(parameter_list, list):
            continue

        for parameter_value in parameter_list:
            parameter = dereference_cell14(
                document,
                parameter_value,
            )

            if not isinstance(parameter, dict):
                continue

            name = parameter.get("name")
            location = parameter.get("in")

            if name is None or location is None:
                continue

            effective[
                (
                    str(name),
                    str(location).lower(),
                )
            ] = parameter

    return list(effective.values())


def collect_schema_roots_cell14(document):
    roots = []
    seen_ids = set()

    def add_root(value):
        if not isinstance(value, dict):
            return

        object_id = id(value)

        if object_id in seen_ids:
            return

        seen_ids.add(object_id)
        roots.append(value)

    components = document.get(
        "components",
        {},
    )

    if isinstance(components, dict):
        schemas = components.get(
            "schemas",
            {},
        )

        if isinstance(schemas, dict):
            for schema in schemas.values():
                add_root(schema)

    def walk(value):
        if isinstance(value, dict):
            for key, nested_value in value.items():
                if (
                    key == "schema"
                    and isinstance(
                        nested_value,
                        dict,
                    )
                ):
                    add_root(nested_value)

                walk(nested_value)

        elif isinstance(value, list):
            for nested_value in value:
                walk(nested_value)

    walk(document)

    return roots


def schema_metrics_cell14(document):
    metrics = {
        "schema_object_count": 0,
        "schema_property_count": 0,
        "required_property_count": 0,
        "maximum_schema_depth": 0,
        "array_schema_count": 0,
        "allof_occurrence_count": 0,
        "anyof_occurrence_count": 0,
        "oneof_occurrence_count": 0,
        "not_occurrence_count": 0,
    }

    visited = set()

    def visit(schema, depth):
        if not isinstance(schema, dict):
            return

        object_id = id(schema)

        if object_id in visited:
            return

        visited.add(object_id)

        metrics["schema_object_count"] += 1
        metrics["maximum_schema_depth"] = max(
            metrics["maximum_schema_depth"],
            depth,
        )

        properties = schema.get("properties")

        if isinstance(properties, dict):
            metrics[
                "schema_property_count"
            ] += len(properties)

            for child in properties.values():
                visit(child, depth + 1)

        required = schema.get("required")

        if isinstance(required, list):
            metrics[
                "required_property_count"
            ] += len(required)

        schema_type = schema.get("type")

        if (
            schema_type == "array"
            or (
                isinstance(schema_type, list)
                and "array" in schema_type
            )
        ):
            metrics["array_schema_count"] += 1

        items = schema.get("items")

        if isinstance(items, dict):
            visit(items, depth + 1)

        prefix_items = schema.get("prefixItems")

        if isinstance(prefix_items, list):
            for child in prefix_items:
                visit(child, depth + 1)

        for keyword, metric_name in [
            (
                "allOf",
                "allof_occurrence_count",
            ),
            (
                "anyOf",
                "anyof_occurrence_count",
            ),
            (
                "oneOf",
                "oneof_occurrence_count",
            ),
        ]:
            children = schema.get(keyword)

            if isinstance(children, list):
                metrics[metric_name] += len(children)

                for child in children:
                    visit(child, depth + 1)

        not_schema = schema.get("not")

        if isinstance(not_schema, dict):
            metrics["not_occurrence_count"] += 1
            visit(not_schema, depth + 1)

        for keyword in [
            "additionalProperties",
            "contains",
            "propertyNames",
            "unevaluatedProperties",
            "unevaluatedItems",
            "if",
            "then",
            "else",
        ]:
            child = schema.get(keyword)

            if isinstance(child, dict):
                visit(child, depth + 1)

        dependent_schemas = schema.get(
            "dependentSchemas"
        )

        if isinstance(dependent_schemas, dict):
            for child in dependent_schemas.values():
                visit(child, depth + 1)

    for root in collect_schema_roots_cell14(
        document
    ):
        visit(root, 1)

    return metrics


# ---------------------------------------------------------------------
# Extract operation-level features
# ---------------------------------------------------------------------

def extract_operation_features_cell14(
    document,
    expected_path,
    expected_method,
):
    (
        observed_path,
        observed_method,
        path_item,
        operation,
    ) = locate_operation_cell14(
        document,
        expected_path,
        expected_method,
    )

    parameters = effective_parameters_cell14(
        document,
        path_item,
        operation,
    )

    location_counts = Counter()
    required_location_counts = Counter()

    for parameter in parameters:
        location = str(
            parameter.get("in", "unknown")
        ).lower()

        location_counts[location] += 1

        if (
            parameter.get("required") is True
            or location == "path"
        ):
            required_location_counts[
                location
            ] += 1

    request_body = dereference_cell14(
        document,
        operation.get("requestBody"),
    )

    request_body_present = isinstance(
        request_body,
        dict,
    )

    request_media_types = []

    if request_body_present:
        request_content = request_body.get(
            "content",
            {},
        )

        if isinstance(request_content, dict):
            request_media_types = sorted(
                str(value)
                for value in request_content
            )

    responses = operation.get(
        "responses",
        {},
    )

    if not isinstance(responses, dict):
        responses = {}

    response_codes = sorted(
        str(value)
        for value in responses
    )

    response_media_types = set()

    for response_value in responses.values():
        response = dereference_cell14(
            document,
            response_value,
        )

        if not isinstance(response, dict):
            continue

        response_content = response.get(
            "content",
            {},
        )

        if isinstance(response_content, dict):
            response_media_types.update(
                str(value)
                for value in response_content
            )

    effective_security = (
        operation["security"]
        if "security" in operation
        else document.get("security", [])
    )

    if not isinstance(effective_security, list):
        effective_security = []

    security_scheme_names = set()

    for requirement in effective_security:
        if isinstance(requirement, dict):
            security_scheme_names.update(
                str(value)
                for value in requirement
            )

    components = document.get(
        "components",
        {},
    )

    security_schemes = (
        components.get(
            "securitySchemes",
            {},
        )
        if isinstance(components, dict)
        else {}
    )

    if not isinstance(security_schemes, dict):
        security_schemes = {}

    security_types = set()
    http_security_schemes = set()

    for name in security_scheme_names:
        scheme = dereference_cell14(
            document,
            security_schemes.get(name),
        )

        if not isinstance(scheme, dict):
            security_types.add("unresolved")
            continue

        scheme_type = str(
            scheme.get("type", "unknown")
        ).lower()

        security_types.add(scheme_type)

        if scheme_type == "http":
            http_scheme = scheme.get("scheme")

            if http_scheme is not None:
                http_security_schemes.add(
                    str(http_scheme).lower()
                )

    metrics = schema_metrics_cell14(document)

    metrics.update(
        {
            "observed_path": observed_path,
            "observed_method": observed_method,
            "source_operation_id":
                operation.get("operationId"),
            "operation_has_summary":
                isinstance(
                    operation.get("summary"),
                    str,
                ),
            "operation_has_description":
                isinstance(
                    operation.get("description"),
                    str,
                ),
            "parameter_count":
                len(parameters),
            "path_parameter_count":
                location_counts["path"],
            "query_parameter_count":
                location_counts["query"],
            "header_parameter_count":
                location_counts["header"],
            "cookie_parameter_count":
                location_counts["cookie"],
            "other_parameter_count":
                sum(
                    count
                    for location, count
                    in location_counts.items()
                    if location not in {
                        "path",
                        "query",
                        "header",
                        "cookie",
                    }
                ),
            "required_parameter_count":
                sum(
                    required_location_counts.values()
                ),
            "required_path_parameter_count":
                required_location_counts["path"],
            "required_query_parameter_count":
                required_location_counts["query"],
            "required_header_parameter_count":
                required_location_counts["header"],
            "required_cookie_parameter_count":
                required_location_counts["cookie"],
            "request_body_present":
                request_body_present,
            "request_body_required": (
                request_body.get("required") is True
                if request_body_present
                else False
            ),
            "request_media_type_count":
                len(request_media_types),
            "request_media_types":
                request_media_types,
            "response_entry_count":
                len(responses),
            "response_codes":
                response_codes,
            "response_media_type_count":
                len(response_media_types),
            "response_media_types":
                sorted(response_media_types),
            "security_requirement_count":
                len(effective_security),
            "security_scheme_count":
                len(security_scheme_names),
            "security_scheme_names":
                sorted(security_scheme_names),
            "security_types":
                sorted(security_types),
            "http_security_schemes":
                sorted(http_security_schemes),
            "reference_count":
                count_references_cell14(document),
        }
    )

    return metrics


# ---------------------------------------------------------------------
# Output table definition
# ---------------------------------------------------------------------

fieldnames_cell14 = [
    "op_id",
    "spec_id",
    "raw_id",
    "family_id",
    "provider_directory",
    "relative_path",
    "source_class",
    "dialect",
    "canonical_version",
    "method",
    "path",
    "source_operation_id",
    "audit_status",
    "audit_diagnostic",
    "slice_status",
    "slice_sha256",
    "slice_bytes",
    "slice_validation_pass",
    "reference_count",
    "reference_relocation_count",
    "reference_issue_count",
    "operation_has_summary",
    "operation_has_description",
    "parameter_count",
    "path_parameter_count",
    "query_parameter_count",
    "header_parameter_count",
    "cookie_parameter_count",
    "other_parameter_count",
    "required_parameter_count",
    "required_path_parameter_count",
    "required_query_parameter_count",
    "required_header_parameter_count",
    "required_cookie_parameter_count",
    "request_body_present",
    "request_body_required",
    "request_media_type_count",
    "request_media_types",
    "response_entry_count",
    "response_codes",
    "response_media_type_count",
    "response_media_types",
    "security_requirement_count",
    "security_scheme_count",
    "security_scheme_names",
    "security_types",
    "http_security_schemes",
    "schema_object_count",
    "schema_property_count",
    "required_property_count",
    "maximum_schema_depth",
    "array_schema_count",
    "allof_occurrence_count",
    "anyof_occurrence_count",
    "oneof_occurrence_count",
    "not_occurrence_count",
    "summary_annotation_count",
    "description_annotation_count",
    "included_annotation_count",
    "prose_tokens_L0",
    "zero_prose",
    "prompt_status",
    "rendered_prompt_tokens",
    "rendered_prompt_bytes",
    "input_token_limit",
    "within_input_limit",
    "excess_prompt_tokens",
    "rendered_prompt_sha256",
    "token_ids_sha256",
    "eligible_for_experimental_generation",
    "experimental_exclusion_reason",
]


# ---------------------------------------------------------------------
# Build operations.csv
# ---------------------------------------------------------------------

temporary_operations_path_cell14 = Path(
    str(OPERATIONS_CSV_PATH_CELL14) + ".tmp"
)

start_time_cell14 = time.perf_counter()

method_counts_cell14 = Counter()
source_class_counts_cell14 = Counter()
dialect_counts_cell14 = Counter()
eligibility_counts_cell14 = Counter()
exclusion_reason_counts_cell14 = Counter()
eligible_source_class_counts_cell14 = Counter()

audit_failure_count_cell14 = 0
audit_failure_examples_cell14 = []

total_prose_tokens_cell14 = 0
total_summary_annotations_cell14 = 0
total_description_annotations_cell14 = 0
total_annotations_cell14 = 0
total_rendered_prompt_tokens_cell14 = 0

with temporary_operations_path_cell14.open(
    "w",
    encoding="utf-8",
    newline="",
) as output_handle_cell14:
    writer_cell14 = csv.DictWriter(
        output_handle_cell14,
        fieldnames=fieldnames_cell14,
        extrasaction="ignore",
        lineterminator="\n",
    )

    writer_cell14.writeheader()

    for index_cell14, operation_id_cell14 in enumerate(
        ordered_operation_ids_cell14,
        start=1,
    ):
        source_record_cell14 = (
            operation_map_cell14[
                operation_id_cell14
            ]
        )

        slice_record_cell14 = (
            slice_map_cell14[
                operation_id_cell14
            ]
        )

        prose_record_cell14 = (
            prose_map_cell14[
                operation_id_cell14
            ]
        )

        prompt_record_cell14 = (
            prompt_map_cell14[
                operation_id_cell14
            ]
        )

        method_cell14 = str(
            first_value_cell14(
                source_record_cell14.get("method"),
                slice_record_cell14.get("method"),
                prompt_record_cell14.get("method"),
                "",
            )
        ).lower()

        path_cell14 = first_value_cell14(
            source_record_cell14.get("path"),
            slice_record_cell14.get("path"),
            prompt_record_cell14.get("path"),
        )

        spec_id_cell14 = first_value_cell14(
            source_record_cell14.get("spec_id"),
            slice_record_cell14.get("spec_id"),
            prompt_record_cell14.get("spec_id"),
        )

        source_class_cell14 = first_value_cell14(
            source_record_cell14.get(
                "source_class"
            ),
            slice_record_cell14.get(
                "source_class"
            ),
            prompt_record_cell14.get(
                "source_class"
            ),
        )

        dialect_cell14 = first_value_cell14(
            source_record_cell14.get("dialect"),
            slice_record_cell14.get("dialect"),
        )

        slice_status_cell14 = str(
            slice_record_cell14.get(
                "status",
                "UNKNOWN",
            )
        ).upper()

        prompt_status_cell14 = str(
            prompt_record_cell14.get(
                "status",
                "UNKNOWN",
            )
        ).upper()

        prose_record_status_cell14 = (
            get_prose_status_cell14(
                prose_record_cell14
            )
        )

        row_cell14 = {
            "op_id": operation_id_cell14,
            "spec_id": spec_id_cell14,
            "family_id": source_record_cell14["family_id"],
            "raw_id":
                source_record_cell14.get(
                    "raw_id"
                ),
            "provider_directory":
                source_record_cell14.get(
                    "provider_directory"
                ),
            "relative_path":
                source_record_cell14.get(
                    "relative_path"
                ),
            "source_class":
                source_class_cell14,
            "dialect": dialect_cell14,
            "canonical_version":
                first_value_cell14(
                    source_record_cell14.get(
                        "canonical_version"
                    ),
                    source_record_cell14.get(
                        "openapi_version"
                    ),
                ),
            "method": method_cell14,
            "path": path_cell14,
            "source_operation_id":
                first_value_cell14(
                    source_record_cell14.get(
                        "source_operation_id"
                    ),
                    source_record_cell14.get(
                        "operationId"
                    ),
                    source_record_cell14.get(
                        "operation_id"
                    ),
                ),
            "slice_status": slice_status_cell14,
            "slice_sha256":
                first_value_cell14(
                    prompt_record_cell14.get(
                        "slice_sha256"
                    ),
                    slice_record_cell14.get(
                        "slice_sha256"
                    ),
                ),
            "slice_bytes":
                first_value_cell14(
                    slice_record_cell14.get(
                        "slice_bytes"
                    ),
                    slice_record_cell14.get(
                        "bytes"
                    ),
                ),
            "slice_validation_pass":
                boolean_csv_cell14(
                    slice_status_cell14 == "PASS"
                ),
            "reference_relocation_count":
                relocation_counts_cell14[
                    operation_id_cell14
                ],
            "reference_issue_count": len(
                slice_record_cell14.get(
                    "diagnostics",
                    [],
                )
                if isinstance(
                    slice_record_cell14.get(
                        "diagnostics",
                        [],
                    ),
                    list,
                )
                else []
            ),
            "prompt_status":
                prompt_status_cell14,
            "rendered_prompt_tokens":
                prompt_record_cell14.get(
                    "rendered_prompt_tokens"
                ),
            "rendered_prompt_bytes":
                prompt_record_cell14.get(
                    "rendered_prompt_bytes"
                ),
            "input_token_limit":
                prompt_record_cell14.get(
                    "input_token_limit"
                ),
            "within_input_limit":
                boolean_csv_cell14(
                    prompt_record_cell14.get(
                        "within_input_limit"
                    )
                ),
            "excess_prompt_tokens":
                prompt_record_cell14.get(
                    "excess_tokens"
                ),
            "rendered_prompt_sha256":
                prompt_record_cell14.get(
                    "rendered_prompt_sha256"
                ),
            "token_ids_sha256":
                prompt_record_cell14.get(
                    "token_ids_sha256"
                ),
        }

        try:
            if slice_status_cell14 == "PASS":
                slice_path_value_cell14 = (
                    prompt_record_cell14.get(
                        "slice_path"
                    )
                    or slice_record_cell14.get(
                        "slice_path"
                    )
                    or slice_record_cell14.get(
                        "slice_relative_path"
                    )
                )

                if not slice_path_value_cell14:
                    raise RuntimeError(
                        "No slice path was recorded."
                    )

                slice_path_cell14 = Path(
                    str(slice_path_value_cell14)
                )

                if not slice_path_cell14.is_absolute():
                    slice_path_cell14 = (
                        PROJECT_ROOT_CELL14
                        / slice_path_cell14
                    )

                slice_text_cell14 = (
                    slice_path_cell14.read_text(
                        encoding="utf-8"
                    )
                )

                observed_slice_sha_cell14 = (
                    sha256_text_cell14(
                        slice_text_cell14
                    )
                )

                expected_slice_sha_cell14 = (
                    row_cell14["slice_sha256"]
                )

                if (
                    expected_slice_sha_cell14
                    and observed_slice_sha_cell14
                    != expected_slice_sha_cell14
                ):
                    raise RuntimeError(
                        "Slice hash mismatch: expected "
                        f"{expected_slice_sha_cell14}, "
                        "observed "
                        f"{observed_slice_sha_cell14}."
                    )

                slice_document_cell14 = json.loads(
                    slice_text_cell14
                )

                features_cell14 = (
                    extract_operation_features_cell14(
                        slice_document_cell14,
                        path_cell14,
                        method_cell14,
                    )
                )

                row_cell14.update(features_cell14)

                row_cell14["slice_bytes"] = len(
                    slice_text_cell14.encode(
                        "utf-8"
                    )
                )

                if (
                    row_cell14[
                        "canonical_version"
                    ]
                    is None
                ):
                    row_cell14[
                        "canonical_version"
                    ] = slice_document_cell14.get(
                        "openapi"
                    )

                row_cell14["audit_status"] = "PASS"
                row_cell14["audit_diagnostic"] = ""

            else:
                row_cell14["audit_status"] = (
                    "NOT_APPLICABLE_EXCLUDED_SLICE"
                )

                row_cell14["audit_diagnostic"] = (
                    compact_json_cell14(
                        slice_record_cell14.get(
                            "diagnostics",
                            [],
                        )
                    )
                )

        except Exception as error:
            audit_failure_count_cell14 += 1

            row_cell14["audit_status"] = "FAILED"
            row_cell14[
                "audit_diagnostic"
            ] = repr(error)

            if len(
                audit_failure_examples_cell14
            ) < 20:
                audit_failure_examples_cell14.append(
                    {
                        "op_id":
                            operation_id_cell14,
                        "spec_id":
                            spec_id_cell14,
                        "method":
                            method_cell14,
                        "path":
                            path_cell14,
                        "diagnostic":
                            repr(error),
                    }
                )

        if use_prose_field_fallback_cell14:
            prose_values_cell14 = dict(
                prose_field_aggregates_cell14.get(
                    operation_id_cell14,
                    {
                        "summary_count": 0,
                        "description_count": 0,
                        "annotation_count": 0,
                        "prose_tokens": 0,
                    },
                )
            )
        else:
            prose_values_cell14 = (
                prose_values_from_record_cell14(
                    prose_record_cell14
                )
            )

        if prose_record_status_cell14 == "PASS":
            summary_count_cell14 = int(
                prose_values_cell14[
                    "summary_count"
                ]
                or 0
            )

            description_count_cell14 = int(
                prose_values_cell14[
                    "description_count"
                ]
                or 0
            )

            annotation_value_cell14 = (
                prose_values_cell14[
                    "annotation_count"
                ]
            )

            if annotation_value_cell14 is None:
                annotation_count_cell14 = (
                    summary_count_cell14
                    + description_count_cell14
                )
            else:
                annotation_count_cell14 = int(
                    annotation_value_cell14
                )

            prose_tokens_cell14 = int(
                prose_values_cell14[
                    "prose_tokens"
                ]
                or 0
            )

            row_cell14.update(
                {
                    "summary_annotation_count":
                        summary_count_cell14,
                    "description_annotation_count":
                        description_count_cell14,
                    "included_annotation_count":
                        annotation_count_cell14,
                    "prose_tokens_L0":
                        prose_tokens_cell14,
                    "zero_prose":
                        boolean_csv_cell14(
                            prose_tokens_cell14 == 0
                        ),
                }
            )

            total_summary_annotations_cell14 += (
                summary_count_cell14
            )

            total_description_annotations_cell14 += (
                description_count_cell14
            )

            total_annotations_cell14 += (
                annotation_count_cell14
            )

            total_prose_tokens_cell14 += (
                prose_tokens_cell14
            )

        else:
            row_cell14.update(
                {
                    "summary_annotation_count": "",
                    "description_annotation_count": "",
                    "included_annotation_count": "",
                    "prose_tokens_L0": "",
                    "zero_prose": "",
                }
            )

        within_input_limit_cell14 = (
            prompt_record_cell14.get(
                "within_input_limit"
            )
            is True
        )

        eligible_cell14 = (
            slice_status_cell14 == "PASS"
            and prompt_status_cell14
            == "PASS_WITHIN_INPUT_LIMIT"
            and within_input_limit_cell14
            and row_cell14["audit_status"]
            == "PASS"
            and prose_record_status_cell14
            == "PASS"
        )

        if eligible_cell14:
            exclusion_reason_cell14 = ""

            eligibility_counts_cell14[
                "ELIGIBLE"
            ] += 1

            eligible_source_class_counts_cell14[
                str(source_class_cell14)
            ] += 1

        elif slice_status_cell14 != "PASS":
            exclusion_reason_cell14 = (
                "EXCLUDED_INVALID_L0_SLICE"
            )

        elif (
            prompt_status_cell14
            == "EXCEEDS_RENDERED_INPUT_LIMIT"
        ):
            exclusion_reason_cell14 = (
                "EXCLUDED_RENDERED_PROMPT_TOO_LARGE"
            )

        elif row_cell14["audit_status"] == "FAILED":
            exclusion_reason_cell14 = (
                "EXCLUDED_OPERATION_AUDIT_FAILURE"
            )

        elif prose_record_status_cell14 != "PASS":
            exclusion_reason_cell14 = (
                "EXCLUDED_PROSE_MEASUREMENT_FAILURE"
            )

        else:
            exclusion_reason_cell14 = (
                "EXCLUDED_UNCLASSIFIED"
            )

        if not eligible_cell14:
            eligibility_counts_cell14[
                "INELIGIBLE"
            ] += 1

            exclusion_reason_counts_cell14[
                exclusion_reason_cell14
            ] += 1

        row_cell14[
            "eligible_for_experimental_generation"
        ] = boolean_csv_cell14(
            eligible_cell14
        )

        row_cell14[
            "experimental_exclusion_reason"
        ] = exclusion_reason_cell14

        for list_field_cell14 in [
            "request_media_types",
            "response_codes",
            "response_media_types",
            "security_scheme_names",
            "security_types",
            "http_security_schemes",
        ]:
            value_cell14 = row_cell14.get(
                list_field_cell14
            )

            if isinstance(value_cell14, list):
                row_cell14[
                    list_field_cell14
                ] = compact_json_cell14(
                    value_cell14
                )

        method_counts_cell14[
            method_cell14
        ] += 1

        source_class_counts_cell14[
            str(source_class_cell14)
        ] += 1

        dialect_counts_cell14[
            str(dialect_cell14)
        ] += 1

        rendered_tokens_cell14 = (
            prompt_record_cell14.get(
                "rendered_prompt_tokens"
            )
        )

        if rendered_tokens_cell14 is not None:
            total_rendered_prompt_tokens_cell14 += int(
                rendered_tokens_cell14
            )

        writer_cell14.writerow(row_cell14)

        if (
            index_cell14 % 1000 == 0
            or index_cell14
            == expected_operation_count_cell14
        ):
            print(
                json.dumps(
                    {
                        "processed": index_cell14,
                        "total":
                            expected_operation_count_cell14,
                        "eligible":
                            eligibility_counts_cell14[
                                "ELIGIBLE"
                            ],
                        "ineligible":
                            eligibility_counts_cell14[
                                "INELIGIBLE"
                            ],
                        "audit_failure_count":
                            audit_failure_count_cell14,
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - start_time_cell14,
                            1,
                        ),
                    }
                )
            )

os.replace(
    temporary_operations_path_cell14,
    OPERATIONS_CSV_PATH_CELL14,
)


# ---------------------------------------------------------------------
# Final reconciliation
# ---------------------------------------------------------------------

expected_prose_statistics_cell14 = (
    prose_report_cell14[
        "prose_token_statistics"
    ]
)

expected_prompt_statistics_cell14 = (
    prompt_report_cell14[
        "rendered_prompt_token_statistics"
    ]
)

expected_eligible_count_cell14 = int(
    prompt_report_cell14[
        "within_input_limit_count"
    ]
)

expected_prompt_exclusion_count_cell14 = int(
    prompt_report_cell14[
        "exceeds_input_limit_count"
    ]
)

expected_slice_exclusion_count_cell14 = int(
    prompt_report_cell14[
        "excluded_slice_count"
    ]
)

reconciliation_checks_cell14 = {
    "operation_count_matches": (
        sum(method_counts_cell14.values())
        == expected_operation_count_cell14
    ),
    "eligible_count_matches": (
        eligibility_counts_cell14["ELIGIBLE"]
        == expected_eligible_count_cell14
    ),
    "prompt_limit_exclusion_count_matches": (
        exclusion_reason_counts_cell14[
            "EXCLUDED_RENDERED_PROMPT_TOO_LARGE"
        ]
        == expected_prompt_exclusion_count_cell14
    ),
    "slice_exclusion_count_matches": (
        exclusion_reason_counts_cell14[
            "EXCLUDED_INVALID_L0_SLICE"
        ]
        == expected_slice_exclusion_count_cell14
    ),
    "prose_token_total_matches": (
        total_prose_tokens_cell14
        == int(
            expected_prose_statistics_cell14[
                "total"
            ]
        )
    ),
    "summary_annotation_total_matches": (
        total_summary_annotations_cell14
        == int(
            prose_report_cell14[
                "summary_annotation_count"
            ]
        )
    ),
    "description_annotation_total_matches": (
        total_description_annotations_cell14
        == int(
            prose_report_cell14[
                "description_annotation_count"
            ]
        )
    ),
    "included_annotation_total_matches": (
        total_annotations_cell14
        == int(
            prose_report_cell14[
                "included_annotation_count"
            ]
        )
    ),
    "rendered_prompt_token_total_matches": (
        total_rendered_prompt_tokens_cell14
        == int(
            expected_prompt_statistics_cell14[
                "total"
            ]
        )
    ),
    "no_operation_audit_failures": (
        audit_failure_count_cell14 == 0
    ),
}

status_cell14 = (
    "PASS"
    if all(
        reconciliation_checks_cell14.values()
    )
    else "BLOCKED"
)

duration_seconds_cell14 = (
    time.perf_counter()
    - start_time_cell14
)

report_cell14 = {
    "status": status_cell14,
    "stage": "OPERATION_LEVEL_AUDIT",
    "operation_count":
        expected_operation_count_cell14,
    "eligible_operation_count":
        eligibility_counts_cell14["ELIGIBLE"],
    "ineligible_operation_count":
        eligibility_counts_cell14[
            "INELIGIBLE"
        ],
    "eligibility_counts": dict(
        sorted(
            eligibility_counts_cell14.items()
        )
    ),
    "exclusion_reason_counts": dict(
        sorted(
            exclusion_reason_counts_cell14.items()
        )
    ),
    "eligible_source_class_counts": dict(
        sorted(
            eligible_source_class_counts_cell14.items()
        )
    ),
    "method_counts": dict(
        sorted(method_counts_cell14.items())
    ),
    "source_class_counts": dict(
        sorted(source_class_counts_cell14.items())
    ),
    "dialect_counts": dict(
        sorted(dialect_counts_cell14.items())
    ),
    "totals": {
        "summary_annotations":
            total_summary_annotations_cell14,
        "description_annotations":
            total_description_annotations_cell14,
        "included_annotations":
            total_annotations_cell14,
        "prose_tokens_L0":
            total_prose_tokens_cell14,
        "rendered_prompt_tokens":
            total_rendered_prompt_tokens_cell14,
        "reference_relocations":
            sum(
                relocation_counts_cell14.values()
            ),
    },
    "operation_audit_failure_count":
        audit_failure_count_cell14,
    "operation_audit_failure_examples":
        audit_failure_examples_cell14,
    "reconciliation_checks":
        reconciliation_checks_cell14,
    "operations_csv": {
        "path":
            str(OPERATIONS_CSV_PATH_CELL14),
        "bytes":
            OPERATIONS_CSV_PATH_CELL14
            .stat().st_size,
        "sha256":
            sha256_file_cell14(
                OPERATIONS_CSV_PATH_CELL14
            ),
        "row_count":
            expected_operation_count_cell14,
        "column_count":
            len(fieldnames_cell14),
        "columns":
            fieldnames_cell14,
    },
    "input_hashes": {
        "operation_inventory":
            sha256_file_cell14(
                OPERATION_INVENTORY_PATH_CELL14
            ),
        "slice_records":
            sha256_file_cell14(
                SLICE_RECORDS_PATH_CELL14
            ),
        "prose_measurements":
            sha256_file_cell14(
                PROSE_MEASUREMENTS_PATH_CELL14
            ),
        "prompt_measurements":
            sha256_file_cell14(
                PROMPT_MEASUREMENTS_PATH_CELL14
            ),
        "reference_relocations": (
            sha256_file_cell14(
                REFERENCE_RELOCATIONS_PATH_CELL14
            )
            if REFERENCE_RELOCATIONS_PATH_CELL14
            .is_file()
            else None
        ),
    },
    "duration_seconds": round(
        duration_seconds_cell14,
        3,
    ),
}

write_json_atomic_cell14(
    REPORT_PATH_CELL14,
    report_cell14,
)

report_cell14["report_path"] = str(
    REPORT_PATH_CELL14
)

report_cell14["report_sha256"] = (
    sha256_file_cell14(
        REPORT_PATH_CELL14
    )
)

print(
    json.dumps(
        report_cell14,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell14 != "PASS":
    raise RuntimeError(
        "Operation-level audit reconciliation failed. "
        "Review reconciliation_checks and failure "
        "examples above."
    )

print("\nNOTEBOOK 02 — CELL 14: PASS")

{"processed": 1000, "total": 58585, "eligible": 760, "ineligible": 240, "audit_failure_count": 0, "elapsed_seconds": 1.3}
{"processed": 2000, "total": 58585, "eligible": 1623, "ineligible": 377, "audit_failure_count": 0, "elapsed_seconds": 2.2}
{"processed": 3000, "total": 58585, "eligible": 2554, "ineligible": 446, "audit_failure_count": 0, "elapsed_seconds": 2.9}
{"processed": 4000, "total": 58585, "eligible": 3514, "ineligible": 486, "audit_failure_count": 0, "elapsed_seconds": 3.7}
{"processed": 5000, "total": 58585, "eligible": 4464, "ineligible": 536, "audit_failure_count": 0, "elapsed_seconds": 4.4}
{"processed": 6000, "total": 58585, "eligible": 5391, "ineligible": 609, "audit_failure_count": 0, "elapsed_seconds": 5.4}
{"processed": 7000, "total": 58585, "eligible": 6283, "ineligible": 717, "audit_failure_count": 0, "elapsed_seconds": 6.2}
{"processed": 8000, "total": 58585, "eligible": 7260, "ineligible": 740, "audit_failure_count": 0, "elapsed_seconds": 6.9}
{"processed": 900

In [16]:
# Notebook 02 — Cell 15
# Complete operation metadata from the frozen specification registry,
# then produce descriptive statistics and prose-token plots.
#
# This cell does not select prose strata or sampling thresholds.

from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import os
import time

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL15 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

OPERATIONS_CSV_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "data/audits/operations.csv"
)

SPECIFICATION_REGISTRY_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "data/inventory/eligible_specifications.jsonl"
)

OPERATION_AUDIT_REPORT_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/operation_audit_summary.json"
)

METADATA_COMPLETION_REPORT_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/operation_metadata_completion.json"
)

NUMERICAL_STATISTICS_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/numerical_operation_statistics.csv"
)

CATEGORICAL_STATISTICS_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/categorical_operation_statistics.csv"
)

DISTRIBUTION_REPORT_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/prose_distribution_summary.json"
)

FINAL_REPORT_PATH_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/audit/descriptive_statistics.json"
)

FIGURE_ROOT_CELL15 = (
    PROJECT_ROOT_CELL15
    / "reports/figures"
)

LINEAR_FIGURE_PATH_CELL15 = (
    FIGURE_ROOT_CELL15
    / "prose_tokens_distribution.png"
)

LOG_FIGURE_PATH_CELL15 = (
    FIGURE_ROOT_CELL15
    / "prose_tokens_distribution_log.png"
)

for directory_cell15 in [
    NUMERICAL_STATISTICS_PATH_CELL15.parent,
    FIGURE_ROOT_CELL15,
]:
    directory_cell15.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell15(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def write_json_atomic_cell15(path, value):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def missing_series_cell15(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(
            {
                "",
                "none",
                "nan",
                "null",
                "<na>",
            }
        )
    )


def boolean_series_cell15(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


def finite_or_none_cell15(value):
    if value is None:
        return None

    try:
        result = float(value)
    except Exception:
        return None

    if not math.isfinite(result):
        return None

    return result


# ---------------------------------------------------------------------
# Validate inputs
# ---------------------------------------------------------------------

PROMPT_REPORT_PATH_CELL15 = (
    PROJECT_ROOT_CELL15 / "reports/audit/rendered_prompt_measurement.json"
)
PROMPT_LEDGER_PATH_CELL15 = (
    PROJECT_ROOT_CELL15 / "data/audits/rendered_prompt_measurements.jsonl"
)

required_paths_cell15 = [
    PROMPT_REPORT_PATH_CELL15,
    PROMPT_LEDGER_PATH_CELL15,
    OPERATIONS_CSV_PATH_CELL15,
    SPECIFICATION_REGISTRY_PATH_CELL15,
    OPERATION_AUDIT_REPORT_PATH_CELL15,
]

missing_paths_cell15 = [
    str(path)
    for path in required_paths_cell15
    if not path.is_file()
]

if missing_paths_cell15:
    raise RuntimeError(
        "Missing required Cell 15 inputs:\n"
        + "\n".join(missing_paths_cell15)
    )


# ---------------------------------------------------------------------
# Load the frozen specification registry using its actual schema
# ---------------------------------------------------------------------

specification_metadata_cell15 = {}

with SPECIFICATION_REGISTRY_PATH_CELL15.open(
    "r",
    encoding="utf-8",
) as handle_cell15:
    for line_number_cell15, line_cell15 in enumerate(
        handle_cell15,
        start=1,
    ):
        if not line_cell15.strip():
            continue

        record_cell15 = json.loads(line_cell15)

        specification_id_cell15 = record_cell15.get(
            "spec_id"
        )

        if specification_id_cell15 is None:
            raise RuntimeError(
                "Missing spec_id in specification "
                f"registry line {line_number_cell15}."
            )

        specification_id_cell15 = str(
            specification_id_cell15
        )

        if (
            specification_id_cell15
            in specification_metadata_cell15
        ):
            raise RuntimeError(
                "Duplicate specification ID: "
                f"{specification_id_cell15}"
            )

        source_dialect_cell15 = first_value_cell15 = (
            record_cell15.get("source_dialect")
            if record_cell15.get("source_dialect")
            is not None
            else record_cell15.get("dialect")
        )

        canonical_version_cell15 = (
            record_cell15.get(
                "canonical_openapi_version"
            )
            if record_cell15.get(
                "canonical_openapi_version"
            )
            is not None
            else record_cell15.get(
                "canonical_version"
            )
        )

        specification_metadata_cell15[
            specification_id_cell15
        ] = {
            "source_dialect":
                source_dialect_cell15,
            "canonical_openapi_version":
                canonical_version_cell15,
            "source_class":
                record_cell15.get(
                    "source_class"
                ),
        }

if len(specification_metadata_cell15) != 1970:
    raise RuntimeError(
        "Expected 1,970 frozen specification "
        f"records; observed "
        f"{len(specification_metadata_cell15)}."
    )

missing_registry_dialect_cell15 = [
    specification_id
    for specification_id, metadata
    in specification_metadata_cell15.items()
    if metadata["source_dialect"] is None
]

missing_registry_version_cell15 = [
    specification_id
    for specification_id, metadata
    in specification_metadata_cell15.items()
    if metadata[
        "canonical_openapi_version"
    ] is None
]

if missing_registry_dialect_cell15:
    raise RuntimeError(
        "Frozen registry has missing source dialects: "
        + ", ".join(
            missing_registry_dialect_cell15[:20]
        )
    )

if missing_registry_version_cell15:
    raise RuntimeError(
        "Frozen registry has missing canonical versions: "
        + ", ".join(
            missing_registry_version_cell15[:20]
        )
    )


# ---------------------------------------------------------------------
# Load operations.csv as text to preserve IDs and exact categories
# ---------------------------------------------------------------------

start_time_cell15 = time.perf_counter()

operations_sha_before_cell15 = (
    sha256_file_cell15(
        OPERATIONS_CSV_PATH_CELL15
    )
)

operations_cell15 = pd.read_csv(
    OPERATIONS_CSV_PATH_CELL15,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    low_memory=False,
)

if len(operations_cell15) != 58585:
    raise RuntimeError(
        "Expected 58,585 operation rows; observed "
        f"{len(operations_cell15)}."
    )

if operations_cell15["op_id"].nunique() != 58585:
    raise RuntimeError(
        "operations.csv contains duplicate op_id values."
    )

unknown_spec_ids_cell15 = sorted(
    set(operations_cell15["spec_id"])
    - set(specification_metadata_cell15)
)

if unknown_spec_ids_cell15:
    raise RuntimeError(
        "Operations reference specification IDs absent "
        "from the frozen registry: "
        + ", ".join(
            unknown_spec_ids_cell15[:20]
        )
    )

dialect_mapping_cell15 = {
    specification_id:
        str(metadata["source_dialect"])
    for specification_id, metadata
    in specification_metadata_cell15.items()
}

version_mapping_cell15 = {
    specification_id:
        str(
            metadata[
                "canonical_openapi_version"
            ]
        )
    for specification_id, metadata
    in specification_metadata_cell15.items()
}

source_class_mapping_cell15 = {
    specification_id:
        str(metadata["source_class"])
    for specification_id, metadata
    in specification_metadata_cell15.items()
}


# ---------------------------------------------------------------------
# Fill metadata using exact spec_id joins
# ---------------------------------------------------------------------

dialect_missing_before_cell15 = (
    missing_series_cell15(
        operations_cell15["dialect"]
    )
)

version_missing_before_cell15 = (
    missing_series_cell15(
        operations_cell15[
            "canonical_version"
        ]
    )
)

source_class_missing_before_cell15 = (
    missing_series_cell15(
        operations_cell15["source_class"]
    )
)

operations_cell15.loc[
    dialect_missing_before_cell15,
    "dialect",
] = (
    operations_cell15.loc[
        dialect_missing_before_cell15,
        "spec_id",
    ].map(dialect_mapping_cell15)
)

operations_cell15.loc[
    version_missing_before_cell15,
    "canonical_version",
] = (
    operations_cell15.loc[
        version_missing_before_cell15,
        "spec_id",
    ].map(version_mapping_cell15)
)

operations_cell15.loc[
    source_class_missing_before_cell15,
    "source_class",
] = (
    operations_cell15.loc[
        source_class_missing_before_cell15,
        "spec_id",
    ].map(source_class_mapping_cell15)
)

dialect_missing_after_cell15 = (
    missing_series_cell15(
        operations_cell15["dialect"]
    )
)

version_missing_after_cell15 = (
    missing_series_cell15(
        operations_cell15[
            "canonical_version"
        ]
    )
)

source_class_missing_after_cell15 = (
    missing_series_cell15(
        operations_cell15["source_class"]
    )
)

if dialect_missing_after_cell15.any():
    raise RuntimeError(
        "Dialect metadata remains missing for "
        f"{int(dialect_missing_after_cell15.sum())} "
        "operation rows."
    )

if version_missing_after_cell15.any():
    raise RuntimeError(
        "Canonical version metadata remains missing for "
        f"{int(version_missing_after_cell15.sum())} "
        "operation rows."
    )

if source_class_missing_after_cell15.any():
    raise RuntimeError(
        "Source-class metadata remains missing for "
        f"{int(source_class_missing_after_cell15.sum())} "
        "operation rows."
    )

# Reconcile exact per-operation prompt evidence before refreshing exports.
prompt_report_cell15 = json.loads(PROMPT_REPORT_PATH_CELL15.read_text(encoding="utf-8"))
operation_audit_report_cell15 = json.loads(
    OPERATION_AUDIT_REPORT_PATH_CELL15.read_text(encoding="utf-8")
)
if prompt_report_cell15.get("status") != "PASS" or operation_audit_report_cell15.get("status") != "PASS":
    raise RuntimeError("Notebook 02 Cells 13 and 14 must both have passing reports.")

limit_cell15 = prompt_report_cell15["input_token_limit"]
if limit_cell15 != 5120 or prompt_report_cell15["unexpected_failure_count"] != 0:
    raise RuntimeError("Prompt measurement failed or used a different input limit.")

prompt_columns_cell15 = [
    "op_id", "slice_status", "prompt_status", "rendered_prompt_tokens",
    "input_token_limit", "within_input_limit", "audit_status",
    "eligible_for_experimental_generation", "experimental_exclusion_reason",
]
operation_prompt_rows_cell15 = {
    row[0]: row[1:]
    for row in operations_cell15[prompt_columns_cell15].itertuples(index=False, name=None)
}
measured_statuses_cell15 = Counter()
with PROMPT_LEDGER_PATH_CELL15.open(encoding="utf-8") as prompt_ledger:
    for line in prompt_ledger:
        if not line.strip():
            continue
        record = json.loads(line)
        op_id = record["op_id"]
        row = operation_prompt_rows_cell15.pop(op_id, None)
        if row is None:
            raise RuntimeError(f"Duplicate or unexpected prompt record: {op_id}")
        slice_status, prompt_status, token_text, row_limit, within_text, audit_status, eligible_text, reason = row
        tokens = record["rendered_prompt_tokens"]
        within = record["within_input_limit"]
        if slice_status == "PASS":
            if type(tokens) is not int or tokens < 0 or audit_status != "PASS":
                raise RuntimeError(f"Invalid prompt count or operation audit: {op_id}")
            is_eligible = tokens <= limit_cell15
            expected_status = ("PASS_WITHIN_INPUT_LIMIT" if is_eligible
                               else "EXCEEDS_RENDERED_INPUT_LIMIT")
            expected_reason = "" if is_eligible else "EXCLUDED_RENDERED_PROMPT_TOO_LARGE"
            expected_within = is_eligible
        else:
            is_eligible = False
            expected_status = "NOT_APPLICABLE_EXCLUDED_SLICE"
            expected_reason = "EXCLUDED_INVALID_L0_SLICE"
            expected_within = None
            if tokens is not None:
                raise RuntimeError(f"Invalid slice unexpectedly has prompt tokens: {op_id}")
        if (
            record["slice_status"] != slice_status
            or record["status"] != prompt_status
            or prompt_status != expected_status
            or record["input_token_limit"] != limit_cell15
            or row_limit != str(limit_cell15)
            or token_text != ("" if tokens is None else str(tokens))
            or within is not expected_within
            or within_text.strip().lower() != ("" if within is None else str(within).lower())
            or eligible_text.strip().lower() != str(is_eligible).lower()
            or reason != expected_reason
        ):
            raise RuntimeError(f"Operation row and frozen prompt evidence disagree: {op_id}")
        measured_statuses_cell15[expected_status] += 1

if operation_prompt_rows_cell15:
    raise RuntimeError("The prompt ledger is missing one or more operations.")

expected_eligible_cell15 = prompt_report_cell15["within_input_limit_count"]
expected_over_limit_cell15 = prompt_report_cell15["exceeds_input_limit_count"]
expected_invalid_cell15 = prompt_report_cell15["excluded_slice_count"]
if (
    measured_statuses_cell15["PASS_WITHIN_INPUT_LIMIT"] != expected_eligible_cell15
    or measured_statuses_cell15["EXCEEDS_RENDERED_INPUT_LIMIT"] != expected_over_limit_cell15
    or measured_statuses_cell15["NOT_APPLICABLE_EXCLUDED_SLICE"] != expected_invalid_cell15
    or sum(measured_statuses_cell15.values()) != prompt_report_cell15["operation_count"]
    or expected_eligible_cell15 + expected_over_limit_cell15 != prompt_report_cell15["measured_operation_count"]
    or operation_audit_report_cell15["operation_count"] != len(operations_cell15)
    or operation_audit_report_cell15["eligible_operation_count"] != expected_eligible_cell15
    or operation_audit_report_cell15["ineligible_operation_count"] != expected_over_limit_cell15 + expected_invalid_cell15
    or operation_audit_report_cell15["exclusion_reason_counts"].get("EXCLUDED_RENDERED_PROMPT_TOO_LARGE", 0) != expected_over_limit_cell15
    or operation_audit_report_cell15["exclusion_reason_counts"].get("EXCLUDED_INVALID_L0_SLICE", 0) != expected_invalid_cell15
):
    raise RuntimeError("Prompt ledger, prompt measurement report, and operation audit counts disagree.")

temporary_operations_path_cell15 = Path(
    str(OPERATIONS_CSV_PATH_CELL15)
    + ".metadata.tmp"
)

operations_cell15.to_csv(
    temporary_operations_path_cell15,
    index=False,
    lineterminator="\n",
)

os.replace(
    temporary_operations_path_cell15,
    OPERATIONS_CSV_PATH_CELL15,
)

operations_sha_after_cell15 = (
    sha256_file_cell15(
        OPERATIONS_CSV_PATH_CELL15
    )
)

dialect_counts_cell15 = {
    str(key): int(value)
    for key, value in (
        operations_cell15[
            "dialect"
        ].value_counts().sort_index().items()
    )
}

version_counts_cell15 = {
    str(key): int(value)
    for key, value in (
        operations_cell15[
            "canonical_version"
        ].value_counts().sort_index().items()
    )
}

source_class_counts_cell15 = {
    str(key): int(value)
    for key, value in (
        operations_cell15[
            "source_class"
        ].value_counts().sort_index().items()
    )
}

metadata_completion_report_cell15 = {
    "status": "PASS",
    "amendment":
        "COMPLETE_OPERATION_METADATA_FROM_FROZEN_SPECIFICATION_REGISTRY",
    "basis": (
        "Exact spec_id join using source_dialect, "
        "canonical_openapi_version, and source_class "
        "from the frozen eligible specification registry."
    ),
    "operation_row_count":
        len(operations_cell15),
    "specification_registry_count":
        len(specification_metadata_cell15),
    "dialect_values_filled":
        int(dialect_missing_before_cell15.sum()),
    "canonical_version_values_filled":
        int(version_missing_before_cell15.sum()),
    "source_class_values_filled":
        int(
            source_class_missing_before_cell15.sum()
        ),
    "dialect_counts":
        dialect_counts_cell15,
    "canonical_version_counts":
        version_counts_cell15,
    "source_class_counts":
        source_class_counts_cell15,
    "operations_sha256_before":
        operations_sha_before_cell15,
    "operations_sha256_after":
        operations_sha_after_cell15,
    "specification_registry_sha256":
        sha256_file_cell15(
            SPECIFICATION_REGISTRY_PATH_CELL15
        ),
    "problems": [],
}

write_json_atomic_cell15(
    METADATA_COMPLETION_REPORT_PATH_CELL15,
    metadata_completion_report_cell15,
)

metadata_completion_sha_cell15 = (
    sha256_file_cell15(
        METADATA_COMPLETION_REPORT_PATH_CELL15
    )
)


# ---------------------------------------------------------------------
# Update Cell 14's operation-audit report
# ---------------------------------------------------------------------

operation_audit_report_cell15[
    "dialect_counts"
] = dialect_counts_cell15

operation_audit_report_cell15[
    "canonical_version_counts"
] = version_counts_cell15

operation_audit_report_cell15[
    "source_class_counts"
] = source_class_counts_cell15

operation_audit_report_cell15[
    "operations_csv"
]["bytes"] = (
    OPERATIONS_CSV_PATH_CELL15.stat().st_size
)

operation_audit_report_cell15[
    "operations_csv"
]["sha256"] = operations_sha_after_cell15

operation_audit_report_cell15[
    "metadata_completion"
] = {
    "path": str(
        METADATA_COMPLETION_REPORT_PATH_CELL15
    ),
    "sha256":
        metadata_completion_sha_cell15,
    "dialect_values_filled":
        int(dialect_missing_before_cell15.sum()),
    "canonical_version_values_filled":
        int(version_missing_before_cell15.sum()),
    "source_class_values_filled":
        int(
            source_class_missing_before_cell15.sum()
        ),
}

write_json_atomic_cell15(
    OPERATION_AUDIT_REPORT_PATH_CELL15,
    operation_audit_report_cell15,
)

updated_operation_audit_sha_cell15 = (
    sha256_file_cell15(
        OPERATION_AUDIT_REPORT_PATH_CELL15
    )
)


# ---------------------------------------------------------------------
# Define analysis populations
# ---------------------------------------------------------------------

eligible_mask_cell15 = boolean_series_cell15(
    operations_cell15[
        "eligible_for_experimental_generation"
    ]
)

valid_slice_mask_cell15 = (
    operations_cell15["slice_status"]
    .astype(str)
    .str.upper()
    .eq("PASS")
)

over_limit_mask_cell15 = (
    operations_cell15["prompt_status"]
    .astype(str)
    .str.upper()
    .eq("EXCEEDS_RENDERED_INPUT_LIMIT")
)

populations_cell15 = {
    "all_operations":
        operations_cell15.index,
    "valid_slices":
        operations_cell15.index[
            valid_slice_mask_cell15
        ],
    "generation_eligible":
        operations_cell15.index[
            eligible_mask_cell15
        ],
    "prompt_over_limit":
        operations_cell15.index[
            over_limit_mask_cell15
        ],
}


# ---------------------------------------------------------------------
# Numerical statistics
# ---------------------------------------------------------------------

numerical_columns_cell15 = [
    "slice_bytes",
    "reference_count",
    "reference_relocation_count",
    "reference_issue_count",
    "parameter_count",
    "path_parameter_count",
    "query_parameter_count",
    "header_parameter_count",
    "cookie_parameter_count",
    "other_parameter_count",
    "required_parameter_count",
    "request_media_type_count",
    "response_entry_count",
    "response_media_type_count",
    "security_requirement_count",
    "security_scheme_count",
    "schema_object_count",
    "schema_property_count",
    "required_property_count",
    "maximum_schema_depth",
    "array_schema_count",
    "allof_occurrence_count",
    "anyof_occurrence_count",
    "oneof_occurrence_count",
    "not_occurrence_count",
    "summary_annotation_count",
    "description_annotation_count",
    "included_annotation_count",
    "prose_tokens_L0",
    "rendered_prompt_tokens",
    "rendered_prompt_bytes",
    "excess_prompt_tokens",
]

numerical_rows_cell15 = []

for population_name_cell15, indices_cell15 in (
    populations_cell15.items()
):
    population_frame_cell15 = (
        operations_cell15.loc[indices_cell15]
    )

    population_size_cell15 = len(
        population_frame_cell15
    )

    for column_cell15 in numerical_columns_cell15:
        series_cell15 = pd.to_numeric(
            population_frame_cell15[
                column_cell15
            ],
            errors="coerce",
        )

        observed_cell15 = series_cell15.dropna()
        observed_count_cell15 = len(
            observed_cell15
        )

        if observed_count_cell15 == 0:
            values_cell15 = {
                "sum": None,
                "mean": None,
                "standard_deviation": None,
                "minimum": None,
                "quartile_25": None,
                "median": None,
                "quartile_75": None,
                "maximum": None,
            }

        else:
            values_cell15 = {
                "sum":
                    finite_or_none_cell15(
                        observed_cell15.sum()
                    ),
                "mean":
                    finite_or_none_cell15(
                        observed_cell15.mean()
                    ),
                "standard_deviation":
                    finite_or_none_cell15(
                        observed_cell15.std(
                            ddof=1
                        )
                        if observed_count_cell15 > 1
                        else 0.0
                    ),
                "minimum":
                    finite_or_none_cell15(
                        observed_cell15.min()
                    ),
                "quartile_25":
                    finite_or_none_cell15(
                        observed_cell15.quantile(
                            0.25,
                            interpolation="linear",
                        )
                    ),
                "median":
                    finite_or_none_cell15(
                        observed_cell15.quantile(
                            0.50,
                            interpolation="linear",
                        )
                    ),
                "quartile_75":
                    finite_or_none_cell15(
                        observed_cell15.quantile(
                            0.75,
                            interpolation="linear",
                        )
                    ),
                "maximum":
                    finite_or_none_cell15(
                        observed_cell15.max()
                    ),
            }

        numerical_rows_cell15.append(
            {
                "population":
                    population_name_cell15,
                "variable":
                    column_cell15,
                "population_size":
                    population_size_cell15,
                "observed_count":
                    observed_count_cell15,
                "missing_count":
                    population_size_cell15
                    - observed_count_cell15,
                "quantile_method": "linear",
                **values_cell15,
            }
        )

numerical_statistics_cell15 = pd.DataFrame(
    numerical_rows_cell15
)

numerical_statistics_cell15.to_csv(
    NUMERICAL_STATISTICS_PATH_CELL15,
    index=False,
    lineterminator="\n",
)


# ---------------------------------------------------------------------
# Categorical counts and percentages
# ---------------------------------------------------------------------

categorical_columns_cell15 = [
    "method",
    "source_class",
    "dialect",
    "canonical_version",
    "audit_status",
    "slice_status",
    "prompt_status",
    "eligible_for_experimental_generation",
    "experimental_exclusion_reason",
    "operation_has_summary",
    "operation_has_description",
    "request_body_present",
    "request_body_required",
    "zero_prose",
]

categorical_rows_cell15 = []

for population_name_cell15, indices_cell15 in (
    populations_cell15.items()
):
    population_frame_cell15 = (
        operations_cell15.loc[indices_cell15]
    )

    population_size_cell15 = len(
        population_frame_cell15
    )

    for column_cell15 in categorical_columns_cell15:
        values_cell15 = (
            population_frame_cell15[
                column_cell15
            ]
            .astype(str)
            .replace(
                {
                    "": "<MISSING>",
                    "None": "<MISSING>",
                    "nan": "<MISSING>",
                    "null": "<MISSING>",
                }
            )
        )

        counts_cell15 = values_cell15.value_counts(
            dropna=False
        )

        for category_cell15, count_cell15 in (
            counts_cell15.items()
        ):
            categorical_rows_cell15.append(
                {
                    "population":
                        population_name_cell15,
                    "variable":
                        column_cell15,
                    "category":
                        str(category_cell15),
                    "count":
                        int(count_cell15),
                    "percentage": (
                        float(count_cell15)
                        / population_size_cell15
                        * 100.0
                        if population_size_cell15
                        else None
                    ),
                    "population_size":
                        population_size_cell15,
                }
            )

categorical_statistics_cell15 = pd.DataFrame(
    categorical_rows_cell15
)

categorical_statistics_cell15.to_csv(
    CATEGORICAL_STATISTICS_PATH_CELL15,
    index=False,
    lineterminator="\n",
)


# ---------------------------------------------------------------------
# Distribution summaries
# ---------------------------------------------------------------------

prose_valid_cell15 = pd.to_numeric(
    operations_cell15.loc[
        valid_slice_mask_cell15,
        "prose_tokens_L0",
    ],
    errors="coerce",
).dropna()

prose_eligible_cell15 = pd.to_numeric(
    operations_cell15.loc[
        eligible_mask_cell15,
        "prose_tokens_L0",
    ],
    errors="coerce",
).dropna()

prompt_valid_cell15 = pd.to_numeric(
    operations_cell15.loc[
        valid_slice_mask_cell15,
        "rendered_prompt_tokens",
    ],
    errors="coerce",
).dropna()

paired_cell15 = operations_cell15.loc[
    valid_slice_mask_cell15,
    [
        "prose_tokens_L0",
        "rendered_prompt_tokens",
    ],
].apply(
    pd.to_numeric,
    errors="coerce",
).dropna()

quantile_probabilities_cell15 = [
    0.00,
    0.01,
    0.05,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95,
    0.99,
    1.00,
]


def distribution_summary_cell15(series):
    return {
        "count": int(series.count()),
        "zero_count": int(
            (series == 0).sum()
        ),
        "minimum":
            finite_or_none_cell15(
                series.min()
            ),
        "mean":
            finite_or_none_cell15(
                series.mean()
            ),
        "standard_deviation":
            finite_or_none_cell15(
                series.std(ddof=1)
            ),
        "median":
            finite_or_none_cell15(
                series.median()
            ),
        "maximum":
            finite_or_none_cell15(
                series.max()
            ),
        "skewness":
            finite_or_none_cell15(
                series.skew()
            ),
        "quantiles": {
            f"{probability:.2f}":
                finite_or_none_cell15(
                    series.quantile(
                        probability,
                        interpolation="linear",
                    )
                )
            for probability
            in quantile_probabilities_cell15
        },
    }


distribution_report_cell15 = {
    "status": "PASS",
    "strata_thresholds_selected": False,
    "interpretation": (
        "These are descriptive corpus measurements only. "
        "No prose strata or sampling thresholds are "
        "selected in Notebook 02."
    ),
    "quantile_method": "linear",
    "valid_slice_prose_tokens":
        distribution_summary_cell15(
            prose_valid_cell15
        ),
    "generation_eligible_prose_tokens":
        distribution_summary_cell15(
            prose_eligible_cell15
        ),
    "valid_slice_rendered_prompt_tokens":
        distribution_summary_cell15(
            prompt_valid_cell15
        ),
    "prose_prompt_correlations": {
        "paired_operation_count":
            len(paired_cell15),
        "pearson":
            finite_or_none_cell15(
                paired_cell15[
                    "prose_tokens_L0"
                ].corr(
                    paired_cell15[
                        "rendered_prompt_tokens"
                    ],
                    method="pearson",
                )
            ),
        "spearman":
            finite_or_none_cell15(
                paired_cell15[
                    "prose_tokens_L0"
                ].corr(
                    paired_cell15[
                        "rendered_prompt_tokens"
                    ],
                    method="spearman",
                )
            ),
    },
}

write_json_atomic_cell15(
    DISTRIBUTION_REPORT_PATH_CELL15,
    distribution_report_cell15,
)


# ---------------------------------------------------------------------
# Linear-scale prose distribution
# ---------------------------------------------------------------------

plt.figure(figsize=(11, 6.5))

plt.hist(
    prose_valid_cell15.to_numpy(),
    bins=100,
    color="#3769a5",
    alpha=0.82,
    edgecolor="white",
    linewidth=0.25,
    label=(
        "All valid L0 slices "
        f"(n={len(prose_valid_cell15):,})"
    ),
)

plt.axvline(
    prose_valid_cell15.median(),
    color="#b33b2e",
    linestyle="--",
    linewidth=1.5,
    label=(
        "Median = "
        f"{prose_valid_cell15.median():,.0f}"
    ),
)

plt.title(
    "Distribution of L0 prose-token measurements"
)

plt.xlabel(
    "Specification-defined summary/description tokens"
)

plt.ylabel("Operation count")
plt.grid(axis="y", alpha=0.20)
plt.legend()
plt.tight_layout()

plt.savefig(
    LINEAR_FIGURE_PATH_CELL15,
    dpi=180,
    bbox_inches="tight",
)

plt.close()


# ---------------------------------------------------------------------
# Log1p-scale prose distribution
# ---------------------------------------------------------------------

plt.figure(figsize=(11, 6.5))

plt.hist(
    np.log1p(
        prose_valid_cell15.to_numpy()
    ),
    bins=100,
    color="#2f855a",
    alpha=0.78,
    edgecolor="white",
    linewidth=0.25,
    label=(
        "All valid slices "
        f"(n={len(prose_valid_cell15):,})"
    ),
)

plt.hist(
    np.log1p(
        prose_eligible_cell15.to_numpy()
    ),
    bins=100,
    color="#dd8a2e",
    alpha=0.42,
    edgecolor="none",
    label=(
        "Generation eligible "
        f"(n={len(prose_eligible_cell15):,})"
    ),
)

plt.title(
    "L0 prose-token distribution on a log1p scale"
)

plt.xlabel("log(1 + prose_tokens_L0)")
plt.ylabel("Operation count")
plt.grid(axis="y", alpha=0.20)
plt.legend()
plt.tight_layout()

plt.savefig(
    LOG_FIGURE_PATH_CELL15,
    dpi=180,
    bbox_inches="tight",
)

plt.close()


# ---------------------------------------------------------------------
# Verification
# ---------------------------------------------------------------------

verification_checks_cell15 = {
    "operation_row_count": (
        len(operations_cell15) == 58585
    ),
    "unique_operation_ids": (
        operations_cell15[
            "op_id"
        ].nunique()
        == 58585
    ),
    "eligible_operation_count": (
        int(eligible_mask_cell15.sum()) == expected_eligible_cell15
    ),
    "valid_slice_count": (
        int(valid_slice_mask_cell15.sum()) == prompt_report_cell15["measured_operation_count"]
    ),
    "prompt_over_limit_count": (
        int(over_limit_mask_cell15.sum()) == expected_over_limit_cell15
    ),
    "dialect_complete": (
        not missing_series_cell15(
            operations_cell15["dialect"]
        ).any()
    ),
    "canonical_version_complete": (
        not missing_series_cell15(
            operations_cell15[
                "canonical_version"
            ]
        ).any()
    ),
    "source_class_complete": (
        not missing_series_cell15(
            operations_cell15["source_class"]
        ).any()
    ),
    "numerical_statistics_nonempty": (
        NUMERICAL_STATISTICS_PATH_CELL15
        .stat().st_size > 0
    ),
    "categorical_statistics_nonempty": (
        CATEGORICAL_STATISTICS_PATH_CELL15
        .stat().st_size > 0
    ),
    "linear_plot_nonempty": (
        LINEAR_FIGURE_PATH_CELL15
        .stat().st_size > 0
    ),
    "log_plot_nonempty": (
        LOG_FIGURE_PATH_CELL15
        .stat().st_size > 0
    ),
    "no_strata_threshold_selected": (
        distribution_report_cell15[
            "strata_thresholds_selected"
        ]
        is False
    ),
}

status_cell15 = (
    "PASS"
    if all(
        verification_checks_cell15.values()
    )
    else "BLOCKED"
)

final_report_cell15 = {
    "status": status_cell15,
    "stage":
        "DESCRIPTIVE_STATISTICS_AND_PLOTS",
    "operation_count":
        len(operations_cell15),
    "valid_slice_count":
        int(valid_slice_mask_cell15.sum()),
    "eligible_operation_count":
        int(eligible_mask_cell15.sum()),
    "prompt_over_limit_count":
        int(over_limit_mask_cell15.sum()),
    "dialect_counts":
        dialect_counts_cell15,
    "canonical_version_counts":
        version_counts_cell15,
    "source_class_counts":
        source_class_counts_cell15,
    "metadata_completion": {
        "path": str(
            METADATA_COMPLETION_REPORT_PATH_CELL15
        ),
        "sha256":
            metadata_completion_sha_cell15,
        "dialect_values_filled":
            int(
                dialect_missing_before_cell15.sum()
            ),
        "canonical_version_values_filled":
            int(
                version_missing_before_cell15.sum()
            ),
        "source_class_values_filled":
            int(
                source_class_missing_before_cell15.sum()
            ),
    },
    "updated_operation_audit": {
        "path": str(
            OPERATION_AUDIT_REPORT_PATH_CELL15
        ),
        "sha256":
            updated_operation_audit_sha_cell15,
    },
    "operations_csv": {
        "path": str(
            OPERATIONS_CSV_PATH_CELL15
        ),
        "bytes":
            OPERATIONS_CSV_PATH_CELL15
            .stat().st_size,
        "sha256":
            operations_sha_after_cell15,
    },
    "numerical_statistics": {
        "path": str(
            NUMERICAL_STATISTICS_PATH_CELL15
        ),
        "row_count":
            len(numerical_statistics_cell15),
        "bytes":
            NUMERICAL_STATISTICS_PATH_CELL15
            .stat().st_size,
        "sha256":
            sha256_file_cell15(
                NUMERICAL_STATISTICS_PATH_CELL15
            ),
    },
    "categorical_statistics": {
        "path": str(
            CATEGORICAL_STATISTICS_PATH_CELL15
        ),
        "row_count":
            len(categorical_statistics_cell15),
        "bytes":
            CATEGORICAL_STATISTICS_PATH_CELL15
            .stat().st_size,
        "sha256":
            sha256_file_cell15(
                CATEGORICAL_STATISTICS_PATH_CELL15
            ),
    },
    "distribution_summary": {
        "path": str(
            DISTRIBUTION_REPORT_PATH_CELL15
        ),
        "sha256":
            sha256_file_cell15(
                DISTRIBUTION_REPORT_PATH_CELL15
            ),
    },
    "figures": [
        {
            "path": str(
                LINEAR_FIGURE_PATH_CELL15
            ),
            "bytes":
                LINEAR_FIGURE_PATH_CELL15
                .stat().st_size,
            "sha256":
                sha256_file_cell15(
                    LINEAR_FIGURE_PATH_CELL15
                ),
        },
        {
            "path": str(
                LOG_FIGURE_PATH_CELL15
            ),
            "bytes":
                LOG_FIGURE_PATH_CELL15
                .stat().st_size,
            "sha256":
                sha256_file_cell15(
                    LOG_FIGURE_PATH_CELL15
                ),
        },
    ],
    "verification_checks":
        verification_checks_cell15,
    "strata_thresholds_selected": False,
    "duration_seconds": round(
        time.perf_counter()
        - start_time_cell15,
        3,
    ),
}

write_json_atomic_cell15(
    FINAL_REPORT_PATH_CELL15,
    final_report_cell15,
)

final_report_cell15["report_path"] = str(
    FINAL_REPORT_PATH_CELL15
)

final_report_cell15["report_sha256"] = (
    sha256_file_cell15(
        FINAL_REPORT_PATH_CELL15
    )
)

print(
    json.dumps(
        final_report_cell15,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell15 != "PASS":
    raise RuntimeError(
        "Descriptive-statistics verification failed. "
        "Review verification_checks above."
    )

print("\nNOTEBOOK 02 — CELL 15: PASS")


{
  "status": "PASS",
  "stage": "DESCRIPTIVE_STATISTICS_AND_PLOTS",
  "operation_count": 58585,
  "valid_slice_count": 58584,
  "eligible_operation_count": 52922,
  "prompt_over_limit_count": 5662,
  "dialect_counts": {
    "openapi_3_0": 54504,
    "openapi_3_1": 1505,
    "swagger_2_0": 2576
  },
  "canonical_version_counts": {
    "3.0.0": 43980,
    "3.0.1": 6435,
    "3.0.2": 2708,
    "3.0.3": 3957,
    "3.1.0": 1505
  },
  "source_class_counts": {
    "converted_swagger": 2576,
    "native_openapi": 56009
  },
  "metadata_completion": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/reports/audit/operation_metadata_completion.json",
    "sha256": "7185674fb97ca2796cee71685eec4b2e0368b0551647c0d2ab4e820c95f63522",
    "dialect_values_filled": 0,
    "canonical_version_values_filled": 0,
    "source_class_values_filled": 0
  },
  "updated_operation_audit": {
    "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/reports/audit/operation_audi

In [17]:
# Notebook 02 — Cell 16
# Deterministically select and manually trace varied operations from:
#
# frozen canonical source -> L0 slice -> prose audit rows
#
# For every selected prose annotation, verify:
# - source and slice pointers resolve
# - source text equals slice text
# - text SHA-256 matches
# - character count matches
# - exact frozen-tokenizer count matches
# - prose-row token sum equals the operation's prose_tokens_L0 score

from pathlib import Path
from collections import defaultdict
from urllib.parse import unquote
import csv
import hashlib
import json
import os
import time

import pandas as pd
import yaml


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL16 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

OPERATIONS_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/operations.csv"
)

PROSE_FIELDS_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/prose_fields.csv"
)

PROMPT_MEASUREMENTS_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/rendered_prompt_measurements.jsonl"
)

SPECIFICATION_REGISTRY_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/inventory/eligible_specifications.jsonl"
)

TRACE_SELECTION_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/manual_trace_selection.csv"
)

TRACE_PROSE_ROWS_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/manual_trace_prose_rows.csv"
)

TRACE_RESULTS_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "data/audits/manual_trace_results.jsonl"
)

TRACE_REPORT_PATH_CELL16 = (
    PROJECT_ROOT_CELL16
    / "reports/audit/manual_trace_summary.json"
)

for directory_cell16 in [
    TRACE_SELECTION_PATH_CELL16.parent,
    TRACE_REPORT_PATH_CELL16.parent,
]:
    directory_cell16.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell16(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sha256_text_cell16(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def write_text_atomic_cell16(path, text):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_json_atomic_cell16(path, value):
    write_text_atomic_cell16(
        path,
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
    )


def integer_cell16(value, default=0):
    if value is None:
        return default

    text = str(value).strip()

    if text.lower() in {
        "",
        "none",
        "nan",
        "null",
    }:
        return default

    return int(float(text))


def true_value_cell16(value):
    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
    }


def compact_json_cell16(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        allow_nan=False,
    )


# ---------------------------------------------------------------------
# Validate inputs
# ---------------------------------------------------------------------

required_paths_cell16 = [
    OPERATIONS_PATH_CELL16,
    PROSE_FIELDS_PATH_CELL16,
    PROMPT_MEASUREMENTS_PATH_CELL16,
    SPECIFICATION_REGISTRY_PATH_CELL16,
]

missing_paths_cell16 = [
    str(path)
    for path in required_paths_cell16
    if not path.is_file()
]

if missing_paths_cell16:
    raise RuntimeError(
        "Missing required Cell 16 inputs:\n"
        + "\n".join(missing_paths_cell16)
    )


# ---------------------------------------------------------------------
# Locate expanded Stage 01 project
# ---------------------------------------------------------------------

stage_01_project_candidates_cell16 = [
    Path(
        "/kaggle/input/datasets/anonuser01/"
        "prose-operation-stage-01-corpus-fresh/"
        "stage_01_corpus_preparation_v1/"
        "prose_operation_experiment"
    ),
    Path(
        "/kaggle/input/prose-operation-stage-01-corpus-fresh/"
        "stage_01_corpus_preparation_v1/"
        "prose_operation_experiment"
    ),
]

if (
    "STAGE_01_PROJECT_ROOT" in globals()
    and globals()["STAGE_01_PROJECT_ROOT"]
    is not None
):
    stage_01_project_candidates_cell16.insert(
        0,
        Path(
            globals()[
                "STAGE_01_PROJECT_ROOT"
            ]
        ),
    )

STAGE_01_PROJECT_ROOT_CELL16 = next(
    (
        path
        for path
        in stage_01_project_candidates_cell16
        if path.is_dir()
    ),
    None,
)

if STAGE_01_PROJECT_ROOT_CELL16 is None:
    raise RuntimeError(
        "Could not locate the expanded Stage 01 project."
    )


# ---------------------------------------------------------------------
# Load exact specification records
# ---------------------------------------------------------------------

specification_map_cell16 = {}

with SPECIFICATION_REGISTRY_PATH_CELL16.open(
    "r",
    encoding="utf-8",
) as handle_cell16:
    for line_cell16 in handle_cell16:
        if not line_cell16.strip():
            continue

        record_cell16 = json.loads(line_cell16)

        specification_id_cell16 = str(
            record_cell16["spec_id"]
        )

        specification_map_cell16[
            specification_id_cell16
        ] = record_cell16

if len(specification_map_cell16) != 1970:
    raise RuntimeError(
        "Expected 1,970 specification records; "
        f"observed {len(specification_map_cell16)}."
    )


# ---------------------------------------------------------------------
# Load operation table and prepare numeric features
# ---------------------------------------------------------------------

operations_cell16 = pd.read_csv(
    OPERATIONS_PATH_CELL16,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    low_memory=False,
)

if len(operations_cell16) != 58585:
    raise RuntimeError(
        "Expected 58,585 operations."
    )

numeric_columns_cell16 = [
    "prose_tokens_L0",
    "rendered_prompt_tokens",
    "parameter_count",
    "security_scheme_count",
    "maximum_schema_depth",
    "schema_property_count",
]

for column_cell16 in numeric_columns_cell16:
    operations_cell16[
        column_cell16 + "__numeric"
    ] = pd.to_numeric(
        operations_cell16[column_cell16],
        errors="coerce",
    )

eligible_mask_cell16 = (
    operations_cell16[
        "eligible_for_experimental_generation"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
)

valid_slice_mask_cell16 = (
    operations_cell16["slice_status"]
    .astype(str)
    .str.upper()
    .eq("PASS")
)

over_limit_mask_cell16 = (
    operations_cell16["prompt_status"]
    .astype(str)
    .str.upper()
    .eq("EXCEEDS_RENDERED_INPUT_LIMIT")
)

eligible_cell16 = operations_cell16.loc[
    eligible_mask_cell16
].copy()

valid_cell16 = operations_cell16.loc[
    valid_slice_mask_cell16
].copy()

over_limit_cell16 = operations_cell16.loc[
    over_limit_mask_cell16
].copy()


# ---------------------------------------------------------------------
# Deterministic trace selection
# ---------------------------------------------------------------------

selected_operation_ids_cell16 = set()
selection_rows_cell16 = []


def select_one_cell16(
    reason,
    frame,
    sort_column,
    ascending=False,
):
    if frame.empty:
        raise RuntimeError(
            f"No candidate operation for trace reason: {reason}"
        )

    sorted_frame_cell16 = frame.sort_values(
        by=[
            sort_column,
            "op_id",
        ],
        ascending=[
            ascending,
            True,
        ],
        kind="mergesort",
    )

    for _, row_cell16 in sorted_frame_cell16.iterrows():
        operation_id_cell16 = str(
            row_cell16["op_id"]
        )

        if (
            operation_id_cell16
            in selected_operation_ids_cell16
        ):
            continue

        selected_operation_ids_cell16.add(
            operation_id_cell16
        )

        selection_rows_cell16.append(
            {
                "selection_order":
                    len(selection_rows_cell16) + 1,
                "selection_reason": reason,
                "op_id":
                    operation_id_cell16,
                "spec_id":
                    row_cell16["spec_id"],
                "source_class":
                    row_cell16["source_class"],
                "dialect":
                    row_cell16["dialect"],
                "canonical_version":
                    row_cell16[
                        "canonical_version"
                    ],
                "method":
                    row_cell16["method"],
                "path":
                    row_cell16["path"],
                "prose_tokens_L0":
                    row_cell16[
                        "prose_tokens_L0"
                    ],
                "rendered_prompt_tokens":
                    row_cell16[
                        "rendered_prompt_tokens"
                    ],
                "eligible_for_experimental_generation":
                    row_cell16[
                        "eligible_for_experimental_generation"
                    ],
            }
        )

        return

    raise RuntimeError(
        "All candidates were already selected for "
        f"trace reason: {reason}"
    )


select_one_cell16(
    "eligible_converted_swagger_high_prose",
    eligible_cell16.loc[
        eligible_cell16["source_class"]
        == "converted_swagger"
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_openapi_3_1_high_prose",
    eligible_cell16.loc[
        eligible_cell16["dialect"]
        == "openapi_3_1"
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_zero_prose",
    eligible_cell16.loc[
        eligible_cell16[
            "prose_tokens_L0__numeric"
        ] == 0
    ],
    "op_id",
    ascending=True,
)

select_one_cell16(
    "eligible_high_parameter_count",
    eligible_cell16,
    "parameter_count__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_required_request_body",
    eligible_cell16.loc[
        eligible_cell16[
            "request_body_required"
        ].astype(str).str.lower().eq("true")
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_with_security_scheme",
    eligible_cell16.loc[
        eligible_cell16[
            "security_scheme_count__numeric"
        ] > 0
    ],
    "security_scheme_count__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_deep_schema",
    eligible_cell16,
    "maximum_schema_depth__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_get_operation",
    eligible_cell16.loc[
        eligible_cell16["method"] == "get"
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_post_operation",
    eligible_cell16.loc[
        eligible_cell16["method"] == "post"
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "eligible_patch_operation",
    eligible_cell16.loc[
        eligible_cell16["method"] == "patch"
    ],
    "prose_tokens_L0__numeric",
    ascending=False,
)

select_one_cell16(
    "smallest_prompt_over_limit",
    over_limit_cell16,
    "rendered_prompt_tokens__numeric",
    ascending=True,
)

eligible_median_cell16 = float(
    eligible_cell16[
        "prose_tokens_L0__numeric"
    ].median()
)

median_candidates_cell16 = (
    eligible_cell16.copy()
)

median_candidates_cell16[
    "median_distance__numeric"
] = (
    median_candidates_cell16[
        "prose_tokens_L0__numeric"
    ]
    - eligible_median_cell16
).abs()

select_one_cell16(
    "eligible_near_corpus_median",
    median_candidates_cell16,
    "median_distance__numeric",
    ascending=True,
)

if len(selection_rows_cell16) != 12:
    raise RuntimeError(
        "Manual trace selection did not produce "
        "exactly 12 unique operations."
    )

selection_frame_cell16 = pd.DataFrame(
    selection_rows_cell16
)

selection_frame_cell16.to_csv(
    TRACE_SELECTION_PATH_CELL16,
    index=False,
    lineterminator="\n",
)


# ---------------------------------------------------------------------
# Load prompt records only for selected operations
# ---------------------------------------------------------------------

selected_set_cell16 = set(
    selection_frame_cell16["op_id"]
)

prompt_records_cell16 = {}

with PROMPT_MEASUREMENTS_PATH_CELL16.open(
    "r",
    encoding="utf-8",
) as handle_cell16:
    for line_cell16 in handle_cell16:
        if not line_cell16.strip():
            continue

        record_cell16 = json.loads(line_cell16)
        operation_id_cell16 = str(
            record_cell16["op_id"]
        )

        if operation_id_cell16 in selected_set_cell16:
            prompt_records_cell16[
                operation_id_cell16
            ] = record_cell16

if set(prompt_records_cell16) != selected_set_cell16:
    raise RuntimeError(
        "Could not locate every selected operation "
        "in rendered_prompt_measurements.jsonl."
    )


# ---------------------------------------------------------------------
# Load exact frozen tokenizer
# ---------------------------------------------------------------------

tokenizer_cell16 = None

for tokenizer_name_cell16 in [
    "tokenizer_cell13",
    "tokenizer_cell10",
    "TOKENIZER",
    "tokenizer",
]:
    candidate_cell16 = globals().get(
        tokenizer_name_cell16
    )

    if (
        candidate_cell16 is not None
        and hasattr(
            candidate_cell16,
            "encode",
        )
    ):
        tokenizer_cell16 = candidate_cell16
        break

if tokenizer_cell16 is None:
    from transformers import AutoTokenizer

    MODEL_ID_CELL16 = (
        "Qwen/Qwen2.5-Coder-7B-Instruct"
    )

    REVISION_CELL16 = (
        "c03e6d358207e414f1eca0bb1891e29f1db0e242"
    )

    try:
        tokenizer_cell16 = (
            AutoTokenizer.from_pretrained(
                MODEL_ID_CELL16,
                revision=REVISION_CELL16,
                local_files_only=True,
                trust_remote_code=False,
            )
        )
    except Exception:
        tokenizer_cell16 = (
            AutoTokenizer.from_pretrained(
                MODEL_ID_CELL16,
                revision=REVISION_CELL16,
                trust_remote_code=False,
            )
        )


# ---------------------------------------------------------------------
# Read selected prose rows from the full prose audit
# ---------------------------------------------------------------------

start_time_cell16 = time.perf_counter()

prose_rows_by_operation_cell16 = defaultdict(
    list
)

prose_field_rows_read_cell16 = 0
selected_prose_row_count_cell16 = 0

with PROSE_FIELDS_PATH_CELL16.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle_cell16:
    reader_cell16 = csv.DictReader(
        handle_cell16
    )

    expected_columns_cell16 = {
        "op_id",
        "slice_pointer",
        "source_pointer",
        "object_kind",
        "field_name",
        "text_sha256",
        "char_count",
        "token_count",
        "included_in_grouping",
    }

    missing_columns_cell16 = (
        expected_columns_cell16
        - set(reader_cell16.fieldnames or [])
    )

    if missing_columns_cell16:
        raise RuntimeError(
            "prose_fields.csv is missing columns: "
            + ", ".join(
                sorted(missing_columns_cell16)
            )
        )

    for row_cell16 in reader_cell16:
        prose_field_rows_read_cell16 += 1

        operation_id_cell16 = str(
            row_cell16["op_id"]
        )

        if operation_id_cell16 not in selected_set_cell16:
            continue

        if not true_value_cell16(
            row_cell16[
                "included_in_grouping"
            ]
        ):
            continue

        prose_rows_by_operation_cell16[
            operation_id_cell16
        ].append(row_cell16)

        selected_prose_row_count_cell16 += 1

        if (
            selected_prose_row_count_cell16
            % 1000 == 0
        ):
            print(
                json.dumps(
                    {
                        "selected_prose_rows_found":
                            selected_prose_row_count_cell16,
                        "total_prose_rows_scanned":
                            prose_field_rows_read_cell16,
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - start_time_cell16,
                            1,
                        ),
                    }
                )
            )

print(
    json.dumps(
        {
            "prose_field_rows_scanned":
                prose_field_rows_read_cell16,
            "selected_prose_rows":
                selected_prose_row_count_cell16,
            "selected_operations":
                len(selected_set_cell16),
            "scan_duration_seconds": round(
                time.perf_counter()
                - start_time_cell16,
                3,
            ),
        }
    )
)


# ---------------------------------------------------------------------
# JSON Pointer resolver
# ---------------------------------------------------------------------

def decode_pointer_token_cell16(token):
    return unquote(token).replace(
        "~1",
        "/",
    ).replace(
        "~0",
        "~",
    )


def resolve_pointer_cell16(
    document,
    pointer,
):
    if pointer is None:
        raise KeyError("Pointer is null.")

    pointer = str(pointer)

    if pointer.startswith("#"):
        pointer = pointer[1:]

    pointer = unquote(pointer)

    if pointer == "":
        return document

    if not pointer.startswith("/"):
        raise KeyError(
            f"Invalid JSON Pointer: {pointer}"
        )

    current = document

    for raw_token_cell16 in pointer.split("/")[1:]:
        token_cell16 = decode_pointer_token_cell16(
            raw_token_cell16
        )

        if isinstance(current, dict):
            if token_cell16 not in current:
                raise KeyError(
                    f"Missing mapping key "
                    f"{token_cell16!r} in pointer "
                    f"{pointer!r}."
                )

            current = current[token_cell16]

        elif isinstance(current, list):
            index_cell16 = int(token_cell16)

            if not 0 <= index_cell16 < len(current):
                raise IndexError(
                    f"List index {index_cell16} "
                    f"outside pointer {pointer!r}."
                )

            current = current[index_cell16]

        else:
            raise KeyError(
                f"Pointer {pointer!r} traverses "
                "through a scalar value."
            )

    return current


# ---------------------------------------------------------------------
# Source operation locator
# ---------------------------------------------------------------------

HTTP_METHODS_CELL16 = {
    "get",
    "put",
    "post",
    "delete",
    "options",
    "head",
    "patch",
    "trace",
}


def locate_operation_cell16(
    document,
    path_value,
    method_value,
):
    paths_cell16 = document.get(
        "paths",
        {},
    )

    method_value = str(
        method_value
    ).lower()

    if (
        isinstance(paths_cell16, dict)
        and path_value in paths_cell16
        and isinstance(
            paths_cell16[path_value],
            dict,
        )
        and method_value
        in paths_cell16[path_value]
    ):
        operation_cell16 = (
            paths_cell16[
                path_value
            ][method_value]
        )

        if isinstance(operation_cell16, dict):
            return operation_cell16

    candidates_cell16 = []

    if isinstance(paths_cell16, dict):
        for candidate_path_cell16, path_item_cell16 in (
            paths_cell16.items()
        ):
            if not isinstance(
                path_item_cell16,
                dict,
            ):
                continue

            for (
                candidate_method_cell16,
                candidate_operation_cell16,
            ) in path_item_cell16.items():
                candidate_method_cell16 = str(
                    candidate_method_cell16
                ).lower()

                if (
                    candidate_method_cell16
                    in HTTP_METHODS_CELL16
                    and isinstance(
                        candidate_operation_cell16,
                        dict,
                    )
                ):
                    candidates_cell16.append(
                        candidate_operation_cell16
                    )

    if len(candidates_cell16) == 1:
        return candidates_cell16[0]

    raise RuntimeError(
        "Could not uniquely locate the selected "
        f"operation {method_value.upper()} {path_value}."
    )


# ---------------------------------------------------------------------
# Execute manual traces
# ---------------------------------------------------------------------

operation_lookup_cell16 = (
    operations_cell16.set_index(
        "op_id",
        drop=False,
    )
)

trace_results_cell16 = []
trace_prose_output_rows_cell16 = []

failure_examples_cell16 = []

for selection_cell16 in selection_rows_cell16:
    operation_id_cell16 = selection_cell16[
        "op_id"
    ]

    operation_row_cell16 = (
        operation_lookup_cell16.loc[
            operation_id_cell16
        ]
    )

    specification_id_cell16 = str(
        operation_row_cell16["spec_id"]
    )

    specification_record_cell16 = (
        specification_map_cell16[
            specification_id_cell16
        ]
    )

    source_relative_path_cell16 = (
        specification_record_cell16[
            "canonical_relative_path"
        ]
    )

    source_path_cell16 = (
        STAGE_01_PROJECT_ROOT_CELL16
        / source_relative_path_cell16
    )

    prompt_record_cell16 = (
        prompt_records_cell16[
            operation_id_cell16
        ]
    )

    slice_path_cell16 = Path(
        prompt_record_cell16[
            "slice_path"
        ]
    )

    checks_cell16 = {
        "source_file_exists":
            source_path_cell16.is_file(),
        "slice_file_exists":
            slice_path_cell16.is_file(),
    }

    diagnostic_cell16 = None
    annotation_failures_cell16 = []

    try:
        if not all(checks_cell16.values()):
            raise FileNotFoundError(
                "Source or slice file is missing."
            )

        source_text_cell16 = (
            source_path_cell16.read_text(
                encoding="utf-8"
            )
        )

        slice_text_cell16 = (
            slice_path_cell16.read_text(
                encoding="utf-8"
            )
        )

        source_document_cell16 = json.loads(
            source_text_cell16
        )

        slice_document_cell16 = json.loads(
            slice_text_cell16
        )

        source_expected_sha_cell16 = (
            specification_record_cell16[
                "canonical_file_sha256"
            ]
        )

        source_observed_sha_cell16 = (
            sha256_text_cell16(
                source_text_cell16
            )
        )

        slice_expected_sha_cell16 = str(
            operation_row_cell16[
                "slice_sha256"
            ]
        )

        slice_observed_sha_cell16 = (
            sha256_text_cell16(
                slice_text_cell16
            )
        )

        checks_cell16[
            "source_sha256_matches"
        ] = (
            source_observed_sha_cell16
            == source_expected_sha_cell16
        )

        checks_cell16[
            "slice_sha256_matches"
        ] = (
            slice_observed_sha_cell16
            == slice_expected_sha_cell16
        )

        source_operation_cell16 = (
            locate_operation_cell16(
                source_document_cell16,
                operation_row_cell16["path"],
                operation_row_cell16["method"],
            )
        )

        slice_operation_cell16 = (
            locate_operation_cell16(
                slice_document_cell16,
                operation_row_cell16["path"],
                operation_row_cell16["method"],
            )
        )

        checks_cell16[
            "source_operation_resolved"
        ] = isinstance(
            source_operation_cell16,
            dict,
        )

        checks_cell16[
            "slice_operation_resolved"
        ] = isinstance(
            slice_operation_cell16,
            dict,
        )

        prose_rows_cell16 = (
            prose_rows_by_operation_cell16.get(
                operation_id_cell16,
                [],
            )
        )

        observed_token_sum_cell16 = 0
        observed_summary_count_cell16 = 0
        observed_description_count_cell16 = 0

        for prose_row_index_cell16, prose_row_cell16 in enumerate(
            prose_rows_cell16,
            start=1,
        ):
            slice_pointer_cell16 = (
                prose_row_cell16[
                    "slice_pointer"
                ]
            )

            source_pointer_cell16 = (
                prose_row_cell16[
                    "source_pointer"
                ]
            )

            row_checks_cell16 = {}

            try:
                slice_value_cell16 = (
                    resolve_pointer_cell16(
                        slice_document_cell16,
                        slice_pointer_cell16,
                    )
                )

                source_value_cell16 = (
                    resolve_pointer_cell16(
                        source_document_cell16,
                        source_pointer_cell16,
                    )
                )

                row_checks_cell16[
                    "slice_value_is_string"
                ] = isinstance(
                    slice_value_cell16,
                    str,
                )

                row_checks_cell16[
                    "source_value_is_string"
                ] = isinstance(
                    source_value_cell16,
                    str,
                )

                row_checks_cell16[
                    "source_slice_text_equal"
                ] = (
                    source_value_cell16
                    == slice_value_cell16
                )

                observed_text_sha_cell16 = (
                    sha256_text_cell16(
                        slice_value_cell16
                    )
                )

                observed_char_count_cell16 = len(
                    slice_value_cell16
                )

                observed_token_count_cell16 = len(
                    tokenizer_cell16.encode(
                        slice_value_cell16,
                        add_special_tokens=False,
                    )
                )

                row_checks_cell16[
                    "text_sha256_matches"
                ] = (
                    observed_text_sha_cell16
                    == prose_row_cell16[
                        "text_sha256"
                    ]
                )

                row_checks_cell16[
                    "character_count_matches"
                ] = (
                    observed_char_count_cell16
                    == integer_cell16(
                        prose_row_cell16[
                            "char_count"
                        ]
                    )
                )

                row_checks_cell16[
                    "token_count_matches"
                ] = (
                    observed_token_count_cell16
                    == integer_cell16(
                        prose_row_cell16[
                            "token_count"
                        ]
                    )
                )

                row_pass_cell16 = all(
                    row_checks_cell16.values()
                )

                observed_token_sum_cell16 += (
                    observed_token_count_cell16
                )

                if (
                    prose_row_cell16[
                        "field_name"
                    ]
                    == "summary"
                ):
                    observed_summary_count_cell16 += 1

                elif (
                    prose_row_cell16[
                        "field_name"
                    ]
                    == "description"
                ):
                    observed_description_count_cell16 += 1

                trace_prose_output_rows_cell16.append(
                    {
                        "selection_reason":
                            selection_cell16[
                                "selection_reason"
                            ],
                        "op_id":
                            operation_id_cell16,
                        "spec_id":
                            specification_id_cell16,
                        "row_number_within_operation":
                            prose_row_index_cell16,
                        "slice_pointer":
                            slice_pointer_cell16,
                        "source_pointer":
                            source_pointer_cell16,
                        "object_kind":
                            prose_row_cell16[
                                "object_kind"
                            ],
                        "field_name":
                            prose_row_cell16[
                                "field_name"
                            ],
                        "text_sha256":
                            observed_text_sha_cell16,
                        "char_count":
                            observed_char_count_cell16,
                        "token_count":
                            observed_token_count_cell16,
                        "text_preview":
                            slice_value_cell16[
                                :120
                            ].replace(
                                "\n",
                                "\\n",
                            ),
                        "source_slice_text_equal":
                            row_checks_cell16[
                                "source_slice_text_equal"
                            ],
                        "text_sha256_matches":
                            row_checks_cell16[
                                "text_sha256_matches"
                            ],
                        "character_count_matches":
                            row_checks_cell16[
                                "character_count_matches"
                            ],
                        "token_count_matches":
                            row_checks_cell16[
                                "token_count_matches"
                            ],
                        "status": (
                            "PASS"
                            if row_pass_cell16
                            else "FAILED"
                        ),
                    }
                )

                if not row_pass_cell16:
                    annotation_failures_cell16.append(
                        {
                            "slice_pointer":
                                slice_pointer_cell16,
                            "source_pointer":
                                source_pointer_cell16,
                            "checks":
                                row_checks_cell16,
                        }
                    )

            except Exception as error_cell16:
                annotation_failures_cell16.append(
                    {
                        "slice_pointer":
                            slice_pointer_cell16,
                        "source_pointer":
                            source_pointer_cell16,
                        "diagnostic":
                            repr(error_cell16),
                    }
                )

        expected_prose_tokens_cell16 = (
            integer_cell16(
                operation_row_cell16[
                    "prose_tokens_L0"
                ]
            )
        )

        expected_summary_count_cell16 = (
            integer_cell16(
                operation_row_cell16[
                    "summary_annotation_count"
                ]
            )
        )

        expected_description_count_cell16 = (
            integer_cell16(
                operation_row_cell16[
                    "description_annotation_count"
                ]
            )
        )

        expected_annotation_count_cell16 = (
            integer_cell16(
                operation_row_cell16[
                    "included_annotation_count"
                ]
            )
        )

        checks_cell16[
            "prose_row_count_matches"
        ] = (
            len(prose_rows_cell16)
            == expected_annotation_count_cell16
        )

        checks_cell16[
            "summary_count_matches"
        ] = (
            observed_summary_count_cell16
            == expected_summary_count_cell16
        )

        checks_cell16[
            "description_count_matches"
        ] = (
            observed_description_count_cell16
            == expected_description_count_cell16
        )

        checks_cell16[
            "row_token_sum_matches_operation_score"
        ] = (
            observed_token_sum_cell16
            == expected_prose_tokens_cell16
        )

        checks_cell16[
            "all_annotation_rows_pass"
        ] = (
            len(annotation_failures_cell16)
            == 0
        )

        trace_status_cell16 = (
            "PASS"
            if all(checks_cell16.values())
            else "FAILED"
        )

    except Exception as error_cell16:
        diagnostic_cell16 = repr(error_cell16)
        trace_status_cell16 = "FAILED"

        prose_rows_cell16 = []
        observed_token_sum_cell16 = 0
        observed_summary_count_cell16 = 0
        observed_description_count_cell16 = 0
        expected_prose_tokens_cell16 = integer_cell16(
            operation_row_cell16[
                "prose_tokens_L0"
            ]
        )
        expected_annotation_count_cell16 = integer_cell16(
            operation_row_cell16[
                "included_annotation_count"
            ]
        )

    trace_result_cell16 = {
        "selection_order":
            selection_cell16[
                "selection_order"
            ],
        "selection_reason":
            selection_cell16[
                "selection_reason"
            ],
        "op_id": operation_id_cell16,
        "spec_id": specification_id_cell16,
        "source_class":
            operation_row_cell16[
                "source_class"
            ],
        "dialect":
            operation_row_cell16["dialect"],
        "method":
            operation_row_cell16["method"],
        "path":
            operation_row_cell16["path"],
        "generation_eligible":
            true_value_cell16(
                operation_row_cell16[
                    "eligible_for_experimental_generation"
                ]
            ),
        "source_path":
            str(source_path_cell16),
        "slice_path":
            str(slice_path_cell16),
        "expected_prose_annotation_count":
            expected_annotation_count_cell16,
        "observed_prose_row_count":
            len(prose_rows_cell16),
        "expected_prose_tokens_L0":
            expected_prose_tokens_cell16,
        "observed_prose_token_sum":
            observed_token_sum_cell16,
        "observed_summary_count":
            observed_summary_count_cell16,
        "observed_description_count":
            observed_description_count_cell16,
        "annotation_failure_count":
            len(annotation_failures_cell16),
        "checks": checks_cell16,
        "diagnostic": diagnostic_cell16,
        "status": trace_status_cell16,
    }

    trace_results_cell16.append(
        trace_result_cell16
    )

    if trace_status_cell16 != "PASS":
        failure_examples_cell16.append(
            trace_result_cell16
        )

    print(
        json.dumps(
            {
                "selection_order":
                    selection_cell16[
                        "selection_order"
                    ],
                "selection_reason":
                    selection_cell16[
                        "selection_reason"
                    ],
                "op_id":
                    operation_id_cell16,
                "prose_rows":
                    len(prose_rows_cell16),
                "token_sum":
                    observed_token_sum_cell16,
                "status":
                    trace_status_cell16,
            }
        )
    )


# ---------------------------------------------------------------------
# Write detailed trace artifacts
# ---------------------------------------------------------------------

write_text_atomic_cell16(
    TRACE_RESULTS_PATH_CELL16,
    "".join(
        json.dumps(
            record_cell16,
            ensure_ascii=False,
            separators=(",", ":"),
            allow_nan=False,
        )
        + "\n"
        for record_cell16
        in trace_results_cell16
    ),
)

trace_prose_fieldnames_cell16 = [
    "selection_reason",
    "op_id",
    "spec_id",
    "row_number_within_operation",
    "slice_pointer",
    "source_pointer",
    "object_kind",
    "field_name",
    "text_sha256",
    "char_count",
    "token_count",
    "text_preview",
    "source_slice_text_equal",
    "text_sha256_matches",
    "character_count_matches",
    "token_count_matches",
    "status",
]

temporary_trace_rows_path_cell16 = Path(
    str(TRACE_PROSE_ROWS_PATH_CELL16)
    + ".tmp"
)

with temporary_trace_rows_path_cell16.open(
    "w",
    encoding="utf-8",
    newline="",
) as handle_cell16:
    writer_cell16 = csv.DictWriter(
        handle_cell16,
        fieldnames=
            trace_prose_fieldnames_cell16,
        extrasaction="ignore",
        lineterminator="\n",
    )

    writer_cell16.writeheader()
    writer_cell16.writerows(
        trace_prose_output_rows_cell16
    )

os.replace(
    temporary_trace_rows_path_cell16,
    TRACE_PROSE_ROWS_PATH_CELL16,
)


# ---------------------------------------------------------------------
# Final report
# ---------------------------------------------------------------------

status_counts_cell16 = Counter(
    record_cell16["status"]
    for record_cell16 in trace_results_cell16
)

verification_checks_cell16 = {
    "selected_operation_count_is_12": (
        len(trace_results_cell16) == 12
    ),
    "selected_operation_ids_unique": (
        len(selected_operation_ids_cell16)
        == 12
    ),
    "all_traces_pass": (
        status_counts_cell16["PASS"] == 12
    ),
    "no_trace_failures": (
        status_counts_cell16["FAILED"] == 0
    ),
    "converted_swagger_represented": any(
        record_cell16["source_class"]
        == "converted_swagger"
        for record_cell16
        in trace_results_cell16
    ),
    "openapi_3_1_represented": any(
        record_cell16["dialect"]
        == "openapi_3_1"
        for record_cell16
        in trace_results_cell16
    ),
    "zero_prose_represented": any(
        record_cell16[
            "expected_prose_tokens_L0"
        ] == 0
        for record_cell16
        in trace_results_cell16
    ),
    "prompt_over_limit_represented": any(
        not record_cell16[
            "generation_eligible"
        ]
        for record_cell16
        in trace_results_cell16
    ),
    "trace_rows_file_exists": (
        TRACE_PROSE_ROWS_PATH_CELL16.is_file()
    ),
}

status_cell16 = (
    "PASS"
    if all(
        verification_checks_cell16.values()
    )
    else "BLOCKED"
)

report_cell16 = {
    "status": status_cell16,
    "stage":
        "MANUAL_SOURCE_SLICE_PROSE_TRACE",
    "selection_policy": (
        "Deterministic criterion-based selection "
        "with lexical op_id tie breaking."
    ),
    "selected_operation_count":
        len(trace_results_cell16),
    "selected_prose_row_count":
        len(trace_prose_output_rows_cell16),
    "status_counts": dict(
        sorted(status_counts_cell16.items())
    ),
    "verification_checks":
        verification_checks_cell16,
    "failure_examples":
        failure_examples_cell16[:20],
    "selection": {
        "path": str(
            TRACE_SELECTION_PATH_CELL16
        ),
        "bytes":
            TRACE_SELECTION_PATH_CELL16
            .stat().st_size,
        "sha256":
            sha256_file_cell16(
                TRACE_SELECTION_PATH_CELL16
            ),
    },
    "trace_results": {
        "path": str(
            TRACE_RESULTS_PATH_CELL16
        ),
        "bytes":
            TRACE_RESULTS_PATH_CELL16
            .stat().st_size,
        "sha256":
            sha256_file_cell16(
                TRACE_RESULTS_PATH_CELL16
            ),
    },
    "trace_prose_rows": {
        "path": str(
            TRACE_PROSE_ROWS_PATH_CELL16
        ),
        "bytes":
            TRACE_PROSE_ROWS_PATH_CELL16
            .stat().st_size,
        "sha256":
            sha256_file_cell16(
                TRACE_PROSE_ROWS_PATH_CELL16
            ),
    },
    "input_hashes": {
        "operations":
            sha256_file_cell16(
                OPERATIONS_PATH_CELL16
            ),
        "prose_fields":
            sha256_file_cell16(
                PROSE_FIELDS_PATH_CELL16
            ),
        "prompt_measurements":
            sha256_file_cell16(
                PROMPT_MEASUREMENTS_PATH_CELL16
            ),
        "specification_registry":
            sha256_file_cell16(
                SPECIFICATION_REGISTRY_PATH_CELL16
            ),
    },
    "duration_seconds": round(
        time.perf_counter()
        - start_time_cell16,
        3,
    ),
}

write_json_atomic_cell16(
    TRACE_REPORT_PATH_CELL16,
    report_cell16,
)

report_cell16["report_path"] = str(
    TRACE_REPORT_PATH_CELL16
)

report_cell16["report_sha256"] = (
    sha256_file_cell16(
        TRACE_REPORT_PATH_CELL16
    )
)

print(
    json.dumps(
        report_cell16,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell16 != "PASS":
    raise RuntimeError(
        "Manual trace verification failed. Review "
        "failure_examples and trace results above."
    )

print("\nNOTEBOOK 02 — CELL 16: PASS")

{"prose_field_rows_scanned": 3306273, "selected_prose_rows": 629, "selected_operations": 12, "scan_duration_seconds": 17.973}
{"selection_order": 1, "selection_reason": "eligible_converted_swagger_high_prose", "op_id": "op_2f49824e15b9b59a3228d124", "prose_rows": 42, "token_sum": 1720, "status": "PASS"}
{"selection_order": 2, "selection_reason": "eligible_openapi_3_1_high_prose", "op_id": "op_4e7547da64ced23218bbb448", "prose_rows": 100, "token_sum": 2670, "status": "PASS"}
{"selection_order": 3, "selection_reason": "eligible_zero_prose", "op_id": "op_018c2508a6266e0056273c19", "prose_rows": 2, "token_sum": 0, "status": "PASS"}
{"selection_order": 4, "selection_reason": "eligible_high_parameter_count", "op_id": "op_c1eb6e8923f10babd8a2c554", "prose_rows": 132, "token_sum": 47, "status": "PASS"}
{"selection_order": 5, "selection_reason": "eligible_required_request_body", "op_id": "op_228909198bbfe2e0cb903891", "prose_rows": 65, "token_sum": 3615, "status": "PASS"}
{"selection_order": 6,

In [18]:
# Notebook 02 — Cell 17
# Freeze and reconcile the final operation eligibility registry.
#
# Outputs:
# - final_eligible_operations.jsonl
# - operation_exclusions.jsonl
# - final_eligibility_reconciliation.json
#
# No prose strata or sampling thresholds are selected here.

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import time

import pandas as pd


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL17 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

OPERATIONS_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/audits/operations.csv"
)

PROMPT_MEASUREMENTS_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/audits/rendered_prompt_measurements.jsonl"
)

SLICE_RECORDS_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/audits/slice_records.jsonl"
)

PROMPT_PAYLOAD_INDEX_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/audits/rendered_prompt_payload_index.jsonl"
)

ELIGIBLE_REGISTRY_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/inventory/final_eligible_operations.jsonl"
)

EXCLUSION_LEDGER_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "data/inventory/operation_exclusions.jsonl"
)

REPORT_PATH_CELL17 = (
    PROJECT_ROOT_CELL17
    / "reports/audit/final_eligibility_reconciliation.json"
)

for directory_cell17 in [
    ELIGIBLE_REGISTRY_PATH_CELL17.parent,
    REPORT_PATH_CELL17.parent,
]:
    directory_cell17.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell17(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def write_text_atomic_cell17(path, text):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_json_atomic_cell17(path, value):
    write_text_atomic_cell17(
        path,
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
    )


def compact_json_line_cell17(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ) + "\n"


def missing_cell17(value):
    if value is None:
        return True

    return str(value).strip().lower() in {
        "",
        "none",
        "nan",
        "null",
        "<na>",
    }


def text_or_none_cell17(value):
    if missing_cell17(value):
        return None

    return str(value)


def integer_or_none_cell17(value):
    if missing_cell17(value):
        return None

    return int(float(str(value)))


def boolean_or_none_cell17(value):
    if missing_cell17(value):
        return None

    normalized = str(value).strip().lower()

    if normalized in {
        "true",
        "1",
        "yes",
    }:
        return True

    if normalized in {
        "false",
        "0",
        "no",
    }:
        return False

    raise ValueError(
        f"Unexpected boolean value: {value!r}"
    )


def relative_project_path_cell17(path_value):
    if missing_cell17(path_value):
        return None

    path = Path(str(path_value))

    if not path.is_absolute():
        return path.as_posix()

    try:
        return path.relative_to(
            PROJECT_ROOT_CELL17
        ).as_posix()

    except ValueError:
        raise RuntimeError(
            "Recorded artifact is outside the Stage 02 "
            f"project root: {path}"
        )


# ---------------------------------------------------------------------
# Validate required inputs
# ---------------------------------------------------------------------

required_paths_cell17 = [
    OPERATIONS_PATH_CELL17,
    PROMPT_MEASUREMENTS_PATH_CELL17,
    SLICE_RECORDS_PATH_CELL17,
    PROMPT_PAYLOAD_INDEX_PATH_CELL17,
]

missing_paths_cell17 = [
    str(path)
    for path in required_paths_cell17
    if not path.is_file()
]

if missing_paths_cell17:
    raise RuntimeError(
        "Missing required Cell 17 inputs:\n"
        + "\n".join(missing_paths_cell17)
    )


# ---------------------------------------------------------------------
# Load the completed operation table
# ---------------------------------------------------------------------

start_time_cell17 = time.perf_counter()

operations_cell17 = pd.read_csv(
    OPERATIONS_PATH_CELL17,
    dtype=str,
    keep_default_na=False,
    na_filter=False,
    low_memory=False,
)

if len(operations_cell17) != 58585:
    raise RuntimeError(
        "Expected 58,585 operation rows."
    )

if operations_cell17["op_id"].nunique() != 58585:
    raise RuntimeError(
        "operations.csv contains duplicate op_id values."
    )

operation_map_cell17 = {
    str(row_cell17["op_id"]):
        row_cell17.to_dict()
    for _, row_cell17
    in operations_cell17.iterrows()
}


# ---------------------------------------------------------------------
# Load slice records
# ---------------------------------------------------------------------

slice_map_cell17 = {}

with SLICE_RECORDS_PATH_CELL17.open(
    "r",
    encoding="utf-8",
) as handle_cell17:
    for line_number_cell17, line_cell17 in enumerate(
        handle_cell17,
        start=1,
    ):
        if not line_cell17.strip():
            continue

        record_cell17 = json.loads(line_cell17)
        operation_id_cell17 = str(
            record_cell17["op_id"]
        )

        if operation_id_cell17 in slice_map_cell17:
            raise RuntimeError(
                "Duplicate op_id in slice records: "
                f"{operation_id_cell17}"
            )

        slice_map_cell17[
            operation_id_cell17
        ] = record_cell17

if len(slice_map_cell17) != 58585:
    raise RuntimeError(
        "Expected 58,585 slice records."
    )


# ---------------------------------------------------------------------
# Load payload chunk metadata
# ---------------------------------------------------------------------

payload_chunk_map_cell17 = {}

with PROMPT_PAYLOAD_INDEX_PATH_CELL17.open(
    "r",
    encoding="utf-8",
) as handle_cell17:
    for line_cell17 in handle_cell17:
        if not line_cell17.strip():
            continue

        record_cell17 = json.loads(line_cell17)

        chunk_number_cell17 = int(
            record_cell17["chunk_number"]
        )

        payload_path_cell17 = Path(
            record_cell17["payload_path"]
        )

        if not payload_path_cell17.is_file():
            raise RuntimeError(
                "Missing rendered-prompt payload chunk: "
                f"{payload_path_cell17}"
            )

        if (
            sha256_file_cell17(
                payload_path_cell17
            )
            != record_cell17[
                "payload_sha256"
            ]
        ):
            raise RuntimeError(
                "Rendered-prompt payload chunk hash "
                f"mismatch: {payload_path_cell17}"
            )

        payload_chunk_map_cell17[
            chunk_number_cell17
        ] = {
            **record_cell17,
            "relative_payload_path":
                relative_project_path_cell17(
                    payload_path_cell17
                ),
        }

if len(payload_chunk_map_cell17) != 235:
    raise RuntimeError(
        "Expected 235 rendered-prompt payload chunks; "
        f"observed {len(payload_chunk_map_cell17)}."
    )


# ---------------------------------------------------------------------
# Load prompt records and assign exact positions inside payload chunks
# ---------------------------------------------------------------------

prompt_map_cell17 = {}
payload_position_by_chunk_cell17 = Counter()
prompt_record_count_cell17 = 0

with PROMPT_MEASUREMENTS_PATH_CELL17.open(
    "r",
    encoding="utf-8",
) as handle_cell17:
    for global_index_cell17, line_cell17 in enumerate(
        handle_cell17
    ):
        if not line_cell17.strip():
            continue

        record_cell17 = json.loads(line_cell17)
        operation_id_cell17 = str(
            record_cell17["op_id"]
        )

        if operation_id_cell17 in prompt_map_cell17:
            raise RuntimeError(
                "Duplicate op_id in prompt measurements: "
                f"{operation_id_cell17}"
            )

        chunk_number_cell17 = (
            global_index_cell17 // 250
        )

        has_payload_cell17 = (
            record_cell17.get(
                "rendered_prompt_tokens"
            )
            is not None
        )

        if has_payload_cell17:
            payload_record_index_cell17 = (
                payload_position_by_chunk_cell17[
                    chunk_number_cell17
                ]
            )

            payload_position_by_chunk_cell17[
                chunk_number_cell17
            ] += 1

            payload_chunk_cell17 = (
                payload_chunk_map_cell17[
                    chunk_number_cell17
                ]
            )

            record_cell17[
                "payload_chunk_relative_path"
            ] = payload_chunk_cell17[
                "relative_payload_path"
            ]

            record_cell17[
                "payload_chunk_sha256"
            ] = payload_chunk_cell17[
                "payload_sha256"
            ]

            record_cell17[
                "payload_record_index"
            ] = payload_record_index_cell17

        else:
            record_cell17[
                "payload_chunk_relative_path"
            ] = None

            record_cell17[
                "payload_chunk_sha256"
            ] = None

            record_cell17[
                "payload_record_index"
            ] = None

        prompt_map_cell17[
            operation_id_cell17
        ] = record_cell17

        prompt_record_count_cell17 += 1

if prompt_record_count_cell17 != 58585:
    raise RuntimeError(
        "Expected 58,585 prompt records."
    )

for (
    chunk_number_cell17,
    chunk_metadata_cell17,
) in payload_chunk_map_cell17.items():
    observed_count_cell17 = (
        payload_position_by_chunk_cell17[
            chunk_number_cell17
        ]
    )

    expected_count_cell17 = int(
        chunk_metadata_cell17[
            "payload_record_count"
        ]
    )

    if observed_count_cell17 != expected_count_cell17:
        raise RuntimeError(
            "Payload position reconciliation failed "
            f"for chunk {chunk_number_cell17}: "
            f"expected {expected_count_cell17}, "
            f"observed {observed_count_cell17}."
        )


# ---------------------------------------------------------------------
# Build portable eligibility and exclusion records
# ---------------------------------------------------------------------

import csv
with (STAGE_01_PROJECT_ROOT / "data/inventory/families.csv").open(newline="", encoding="utf-8") as f:
    family_rows = list(csv.DictReader(f))
families_by_spec = {row["spec_id"]: row["family_id"] for row in family_rows}
if (len(families_by_spec) != len(family_rows)
        or any(not row["family_id"].startswith("family_")
               or row["review_status"] not in {"AUTO_VERIFIED", "HUMAN_VERIFIED"}
               for row in family_rows)):
    raise RuntimeError("Stage 01 family mapping is incomplete or ambiguous.")
import shutil
shutil.copyfile(STAGE_01_PROJECT_ROOT / "data/inventory/families.csv",
                PROJECT_ROOT / "data/inventory/families.csv")

eligible_records_cell17 = []
exclusion_records_cell17 = []

eligibility_counts_cell17 = Counter()
eligible_method_counts_cell17 = Counter()
eligible_source_class_counts_cell17 = Counter()
eligible_dialect_counts_cell17 = Counter()
eligible_version_counts_cell17 = Counter()

exclusion_reason_counts_cell17 = Counter()
exclusion_source_class_counts_cell17 = Counter()
exclusion_method_counts_cell17 = Counter()

zero_prose_eligible_count_cell17 = 0
payload_reference_problem_count_cell17 = 0

for operation_id_cell17 in sorted(
    operation_map_cell17
):
    operation_cell17 = (
        operation_map_cell17[
            operation_id_cell17
        ]
    )

    if operation_cell17["family_id"] != families_by_spec[operation_cell17["spec_id"]]:
        raise RuntimeError("The operation audit lost its source family ID.")

    slice_cell17 = slice_map_cell17[
        operation_id_cell17
    ]

    prompt_cell17 = prompt_map_cell17[
        operation_id_cell17
    ]

    eligible_cell17 = (
        boolean_or_none_cell17(
            operation_cell17[
                "eligible_for_experimental_generation"
            ]
        )
        is True
    )

    common_record_cell17 = {
        "eligibility_version": "1.0",
        "op_id": operation_id_cell17,
        "family_id": families_by_spec[operation_cell17["spec_id"]],
        "spec_id":
            text_or_none_cell17(
                operation_cell17.get(
                    "spec_id"
                )
            ),
        "raw_id":
            text_or_none_cell17(
                operation_cell17.get(
                    "raw_id"
                )
            ),
        "provider_directory":
            text_or_none_cell17(
                operation_cell17.get(
                    "provider_directory"
                )
            ),
        "relative_path":
            text_or_none_cell17(
                operation_cell17.get(
                    "relative_path"
                )
            ),
        "source_class":
            text_or_none_cell17(
                operation_cell17.get(
                    "source_class"
                )
            ),
        "source_dialect":
            text_or_none_cell17(
                operation_cell17.get(
                    "dialect"
                )
            ),
        "canonical_openapi_version":
            text_or_none_cell17(
                operation_cell17.get(
                    "canonical_version"
                )
            ),
        "method":
            text_or_none_cell17(
                operation_cell17.get(
                    "method"
                )
            ),
        "path":
            text_or_none_cell17(
                operation_cell17.get(
                    "path"
                )
            ),
        "source_operation_id":
            text_or_none_cell17(
                operation_cell17.get(
                    "source_operation_id"
                )
            ),
        "slice_status":
            text_or_none_cell17(
                operation_cell17.get(
                    "slice_status"
                )
            ),
        "slice_relative_path":
            relative_project_path_cell17(
                prompt_cell17.get(
                    "slice_path"
                )
            ),
        "slice_sha256":
            text_or_none_cell17(
                operation_cell17.get(
                    "slice_sha256"
                )
            ),
        "slice_bytes":
            integer_or_none_cell17(
                operation_cell17.get(
                    "slice_bytes"
                )
            ),
        "reference_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "reference_count"
                )
            ),
        "reference_relocation_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "reference_relocation_count"
                )
            ),
        "parameter_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "parameter_count"
                )
            ),
        "request_body_present":
            boolean_or_none_cell17(
                operation_cell17.get(
                    "request_body_present"
                )
            ),
        "request_body_required":
            boolean_or_none_cell17(
                operation_cell17.get(
                    "request_body_required"
                )
            ),
        "response_entry_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "response_entry_count"
                )
            ),
        "security_scheme_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "security_scheme_count"
                )
            ),
        "schema_object_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "schema_object_count"
                )
            ),
        "schema_property_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "schema_property_count"
                )
            ),
        "maximum_schema_depth":
            integer_or_none_cell17(
                operation_cell17.get(
                    "maximum_schema_depth"
                )
            ),
        "summary_annotation_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "summary_annotation_count"
                )
            ),
        "description_annotation_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "description_annotation_count"
                )
            ),
        "included_annotation_count":
            integer_or_none_cell17(
                operation_cell17.get(
                    "included_annotation_count"
                )
            ),
        "prose_tokens_L0":
            integer_or_none_cell17(
                operation_cell17.get(
                    "prose_tokens_L0"
                )
            ),
        "prompt_status":
            text_or_none_cell17(
                prompt_cell17.get(
                    "status"
                )
            ),
        "rendered_prompt_tokens":
            integer_or_none_cell17(
                prompt_cell17.get(
                    "rendered_prompt_tokens"
                )
            ),
        "rendered_prompt_bytes":
            integer_or_none_cell17(
                prompt_cell17.get(
                    "rendered_prompt_bytes"
                )
            ),
        "input_token_limit":
            integer_or_none_cell17(
                prompt_cell17.get(
                    "input_token_limit"
                )
            ),
        "within_input_limit":
            prompt_cell17.get(
                "within_input_limit"
            ),
        "boundary":
            text_or_none_cell17(
                prompt_cell17.get(
                    "boundary"
                )
            ),
        "rendered_prompt_sha256":
            text_or_none_cell17(
                prompt_cell17.get(
                    "rendered_prompt_sha256"
                )
            ),
        "token_ids_sha256":
            text_or_none_cell17(
                prompt_cell17.get(
                    "token_ids_sha256"
                )
            ),
        "payload_chunk_relative_path":
            prompt_cell17.get(
                "payload_chunk_relative_path"
            ),
        "payload_chunk_sha256":
            prompt_cell17.get(
                "payload_chunk_sha256"
            ),
        "payload_record_index":
            prompt_cell17.get(
                "payload_record_index"
            ),
    }

    if eligible_cell17:
        eligibility_counts_cell17[
            "ELIGIBLE"
        ] += 1

        common_record_cell17[
            "eligible_for_experimental_generation"
        ] = True

        common_record_cell17[
            "eligibility_basis"
        ] = (
            "valid_L0_slice_and_rendered_prompt_within_5120_tokens"
        )

        if (
            common_record_cell17[
                "payload_chunk_relative_path"
            ]
            is None
            or common_record_cell17[
                "payload_record_index"
            ]
            is None
        ):
            payload_reference_problem_count_cell17 += 1

        eligible_records_cell17.append(
            common_record_cell17
        )

        eligible_method_counts_cell17[
            common_record_cell17["method"]
        ] += 1

        eligible_source_class_counts_cell17[
            common_record_cell17[
                "source_class"
            ]
        ] += 1

        eligible_dialect_counts_cell17[
            common_record_cell17[
                "source_dialect"
            ]
        ] += 1

        eligible_version_counts_cell17[
            common_record_cell17[
                "canonical_openapi_version"
            ]
        ] += 1

        if (
            common_record_cell17[
                "prose_tokens_L0"
            ] == 0
        ):
            zero_prose_eligible_count_cell17 += 1

    else:
        eligibility_counts_cell17[
            "INELIGIBLE"
        ] += 1

        exclusion_reason_cell17 = (
            text_or_none_cell17(
                operation_cell17.get(
                    "experimental_exclusion_reason"
                )
            )
        )

        if exclusion_reason_cell17 is None:
            exclusion_reason_cell17 = (
                "EXCLUDED_UNCLASSIFIED"
            )

        exclusion_record_cell17 = {
            **common_record_cell17,
            "eligible_for_experimental_generation":
                False,
            "exclusion_reason":
                exclusion_reason_cell17,
            "excess_prompt_tokens":
                integer_or_none_cell17(
                    operation_cell17.get(
                        "excess_prompt_tokens"
                    )
                ),
            "slice_diagnostics":
                slice_cell17.get(
                    "diagnostics"
                ),
        }

        exclusion_records_cell17.append(
            exclusion_record_cell17
        )

        exclusion_reason_counts_cell17[
            exclusion_reason_cell17
        ] += 1

        exclusion_source_class_counts_cell17[
            common_record_cell17[
                "source_class"
            ]
        ] += 1

        exclusion_method_counts_cell17[
            common_record_cell17["method"]
        ] += 1


# ---------------------------------------------------------------------
# Write deterministic registries
# ---------------------------------------------------------------------

write_text_atomic_cell17(
    ELIGIBLE_REGISTRY_PATH_CELL17,
    "".join(
        compact_json_line_cell17(record)
        for record in eligible_records_cell17
    ),
)

write_text_atomic_cell17(
    EXCLUSION_LEDGER_PATH_CELL17,
    "".join(
        compact_json_line_cell17(record)
        for record in exclusion_records_cell17
    ),
)


# ---------------------------------------------------------------------
# Reconciliation
# ---------------------------------------------------------------------

eligible_operation_ids_cell17 = {
    record["op_id"]
    for record in eligible_records_cell17
}

excluded_operation_ids_cell17 = {
    record["op_id"]
    for record in exclusion_records_cell17
}

eligible_specification_ids_cell17 = {
    record["spec_id"]
    for record in eligible_records_cell17
}

descriptive_cell17 = json.loads(
    (PROJECT_ROOT_CELL17 / "reports/audit/descriptive_statistics.json").read_text(encoding="utf-8")
)
operation_audit_cell17 = json.loads(
    (PROJECT_ROOT_CELL17 / "reports/audit/operation_audit_summary.json").read_text(encoding="utf-8")
)
if descriptive_cell17.get("status") != "PASS" or operation_audit_cell17.get("status") != "PASS":
    raise RuntimeError("Cells 14 and 15 must pass before eligibility reconciliation.")
if sha256_file_cell17(OPERATIONS_PATH_CELL17) != descriptive_cell17["operations_csv"]["sha256"]:
    raise RuntimeError("operations.csv differs from the file verified by Cell 15.")

reconciliation_checks_cell17 = {
    "operation_count_accounted": (
        len(eligible_records_cell17)
        + len(exclusion_records_cell17)
        == 58585
    ),
    "eligible_count_matches": (
        len(eligible_records_cell17) == descriptive_cell17["eligible_operation_count"]
    ),
    "exclusion_count_matches": (
        len(exclusion_records_cell17) == operation_audit_cell17["ineligible_operation_count"]
    ),
    "eligible_ids_unique": (
        len(eligible_operation_ids_cell17)
        == len(eligible_records_cell17)
    ),
    "excluded_ids_unique": (
        len(excluded_operation_ids_cell17)
        == len(exclusion_records_cell17)
    ),
    "eligible_and_excluded_disjoint": (
        not (
            eligible_operation_ids_cell17
            & excluded_operation_ids_cell17
        )
    ),
    "all_operation_ids_present": (
        eligible_operation_ids_cell17
        | excluded_operation_ids_cell17
        == set(operation_map_cell17)
    ),
    "invalid_slice_exclusion_count": (
        exclusion_reason_counts_cell17["EXCLUDED_INVALID_L0_SLICE"]
        == descriptive_cell17["operation_count"] - descriptive_cell17["valid_slice_count"]
    ),
    "prompt_limit_exclusion_count": (
        exclusion_reason_counts_cell17["EXCLUDED_RENDERED_PROMPT_TOO_LARGE"]
        == descriptive_cell17["prompt_over_limit_count"]
    ),
    "converted_swagger_eligible_count": (
        eligible_source_class_counts_cell17["converted_swagger"]
        == operation_audit_cell17["eligible_source_class_counts"].get("converted_swagger", 0)
    ),
    "native_openapi_eligible_count": (
        eligible_source_class_counts_cell17["native_openapi"]
        == operation_audit_cell17["eligible_source_class_counts"].get("native_openapi", 0)
    ),
    "payload_references_complete": (
        payload_reference_problem_count_cell17
        == 0
    ),
    "no_strata_selected": True,
}

status_cell17 = (
    "PASS"
    if all(
        reconciliation_checks_cell17.values()
    )
    else "BLOCKED"
)

report_cell17 = {
    "status": status_cell17,
    "stage":
        "FINAL_OPERATION_ELIGIBILITY_RECONCILIATION",
    "eligibility_version": "1.0",
    "operation_count": 58585,
    "eligible_operation_count":
        len(eligible_records_cell17),
    "excluded_operation_count":
        len(exclusion_records_cell17),
    "eligible_specification_count":
        len(eligible_specification_ids_cell17),
    "zero_prose_eligible_operation_count":
        zero_prose_eligible_count_cell17,
    "eligibility_counts": dict(
        sorted(eligibility_counts_cell17.items())
    ),
    "eligible_method_counts": dict(
        sorted(
            eligible_method_counts_cell17.items()
        )
    ),
    "eligible_source_class_counts": dict(
        sorted(
            eligible_source_class_counts_cell17.items()
        )
    ),
    "eligible_dialect_counts": dict(
        sorted(
            eligible_dialect_counts_cell17.items()
        )
    ),
    "eligible_canonical_version_counts": dict(
        sorted(
            eligible_version_counts_cell17.items()
        )
    ),
    "exclusion_reason_counts": dict(
        sorted(
            exclusion_reason_counts_cell17.items()
        )
    ),
    "exclusion_source_class_counts": dict(
        sorted(
            exclusion_source_class_counts_cell17.items()
        )
    ),
    "exclusion_method_counts": dict(
        sorted(
            exclusion_method_counts_cell17.items()
        )
    ),
    "payload_reference_problem_count":
        payload_reference_problem_count_cell17,
    "reconciliation_checks":
        reconciliation_checks_cell17,
    "strata_thresholds_selected": False,
    "model_outputs_used_for_eligibility": False,
    "eligible_registry": {
        "path": str(
            ELIGIBLE_REGISTRY_PATH_CELL17
        ),
        "bytes":
            ELIGIBLE_REGISTRY_PATH_CELL17
            .stat().st_size,
        "sha256":
            sha256_file_cell17(
                ELIGIBLE_REGISTRY_PATH_CELL17
            ),
        "record_count":
            len(eligible_records_cell17),
    },
    "exclusion_ledger": {
        "path": str(
            EXCLUSION_LEDGER_PATH_CELL17
        ),
        "bytes":
            EXCLUSION_LEDGER_PATH_CELL17
            .stat().st_size,
        "sha256":
            sha256_file_cell17(
                EXCLUSION_LEDGER_PATH_CELL17
            ),
        "record_count":
            len(exclusion_records_cell17),
    },
    "input_hashes": {
        "operations":
            sha256_file_cell17(
                OPERATIONS_PATH_CELL17
            ),
        "prompt_measurements":
            sha256_file_cell17(
                PROMPT_MEASUREMENTS_PATH_CELL17
            ),
        "slice_records":
            sha256_file_cell17(
                SLICE_RECORDS_PATH_CELL17
            ),
        "prompt_payload_index":
            sha256_file_cell17(
                PROMPT_PAYLOAD_INDEX_PATH_CELL17
            ),
    },
    "duration_seconds": round(
        time.perf_counter()
        - start_time_cell17,
        3,
    ),
    "interpretation": (
        "Only operations with a valid L0 slice and an "
        "exact rendered prompt of at most 5,120 tokens "
        "are eligible for later experimental sampling."
    ),
}

write_json_atomic_cell17(
    REPORT_PATH_CELL17,
    report_cell17,
)

report_cell17["report_path"] = str(
    REPORT_PATH_CELL17
)

report_cell17["report_sha256"] = (
    sha256_file_cell17(
        REPORT_PATH_CELL17
    )
)

print(
    json.dumps(
        report_cell17,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell17 != "PASS":
    raise RuntimeError(
        "Final operation eligibility reconciliation "
        "failed. Review reconciliation_checks above."
    )

print("\nNOTEBOOK 02 — CELL 17: PASS")


{
  "status": "PASS",
  "stage": "FINAL_OPERATION_ELIGIBILITY_RECONCILIATION",
  "eligibility_version": "1.0",
  "operation_count": 58585,
  "eligible_operation_count": 52922,
  "excluded_operation_count": 5663,
  "eligible_specification_count": 1925,
  "zero_prose_eligible_operation_count": 311,
  "eligibility_counts": {
    "ELIGIBLE": 52922,
    "INELIGIBLE": 5663
  },
  "eligible_method_counts": {
    "delete": 4790,
    "get": 24181,
    "head": 49,
    "options": 67,
    "patch": 1810,
    "post": 18474,
    "put": 3546,
    "trace": 5
  },
  "eligible_source_class_counts": {
    "converted_swagger": 2517,
    "native_openapi": 50405
  },
  "eligible_dialect_counts": {
    "openapi_3_0": 49282,
    "openapi_3_1": 1123,
    "swagger_2_0": 2517
  },
  "eligible_canonical_version_counts": {
    "3.0.0": 39462,
    "3.0.1": 6167,
    "3.0.2": 2430,
    "3.0.3": 3740,
    "3.1.0": 1123
  },
  "exclusion_reason_counts": {
    "EXCLUDED_INVALID_L0_SLICE": 1,
    "EXCLUDED_RENDERED_PROMP

In [19]:
# Notebook 02 — Cell 18
# Final Stage 02 validation before manifest creation and packaging.
#
# This cell verifies:
# - required artifacts exist and are nonempty
# - recorded hashes still match
# - operation IDs and eligibility ledgers reconcile
# - all valid slice paths exist
# - prompt payload chunks and record positions remain valid
# - downstream Stage 03 may begin
#
# It does not build the manifest or ZIP; those are Cells 19 and 20.

from pathlib import Path
from collections import Counter
import csv
import hashlib
import json
import os
import time

import yaml


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL18 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

REPORT_ROOT_CELL18 = (
    PROJECT_ROOT_CELL18
    / "reports/audit"
)

DATA_ROOT_CELL18 = (
    PROJECT_ROOT_CELL18
    / "data"
)

CONFIG_ROOT_CELL18 = (
    PROJECT_ROOT_CELL18
    / "config"
)

OPERATIONS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/operations.csv"
)

ELIGIBLE_REGISTRY_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "inventory/final_eligible_operations.jsonl"
)

EXCLUSION_LEDGER_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "inventory/operation_exclusions.jsonl"
)

PROSE_FIELDS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/prose_fields.csv"
)

PROSE_MEASUREMENTS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/prose_measurements.jsonl"
)

PROMPT_MEASUREMENTS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/rendered_prompt_measurements.jsonl"
)

PROMPT_PAYLOAD_INDEX_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/rendered_prompt_payload_index.jsonl"
)

PROMPT_PAYLOAD_ROOT_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/rendered_prompt_payload_chunks"
)

SLICE_RECORDS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/slice_records.jsonl"
)

POINTER_MAPS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/slice_pointer_maps.jsonl"
)

REFERENCE_RELOCATIONS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/reference_relocations.jsonl"
)

MANUAL_SELECTION_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/manual_trace_selection.csv"
)

MANUAL_RESULTS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/manual_trace_results.jsonl"
)

MANUAL_ROWS_PATH_CELL18 = (
    DATA_ROOT_CELL18
    / "audits/manual_trace_prose_rows.csv"
)

FULL_SLICE_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "full_slice_construction.json"
)

PROSE_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "prose_measurement.json"
)

PROMPT_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "rendered_prompt_measurement.json"
)

OPERATION_AUDIT_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "operation_audit_summary.json"
)

DESCRIPTIVE_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "descriptive_statistics.json"
)

MANUAL_TRACE_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "manual_trace_summary.json"
)

ELIGIBILITY_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "final_eligibility_reconciliation.json"
)

NUMERICAL_STATISTICS_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "numerical_operation_statistics.csv"
)

CATEGORICAL_STATISTICS_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "categorical_operation_statistics.csv"
)

DISTRIBUTION_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "prose_distribution_summary.json"
)

LINEAR_FIGURE_PATH_CELL18 = (
    PROJECT_ROOT_CELL18
    / "reports/figures/prose_tokens_distribution.png"
)

LOG_FIGURE_PATH_CELL18 = (
    PROJECT_ROOT_CELL18
    / "reports/figures/prose_tokens_distribution_log.png"
)

STAGE_02_POLICY_PATH_CELL18 = (
    CONFIG_ROOT_CELL18
    / "stage_02.yaml"
)

FINAL_REPORT_PATH_CELL18 = (
    REPORT_ROOT_CELL18
    / "stage_02_final_validation.json"
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell18(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def write_json_atomic_cell18(path, value):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def nonempty_file_cell18(path):
    path = Path(path)

    return (
        path.is_file()
        and path.stat().st_size > 0
    )


def relative_project_path_cell18(path):
    return Path(path).relative_to(
        PROJECT_ROOT_CELL18
    ).as_posix()


def load_json_cell18(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def add_problem_cell18(
    problems,
    problem_type,
    path=None,
    expected=None,
    observed=None,
    diagnostic=None,
):
    record = {
        "type": problem_type,
    }

    if path is not None:
        record["path"] = str(path)

    if expected is not None:
        record["expected"] = expected

    if observed is not None:
        record["observed"] = observed

    if diagnostic is not None:
        record["diagnostic"] = diagnostic

    problems.append(record)


# ---------------------------------------------------------------------
# Required artifacts
# ---------------------------------------------------------------------

required_artifacts_cell18 = [
    STAGE_02_POLICY_PATH_CELL18,
    CONFIG_ROOT_CELL18 / "environment.json",
    CONFIG_ROOT_CELL18 / "requirements.lock.txt",
    DATA_ROOT_CELL18
    / "inventory/eligible_specifications.jsonl",
    DATA_ROOT_CELL18
    / "inventory/operations_enumerated.jsonl",
    DATA_ROOT_CELL18
    / "inventory/specification_operation_counts.csv",
    ELIGIBLE_REGISTRY_PATH_CELL18,
    EXCLUSION_LEDGER_PATH_CELL18,
    OPERATIONS_PATH_CELL18,
    PROSE_FIELDS_PATH_CELL18,
    PROSE_MEASUREMENTS_PATH_CELL18,
    PROMPT_MEASUREMENTS_PATH_CELL18,
    PROMPT_PAYLOAD_INDEX_PATH_CELL18,
    SLICE_RECORDS_PATH_CELL18,
    POINTER_MAPS_PATH_CELL18,
    REFERENCE_RELOCATIONS_PATH_CELL18,
    MANUAL_SELECTION_PATH_CELL18,
    MANUAL_RESULTS_PATH_CELL18,
    MANUAL_ROWS_PATH_CELL18,
    FULL_SLICE_REPORT_PATH_CELL18,
    PROSE_REPORT_PATH_CELL18,
    PROMPT_REPORT_PATH_CELL18,
    OPERATION_AUDIT_REPORT_PATH_CELL18,
    DESCRIPTIVE_REPORT_PATH_CELL18,
    MANUAL_TRACE_REPORT_PATH_CELL18,
    ELIGIBILITY_REPORT_PATH_CELL18,
    NUMERICAL_STATISTICS_PATH_CELL18,
    CATEGORICAL_STATISTICS_PATH_CELL18,
    DISTRIBUTION_REPORT_PATH_CELL18,
    LINEAR_FIGURE_PATH_CELL18,
    LOG_FIGURE_PATH_CELL18,
]

problems_cell18 = []

for artifact_path_cell18 in required_artifacts_cell18:
    if not nonempty_file_cell18(
        artifact_path_cell18
    ):
        add_problem_cell18(
            problems_cell18,
            "MISSING_OR_EMPTY_REQUIRED_ARTIFACT",
            path=artifact_path_cell18,
        )


if problems_cell18:
    preliminary_report_cell18 = {
        "status": "BLOCKED",
        "stage": "02",
        "problems": problems_cell18,
    }

    write_json_atomic_cell18(
        FINAL_REPORT_PATH_CELL18,
        preliminary_report_cell18,
    )

    print(
        json.dumps(
            preliminary_report_cell18,
            indent=2,
        )
    )

    raise RuntimeError(
        "Required Stage 02 artifacts are missing "
        "or empty."
    )


# ---------------------------------------------------------------------
# Load authoritative reports
# ---------------------------------------------------------------------

start_time_cell18 = time.perf_counter()

full_slice_report_cell18 = load_json_cell18(
    FULL_SLICE_REPORT_PATH_CELL18
)

prose_report_cell18 = load_json_cell18(
    PROSE_REPORT_PATH_CELL18
)

prompt_report_cell18 = load_json_cell18(
    PROMPT_REPORT_PATH_CELL18
)

operation_audit_report_cell18 = (
    load_json_cell18(
        OPERATION_AUDIT_REPORT_PATH_CELL18
    )
)

descriptive_report_cell18 = load_json_cell18(
    DESCRIPTIVE_REPORT_PATH_CELL18
)

manual_trace_report_cell18 = load_json_cell18(
    MANUAL_TRACE_REPORT_PATH_CELL18
)

eligibility_report_cell18 = load_json_cell18(
    ELIGIBILITY_REPORT_PATH_CELL18
)

stage_02_policy_cell18 = yaml.safe_load(
    STAGE_02_POLICY_PATH_CELL18.read_text(
        encoding="utf-8"
    )
)


# ---------------------------------------------------------------------
# Verify report statuses
# ---------------------------------------------------------------------

reports_to_check_cell18 = {
    "full_slice_construction":
        full_slice_report_cell18,
    "prose_measurement":
        prose_report_cell18,
    "rendered_prompt_measurement":
        prompt_report_cell18,
    "operation_audit":
        operation_audit_report_cell18,
    "descriptive_statistics":
        descriptive_report_cell18,
    "manual_trace":
        manual_trace_report_cell18,
    "eligibility_reconciliation":
        eligibility_report_cell18,
}

report_statuses_cell18 = {}

for report_name_cell18, report_cell18 in (
    reports_to_check_cell18.items()
):
    status_cell18 = str(
        report_cell18.get(
            "status",
            "MISSING",
        )
    ).upper()

    report_statuses_cell18[
        report_name_cell18
    ] = status_cell18

    if status_cell18 != "PASS":
        add_problem_cell18(
            problems_cell18,
            "REPORT_STATUS_NOT_PASS",
            path=report_name_cell18,
            expected="PASS",
            observed=status_cell18,
        )


# ---------------------------------------------------------------------
# Verify recorded hashes
# ---------------------------------------------------------------------

hash_expectations_cell18 = [
    (
        SLICE_RECORDS_PATH_CELL18,
        full_slice_report_cell18[
            "slice_records"
        ]["sha256"],
        "slice_records",
    ),
    (
        POINTER_MAPS_PATH_CELL18,
        full_slice_report_cell18[
            "pointer_maps"
        ]["sha256"],
        "slice_pointer_maps",
    ),
    (
        REFERENCE_RELOCATIONS_PATH_CELL18,
        full_slice_report_cell18[
            "reference_relocations"
        ]["sha256"],
        "reference_relocations",
    ),
    (
        PROSE_FIELDS_PATH_CELL18,
        prose_report_cell18[
            "prose_fields"
        ]["sha256"],
        "prose_fields",
    ),
    (
        PROSE_MEASUREMENTS_PATH_CELL18,
        prose_report_cell18[
            "prose_measurements"
        ]["sha256"],
        "prose_measurements",
    ),
    (
        PROMPT_MEASUREMENTS_PATH_CELL18,
        prompt_report_cell18[
            "measurements"
        ]["sha256"],
        "prompt_measurements",
    ),
    (
        PROMPT_PAYLOAD_INDEX_PATH_CELL18,
        prompt_report_cell18[
            "exact_payload_storage"
        ]["index_sha256"],
        "prompt_payload_index",
    ),
    (
        OPERATIONS_PATH_CELL18,
        operation_audit_report_cell18[
            "operations_csv"
        ]["sha256"],
        "operations_csv_operation_audit",
    ),
    (
        OPERATIONS_PATH_CELL18,
        descriptive_report_cell18[
            "operations_csv"
        ]["sha256"],
        "operations_csv_descriptive_report",
    ),
    (
        ELIGIBLE_REGISTRY_PATH_CELL18,
        eligibility_report_cell18[
            "eligible_registry"
        ]["sha256"],
        "eligible_registry",
    ),
    (
        EXCLUSION_LEDGER_PATH_CELL18,
        eligibility_report_cell18[
            "exclusion_ledger"
        ]["sha256"],
        "exclusion_ledger",
    ),
    (
        MANUAL_SELECTION_PATH_CELL18,
        manual_trace_report_cell18[
            "selection"
        ]["sha256"],
        "manual_trace_selection",
    ),
    (
        MANUAL_RESULTS_PATH_CELL18,
        manual_trace_report_cell18[
            "trace_results"
        ]["sha256"],
        "manual_trace_results",
    ),
    (
        MANUAL_ROWS_PATH_CELL18,
        manual_trace_report_cell18[
            "trace_prose_rows"
        ]["sha256"],
        "manual_trace_prose_rows",
    ),
    (
        NUMERICAL_STATISTICS_PATH_CELL18,
        descriptive_report_cell18[
            "numerical_statistics"
        ]["sha256"],
        "numerical_statistics",
    ),
    (
        CATEGORICAL_STATISTICS_PATH_CELL18,
        descriptive_report_cell18[
            "categorical_statistics"
        ]["sha256"],
        "categorical_statistics",
    ),
    (
        DISTRIBUTION_REPORT_PATH_CELL18,
        descriptive_report_cell18[
            "distribution_summary"
        ]["sha256"],
        "distribution_summary",
    ),
]

hash_verification_records_cell18 = []

for (
    artifact_path_cell18,
    expected_sha_cell18,
    label_cell18,
) in hash_expectations_cell18:
    observed_sha_cell18 = sha256_file_cell18(
        artifact_path_cell18
    )

    matched_cell18 = (
        observed_sha_cell18
        == expected_sha_cell18
    )

    hash_verification_records_cell18.append(
        {
            "label": label_cell18,
            "path": str(artifact_path_cell18),
            "expected_sha256":
                expected_sha_cell18,
            "observed_sha256":
                observed_sha_cell18,
            "matches": matched_cell18,
        }
    )

    if not matched_cell18:
        add_problem_cell18(
            problems_cell18,
            "ARTIFACT_HASH_MISMATCH",
            path=artifact_path_cell18,
            expected=expected_sha_cell18,
            observed=observed_sha_cell18,
        )

    if (
        len(hash_verification_records_cell18)
        % 5 == 0
    ):
        print(
            json.dumps(
                {
                    "hashes_verified":
                        len(
                            hash_verification_records_cell18
                        ),
                    "hash_total":
                        len(
                            hash_expectations_cell18
                        ),
                    "problem_count":
                        len(problems_cell18),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - start_time_cell18,
                        1,
                    ),
                }
            )
        )


# ---------------------------------------------------------------------
# Validate operations.csv
# ---------------------------------------------------------------------

operation_count_cell18 = 0
operation_ids_cell18 = set()
eligible_operation_ids_from_csv_cell18 = set()
excluded_operation_ids_from_csv_cell18 = set()

csv_exclusion_reason_counts_cell18 = Counter()
csv_source_class_counts_cell18 = Counter()
csv_dialect_counts_cell18 = Counter()

with OPERATIONS_PATH_CELL18.open(
    "r",
    encoding="utf-8",
    newline="",
) as handle_cell18:
    reader_cell18 = csv.DictReader(
        handle_cell18
    )

    required_columns_cell18 = {
        "op_id",
        "spec_id",
        "source_class",
        "dialect",
        "canonical_version",
        "slice_status",
        "prompt_status",
        "prose_tokens_L0",
        "rendered_prompt_tokens",
        "eligible_for_experimental_generation",
        "experimental_exclusion_reason",
    }

    missing_columns_cell18 = (
        required_columns_cell18
        - set(reader_cell18.fieldnames or [])
    )

    if missing_columns_cell18:
        add_problem_cell18(
            problems_cell18,
            "OPERATIONS_CSV_MISSING_COLUMNS",
            diagnostic=", ".join(
                sorted(missing_columns_cell18)
            ),
        )

    for row_cell18 in reader_cell18:
        operation_count_cell18 += 1

        operation_id_cell18 = str(
            row_cell18["op_id"]
        )

        if operation_id_cell18 in operation_ids_cell18:
            add_problem_cell18(
                problems_cell18,
                "DUPLICATE_OPERATION_ID",
                observed=operation_id_cell18,
            )

        operation_ids_cell18.add(
            operation_id_cell18
        )

        eligible_cell18 = (
            str(
                row_cell18[
                    "eligible_for_experimental_generation"
                ]
            ).strip().lower()
            == "true"
        )

        if eligible_cell18:
            eligible_operation_ids_from_csv_cell18.add(
                operation_id_cell18
            )
        else:
            excluded_operation_ids_from_csv_cell18.add(
                operation_id_cell18
            )

            csv_exclusion_reason_counts_cell18[
                row_cell18[
                    "experimental_exclusion_reason"
                ]
            ] += 1

        csv_source_class_counts_cell18[
            row_cell18["source_class"]
        ] += 1

        csv_dialect_counts_cell18[
            row_cell18["dialect"]
        ] += 1


# ---------------------------------------------------------------------
# Validate eligible and exclusion ledgers
# ---------------------------------------------------------------------

eligible_registry_ids_cell18 = set()
excluded_ledger_ids_cell18 = set()

eligible_slice_paths_cell18 = {}
excluded_slice_paths_cell18 = {}

eligible_payload_references_cell18 = []
excluded_payload_references_cell18 = []

eligible_record_count_cell18 = 0
exclusion_record_count_cell18 = 0

with ELIGIBLE_REGISTRY_PATH_CELL18.open(
    "r",
    encoding="utf-8",
) as handle_cell18:
    for line_cell18 in handle_cell18:
        if not line_cell18.strip():
            continue

        record_cell18 = json.loads(line_cell18)

        operation_id_cell18 = str(
            record_cell18["op_id"]
        )

        eligible_record_count_cell18 += 1

        if operation_id_cell18 in eligible_registry_ids_cell18:
            add_problem_cell18(
                problems_cell18,
                "DUPLICATE_ELIGIBLE_OPERATION_ID",
                observed=operation_id_cell18,
            )

        eligible_registry_ids_cell18.add(
            operation_id_cell18
        )

        slice_relative_path_cell18 = (
            record_cell18.get(
                "slice_relative_path"
            )
        )

        eligible_slice_paths_cell18[
            operation_id_cell18
        ] = slice_relative_path_cell18

        eligible_payload_references_cell18.append(
            {
                "op_id": operation_id_cell18,
                "chunk":
                    record_cell18.get(
                        "payload_chunk_relative_path"
                    ),
                "record_index":
                    record_cell18.get(
                        "payload_record_index"
                    ),
            }
        )

with EXCLUSION_LEDGER_PATH_CELL18.open(
    "r",
    encoding="utf-8",
) as handle_cell18:
    for line_cell18 in handle_cell18:
        if not line_cell18.strip():
            continue

        record_cell18 = json.loads(line_cell18)

        operation_id_cell18 = str(
            record_cell18["op_id"]
        )

        exclusion_record_count_cell18 += 1

        if operation_id_cell18 in excluded_ledger_ids_cell18:
            add_problem_cell18(
                problems_cell18,
                "DUPLICATE_EXCLUDED_OPERATION_ID",
                observed=operation_id_cell18,
            )

        excluded_ledger_ids_cell18.add(
            operation_id_cell18
        )

        slice_relative_path_cell18 = (
            record_cell18.get(
                "slice_relative_path"
            )
        )

        if slice_relative_path_cell18:
            excluded_slice_paths_cell18[
                operation_id_cell18
            ] = slice_relative_path_cell18

        if (
            record_cell18.get(
                "rendered_prompt_tokens"
            )
            is not None
        ):
            excluded_payload_references_cell18.append(
                {
                    "op_id": operation_id_cell18,
                    "chunk":
                        record_cell18.get(
                            "payload_chunk_relative_path"
                        ),
                    "record_index":
                        record_cell18.get(
                            "payload_record_index"
                        ),
                }
            )


# ---------------------------------------------------------------------
# Validate slice-file references
# ---------------------------------------------------------------------

all_valid_slice_references_cell18 = {
    **eligible_slice_paths_cell18,
    **excluded_slice_paths_cell18,
}

missing_slice_examples_cell18 = []
unique_slice_paths_cell18 = set()

for (
    operation_id_cell18,
    relative_path_cell18,
) in all_valid_slice_references_cell18.items():
    if not relative_path_cell18:
        if len(missing_slice_examples_cell18) < 20:
            missing_slice_examples_cell18.append(
                {
                    "op_id": operation_id_cell18,
                    "problem":
                        "MISSING_SLICE_RELATIVE_PATH",
                }
            )

        continue

    slice_path_cell18 = (
        PROJECT_ROOT_CELL18
        / relative_path_cell18
    )

    unique_slice_paths_cell18.add(
        str(relative_path_cell18)
    )

    if not nonempty_file_cell18(
        slice_path_cell18
    ):
        if len(missing_slice_examples_cell18) < 20:
            missing_slice_examples_cell18.append(
                {
                    "op_id": operation_id_cell18,
                    "path":
                        str(slice_path_cell18),
                    "problem":
                        "SLICE_FILE_MISSING_OR_EMPTY",
                }
            )

if missing_slice_examples_cell18:
    add_problem_cell18(
        problems_cell18,
        "SLICE_REFERENCE_VALIDATION_FAILED",
        observed=len(
            missing_slice_examples_cell18
        ),
        diagnostic=json.dumps(
            missing_slice_examples_cell18,
            ensure_ascii=False,
        ),
    )


# ---------------------------------------------------------------------
# Validate payload references
# ---------------------------------------------------------------------

payload_chunk_metadata_cell18 = {}

with PROMPT_PAYLOAD_INDEX_PATH_CELL18.open(
    "r",
    encoding="utf-8",
) as handle_cell18:
    for line_cell18 in handle_cell18:
        if not line_cell18.strip():
            continue

        record_cell18 = json.loads(line_cell18)

        payload_path_cell18 = Path(
            record_cell18["payload_path"]
        )

        relative_payload_path_cell18 = (
            relative_project_path_cell18(
                payload_path_cell18
            )
        )

        payload_chunk_metadata_cell18[
            relative_payload_path_cell18
        ] = {
            "record_count": int(
                record_cell18[
                    "payload_record_count"
                ]
            ),
            "sha256":
                record_cell18[
                    "payload_sha256"
                ],
            "path":
                payload_path_cell18,
        }

payload_reference_problem_examples_cell18 = []

for payload_reference_cell18 in (
    eligible_payload_references_cell18
    + excluded_payload_references_cell18
):
    chunk_cell18 = payload_reference_cell18[
        "chunk"
    ]

    record_index_cell18 = (
        payload_reference_cell18[
            "record_index"
        ]
    )

    chunk_metadata_cell18 = (
        payload_chunk_metadata_cell18.get(
            chunk_cell18
        )
    )

    if chunk_metadata_cell18 is None:
        if len(
            payload_reference_problem_examples_cell18
        ) < 20:
            payload_reference_problem_examples_cell18.append(
                {
                    **payload_reference_cell18,
                    "problem":
                        "PAYLOAD_CHUNK_NOT_IN_INDEX",
                }
            )

        continue

    if (
        record_index_cell18 is None
        or not (
            0
            <= int(record_index_cell18)
            < chunk_metadata_cell18[
                "record_count"
            ]
        )
    ):
        if len(
            payload_reference_problem_examples_cell18
        ) < 20:
            payload_reference_problem_examples_cell18.append(
                {
                    **payload_reference_cell18,
                    "problem":
                        "PAYLOAD_RECORD_INDEX_OUT_OF_RANGE",
                }
            )

if payload_reference_problem_examples_cell18:
    add_problem_cell18(
        problems_cell18,
        "PAYLOAD_REFERENCE_VALIDATION_FAILED",
        observed=len(
            payload_reference_problem_examples_cell18
        ),
        diagnostic=json.dumps(
            payload_reference_problem_examples_cell18,
            ensure_ascii=False,
        ),
    )


# ---------------------------------------------------------------------
# Cross-artifact reconciliation checks
# ---------------------------------------------------------------------

reconciliation_checks_cell18 = {
    "operation_count_is_58585": (
        operation_count_cell18 == 58585
    ),
    "unique_operation_count_is_58585": (
        len(operation_ids_cell18) == 58585
    ),
    "eligible_csv_count_matches_prompt_measurement": (
        len(eligible_operation_ids_from_csv_cell18) == prompt_report_cell18["within_input_limit_count"]
    ),
    "excluded_csv_count_matches_prompt_measurement": (
        len(excluded_operation_ids_from_csv_cell18)
        == prompt_report_cell18["exceeds_input_limit_count"] + prompt_report_cell18["excluded_slice_count"]
    ),
    "eligible_registry_count_matches_cell17": (
        eligible_record_count_cell18 == eligibility_report_cell18["eligible_operation_count"]
    ),
    "exclusion_ledger_count_matches_cell17": (
        exclusion_record_count_cell18 == eligibility_report_cell18["excluded_operation_count"]
    ),
    "eligible_registry_matches_csv": (
        eligible_registry_ids_cell18
        == eligible_operation_ids_from_csv_cell18
    ),
    "exclusion_ledger_matches_csv": (
        excluded_ledger_ids_cell18
        == excluded_operation_ids_from_csv_cell18
    ),
    "eligible_and_excluded_disjoint": (
        not (
            eligible_registry_ids_cell18
            & excluded_ledger_ids_cell18
        )
    ),
    "all_operations_accounted": (
        eligible_registry_ids_cell18
        | excluded_ledger_ids_cell18
        == operation_ids_cell18
    ),
    "valid_slice_reference_count_is_58584": (
        len(all_valid_slice_references_cell18)
        == 58584
    ),
    "unique_valid_slice_path_count_is_58584": (
        len(unique_slice_paths_cell18)
        == 58584
    ),
    "payload_chunk_count_is_235": (
        len(payload_chunk_metadata_cell18)
        == 235
    ),
    "payload_reference_count_is_58584": (
        len(
            eligible_payload_references_cell18
        )
        + len(
            excluded_payload_references_cell18
        )
        == 58584
    ),
    "prompt_limit_exclusion_count_matches_measurement": (
        csv_exclusion_reason_counts_cell18["EXCLUDED_RENDERED_PROMPT_TOO_LARGE"]
        == prompt_report_cell18["exceeds_input_limit_count"]
        == descriptive_report_cell18["prompt_over_limit_count"]
    ),
    "invalid_slice_exclusion_count_is_1": (
        csv_exclusion_reason_counts_cell18[
            "EXCLUDED_INVALID_L0_SLICE"
        ]
        == 1
    ),
    "source_class_counts_match": (
        dict(
            sorted(
                csv_source_class_counts_cell18.items()
            )
        )
        == {
            "converted_swagger": 2576,
            "native_openapi": 56009,
        }
    ),
    "dialect_counts_match": (
        dict(
            sorted(
                csv_dialect_counts_cell18.items()
            )
        )
        == {
            "openapi_3_0": 54504,
            "openapi_3_1": 1505,
            "swagger_2_0": 2576,
        }
    ),
    "all_registered_hashes_match": all(
        record_cell18["matches"]
        for record_cell18
        in hash_verification_records_cell18
    ),
    "all_required_reports_pass": all(
        value_cell18 == "PASS"
        for value_cell18
        in report_statuses_cell18.values()
    ),
    "manual_trace_count_is_12": (
        manual_trace_report_cell18[
            "selected_operation_count"
        ]
        == 12
    ),
    "manual_traces_all_pass": (
        manual_trace_report_cell18[
            "status_counts"
        ].get("PASS")
        == 12
    ),
    "strata_thresholds_not_selected": (
        eligibility_report_cell18[
            "strata_thresholds_selected"
        ]
        is False
        and descriptive_report_cell18[
            "strata_thresholds_selected"
        ]
        is False
    ),
    "model_outputs_not_used": (
        eligibility_report_cell18[
            "model_outputs_used_for_eligibility"
        ]
        is False
    ),
}

for (
    check_name_cell18,
    check_pass_cell18,
) in reconciliation_checks_cell18.items():
    if not check_pass_cell18:
        add_problem_cell18(
            problems_cell18,
            "RECONCILIATION_CHECK_FAILED",
            path=check_name_cell18,
            expected=True,
            observed=False,
        )


# ---------------------------------------------------------------------
# Final gate
# ---------------------------------------------------------------------

status_cell18 = (
    "PASS"
    if not problems_cell18
    else "BLOCKED"
)

report_cell18 = {
    "status": status_cell18,
    "stage": "02",
    "gate":
        "FINAL_PRE_MANIFEST_VALIDATION",
    "operation_count":
        operation_count_cell18,
    "eligible_operation_count":
        len(eligible_registry_ids_cell18),
    "excluded_operation_count":
        len(excluded_ledger_ids_cell18),
    "valid_slice_count":
        len(all_valid_slice_references_cell18),
    "prompt_payload_chunk_count":
        len(payload_chunk_metadata_cell18),
    "required_artifact_count":
        len(required_artifacts_cell18),
    "required_artifacts_nonempty": (
        len(
            [
                path
                for path in required_artifacts_cell18
                if nonempty_file_cell18(path)
            ]
        )
    ),
    "hash_verification_count":
        len(hash_verification_records_cell18),
    "hash_verifications":
        hash_verification_records_cell18,
    "report_statuses":
        report_statuses_cell18,
    "source_class_counts": dict(
        sorted(
            csv_source_class_counts_cell18.items()
        )
    ),
    "dialect_counts": dict(
        sorted(
            csv_dialect_counts_cell18.items()
        )
    ),
    "exclusion_reason_counts": dict(
        sorted(
            csv_exclusion_reason_counts_cell18.items()
        )
    ),
    "reconciliation_checks":
        reconciliation_checks_cell18,
    "problem_count":
        len(problems_cell18),
    "problems":
        problems_cell18[:100],
    "may_begin_notebook_03": (
        status_cell18 == "PASS"
    ),
    "may_begin_experimental_generation": False,
    "strata_thresholds_selected": False,
    "interpretation": (
        "Stage 02 corpus slicing, prose measurement, "
        "prompt qualification, eligibility filtering, "
        "statistics, and manual traces are complete. "
        "Notebook 03 may construct paired inputs and establish the eligible frame."
        if status_cell18 == "PASS"
        else
        "Stage 02 has unresolved validation problems."
    ),
    "duration_seconds": round(
        time.perf_counter()
        - start_time_cell18,
        3,
    ),
}

write_json_atomic_cell18(
    FINAL_REPORT_PATH_CELL18,
    report_cell18,
)

report_cell18["report_path"] = str(
    FINAL_REPORT_PATH_CELL18
)

report_cell18["report_sha256"] = (
    sha256_file_cell18(
        FINAL_REPORT_PATH_CELL18
    )
)

print(
    json.dumps(
        report_cell18,
        indent=2,
        ensure_ascii=False,
    )
)

if status_cell18 != "PASS":
    raise RuntimeError(
        "Stage 02 final validation failed. "
        "Review problems and reconciliation_checks."
    )

print("\nNOTEBOOK 02 — CELL 18: PASS")


{"hashes_verified": 5, "hash_total": 17, "problem_count": 0, "elapsed_seconds": 4.4}
{"hashes_verified": 10, "hash_total": 17, "problem_count": 0, "elapsed_seconds": 5.2}
{"hashes_verified": 15, "hash_total": 17, "problem_count": 0, "elapsed_seconds": 5.2}
{
  "status": "PASS",
  "stage": "02",
  "gate": "FINAL_PRE_MANIFEST_VALIDATION",
  "operation_count": 58585,
  "eligible_operation_count": 52922,
  "excluded_operation_count": 5663,
  "valid_slice_count": 58584,
  "prompt_payload_chunk_count": 235,
  "required_artifact_count": 31,
  "required_artifacts_nonempty": 31,
  "hash_verification_count": 17,
  "hash_verifications": [
    {
      "label": "slice_records",
      "path": "/kaggle/working/fresh/stage_02/prose_operation_experiment/data/audits/slice_records.jsonl",
      "expected_sha256": "f0cae128c491d57349829840ae0fae681929c72a201b458429bb47115f872d76",
      "observed_sha256": "f0cae128c491d57349829840ae0fae681929c72a201b458429bb47115f872d76",
      "matches": true
    },
    

In [20]:
# Notebook 02 — Cell 19
# Create the internal Stage 02 artifact manifest.
#
# Included:
# - frozen configuration
# - inventories and final eligibility ledgers
# - L0 slices and pointer maps
# - prose and rendered-prompt audit artifacts
# - exact compressed prompt/token payloads
# - reports, figures, qualifications, and final validation
#
# Excluded:
# - resumable intermediate checkpoint files
# - temporary files
# - Python caches
#
# The internal manifest does not include itself.

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import time


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL19 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

REGISTRY_ROOT_CELL19 = (
    PROJECT_ROOT_CELL19
    / "registries"
)

MANIFEST_PATH_CELL19 = (
    REGISTRY_ROOT_CELL19
    / "02_manifest.json"
)

FINAL_VALIDATION_PATH_CELL19 = (
    PROJECT_ROOT_CELL19
    / "reports/audit/stage_02_final_validation.json"
)

ELIGIBILITY_REPORT_PATH_CELL19 = (
    PROJECT_ROOT_CELL19
    / "reports/audit/final_eligibility_reconciliation.json"
)

STAGE_02_POLICY_PATH_CELL19 = (
    PROJECT_ROOT_CELL19
    / "config/stage_02.yaml"
)

REGISTRY_ROOT_CELL19.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell19(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def sha256_text_cell19(value):
    return hashlib.sha256(
        value.encode("utf-8")
    ).hexdigest()


def canonical_json_cell19(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    )


def write_json_atomic_cell19(path, value):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        path,
    )


def relative_path_cell19(path):
    return Path(path).relative_to(
        PROJECT_ROOT_CELL19
    ).as_posix()


def should_exclude_cell19(path):
    relative_path = Path(path).relative_to(
        PROJECT_ROOT_CELL19
    )

    relative_text = relative_path.as_posix()
    parts = set(relative_path.parts)

    if path == MANIFEST_PATH_CELL19:
        return True

    if (
        len(relative_path.parts) >= 2
        and relative_path.parts[0] == "data"
        and relative_path.parts[1]
        == "checkpoints"
    ):
        return True

    if "__pycache__" in parts:
        return True

    if ".ipynb_checkpoints" in parts:
        return True

    if relative_text.endswith(".tmp"):
        return True

    if ".tmp." in relative_text:
        return True

    if relative_text.endswith("~"):
        return True

    return False


# ---------------------------------------------------------------------
# Validate final gate
# ---------------------------------------------------------------------

required_gate_files_cell19 = [
    FINAL_VALIDATION_PATH_CELL19,
    ELIGIBILITY_REPORT_PATH_CELL19,
    STAGE_02_POLICY_PATH_CELL19,
]

missing_gate_files_cell19 = [
    str(path)
    for path in required_gate_files_cell19
    if not path.is_file()
    or path.stat().st_size == 0
]

if missing_gate_files_cell19:
    raise RuntimeError(
        "Missing final Stage 02 gate files:\n"
        + "\n".join(missing_gate_files_cell19)
    )

final_validation_cell19 = json.loads(
    FINAL_VALIDATION_PATH_CELL19.read_text(
        encoding="utf-8"
    )
)

eligibility_report_cell19 = json.loads(
    ELIGIBILITY_REPORT_PATH_CELL19.read_text(
        encoding="utf-8"
    )
)

if (
    final_validation_cell19.get("status")
    != "PASS"
):
    raise RuntimeError(
        "Stage 02 final validation is not PASS."
    )

if (
    eligibility_report_cell19.get("status")
    != "PASS"
):
    raise RuntimeError(
        "Final eligibility reconciliation is not PASS."
    )


# ---------------------------------------------------------------------
# Discover final artifact set
# ---------------------------------------------------------------------

candidate_files_cell19 = sorted(
    (
        path
        for path in PROJECT_ROOT_CELL19.rglob("*")
        if path.is_file()
        and not should_exclude_cell19(path)
    ),
    key=lambda path: relative_path_cell19(path),
)

if not candidate_files_cell19:
    raise RuntimeError(
        "No Stage 02 artifacts were discovered."
    )

required_relative_paths_cell19 = {
    "config/stage_02.yaml",
    "data/audits/operations.csv",
    "data/audits/prose_fields.csv",
    "data/audits/prose_measurements.jsonl",
    "data/audits/rendered_prompt_measurements.jsonl",
    "data/audits/rendered_prompt_payload_index.jsonl",
    "data/audits/slice_records.jsonl",
    "data/audits/slice_pointer_maps.jsonl",
    "data/inventory/final_eligible_operations.jsonl",
    "data/inventory/families.csv",
    "data/inventory/operation_exclusions.jsonl",
    "reports/audit/stage_02_final_validation.json",
    "reports/audit/final_eligibility_reconciliation.json",
    "reports/audit/manual_trace_summary.json",
    "reports/audit/descriptive_statistics.json",
    "reports/figures/prose_tokens_distribution.png",
    "reports/figures/prose_tokens_distribution_log.png",
}

discovered_relative_paths_cell19 = {
    relative_path_cell19(path)
    for path in candidate_files_cell19
}

missing_required_members_cell19 = sorted(
    required_relative_paths_cell19
    - discovered_relative_paths_cell19
)

if missing_required_members_cell19:
    raise RuntimeError(
        "Required artifacts are absent from the "
        "manifest candidate set:\n"
        + "\n".join(
            missing_required_members_cell19
        )
    )


# ---------------------------------------------------------------------
# Hash artifacts
# ---------------------------------------------------------------------

start_time_cell19 = time.perf_counter()

artifact_records_cell19 = []
verification_problems_cell19 = []

top_level_counts_cell19 = Counter()
extension_counts_cell19 = Counter()

total_artifact_bytes_cell19 = 0

for index_cell19, path_cell19 in enumerate(
    candidate_files_cell19,
    start=1,
):
    relative_cell19 = relative_path_cell19(
        path_cell19
    )

    stat_before_cell19 = path_cell19.stat()

    observed_sha_cell19 = sha256_file_cell19(
        path_cell19
    )

    stat_after_cell19 = path_cell19.stat()

    if (
        stat_before_cell19.st_size
        != stat_after_cell19.st_size
        or stat_before_cell19.st_mtime_ns
        != stat_after_cell19.st_mtime_ns
    ):
        verification_problems_cell19.append(
            {
                "path": relative_cell19,
                "problem":
                    "FILE_CHANGED_DURING_HASHING",
                "size_before":
                    stat_before_cell19.st_size,
                "size_after":
                    stat_after_cell19.st_size,
                "mtime_ns_before":
                    stat_before_cell19.st_mtime_ns,
                "mtime_ns_after":
                    stat_after_cell19.st_mtime_ns,
            }
        )

    artifact_record_cell19 = {
        "path": relative_cell19,
        "bytes": stat_after_cell19.st_size,
        "sha256": observed_sha_cell19,
    }

    artifact_records_cell19.append(
        artifact_record_cell19
    )

    total_artifact_bytes_cell19 += (
        stat_after_cell19.st_size
    )

    top_level_cell19 = (
        relative_cell19.split("/", 1)[0]
    )

    top_level_counts_cell19[
        top_level_cell19
    ] += 1

    extension_cell19 = (
        path_cell19.suffix.lower()
        if path_cell19.suffix
        else "<none>"
    )

    extension_counts_cell19[
        extension_cell19
    ] += 1

    if (
        index_cell19 % 1000 == 0
        or index_cell19
        == len(candidate_files_cell19)
    ):
        print(
            json.dumps(
                {
                    "hashed":
                        index_cell19,
                    "total":
                        len(
                            candidate_files_cell19
                        ),
                    "artifact_gib": round(
                        total_artifact_bytes_cell19
                        / (1024 ** 3),
                        3,
                    ),
                    "verification_problem_count":
                        len(
                            verification_problems_cell19
                        ),
                    "elapsed_seconds": round(
                        time.perf_counter()
                        - start_time_cell19,
                        1,
                    ),
                }
            )
        )


# ---------------------------------------------------------------------
# Configuration hash
# ---------------------------------------------------------------------

configuration_records_cell19 = [
    record
    for record in artifact_records_cell19
    if record["path"].startswith(
        "config/"
    )
]

if not configuration_records_cell19:
    raise RuntimeError(
        "Manifest contains no configuration files."
    )

configuration_hash_cell19 = sha256_text_cell19(
    canonical_json_cell19(
        configuration_records_cell19
    )
)

project_tree_hash_cell19 = sha256_text_cell19(
    canonical_json_cell19(
        artifact_records_cell19
    )
)


# ---------------------------------------------------------------------
# Count packaged structural artifact groups
# ---------------------------------------------------------------------

slice_artifact_count_cell19 = sum(
    1
    for record in artifact_records_cell19
    if (
        record["path"].startswith(
            "data/slices/"
        )
        or record["path"].startswith(
            "data/l0_slices/"
        )
    )
    and record["path"].endswith(".json")
)

prompt_payload_chunk_count_cell19 = sum(
    1
    for record in artifact_records_cell19
    if record["path"].startswith(
        "data/audits/"
        "rendered_prompt_payload_chunks/"
    )
    and record["path"].endswith(
        ".jsonl.gz"
    )
)

if prompt_payload_chunk_count_cell19 != 235:
    verification_problems_cell19.append(
        {
            "problem":
                "PROMPT_PAYLOAD_CHUNK_COUNT_MISMATCH",
            "expected": 235,
            "observed":
                prompt_payload_chunk_count_cell19,
        }
    )


# ---------------------------------------------------------------------
# Upstream provenance
# ---------------------------------------------------------------------

upstream_manifest_records_cell19 = []

upstream_verification_path_cell19 = (
    PROJECT_ROOT_CELL19
    / "reports/audit/"
    "upstream_stage_00_01_verification.json"
)

if upstream_verification_path_cell19.is_file():
    upstream_verification_cell19 = json.loads(
        upstream_verification_path_cell19.read_text(
            encoding="utf-8"
        )
    )

    for stage_key_cell19 in [
        "stage_00",
        "stage_01",
    ]:
        stage_record_cell19 = (
            upstream_verification_cell19.get(
                stage_key_cell19,
                {}
            )
        )

        upstream_manifest_records_cell19.append(
            {
                "stage": stage_key_cell19,
                "internal_manifest_path":
                    stage_record_cell19.get(
                        "internal_manifest_path"
                    ),
                "internal_manifest_sha256":
                    stage_record_cell19.get(
                        "observed_internal_manifest_sha256"
                    ),
                "verified_artifact_count":
                    stage_record_cell19.get(
                        "verified_artifact_count"
                    ),
                "status":
                    stage_record_cell19.get(
                        "status"
                    ),
            }
        )


# ---------------------------------------------------------------------
# Build manifest
# ---------------------------------------------------------------------

status_cell19 = (
    "PASS"
    if not verification_problems_cell19
    else "BLOCKED"
)

manifest_cell19 = {
    "manifest_version": "1.0",
    "stage": "02",
    "design_id": "openapi-fresh-pilot-v1",
    "upstream_manifest_sha256": {
        "00": sha256_file_cell19(STAGE_00_INTERNAL_MANIFEST_PATH),
        "01": sha256_file_cell19(STAGE_01_INTERNAL_MANIFEST_PATH),
    },
    "stage_name":
        "operation_slicing_and_prose_measurement",
    "stage_status": status_cell19,
    "project_root_name":
        PROJECT_ROOT_CELL19.name,
    "artifact_count":
        len(artifact_records_cell19),
    "artifact_bytes":
        total_artifact_bytes_cell19,
    "artifact_gib": round(
        total_artifact_bytes_cell19
        / (1024 ** 3),
        6,
    ),
    "configuration_file_count":
        len(configuration_records_cell19),
    "configuration_hash":
        configuration_hash_cell19,
    "project_tree_hash":
        project_tree_hash_cell19,
    "operation_count": final_validation_cell19["operation_count"],
    "valid_slice_count": final_validation_cell19["valid_slice_count"],
    "eligible_operation_count": final_validation_cell19["eligible_operation_count"],
    "excluded_operation_count": final_validation_cell19["excluded_operation_count"],
    "eligible_specification_count": eligibility_report_cell19["eligible_specification_count"],
    "converted_swagger_eligible_operation_count":
        eligibility_report_cell19["eligible_source_class_counts"].get("converted_swagger", 0),
    "native_openapi_eligible_operation_count":
        eligibility_report_cell19["eligible_source_class_counts"].get("native_openapi", 0),
    "prompt_over_limit_exclusion_count":
        final_validation_cell19["exclusion_reason_counts"].get("EXCLUDED_RENDERED_PROMPT_TOO_LARGE", 0),
    "invalid_slice_exclusion_count":
        final_validation_cell19["exclusion_reason_counts"].get("EXCLUDED_INVALID_L0_SLICE", 0),
    "prompt_payload_chunk_count":
        prompt_payload_chunk_count_cell19,
    "detected_slice_artifact_count":
        slice_artifact_count_cell19,
    "top_level_artifact_counts": dict(
        sorted(top_level_counts_cell19.items())
    ),
    "extension_counts": dict(
        sorted(extension_counts_cell19.items())
    ),
    "excluded_from_manifest": [
        "data/checkpoints/**",
        "**/__pycache__/**",
        "**/.ipynb_checkpoints/**",
        "*.tmp",
        "02_manifest.json itself",
    ],
    "upstream_manifests":
        upstream_manifest_records_cell19,
    "final_validation": {
        "path": relative_path_cell19(
            FINAL_VALIDATION_PATH_CELL19
        ),
        "sha256": sha256_file_cell19(
            FINAL_VALIDATION_PATH_CELL19
        ),
        "status":
            final_validation_cell19[
                "status"
            ],
        "may_begin_notebook_03":
            final_validation_cell19[
                "may_begin_notebook_03"
            ],
    },
    "eligibility_reconciliation": {
        "path": relative_path_cell19(
            ELIGIBILITY_REPORT_PATH_CELL19
        ),
        "sha256": sha256_file_cell19(
            ELIGIBILITY_REPORT_PATH_CELL19
        ),
        "status":
            eligibility_report_cell19[
                "status"
            ],
    },
    "strata_thresholds_selected": False,
    "model_outputs_used_for_eligibility": False,
    "verification_problem_count":
        len(verification_problems_cell19),
    "verification_problems":
        verification_problems_cell19,
    "artifacts": artifact_records_cell19,
}

write_json_atomic_cell19(
    MANIFEST_PATH_CELL19,
    manifest_cell19,
)

manifest_sha256_cell19 = sha256_file_cell19(
    MANIFEST_PATH_CELL19
)

manifest_bytes_cell19 = (
    MANIFEST_PATH_CELL19.stat().st_size
)


# ---------------------------------------------------------------------
# Re-read manifest and perform structural verification
# ---------------------------------------------------------------------

reloaded_manifest_cell19 = json.loads(
    MANIFEST_PATH_CELL19.read_text(
        encoding="utf-8"
    )
)

structural_checks_cell19 = {
    "manifest_status_pass": (
        reloaded_manifest_cell19[
            "stage_status"
        ]
        == "PASS"
    ),
    "artifact_count_matches": (
        reloaded_manifest_cell19[
            "artifact_count"
        ]
        == len(
            reloaded_manifest_cell19[
                "artifacts"
            ]
        )
    ),
    "artifact_paths_unique": (
        len(
            {
                record["path"]
                for record
                in reloaded_manifest_cell19[
                    "artifacts"
                ]
            }
        )
        == reloaded_manifest_cell19[
            "artifact_count"
        ]
    ),
    "artifact_bytes_match": (
        sum(
            record["bytes"]
            for record
            in reloaded_manifest_cell19[
                "artifacts"
            ]
        )
        == reloaded_manifest_cell19[
            "artifact_bytes"
        ]
    ),
    "configuration_hash_present": (
        len(
            reloaded_manifest_cell19[
                "configuration_hash"
            ]
        )
        == 64
    ),
    "project_tree_hash_present": (
        len(
            reloaded_manifest_cell19[
                "project_tree_hash"
            ]
        )
        == 64
    ),
    "eligible_count_matches": (
        reloaded_manifest_cell19["eligible_operation_count"]
        == eligibility_report_cell19["eligible_operation_count"]
        == final_validation_cell19["eligible_operation_count"]
    ),
    "excluded_count_matches": (
        reloaded_manifest_cell19["excluded_operation_count"]
        == eligibility_report_cell19["excluded_operation_count"]
        == final_validation_cell19["excluded_operation_count"]
    ),
    "operation_counts_reconcile": (
        reloaded_manifest_cell19["eligible_operation_count"]
        + reloaded_manifest_cell19["excluded_operation_count"]
        == reloaded_manifest_cell19["operation_count"]
        and reloaded_manifest_cell19["eligible_operation_count"]
        + reloaded_manifest_cell19["prompt_over_limit_exclusion_count"]
        == reloaded_manifest_cell19["valid_slice_count"]
        and reloaded_manifest_cell19["valid_slice_count"]
        + reloaded_manifest_cell19["invalid_slice_exclusion_count"]
        == reloaded_manifest_cell19["operation_count"]
        and reloaded_manifest_cell19["converted_swagger_eligible_operation_count"]
        + reloaded_manifest_cell19["native_openapi_eligible_operation_count"]
        == reloaded_manifest_cell19["eligible_operation_count"]
    ),
    "payload_chunk_count_matches": (
        reloaded_manifest_cell19[
            "prompt_payload_chunk_count"
        ]
        == 235
    ),
    "no_verification_problems": (
        reloaded_manifest_cell19[
            "verification_problem_count"
        ]
        == 0
    ),
}

if not all(
    structural_checks_cell19.values()
):
    raise RuntimeError(
        "Stage 02 manifest structural validation "
        "failed:\n"
        + json.dumps(
            structural_checks_cell19,
            indent=2,
        )
    )

summary_cell19 = {
    "stage_status": status_cell19,
    "manifest_path":
        str(MANIFEST_PATH_CELL19),
    "manifest_sha256":
        manifest_sha256_cell19,
    "manifest_bytes":
        manifest_bytes_cell19,
    "artifact_count":
        len(artifact_records_cell19),
    "artifact_bytes":
        total_artifact_bytes_cell19,
    "artifact_gib": round(
        total_artifact_bytes_cell19
        / (1024 ** 3),
        6,
    ),
    "configuration_hash":
        configuration_hash_cell19,
    "project_tree_hash":
        project_tree_hash_cell19,
    "prompt_payload_chunk_count":
        prompt_payload_chunk_count_cell19,
    "detected_slice_artifact_count":
        slice_artifact_count_cell19,
    "verification_problem_count":
        len(verification_problems_cell19),
    "structural_checks":
        structural_checks_cell19,
    "duration_seconds": round(
        time.perf_counter()
        - start_time_cell19,
        3,
    ),
    "status": (
        "PASS"
        if (
            status_cell19 == "PASS"
            and all(
                structural_checks_cell19.values()
            )
        )
        else "BLOCKED"
    ),
}

print(
    json.dumps(
        summary_cell19,
        indent=2,
        ensure_ascii=False,
    )
)

if summary_cell19["status"] != "PASS":
    raise RuntimeError(
        "Stage 02 manifest creation failed."
    )

print("\nNOTEBOOK 02 — CELL 19: PASS")


{"hashed": 1000, "total": 58881, "artifact_gib": 1.733, "verification_problem_count": 0, "elapsed_seconds": 6.6}
{"hashed": 2000, "total": 58881, "artifact_gib": 1.746, "verification_problem_count": 0, "elapsed_seconds": 6.8}
{"hashed": 3000, "total": 58881, "artifact_gib": 1.758, "verification_problem_count": 0, "elapsed_seconds": 6.9}
{"hashed": 4000, "total": 58881, "artifact_gib": 1.774, "verification_problem_count": 0, "elapsed_seconds": 7.1}
{"hashed": 5000, "total": 58881, "artifact_gib": 1.788, "verification_problem_count": 0, "elapsed_seconds": 7.3}
{"hashed": 6000, "total": 58881, "artifact_gib": 1.799, "verification_problem_count": 0, "elapsed_seconds": 7.5}
{"hashed": 7000, "total": 58881, "artifact_gib": 1.815, "verification_problem_count": 0, "elapsed_seconds": 7.6}
{"hashed": 8000, "total": 58881, "artifact_gib": 1.828, "verification_problem_count": 0, "elapsed_seconds": 7.8}
{"hashed": 9000, "total": 58881, "artifact_gib": 1.842, "verification_problem_count": 0, "elapse

In [21]:
# Notebook 02 — Cell 20
# Package and verify the final Stage 02 Kaggle Dataset.
#
# The ZIP contains:
# - every artifact registered in 02_manifest.json
# - 02_manifest.json itself
#
# The archive is verified by:
# - checking every source hash while packaging
# - checking the exact ZIP member list
# - reading every ZIP member
# - checking every archived member hash
# - validating CRC through complete ZIP-member reads

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import time
import zipfile


# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

PROJECT_ROOT_CELL20 = Path(
    "/kaggle/working/fresh/stage_02/prose_operation_experiment"
)

INTERNAL_MANIFEST_PATH_CELL20 = (
    PROJECT_ROOT_CELL20
    / "registries/02_manifest.json"
)

DATASET_OUTPUT_ROOT_CELL20 = Path(
    "/kaggle/working/stage_02_fresh_dataset"
)

ARCHIVE_NAME_CELL20 = (
    "stage_02_operation_slicing_v1.zip"
)

ARCHIVE_PATH_CELL20 = (
    DATASET_OUTPUT_ROOT_CELL20
    / ARCHIVE_NAME_CELL20
)

EXTERNAL_MANIFEST_PATH_CELL20 = (
    DATASET_OUTPUT_ROOT_CELL20
    / "stage_02_manifest.json"
)

CHECKSUM_PATH_CELL20 = (
    DATASET_OUTPUT_ROOT_CELL20
    / "stage_02_checksums.sha256"
)

README_PATH_CELL20 = (
    DATASET_OUTPUT_ROOT_CELL20
    / "README.md"
)

DATASET_OUTPUT_ROOT_CELL20.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file_cell20(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def write_text_atomic_cell20(path, text):
    path = Path(path)
    temporary_path = path.with_name(
        path.name + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
        newline="\n",
    )

    os.replace(
        temporary_path,
        path,
    )


def write_json_atomic_cell20(path, value):
    write_text_atomic_cell20(
        path,
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
    )


def compression_type_cell20(path):
    already_compressed_suffixes = {
        ".gz",
        ".zip",
        ".png",
        ".jpg",
        ".jpeg",
        ".gif",
        ".webp",
        ".pdf",
        ".parquet",
        ".safetensors",
    }

    if path.suffix.lower() in already_compressed_suffixes:
        return zipfile.ZIP_STORED

    return zipfile.ZIP_DEFLATED


def archive_member_name_cell20(relative_path):
    return (
        f"{PROJECT_ROOT_CELL20.name}/"
        f"{relative_path}"
    )


# ---------------------------------------------------------------------
# Validate internal manifest
# ---------------------------------------------------------------------

if not INTERNAL_MANIFEST_PATH_CELL20.is_file():
    raise RuntimeError(
        "Stage 02 internal manifest is missing."
    )

internal_manifest_cell20 = json.loads(
    INTERNAL_MANIFEST_PATH_CELL20.read_text(
        encoding="utf-8"
    )
)

if (
    internal_manifest_cell20.get(
        "stage_status"
    )
    != "PASS"
):
    raise RuntimeError(
        "Stage 02 internal manifest is not PASS."
    )

manifest_artifacts_cell20 = (
    internal_manifest_cell20[
        "artifacts"
    ]
)

expected_artifact_count_cell20 = int(
    internal_manifest_cell20[
        "artifact_count"
    ]
)

if (
    len(manifest_artifacts_cell20)
    != expected_artifact_count_cell20
):
    raise RuntimeError(
        "Internal manifest artifact count mismatch."
    )

artifact_paths_cell20 = [
    str(record["path"])
    for record in manifest_artifacts_cell20
]

if (
    len(set(artifact_paths_cell20))
    != len(artifact_paths_cell20)
):
    raise RuntimeError(
        "Internal manifest contains duplicate paths."
    )

internal_manifest_sha256_cell20 = (
    sha256_file_cell20(
        INTERNAL_MANIFEST_PATH_CELL20
    )
)


# ---------------------------------------------------------------------
# Prepare exact expected archive member map
# ---------------------------------------------------------------------

expected_members_cell20 = {}

for artifact_record_cell20 in (
    manifest_artifacts_cell20
):
    relative_path_cell20 = str(
        artifact_record_cell20["path"]
    )

    member_name_cell20 = (
        archive_member_name_cell20(
            relative_path_cell20
        )
    )

    expected_members_cell20[
        member_name_cell20
    ] = {
        "source_path":
            PROJECT_ROOT_CELL20
            / relative_path_cell20,
        "expected_bytes":
            int(
                artifact_record_cell20[
                    "bytes"
                ]
            ),
        "expected_sha256":
            str(
                artifact_record_cell20[
                    "sha256"
                ]
            ),
    }

manifest_member_name_cell20 = (
    archive_member_name_cell20(
        "registries/02_manifest.json"
    )
)

if manifest_member_name_cell20 in (
    expected_members_cell20
):
    raise RuntimeError(
        "The internal manifest unexpectedly "
        "included itself."
    )

expected_members_cell20[
    manifest_member_name_cell20
] = {
    "source_path":
        INTERNAL_MANIFEST_PATH_CELL20,
    "expected_bytes":
        INTERNAL_MANIFEST_PATH_CELL20
        .stat().st_size,
    "expected_sha256":
        internal_manifest_sha256_cell20,
}

expected_member_count_cell20 = len(
    expected_members_cell20
)

if (
    expected_member_count_cell20
    != expected_artifact_count_cell20 + 1
):
    raise RuntimeError(
        "Expected archive member count is inconsistent."
    )


# ---------------------------------------------------------------------
# Create archive while re-verifying every source
# ---------------------------------------------------------------------

temporary_archive_path_cell20 = (
    ARCHIVE_PATH_CELL20.with_name(
        ARCHIVE_PATH_CELL20.name + ".tmp"
    )
)

if temporary_archive_path_cell20.exists():
    temporary_archive_path_cell20.unlink()

packaging_start_cell20 = time.perf_counter()

packaged_source_bytes_cell20 = 0
stored_member_count_cell20 = 0
deflated_member_count_cell20 = 0

with zipfile.ZipFile(
    temporary_archive_path_cell20,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
    allowZip64=True,
) as archive_cell20:

    for index_cell20, (
        member_name_cell20,
        member_record_cell20,
    ) in enumerate(
        sorted(expected_members_cell20.items()),
        start=1,
    ):
        source_path_cell20 = (
            member_record_cell20[
                "source_path"
            ]
        )

        if not source_path_cell20.is_file():
            raise RuntimeError(
                "Manifest source file is missing: "
                f"{source_path_cell20}"
            )

        observed_bytes_cell20 = (
            source_path_cell20.stat().st_size
        )

        if (
            observed_bytes_cell20
            != member_record_cell20[
                "expected_bytes"
            ]
        ):
            raise RuntimeError(
                "Source size changed after manifest "
                f"creation: {source_path_cell20}"
            )

        compression_cell20 = (
            compression_type_cell20(
                source_path_cell20
            )
        )

        if compression_cell20 == zipfile.ZIP_STORED:
            stored_member_count_cell20 += 1
        else:
            deflated_member_count_cell20 += 1

        zip_info_cell20 = zipfile.ZipInfo(
            filename=member_name_cell20,
            date_time=(1980, 1, 1, 0, 0, 0),
        )

        zip_info_cell20.compress_type = (
            compression_cell20
        )

        zip_info_cell20.create_system = 3
        zip_info_cell20.external_attr = (
            0o100644 << 16
        )

        source_digest_cell20 = hashlib.sha256()
        source_bytes_written_cell20 = 0

        with source_path_cell20.open(
            "rb"
        ) as source_handle_cell20, archive_cell20.open(
            zip_info_cell20,
            mode="w",
            force_zip64=True,
        ) as member_handle_cell20:

            while True:
                block_cell20 = (
                    source_handle_cell20.read(
                        1024 * 1024
                    )
                )

                if not block_cell20:
                    break

                source_digest_cell20.update(
                    block_cell20
                )

                member_handle_cell20.write(
                    block_cell20
                )

                source_bytes_written_cell20 += len(
                    block_cell20
                )

        observed_sha256_cell20 = (
            source_digest_cell20.hexdigest()
        )

        if (
            observed_sha256_cell20
            != member_record_cell20[
                "expected_sha256"
            ]
        ):
            raise RuntimeError(
                "Source hash changed after manifest "
                f"creation: {source_path_cell20}"
            )

        if (
            source_bytes_written_cell20
            != member_record_cell20[
                "expected_bytes"
            ]
        ):
            raise RuntimeError(
                "Source byte count changed while "
                f"packaging: {source_path_cell20}"
            )

        packaged_source_bytes_cell20 += (
            source_bytes_written_cell20
        )

        if (
            index_cell20 % 1000 == 0
            or index_cell20
            == expected_member_count_cell20
        ):
            elapsed_cell20 = (
                time.perf_counter()
                - packaging_start_cell20
            )

            current_archive_bytes_cell20 = (
                temporary_archive_path_cell20
                .stat().st_size
                if temporary_archive_path_cell20
                .exists()
                else 0
            )

            print(
                json.dumps(
                    {
                        "packaged":
                            index_cell20,
                        "total":
                            expected_member_count_cell20,
                        "source_gib": round(
                            packaged_source_bytes_cell20
                            / (1024 ** 3),
                            3,
                        ),
                        "current_archive_gib": round(
                            current_archive_bytes_cell20
                            / (1024 ** 3),
                            3,
                        ),
                        "elapsed_seconds": round(
                            elapsed_cell20,
                            1,
                        ),
                    }
                )
            )

os.replace(
    temporary_archive_path_cell20,
    ARCHIVE_PATH_CELL20,
)


# ---------------------------------------------------------------------
# Verify member list, CRC, uncompressed sizes, and member hashes
# ---------------------------------------------------------------------

verification_start_cell20 = time.perf_counter()

member_hash_problem_examples_cell20 = []
member_size_problem_examples_cell20 = []
duplicate_member_names_cell20 = []

verified_member_count_cell20 = 0
verified_uncompressed_bytes_cell20 = 0
zip_crc_failure_cell20 = None

with zipfile.ZipFile(
    ARCHIVE_PATH_CELL20,
    mode="r",
    allowZip64=True,
) as archive_cell20:

    archive_infos_cell20 = (
        archive_cell20.infolist()
    )

    archive_member_names_cell20 = [
        info_cell20.filename
        for info_cell20
        in archive_infos_cell20
    ]

    archive_member_name_counts_cell20 = (
        Counter(archive_member_names_cell20)
    )

    duplicate_member_names_cell20 = sorted(
        name_cell20
        for name_cell20, count_cell20
        in archive_member_name_counts_cell20.items()
        if count_cell20 > 1
    )

    expected_name_set_cell20 = set(
        expected_members_cell20
    )

    observed_name_set_cell20 = set(
        archive_member_names_cell20
    )

    member_list_matches_cell20 = (
        expected_name_set_cell20
        == observed_name_set_cell20
        and not duplicate_member_names_cell20
        and len(archive_member_names_cell20)
        == expected_member_count_cell20
    )

    missing_archive_members_cell20 = sorted(
        expected_name_set_cell20
        - observed_name_set_cell20
    )

    unexpected_archive_members_cell20 = sorted(
        observed_name_set_cell20
        - expected_name_set_cell20
    )

    for index_cell20, info_cell20 in enumerate(
        archive_infos_cell20,
        start=1,
    ):
        member_name_cell20 = info_cell20.filename

        expected_member_cell20 = (
            expected_members_cell20.get(
                member_name_cell20
            )
        )

        if expected_member_cell20 is None:
            continue

        if (
            info_cell20.file_size
            != expected_member_cell20[
                "expected_bytes"
            ]
        ):
            if len(
                member_size_problem_examples_cell20
            ) < 20:
                member_size_problem_examples_cell20.append(
                    {
                        "member":
                            member_name_cell20,
                        "expected":
                            expected_member_cell20[
                                "expected_bytes"
                            ],
                        "observed":
                            info_cell20.file_size,
                    }
                )

        member_digest_cell20 = hashlib.sha256()
        member_bytes_read_cell20 = 0

        try:
            with archive_cell20.open(
                info_cell20,
                mode="r",
            ) as member_handle_cell20:
                while True:
                    block_cell20 = (
                        member_handle_cell20.read(
                            1024 * 1024
                        )
                    )

                    if not block_cell20:
                        break

                    member_digest_cell20.update(
                        block_cell20
                    )

                    member_bytes_read_cell20 += len(
                        block_cell20
                    )

        except Exception as error_cell20:
            zip_crc_failure_cell20 = {
                "member": member_name_cell20,
                "diagnostic": repr(error_cell20),
            }
            break

        observed_member_sha_cell20 = (
            member_digest_cell20.hexdigest()
        )

        if (
            observed_member_sha_cell20
            != expected_member_cell20[
                "expected_sha256"
            ]
        ):
            if len(
                member_hash_problem_examples_cell20
            ) < 20:
                member_hash_problem_examples_cell20.append(
                    {
                        "member":
                            member_name_cell20,
                        "expected":
                            expected_member_cell20[
                                "expected_sha256"
                            ],
                        "observed":
                            observed_member_sha_cell20,
                    }
                )

        if (
            member_bytes_read_cell20
            != expected_member_cell20[
                "expected_bytes"
            ]
        ):
            if len(
                member_size_problem_examples_cell20
            ) < 20:
                member_size_problem_examples_cell20.append(
                    {
                        "member":
                            member_name_cell20,
                        "expected":
                            expected_member_cell20[
                                "expected_bytes"
                            ],
                        "observed":
                            member_bytes_read_cell20,
                    }
                )

        verified_member_count_cell20 += 1
        verified_uncompressed_bytes_cell20 += (
            member_bytes_read_cell20
        )

        if (
            index_cell20 % 1000 == 0
            or index_cell20
            == len(archive_infos_cell20)
        ):
            print(
                json.dumps(
                    {
                        "verified":
                            index_cell20,
                        "total":
                            len(archive_infos_cell20),
                        "verified_gib": round(
                            verified_uncompressed_bytes_cell20
                            / (1024 ** 3),
                            3,
                        ),
                        "hash_problem_count":
                            len(
                                member_hash_problem_examples_cell20
                            ),
                        "size_problem_count":
                            len(
                                member_size_problem_examples_cell20
                            ),
                        "elapsed_seconds": round(
                            time.perf_counter()
                            - verification_start_cell20,
                            1,
                        ),
                    }
                )
            )


# ---------------------------------------------------------------------
# Final archive hash
# ---------------------------------------------------------------------

archive_sha256_cell20 = sha256_file_cell20(
    ARCHIVE_PATH_CELL20
)

archive_bytes_cell20 = (
    ARCHIVE_PATH_CELL20.stat().st_size
)


# ---------------------------------------------------------------------
# Verification gate
# ---------------------------------------------------------------------

verification_checks_cell20 = {
    "internal_manifest_pass": (
        internal_manifest_cell20[
            "stage_status"
        ]
        == "PASS"
    ),
    "archive_member_count_matches": (
        expected_member_count_cell20
        == expected_artifact_count_cell20 + 1
    ),
    "member_list_matches": (
        member_list_matches_cell20
    ),
    "no_duplicate_member_names": (
        len(duplicate_member_names_cell20)
        == 0
    ),
    "no_missing_members": (
        len(missing_archive_members_cell20)
        == 0
    ),
    "no_unexpected_members": (
        len(unexpected_archive_members_cell20)
        == 0
    ),
    "all_member_hashes_verified": (
        verified_member_count_cell20
        == expected_member_count_cell20
        and not member_hash_problem_examples_cell20
    ),
    "all_member_sizes_verified": (
        not member_size_problem_examples_cell20
    ),
    "zip_crc_pass": (
        zip_crc_failure_cell20 is None
    ),
    "uncompressed_bytes_match": (
        verified_uncompressed_bytes_cell20
        == sum(
            record_cell20[
                "expected_bytes"
            ]
            for record_cell20
            in expected_members_cell20.values()
        )
    ),
}

stage_status_cell20 = (
    "PASS"
    if all(
        verification_checks_cell20.values()
    )
    else "BLOCKED"
)

if stage_status_cell20 != "PASS":
    failure_report_cell20 = {
        "status": "BLOCKED",
        "verification_checks":
            verification_checks_cell20,
        "duplicate_member_names":
            duplicate_member_names_cell20[:20],
        "missing_archive_members":
            missing_archive_members_cell20[:20],
        "unexpected_archive_members":
            unexpected_archive_members_cell20[:20],
        "member_hash_problem_examples":
            member_hash_problem_examples_cell20,
        "member_size_problem_examples":
            member_size_problem_examples_cell20,
        "zip_crc_failure":
            zip_crc_failure_cell20,
    }

    print(
        json.dumps(
            failure_report_cell20,
            indent=2,
        )
    )

    raise RuntimeError(
        "Stage 02 archive verification failed."
    )


# ---------------------------------------------------------------------
# README
# ---------------------------------------------------------------------

readme_text_cell20 = f"""# Prose-operation experiment — Stage 02

This dataset contains the complete verified output of Notebook 02:
operation-level L0 slicing, exact prose measurement, rendered-prompt
qualification, descriptive statistics, manual source-to-slice traces,
and the final operation eligibility ledger.

## Final counts

- Source operations: {internal_manifest_cell20["operation_count"]:,}
- Valid L0 slices: {internal_manifest_cell20["valid_slice_count"]:,}
- Generation-eligible operations: {internal_manifest_cell20["eligible_operation_count"]:,}
- Prompt-length exclusions: {internal_manifest_cell20["prompt_over_limit_exclusion_count"]:,}
- Invalid-reference exclusions: {internal_manifest_cell20["invalid_slice_exclusion_count"]:,}
- Eligible converted-Swagger operations: {internal_manifest_cell20["converted_swagger_eligible_operation_count"]:,}
- Eligible native-OpenAPI operations: {internal_manifest_cell20["native_openapi_eligible_operation_count"]:,}
- Exact rendered-prompt payload chunks: {internal_manifest_cell20["prompt_payload_chunk_count"]:,}

No prose strata or sampling thresholds were selected in Stage 02.
Notebook 03 establishes the paired eligible frame. Notebook 04 sets the strata.

## Archive

- File: `{ARCHIVE_NAME_CELL20}`
- SHA-256: `{archive_sha256_cell20}`
- Archive members: {expected_member_count_cell20:,}
- Internal manifest artifacts: {expected_artifact_count_cell20:,}
- Uncompressed project bytes: {internal_manifest_cell20["artifact_bytes"]:,}

The archive root is:

`prose_operation_experiment/`

The internal scientific manifest is:

`prose_operation_experiment/registries/02_manifest.json`

Use the internal manifest to verify individual files after extraction.
"""

write_text_atomic_cell20(
    README_PATH_CELL20,
    readme_text_cell20,
)

readme_sha256_cell20 = sha256_file_cell20(
    README_PATH_CELL20
)


# ---------------------------------------------------------------------
# External dataset manifest
# ---------------------------------------------------------------------

external_manifest_cell20 = {
    "manifest_version": "1.0",
    "stage": "02",
    "stage_name":
        "operation_slicing_and_prose_measurement",
    "stage_status": "PASS",
    "archive": {
        "name": ARCHIVE_NAME_CELL20,
        "bytes": archive_bytes_cell20,
        "sha256": archive_sha256_cell20,
        "member_count":
            expected_member_count_cell20,
        "uncompressed_member_bytes":
            verified_uncompressed_bytes_cell20,
        "zip64": True,
        "crc_verified": True,
        "member_hashes_verified": True,
    },
    "internal_manifest": {
        "archive_member":
            manifest_member_name_cell20,
        "bytes":
            INTERNAL_MANIFEST_PATH_CELL20
            .stat().st_size,
        "sha256":
            internal_manifest_sha256_cell20,
        "artifact_count":
            expected_artifact_count_cell20,
        "artifact_bytes":
            internal_manifest_cell20[
                "artifact_bytes"
            ],
        "configuration_hash":
            internal_manifest_cell20[
                "configuration_hash"
            ],
        "project_tree_hash":
            internal_manifest_cell20[
                "project_tree_hash"
            ],
    },
    "final_counts": {
        "operation_count": internal_manifest_cell20["operation_count"],
        "valid_slice_count": internal_manifest_cell20["valid_slice_count"],
        "eligible_operation_count": internal_manifest_cell20["eligible_operation_count"],
        "excluded_operation_count": internal_manifest_cell20["excluded_operation_count"],
        "eligible_specification_count": internal_manifest_cell20["eligible_specification_count"],
        "eligible_converted_swagger_count":
            internal_manifest_cell20["converted_swagger_eligible_operation_count"],
        "eligible_native_openapi_count":
            internal_manifest_cell20["native_openapi_eligible_operation_count"],
    },
    "verification_checks":
        verification_checks_cell20,
    "readme": {
        "name": README_PATH_CELL20.name,
        "sha256": readme_sha256_cell20,
    },
    "may_begin_notebook_03": True,
    "may_begin_experimental_generation": False,
    "strata_thresholds_selected": False,
}

write_json_atomic_cell20(
    EXTERNAL_MANIFEST_PATH_CELL20,
    external_manifest_cell20,
)

external_manifest_sha256_cell20 = (
    sha256_file_cell20(
        EXTERNAL_MANIFEST_PATH_CELL20
    )
)


# ---------------------------------------------------------------------
# Dataset checksums
# ---------------------------------------------------------------------

checksum_text_cell20 = (
    f"{archive_sha256_cell20}  "
    f"{ARCHIVE_PATH_CELL20.name}\n"
    f"{external_manifest_sha256_cell20}  "
    f"{EXTERNAL_MANIFEST_PATH_CELL20.name}\n"
    f"{readme_sha256_cell20}  "
    f"{README_PATH_CELL20.name}\n"
)

write_text_atomic_cell20(
    CHECKSUM_PATH_CELL20,
    checksum_text_cell20,
)

checksum_sha256_cell20 = sha256_file_cell20(
    CHECKSUM_PATH_CELL20
)


# ---------------------------------------------------------------------
# Final dataset inventory
# ---------------------------------------------------------------------

dataset_files_cell20 = []

for path_cell20 in sorted(
    DATASET_OUTPUT_ROOT_CELL20.iterdir(),
    key=lambda value: value.name,
):
    if not path_cell20.is_file():
        continue

    dataset_files_cell20.append(
        {
            "name": path_cell20.name,
            "bytes": path_cell20.stat().st_size,
            "sha256":
                sha256_file_cell20(
                    path_cell20
                ),
        }
    )

expected_dataset_names_cell20 = {
    "README.md",
    "stage_02_checksums.sha256",
    "stage_02_manifest.json",
    ARCHIVE_NAME_CELL20,
}

observed_dataset_names_cell20 = {
    record_cell20["name"]
    for record_cell20
    in dataset_files_cell20
}

if (
    observed_dataset_names_cell20
    != expected_dataset_names_cell20
):
    raise RuntimeError(
        "Stage 02 dataset output directory contains "
        "an unexpected file set:\n"
        + json.dumps(
            {
                "expected": sorted(
                    expected_dataset_names_cell20
                ),
                "observed": sorted(
                    observed_dataset_names_cell20
                ),
            },
            indent=2,
        )
    )

duration_seconds_cell20 = (
    time.perf_counter()
    - packaging_start_cell20
)

summary_cell20 = {
    "status": "PASS",
    "stage": "02",
    "notebook":
        "02_operation_slicing_and_prose_measurement",
    "operation_count": internal_manifest_cell20["operation_count"],
    "valid_slice_count": internal_manifest_cell20["valid_slice_count"],
    "eligible_operation_count": internal_manifest_cell20["eligible_operation_count"],
    "excluded_operation_count": internal_manifest_cell20["excluded_operation_count"],
    "internal_manifest_artifact_count":
        expected_artifact_count_cell20,
    "archive_member_count":
        expected_member_count_cell20,
    "archive_path":
        str(ARCHIVE_PATH_CELL20),
    "archive_bytes":
        archive_bytes_cell20,
    "archive_sha256":
        archive_sha256_cell20,
    "zip_crc_failure":
        zip_crc_failure_cell20,
    "member_list_matches":
        member_list_matches_cell20,
    "member_hashes_verified":
        not member_hash_problem_examples_cell20,
    "member_sizes_verified":
        not member_size_problem_examples_cell20,
    "stored_member_count":
        stored_member_count_cell20,
    "deflated_member_count":
        deflated_member_count_cell20,
    "internal_manifest_path":
        str(INTERNAL_MANIFEST_PATH_CELL20),
    "internal_manifest_sha256":
        internal_manifest_sha256_cell20,
    "external_manifest_path":
        str(EXTERNAL_MANIFEST_PATH_CELL20),
    "external_manifest_sha256":
        external_manifest_sha256_cell20,
    "checksum_path":
        str(CHECKSUM_PATH_CELL20),
    "checksum_sha256":
        checksum_sha256_cell20,
    "readme_path":
        str(README_PATH_CELL20),
    "readme_sha256":
        readme_sha256_cell20,
    "dataset_output_root":
        str(DATASET_OUTPUT_ROOT_CELL20),
    "dataset_files":
        dataset_files_cell20,
    "verification_checks":
        verification_checks_cell20,
    "duration_seconds": round(
        duration_seconds_cell20,
        3,
    ),
    "next_action": (
        "Publish stage_02_dataset as the Stage 02 "
        "Kaggle Dataset, then Quick Save Notebook 02."
    ),
}

print(
    json.dumps(
        summary_cell20,
        indent=2,
        ensure_ascii=False,
    )
)

print("\nNOTEBOOK 02 — CELL 20: PASS")


{"packaged": 1000, "total": 58882, "source_gib": 1.733, "current_archive_gib": 0.481, "elapsed_seconds": 33.8}
{"packaged": 2000, "total": 58882, "source_gib": 1.746, "current_archive_gib": 0.484, "elapsed_seconds": 34.9}
{"packaged": 3000, "total": 58882, "source_gib": 1.758, "current_archive_gib": 0.487, "elapsed_seconds": 36.0}
{"packaged": 4000, "total": 58882, "source_gib": 1.774, "current_archive_gib": 0.49, "elapsed_seconds": 37.4}
{"packaged": 5000, "total": 58882, "source_gib": 1.788, "current_archive_gib": 0.493, "elapsed_seconds": 38.4}
{"packaged": 6000, "total": 58882, "source_gib": 1.799, "current_archive_gib": 0.496, "elapsed_seconds": 39.2}
{"packaged": 7000, "total": 58882, "source_gib": 1.815, "current_archive_gib": 0.499, "elapsed_seconds": 40.2}
{"packaged": 8000, "total": 58882, "source_gib": 1.828, "current_archive_gib": 0.502, "elapsed_seconds": 41.1}
{"packaged": 9000, "total": 58882, "source_gib": 1.842, "current_archive_gib": 0.505, "elapsed_seconds": 42.0}
{"